# Build the Cached Evidence Database for Consolidation

This notebook constructs the **shared structured evidence database** used by all subsequent consolidation experiments.

Its purpose is to ensure that the final reasoner does **not** repeatedly process raw audiovisual conversations during the comparison of consolidation strategies.

Instead, participant-level semantic, temporal, and participation evidence is computed in advance, audited, and cached. The later consolidation notebooks therefore operate only over the same frozen structured textual observations.

This enables a controlled comparison of:

- Binary-only consolidation;
- Structured R1;
- Free-form rationale;
- Structured R1 + free-form rationale.

Across those experiments, the underlying evidence database remains fixed while the consolidation/output format is varied.

---

## 1. Deterministic source selection and frozen temporal reference

The notebook first reconstructs the eligible NORMAL-conversation pool using the temporal preprocessing criteria developed earlier.

A conversation is eligible only when both participants have sufficient merged VAD speech activity in the first 120 seconds.

The resulting pool contains:

```text
167 eligible conversations
```

These are deterministically divided into:

```text
50 conversations  → frozen temporal reference
100 conversations → consolidation development sources
```

with zero overlap between the two groups.

The 50 reference conversations are used only to compute frozen NORMAL / synthetic-LAG temporal reference statistics.

The 100 development conversations provide the source groups from which the final 400 consolidation cases are constructed.

---

## 2. Construction of the 400-case consolidation set

Each of the 100 development source conversations contributes one source group containing four case families.

### NORMAL

The original dyadic participant pairing is retained:

```text
100 NORMAL cases
```

### WRONG PARTNER

Participant A is retained from its original source conversation while Participant B is deterministically replaced by a participant from another source conversation.

The construction is audited to ensure:

- no same-conversation Wrong Partner pairings;
- 100 unique Participant-A sources;
- 100 unique Participant-B sources;
- no use of the 50 frozen reference conversations.

This produces:

```text
100 WRONG_PARTNER cases
```

### LAG

Synthetic temporal anomalies are generated from the original NORMAL source pairs:

```text
50 cases with +2 s Participant-B lag
50 cases with +3 s Participant-B lag
```

for a total of:

```text
100 LAG cases
```

### SILENT PARTNER

Each development source contributes Participant A, while Participant B is replaced by a verified dataset-native silent participant whose metadata contain no VAD speech intervals.

This produces:

```text
100 SILENT_PARTNER cases
```

The final development database therefore contains:

```text
100 NORMAL
100 WRONG_PARTNER
100 LAG
100 SILENT_PARTNER
-------------------
400 total cases
```

---

## 3. Temporal evidence generation

The temporal pipeline is applied before consolidation so that every case contains a common cached representation.

### Participant turn preprocessing

VAD intervals are:

1. clipped to the 0–120 s analysis window;
2. merged using the established gap rule;
3. independently filtered for operational backchannel-like turns.

The final participant turn sequences are stored as:

```text
participant_A_filtered_turns
participant_B_filtered_turns
```

### Local temporal evidence

The notebook computes the local timing representation used by the later reasoner, including:

- signed A-end-to-B-start response offsets;
- offset distribution statistics;
- threshold-count statistics;
- clean overlap duration and percentage.

These fields are generated once and cached.

### Global temporal evidence

A global Participant-B correction search is also performed and cached.

The resulting global representation includes:

```text
best_B_correction_shift_seconds
estimated_B_lateness_seconds
alignment_score_gain_vs_zero
best_num_bilateral_events
best_event_coverage_percent
```

Frozen temporal reference statistics are calculated only from the separate 50-conversation reference set.

---

## 4. Semantic evidence generation

Each participant is represented semantically over two synchronized 60-second segments:

```text
Segment 0: 0–60 s
Segment 1: 60–120 s
```

For every participant-segment pair, the database stores two semantic resolutions.

### Coarse summary

The coarse representation contains:

```text
speech_content_summary
apparent_topic
```

### Focused summary

The focused representation contains the richer participant-centric semantic description generated by the focused semantic-analysis prompt.

Existing semantic caches are reused where available.

Missing summaries are explicitly identified, generated, retried where necessary, and inserted into the final database.

LAG cases inherit the **exact semantic summaries of their corresponding NORMAL source conversations**, because only their temporal alignment has been altered.

The notebook performs repeated coverage audits until the final 400-case database has complete coarse and focused semantic coverage.

---

## 5. Participation evidence

A participant-level `speaks` field is added from the final filtered VAD turns:

```text
speaks = True
    if the participant has at least one filtered speaking turn

speaks = False
    otherwise
```

This produces the expected participation structure:

```text
NORMAL:         A speaks=True, B speaks=True
LAG:            A speaks=True, B speaks=True
WRONG_PARTNER:  A speaks=True, B speaks=True
SILENT_PARTNER: A speaks=True, B speaks=False
```

Any legacy `speaks` field embedded inside focused semantic summaries is removed so that participation grounding comes only from the participant-level VAD-derived field.

---

## 6. Final cached evidence database

After all temporal, semantic, and participation enrichment steps, the notebook produces the final reusable database:

```text
consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
```

with the verified composition:

```text
normal          100
wrong_partner   100
lag_2sec         50
lag_3sec         50
silent_partner  100
-------------------
total           400
```

Each case therefore contains the structured observations required by the later consolidation reasoner:

```text
participant identity / source metadata
participant-level speaks
filtered speaking turns
local temporal features
global temporal features
coarse semantic summaries
focused semantic summaries
gold evaluation metadata
```

Gold metadata are retained in the stored case records for evaluation and auditing, but the later consolidation notebooks explicitly construct model-facing payloads that exclude hidden case identity and gold-label information.

---

# Role in the Consolidation Experiments

This notebook performs **data and evidence preparation only**.

It does not evaluate Binary-only, Structured R1, rationale-only, or R1+rationale reasoning.

Instead, it freezes the common observations used by all of them:

```text
Raw audiovisual conversations
        ↓
participant-centric preprocessing
        ↓
semantic + temporal + participation evidence
        ↓
cached 400-case structured database
        ↓
-----------------------------------------
        ↓
Binary-only
Structured R1
Free-form rationale
Structured R1 + rationale
```

This separation is methodologically important: the consolidation strategies can be compared using **identical underlying observations**, so changes in performance can be attributed to the reasoner's consolidation/output policy rather than to different upstream feature extraction runs.

> **Reproducibility note:** every code cell, saved output, execution count, code-cell metadata field, intermediate cache, audit, semantic-generation result, and final database output is preserved exactly as executed. Only this introductory Markdown cell has been rewritten for the public repository.

## Methodological role of this step

The dataset is divided into two disjoint groups:

### Frozen reference conversations

Exactly **50 eligible normal conversations** are selected deterministically. They are used only to construct the frozen temporal reference profiles:

- `NORMAL`
- `LAG_1`
- `LAG_2`
- `LAG_3`

These conversations are not part of the held-out consolidation evaluation set.

### Held-out conversations

Exactly **100 different eligible normal conversations** are selected after removing the reference conversations. These are the normal source conversations that will later form the basis of the consolidation dataset.

At this stage the original lag notebook also constructs its held-out synthetic lag cases and computes their temporal features. This behaviour is retained exactly because the goal of this notebook is to reproduce the original split and preprocessing pipeline without changing its implementation.

### Important isolation property

The notebook asserts that the reference conversation IDs and the held-out conversation IDs are disjoint. Therefore, statistics calculated from the 50 reference conversations do not use the 100 held-out conversations.

## 1. Mount Google Drive and configure the experiment

This cell:

- mounts Google Drive;
- defines the dataset and output directories;
- fixes all random seeds and sample sizes;
- defines the VAD eligibility criteria;
- defines the 120-second analysis window;
- defines all paths for the split manifests, temporal-feature caches, and frozen statistics.

The code is copied verbatim from the source notebook. Some inert model-related constants remain because the complete original configuration cell is preserved, but no model-loading or inference cells are included later.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from functools import lru_cache
import hashlib
import json
import random
import re

import pandas as pd
from tqdm.auto import tqdm

DATA_ROOT = Path(
    "/content/drive/MyDrive/seamless_download/data"
)

OUT_DIR = Path(
    "/content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
NUM_REFERENCE_CONVERSATIONS = 50
NUM_INFERENCE_CONVERSATIONS = 100

INFERENCE_LAG_COUNTS = {
    1.0: 30,
    2.0: 35,
    3.0: 35,
}

REFERENCE_LAG_SECONDS = [
    1.0,
    2.0,
    3.0,
]

MAX_SECONDS = 120.0
MIN_TURNS_PER_PARTICIPANT = 5
MIN_SPEECH_SECONDS_PER_PARTICIPANT = 10.0

MERGE_GAP_SECONDS = 0.75
BOUNDARY_SAFE_MODE = False

PROMPT_VERSION = "mixed_lag_frozen_reference_v1"
MODEL_ID = "Qwen/Qwen2.5-Omni-7B"
MAX_NEW_TOKENS = 260

# None runs all 210 held-out inference cases.
PILOT_MAX_CASES = None

REFERENCE_SELECTION_PATH = OUT_DIR / "reference_50_conversations.json"
INFERENCE_SELECTION_PATH = OUT_DIR / "heldout_100_conversations.json"

REFERENCE_CASES_PATH = OUT_DIR / "reference_cases_normal_lag1_lag2_lag3.json"
INFERENCE_CASES_PATH = OUT_DIR / "inference_cases_100_normal_110_mixed_lag.json"

REFERENCE_FEATURE_CASES_PATH = OUT_DIR / "reference_feature_cases.json"
INFERENCE_FEATURE_CASES_PATH = OUT_DIR / "inference_feature_cases.json"

REFERENCE_BASE_STATS_PATH = OUT_DIR / "frozen_reference_base_statistics.json"
REFERENCE_SHIFT_STATS_PATH = OUT_DIR / "frozen_reference_global_shift_statistics.json"

GLOBAL_SHIFT_CACHE_PATH = OUT_DIR / "global_shift_feature_cache.json"

RESULTS_PATH = OUT_DIR / "qwen_mixed_lag_binary_results.json"
RESULTS_CSV_PATH = OUT_DIR / "qwen_mixed_lag_binary_results.csv"

assert DATA_ROOT.exists(), f"Dataset folder not found: {DATA_ROOT}"
assert sum(INFERENCE_LAG_COUNTS.values()) == 100
assert NUM_REFERENCE_CONVERSATIONS + NUM_INFERENCE_CONVERSATIONS == 150

print("DATA_ROOT:", DATA_ROOT)
print("OUT_DIR:", OUT_DIR)
print("Reference conversations:", NUM_REFERENCE_CONVERSATIONS)
print("Held-out inference conversations:", NUM_INFERENCE_CONVERSATIONS)
print("Held-out lag counts:", INFERENCE_LAG_COUNTS)
print("Boundary-safe mode:", BOUNDARY_SAFE_MODE)

Mounted at /content/drive
DATA_ROOT: /content/drive/MyDrive/seamless_download/data
OUT_DIR: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec
Reference conversations: 50
Held-out inference conversations: 100
Held-out lag counts: {1.0: 30, 2.0: 35, 3.0: 35}
Boundary-safe mode: False


## 2. Discover clean dyadic normal conversations

The dataset is scanned conversation by conversation.

A conversation is retained at this stage only when:

- it is not the dedicated `silent` directory;
- it contains exactly two participant directories;
- both participants have a readable metadata JSON file.

For each participant, the metadata path and the number of raw VAD entries are recorded.

In [ ]:
def find_metadata_json(conversation_id: str, participant_id: str):
    participant_dir = DATA_ROOT / conversation_id / participant_id

    expected = participant_dir / f"{conversation_id}_{participant_id}.json"
    if expected.exists():
        return expected

    # Fallback for small filename differences.
    candidates = [
        p for p in sorted(participant_dir.glob("*.json"))
        if not p.name.endswith(".metadata.json")
    ]
    return candidates[0] if candidates else None


@lru_cache(maxsize=None)
def load_metadata_json(metadata_path_str: str):
    metadata_path = Path(metadata_path_str)
    with metadata_path.open("r", encoding="utf-8") as f:
        return json.load(f)


def scan_normal_conversations():
    records = []

    for conv_dir in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
        if conv_dir.name == "silent":
            continue

        participant_dirs = sorted(
            p for p in conv_dir.iterdir() if p.is_dir()
        )

        # Keep clean dyadic conversations only.
        if len(participant_dirs) != 2:
            continue

        participants = []

        for part_dir in participant_dirs:
            metadata_path = find_metadata_json(
                conv_dir.name,
                part_dir.name,
            )

            if metadata_path is None:
                participants = []
                break

            try:
                metadata = load_metadata_json(str(metadata_path))
            except Exception as exc:
                print("Could not read:", metadata_path, exc)
                participants = []
                break

            vad = metadata.get("metadata:vad", []) or []

            participants.append({
                "conversation_id": conv_dir.name,
                "participant_id": part_dir.name,
                "metadata_path": str(metadata_path),
                "num_raw_vad_entries": len(vad),
            })

        if len(participants) == 2:
            records.append({
                "conversation_id": conv_dir.name,
                "participants": participants,
            })

    return records


normal_conversations = scan_normal_conversations()

print("Clean dyadic normal conversations found:", len(normal_conversations))
print(json.dumps(normal_conversations[:2], indent=2))


Clean dyadic normal conversations found: 206
[
  {
    "conversation_id": "V00_S2017_I00001160",
    "participants": [
      {
        "conversation_id": "V00_S2017_I00001160",
        "participant_id": "P1273A",
        "metadata_path": "/content/drive/MyDrive/seamless_download/data/V00_S2017_I00001160/P1273A/V00_S2017_I00001160_P1273A.json",
        "num_raw_vad_entries": 49
      },
      {
        "conversation_id": "V00_S2017_I00001160",
        "participant_id": "P2072A",
        "metadata_path": "/content/drive/MyDrive/seamless_download/data/V00_S2017_I00001160/P2072A/V00_S2017_I00001160_P2072A.json",
        "num_raw_vad_entries": 33
      }
    ]
  },
  {
    "conversation_id": "V00_S2017_I00001161",
    "participants": [
      {
        "conversation_id": "V00_S2017_I00001161",
        "participant_id": "P1273A",
        "metadata_path": "/content/drive/MyDrive/seamless_download/data/V00_S2017_I00001161/P1273A/V00_S2017_I00001161_P1273A.json",
        "num_raw_vad_entries": 3

## 3. Extract, clip, merge, and summarize VAD turns

The following utilities reproduce the original VAD preprocessing:

1. VAD intervals are clipped to the first 120 seconds.
2. Invalid or empty intervals are removed.
3. Consecutive intervals are merged when their gap is at most `0.75` seconds.
4. Shift and localization utilities are defined for synthetic lag construction.
5. Total speaking time is calculated for eligibility filtering.

No semantic content or transcript is used.

In [ ]:
def merge_ranges(ranges, max_gap=MERGE_GAP_SECONDS):
    if not ranges:
        return []

    clean = sorted(
        [
            {
                "start": float(r["start"]),
                "end": float(r["end"]),
            }
            for r in ranges
            if float(r["end"]) > float(r["start"])
        ],
        key=lambda r: (r["start"], r["end"]),
    )

    if not clean:
        return []

    merged = [dict(clean[0])]

    for current in clean[1:]:
        previous = merged[-1]
        gap = current["start"] - previous["end"]

        if gap <= float(max_gap):
            previous["end"] = max(
                previous["end"],
                current["end"],
            )
        else:
            merged.append(dict(current))

    return [
        {
            "start": round(r["start"], 2),
            "end": round(r["end"], 2),
        }
        for r in merged
    ]


def get_full_merged_vad_turns(
    metadata_path,
    max_seconds=MAX_SECONDS,
):
    metadata = load_metadata_json(str(metadata_path))
    raw_vad = metadata.get("metadata:vad", []) or []

    clipped = []

    for item in raw_vad:
        try:
            start = float(item.get("start", 0.0))
            end = float(item.get("end", start))
        except (TypeError, ValueError):
            continue

        clipped_start = max(0.0, start)
        clipped_end = min(float(max_seconds), end)

        if clipped_end > clipped_start:
            clipped.append({
                "start": clipped_start,
                "end": clipped_end,
            })

    return merge_ranges(
        clipped,
        max_gap=MERGE_GAP_SECONDS,
    )


def shift_ranges(ranges, shift_seconds):
    return [
        {
            "start": round(float(r["start"]) + shift_seconds, 2),
            "end": round(float(r["end"]) + shift_seconds, 2),
        }
        for r in ranges
    ]


def clip_and_localize_ranges(
    ranges,
    global_start,
    global_end,
):
    local = []

    for r in ranges:
        start = max(float(r["start"]), float(global_start))
        end = min(float(r["end"]), float(global_end))

        if end <= start:
            continue

        local.append({
            "start": round(start - global_start, 2),
            "end": round(end - global_start, 2),
        })

    return local


def total_speech_seconds(ranges):
    return round(
        sum(float(r["end"]) - float(r["start"]) for r in ranges),
        2,
    )


## 4. Build the deterministic 50-reference / 100-held-out split

A conversation is eligible only when both participants satisfy:

- at least 5 merged VAD turns;
- at least 10 seconds of total speech in the first 120 seconds.

The split then follows the exact original random procedure:

- reference selection seed: `42`;
- deterministic participant A/B orientation for the 50 references;
- removal of all reference conversation IDs;
- held-out selection seed: `2042`;
- deterministic participant A/B orientation for the 100 held-out conversations.

The code also verifies that the two conversation sets have zero overlap and saves both manifests to Google Drive.

In [ ]:
eligible_conversations = []

for conv in normal_conversations:
    participants = conv["participants"]

    turns_0 = get_full_merged_vad_turns(
        participants[0]["metadata_path"]
    )

    turns_1 = get_full_merged_vad_turns(
        participants[1]["metadata_path"]
    )

    if not turns_0 or not turns_1:
        continue

    if (
        len(turns_0) < MIN_TURNS_PER_PARTICIPANT
        or len(turns_1) < MIN_TURNS_PER_PARTICIPANT
        or total_speech_seconds(turns_0) < MIN_SPEECH_SECONDS_PER_PARTICIPANT
        or total_speech_seconds(turns_1) < MIN_SPEECH_SECONDS_PER_PARTICIPANT
    ):
        continue

    record = dict(conv)
    record["participant_0_full_turns"] = turns_0
    record["participant_1_full_turns"] = turns_1
    eligible_conversations.append(record)

required_conversations = (
    NUM_REFERENCE_CONVERSATIONS
    + NUM_INFERENCE_CONVERSATIONS
)

assert len(eligible_conversations) >= required_conversations, (
    f"Only {len(eligible_conversations)} eligible conversations were found; "
    f"{required_conversations} are required."
)

# Exact recreation of the original deterministic 50-conversation sample.
reference_rng = random.Random(RANDOM_SEED)

selected_reference_conversations = reference_rng.sample(
    eligible_conversations,
    NUM_REFERENCE_CONVERSATIONS,
)


def orient_conversation(conv, reverse_direction):
    if reverse_direction:
        participant_A = conv["participants"][1]
        participant_B = conv["participants"][0]
        turns_A_full = conv["participant_1_full_turns"]
        turns_B_full = conv["participant_0_full_turns"]
    else:
        participant_A = conv["participants"][0]
        participant_B = conv["participants"][1]
        turns_A_full = conv["participant_0_full_turns"]
        turns_B_full = conv["participant_1_full_turns"]

    return {
        "conversation_id": conv["conversation_id"],
        "participant_A": participant_A,
        "participant_B": participant_B,
        "turns_A_full_0_120": turns_A_full,
        "turns_B_full_0_120": turns_B_full,
    }


reference_selection_manifest = []

for conv in selected_reference_conversations:
    reverse_direction = bool(reference_rng.getrandbits(1))

    reference_selection_manifest.append(
        orient_conversation(
            conv,
            reverse_direction,
        )
    )

reference_ids = {
    record["conversation_id"]
    for record in reference_selection_manifest
}

remaining_conversations = [
    conv
    for conv in eligible_conversations
    if conv["conversation_id"] not in reference_ids
]

inference_rng = random.Random(RANDOM_SEED + 2000)

selected_inference_conversations = inference_rng.sample(
    remaining_conversations,
    NUM_INFERENCE_CONVERSATIONS,
)

inference_selection_manifest = []

for conv in selected_inference_conversations:
    reverse_direction = bool(inference_rng.getrandbits(1))

    inference_selection_manifest.append(
        orient_conversation(
            conv,
            reverse_direction,
        )
    )

inference_ids = {
    record["conversation_id"]
    for record in inference_selection_manifest
}

assert reference_ids.isdisjoint(inference_ids)
assert len(reference_selection_manifest) == 50
assert len(inference_selection_manifest) == 100

# Assign 110 lag cases with only the unavoidable 10 repeated source conversations.
assignment_rng = random.Random(RANDOM_SEED + 3000)
assignment_order = list(range(NUM_INFERENCE_CONVERSATIONS))
assignment_rng.shuffle(assignment_order)

lag_3_indices = set(assignment_order[:40])
lag_2_indices = set(assignment_order[40:80])
lag_1_indices = set(assignment_order[80:100])
lag_1_indices.update(
    assignment_rng.sample(
        assignment_order[:80],
        10,
    )
)

for index, record in enumerate(inference_selection_manifest):
    assigned_lags = []

    if index in lag_1_indices:
        assigned_lags.append(1.0)

    if index in lag_2_indices:
        assigned_lags.append(2.0)

    if index in lag_3_indices:
        assigned_lags.append(3.0)

    record["assigned_lag_seconds"] = assigned_lags

assert sum(1.0 in r["assigned_lag_seconds"] for r in inference_selection_manifest) == 30
assert sum(2.0 in r["assigned_lag_seconds"] for r in inference_selection_manifest) == 40
assert sum(3.0 in r["assigned_lag_seconds"] for r in inference_selection_manifest) == 40

REFERENCE_SELECTION_PATH.write_text(
    json.dumps(
        reference_selection_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

INFERENCE_SELECTION_PATH.write_text(
    json.dumps(
        inference_selection_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

# Optional strict verification against the previous +1 s selection manifest.
legacy_selection_path = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_lag_1sec/"
    "selected_50_conversations.json"
)

if legacy_selection_path.exists():
    legacy_manifest = json.loads(
        legacy_selection_path.read_text(
            encoding="utf-8"
        )
    )

    legacy_signature = [
        (
            record["conversation_id"],
            record["participant_A"]["participant_id"],
            record["participant_B"]["participant_id"],
        )
        for record in legacy_manifest
    ]

    current_signature = [
        (
            record["conversation_id"],
            record["participant_A"]["participant_id"],
            record["participant_B"]["participant_id"],
        )
        for record in reference_selection_manifest
    ]

    assert legacy_signature == current_signature, (
        "The recreated reference split does not match the previous 50-conversation split."
    )

    print("Verified: reference 50 exactly match the previous +1 s selection.")
else:
    print(
        "Legacy selection file not found. "
        "The original deterministic selection algorithm was reproduced exactly."
    )

print("Eligible conversations:", len(eligible_conversations))
print("Reference conversations:", len(reference_selection_manifest))
print("Held-out conversations:", len(inference_selection_manifest))
print("Reference/inference overlap:", len(reference_ids & inference_ids))
print("Saved reference selection:", REFERENCE_SELECTION_PATH)
print("Saved inference selection:", INFERENCE_SELECTION_PATH)

Verified: reference 50 exactly match the previous +1 s selection.
Eligible conversations: 167
Reference conversations: 50
Held-out conversations: 100
Reference/inference overlap: 0
Saved reference selection: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reference_50_conversations.json
Saved inference selection: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/heldout_100_conversations.json


## 5. Construct reference and held-out observed cases

For each of the 50 reference conversations, four observed cases are created:

- normal;
- Participant B shifted by +1 second;
- Participant B shifted by +2 seconds;
- Participant B shifted by +3 seconds.

This produces 200 reference-profile cases.

For the 100 held-out conversations, the original notebook creates:

- 100 normal cases;
- 30 lag +1-second cases;
- 40 lag +2-second cases;
- 40 lag +3-second cases.

This produces 210 held-out cases.

Each case receives a stable deterministic ID. The generated case manifests contain construction and gold metadata for internal bookkeeping; those fields are not model inputs in this notebook because no model inference occurs here.

In [ ]:
def stable_case_id(
    conversation_id,
    split_name,
    variant,
    lag_seconds,
):
    payload = (
        f"{conversation_id}|{split_name}|{variant}|"
        f"{float(lag_seconds):.1f}|{MAX_SECONDS}|"
        f"{MERGE_GAP_SECONDS}|{BOUNDARY_SAFE_MODE}|"
        f"{PROMPT_VERSION}"
    )

    digest = hashlib.sha1(
        payload.encode("utf-8")
    ).hexdigest()[:12]

    return f"case_{digest}"


def build_observed_case(
    record,
    split_name,
    lag_seconds,
):
    lag_seconds = float(lag_seconds)

    full_A = record["turns_A_full_0_120"]
    full_B = record["turns_B_full_0_120"]

    observed_B = (
        shift_ranges(full_B, lag_seconds)
        if lag_seconds > 0
        else full_B
    )

    if BOUNDARY_SAFE_MODE:
        boundary_margin = max(REFERENCE_LAG_SECONDS)
        analysis_start = float(boundary_margin)
        analysis_end = float(MAX_SECONDS - boundary_margin)
    else:
        analysis_start = 0.0
        analysis_end = float(MAX_SECONDS)

    analysis_duration = analysis_end - analysis_start

    turns_A = clip_and_localize_ranges(
        full_A,
        analysis_start,
        analysis_end,
    )

    turns_B = clip_and_localize_ranges(
        observed_B,
        analysis_start,
        analysis_end,
    )

    is_lag = lag_seconds > 0

    profile = (
        f"LAG_{int(lag_seconds)}"
        if is_lag
        else "NORMAL"
    )

    variant = (
        f"lag_{int(lag_seconds)}sec"
        if is_lag
        else "normal"
    )

    return {
        "case_id": stable_case_id(
            record["conversation_id"],
            split_name,
            variant,
            lag_seconds,
        ),
        "split_name": split_name,
        "conversation_id": record["conversation_id"],
        "participant_A_id": record["participant_A"]["participant_id"],
        "participant_B_id": record["participant_B"]["participant_id"],
        "variant": variant,
        "reference_profile": profile,
        "gold_label": "LAG" if is_lag else "NORMAL",
        "lag_seconds": lag_seconds,
        "analysis_global_start": analysis_start,
        "analysis_global_end": analysis_end,
        "analysis_duration": analysis_duration,
        "merge_gap_seconds": MERGE_GAP_SECONDS,
        "boundary_safe_mode": BOUNDARY_SAFE_MODE,
        "turns_A": turns_A,
        "turns_B": turns_B,
    }


reference_cases = []

for record in reference_selection_manifest:
    reference_cases.append(
        build_observed_case(
            record,
            split_name="reference",
            lag_seconds=0.0,
        )
    )

    for lag_seconds in REFERENCE_LAG_SECONDS:
        reference_cases.append(
            build_observed_case(
                record,
                split_name="reference",
                lag_seconds=lag_seconds,
            )
        )

inference_cases = []

for record in inference_selection_manifest:
    inference_cases.append(
        build_observed_case(
            record,
            split_name="inference",
            lag_seconds=0.0,
        )
    )

    for lag_seconds in record["assigned_lag_seconds"]:
        inference_cases.append(
            build_observed_case(
                record,
                split_name="inference",
                lag_seconds=lag_seconds,
            )
        )

inference_case_rng = random.Random(RANDOM_SEED + 4000)
inference_case_rng.shuffle(inference_cases)

assert len(reference_cases) == 200
assert len(inference_cases) == 210

assert sum(c["reference_profile"] == "NORMAL" for c in reference_cases) == 50
assert sum(c["reference_profile"] == "LAG_1" for c in reference_cases) == 50
assert sum(c["reference_profile"] == "LAG_2" for c in reference_cases) == 50
assert sum(c["reference_profile"] == "LAG_3" for c in reference_cases) == 50

assert sum(c["reference_profile"] == "NORMAL" for c in inference_cases) == 100
assert sum(c["reference_profile"] == "LAG_1" for c in inference_cases) == 30
assert sum(c["reference_profile"] == "LAG_2" for c in inference_cases) == 40
assert sum(c["reference_profile"] == "LAG_3" for c in inference_cases) == 40

reference_case_conversation_ids = {
    case["conversation_id"]
    for case in reference_cases
}

inference_case_conversation_ids = {
    case["conversation_id"]
    for case in inference_cases
}

assert reference_case_conversation_ids.isdisjoint(
    inference_case_conversation_ids
)

REFERENCE_CASES_PATH.write_text(
    json.dumps(
        reference_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

INFERENCE_CASES_PATH.write_text(
    json.dumps(
        inference_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Reference cases:", len(reference_cases))
print(pd.Series([c["reference_profile"] for c in reference_cases]).value_counts().sort_index())

print("\nHeld-out inference cases:", len(inference_cases))
print(pd.Series([c["reference_profile"] for c in inference_cases]).value_counts().sort_index())

print(
    "\nReference/inference conversation overlap:",
    len(reference_case_conversation_ids & inference_case_conversation_ids),
)

Reference cases: 200
LAG_1     50
LAG_2     50
LAG_3     50
NORMAL    50
Name: count, dtype: int64

Held-out inference cases: 210
LAG_1      30
LAG_2      40
LAG_3      40
NORMAL    100
Name: count, dtype: int64

Reference/inference conversation overlap: 0


## 6. Define the shared signed-offset utilities

These utilities implement the exact local temporal-event representation used by the lag pipeline.

For each valid A-to-B handoff, the signed offset is:

`B_start - A_end`

- negative values represent a slight pre-end start by B;
- zero represents an immediate boundary;
- positive values represent a delayed B response.

The extraction retains only events satisfying the original strict temporal rules.

In [ ]:
import numpy as np
import pandas as pd
from bisect import bisect_left

# Exact limits used by the final experiment.
DURATION_ONLY_MAX_POST_DELAY_SECONDS = 6.0
DURATION_ONLY_MAX_PRESTART_SECONDS = 2.0

def duration_only_sorted_turns(turns):
    """
    Return sorted copies of the supplied turns.
    """

    return sorted(
        [
            {
                "start": float(turn["start"]),
                "end": float(turn["end"]),
            }
            for turn in turns
            if (
                float(turn["end"])
                > float(turn["start"])
            )
        ],
        key=lambda turn: (
            turn["start"],
            turn["end"],
        ),
    )


def duration_only_single_overlap_seconds(
    turn_A,
    turn_B,
):
    return max(
        0.0,
        min(
            float(turn_A["end"]),
            float(turn_B["end"]),
        )
        - max(
            float(turn_A["start"]),
            float(turn_B["start"]),
        ),
    )


def duration_only_total_overlap_seconds(
    turns_A,
    turns_B,
):
    total = 0.0

    for A_turn in turns_A:
        for B_turn in turns_B:
            total += (
                duration_only_single_overlap_seconds(
                    A_turn,
                    B_turn,
                )
            )

    return total


def duration_only_signed_strict_events(
    turns_A,
    turns_B,
    max_post_delay_seconds=(
        DURATION_ONLY_MAX_POST_DELAY_SECONDS
    ),
    max_prestart_seconds=(
        DURATION_ONLY_MAX_PRESTART_SECONDS
    ),
):
    """
    Create at most one signed event for each relevant A_end.

    Negative event:

        A_start < B_start < A_end < B_end

        B starts up to max_prestart_seconds before A_end.

    Positive event:

        A_end <= B_start

        B starts up to max_post_delay_seconds after A_end,
        and A does not restart before or at B_start.

    signed_offset = B_start - A_end
    """

    A_turns = duration_only_sorted_turns(
        turns_A
    )

    B_turns = duration_only_sorted_turns(
        turns_B
    )

    B_starts = [
        float(turn["start"])
        for turn in B_turns
    ]

    events = []

    for A_index, A_turn in enumerate(A_turns):
        A_start = float(
            A_turn["start"]
        )

        A_end = float(
            A_turn["end"]
        )

        # ----------------------------------------------------
        # Negative event:
        # B is already active when A ends.
        # ----------------------------------------------------

        active_B_candidates = []

        for B_index, B_turn in enumerate(B_turns):
            B_start = float(
                B_turn["start"]
            )

            B_end = float(
                B_turn["end"]
            )

            if B_start < A_end < B_end:
                active_B_candidates.append(
                    (
                        B_index,
                        B_start,
                        B_end,
                    )
                )

        if active_B_candidates:
            recent_candidates = [
                (
                    B_index,
                    B_start,
                    B_end,
                )
                for (
                    B_index,
                    B_start,
                    B_end,
                ) in active_B_candidates
                if (
                    A_start < B_start < A_end
                    and (
                        A_end - B_start
                        <= float(
                            max_prestart_seconds
                        )
                    )
                )
            ]

            if recent_candidates:
                # Keep the B start closest to A_end.
                (
                    B_index,
                    B_start,
                    B_end,
                ) = max(
                    recent_candidates,
                    key=lambda item: item[1],
                )

                signed_offset = (
                    B_start - A_end
                )

                events.append({
                    "event_type": (
                        "B_STARTS_SHORTLY_BEFORE_A_END"
                    ),

                    "A_turn_index": int(
                        A_index
                    ),

                    "A_start": round(
                        A_start,
                        2,
                    ),

                    "A_end": round(
                        A_end,
                        2,
                    ),

                    "B_turn_index": int(
                        B_index
                    ),

                    "B_start": round(
                        B_start,
                        2,
                    ),

                    "B_end": round(
                        B_end,
                        2,
                    ),

                    "signed_offset_seconds": round(
                        signed_offset,
                        2,
                    ),
                })

            # Exact same priority as previous experiment:
            # when B is already active at A_end, do not also
            # search for a future positive event.
            continue

        # ----------------------------------------------------
        # Positive event:
        # B begins after A ends.
        # ----------------------------------------------------

        B_index = bisect_left(
            B_starts,
            A_end,
        )

        if B_index >= len(B_starts):
            continue

        B_start = float(
            B_starts[B_index]
        )

        signed_offset = (
            B_start - A_end
        )

        if signed_offset < 0:
            continue

        if (
            signed_offset
            > float(max_post_delay_seconds)
        ):
            continue

        next_A_start = (
            float(
                A_turns[
                    A_index + 1
                ]["start"]
            )
            if (
                A_index + 1
                < len(A_turns)
            )
            else None
        )

        # Same strict exclusion as previous experiment.
        if (
            next_A_start is not None
            and next_A_start <= B_start
        ):
            continue

        B_end = float(
            B_turns[B_index]["end"]
        )

        events.append({
            "event_type": (
                "B_STARTS_AFTER_A_END"
            ),

            "A_turn_index": int(
                A_index
            ),

            "A_start": round(
                A_start,
                2,
            ),

            "A_end": round(
                A_end,
                2,
            ),

            "B_turn_index": int(
                B_index
            ),

            "B_start": round(
                B_start,
                2,
            ),

            "B_end": round(
                B_end,
                2,
            ),

            "signed_offset_seconds": round(
                signed_offset,
                2,
            ),
        })

    return events


def compact_turns_text(turns):
    compact = [
        [
            round(float(turn["start"]), 2),
            round(float(turn["end"]), 2),
        ]
        for turn in turns
    ]

    return json.dumps(
        compact,
        ensure_ascii=False,
        separators=(",", ":"),
    )

## 7. Apply independent operational backchannel filtering

A turn is removed as a backchannel-like candidate only when both conditions hold:

- its duration is at most 1.0 second;
- at least 80% of its duration overlaps speech from the other participant.

Filtering is applied independently to A and B from the observed turns of each case. It does not use the case label or lag magnitude.

In [ ]:
# ============================================================
# NEW EXPERIMENT
#
# Same pipeline as the duration-only signed-offset experiment.
#
# ONLY CHANGE:
#
# Independently remove operational backchannel-like turns:
#
#   duration <= 1.0 sec
#   AND
#   overlap fraction >= 80%
#
# Filtering is performed independently on every observed
# NORMAL or LAG case.
#
# No matching NORMAL case is used.
# No gold label is used during filtering/feature extraction.
# ============================================================

import json
import numpy as np
import pandas as pd


INDEPENDENT_BC_EXPERIMENT_VERSION = (
    "guided_independent_backchannel_filter_signed_offsets_v1"
)

INDEPENDENT_BC_MAX_DURATION_SECONDS = 1.0

INDEPENDENT_BC_MIN_OVERLAP_FRACTION = 0.80

INDEPENDENT_BC_DELAY_THRESHOLD_SECONDS = 1.5

# Same signed-offset limits as the previous experiment
INDEPENDENT_BC_MAX_POST_DELAY_SECONDS = 6.0
INDEPENDENT_BC_MAX_PRESTART_SECONDS = 2.0

# None = all 100 cases
# Set to 4 for a quick pilot
INDEPENDENT_BC_PILOT_MAX_CASES = None

INDEPENDENT_BC_MAX_NEW_TOKENS = 260


INDEPENDENT_BC_CASES_PATH = (
    OUT_DIR
    / "guided_independent_backchannel_signed_offset_cases.json"
)

INDEPENDENT_BC_RESULTS_PATH = (
    OUT_DIR
    / "qwen_guided_independent_backchannel_signed_offset_results.json"
)

INDEPENDENT_BC_RESULTS_CSV_PATH = (
    OUT_DIR
    / "qwen_guided_independent_backchannel_signed_offset_results.csv"
)


INDEPENDENT_BC_PROMPT_VERSION = (
    f"{INDEPENDENT_BC_EXPERIMENT_VERSION}"
    f"_duration{INDEPENDENT_BC_MAX_DURATION_SECONDS}"
    f"_overlap{INDEPENDENT_BC_MIN_OVERLAP_FRACTION}"
    f"_threshold{INDEPENDENT_BC_DELAY_THRESHOLD_SECONDS}"
    f"_postmax{INDEPENDENT_BC_MAX_POST_DELAY_SECONDS}"
    f"_premax{INDEPENDENT_BC_MAX_PRESTART_SECONDS}"
)


print(
    "Independent backchannel prompt version:",
    INDEPENDENT_BC_PROMPT_VERSION,
)


# ============================================================
# Compute the fraction of one target turn covered by
# the other participant's turns
# ============================================================

def independent_bc_overlap_fraction(
    target_turn,
    other_turns,
):
    """
    Return the fraction of target_turn covered by the union
    of the other participant's speaking turns.

    Using the union prevents accidental double counting.
    """

    target_start = float(
        target_turn["start"]
    )

    target_end = float(
        target_turn["end"]
    )

    target_duration = (
        target_end - target_start
    )

    if target_duration <= 0:
        return 0.0

    intersections = []

    for other_turn in other_turns:
        overlap_start = max(
            target_start,
            float(other_turn["start"]),
        )

        overlap_end = min(
            target_end,
            float(other_turn["end"]),
        )

        if overlap_end > overlap_start:
            intersections.append(
                (
                    overlap_start,
                    overlap_end,
                )
            )

    if not intersections:
        return 0.0

    intersections = sorted(
        intersections,
        key=lambda interval: (
            interval[0],
            interval[1],
        ),
    )

    merged_intersections = [
        list(intersections[0])
    ]

    for start, end in intersections[1:]:
        previous = merged_intersections[-1]

        if start <= previous[1]:
            previous[1] = max(
                previous[1],
                end,
            )
        else:
            merged_intersections.append(
                [start, end]
            )

    covered_seconds = sum(
        end - start
        for start, end
        in merged_intersections
    )

    return float(
        min(
            1.0,
            covered_seconds / target_duration,
        )
    )


# ============================================================
# Filter one participant independently
# ============================================================

def independent_bc_filter_stream(
    target_turns,
    other_turns,
    max_duration_seconds=(
        INDEPENDENT_BC_MAX_DURATION_SECONDS
    ),
    min_overlap_fraction=(
        INDEPENDENT_BC_MIN_OVERLAP_FRACTION
    ),
):
    """
    A target turn is removed only when:

        duration <= 1 sec
        AND
        overlap fraction >= 80%

    No semantic information or gold label is used.
    """

    target_turns = duration_only_sorted_turns(
        target_turns
    )

    other_turns = duration_only_sorted_turns(
        other_turns
    )

    filtered_turns = []
    removed_turns = []
    removed_indices = []

    for index, turn in enumerate(target_turns):
        duration = (
            float(turn["end"])
            - float(turn["start"])
        )

        overlap_fraction = (
            independent_bc_overlap_fraction(
                turn,
                other_turns,
            )
        )

        is_backchannel_like = (
            duration
            <= float(max_duration_seconds) + 1e-9
            and overlap_fraction
            >= float(min_overlap_fraction) - 1e-9
        )

        if is_backchannel_like:
            removed_indices.append(
                int(index)
            )

            removed_turns.append({
                "turn_index": int(index),

                "start": round(
                    float(turn["start"]),
                    3,
                ),

                "end": round(
                    float(turn["end"]),
                    3,
                ),

                "duration": round(
                    duration,
                    3,
                ),

                "overlap_fraction": round(
                    overlap_fraction,
                    4,
                ),

                "overlap_percent": round(
                    100.0 * overlap_fraction,
                    2,
                ),
            })

        else:
            filtered_turns.append({
                "start": float(turn["start"]),
                "end": float(turn["end"]),
            })

    return {
        "original_turns": target_turns,
        "filtered_turns": filtered_turns,

        "removed_indices": removed_indices,
        "removed_turns": removed_turns,

        "num_original_turns": len(
            target_turns
        ),

        "num_filtered_turns": len(
            filtered_turns
        ),

        "num_removed_turns": len(
            removed_turns
        ),
    }


# ============================================================
# Filter each observed case independently
# ============================================================

def independently_filter_case_backchannels(
    case,
):
    """
    Detect A backchannels against the observed B turns.

    Detect B backchannels against the observed A turns.

    Both decisions are made from the original observed pair,
    before either participant is filtered.

    The function does not inspect:
        - gold_label
        - variant
        - lag_seconds
        - matching NORMAL cases
    """

    original_A = duration_only_sorted_turns(
        case["turns_A"]
    )

    original_B = duration_only_sorted_turns(
        case["turns_B"]
    )

    A_result = independent_bc_filter_stream(
        target_turns=original_A,
        other_turns=original_B,
    )

    B_result = independent_bc_filter_stream(
        target_turns=original_B,
        other_turns=original_A,
    )

    return {
        "original_turns_A": original_A,
        "original_turns_B": original_B,

        "filtered_turns_A": A_result[
            "filtered_turns"
        ],

        "filtered_turns_B": B_result[
            "filtered_turns"
        ],

        "num_original_A_turns": A_result[
            "num_original_turns"
        ],

        "num_original_B_turns": B_result[
            "num_original_turns"
        ],

        "num_filtered_A_turns": A_result[
            "num_filtered_turns"
        ],

        "num_filtered_B_turns": B_result[
            "num_filtered_turns"
        ],

        "num_removed_A_backchannels": A_result[
            "num_removed_turns"
        ],

        "num_removed_B_backchannels": B_result[
            "num_removed_turns"
        ],

        "removed_A_backchannel_indices": A_result[
            "removed_indices"
        ],

        "removed_B_backchannel_indices": B_result[
            "removed_indices"
        ],

        "removed_A_backchannels": A_result[
            "removed_turns"
        ],

        "removed_B_backchannels": B_result[
            "removed_turns"
        ],
    }

Independent backchannel prompt version: guided_independent_backchannel_filter_signed_offsets_v1_duration1.0_overlap0.8_threshold1.5_postmax6.0_premax2.0


## 8. Compute clean overlap and signed strict-offset features

After independent filtering, the notebook calculates:

- clean overlap duration and percentage;
- the exact signed A-end-to-B-start offset events;
- counts of negative and positive offsets;
- mean, median, minimum, maximum, P75, and P90;
- the number and percentage of offsets above 1.5 seconds;
- turn and removed-backchannel counts;
- the final filtered A and B turn lists.

These are deterministic temporal features derived only from VAD timing.

In [ ]:
# ============================================================
# Overlap after independent backchannel filtering
# ============================================================

def compute_independent_bc_overlap_features(
    original_turns_A,
    original_turns_B,
    filtered_turns_A,
    filtered_turns_B,
    timeline_duration,
):
    raw_overlap = (
        duration_only_total_overlap_seconds(
            original_turns_A,
            original_turns_B,
        )
    )

    clean_overlap = (
        duration_only_total_overlap_seconds(
            filtered_turns_A,
            filtered_turns_B,
        )
    )

    removed_overlap = max(
        0.0,
        raw_overlap - clean_overlap,
    )

    duration = float(
        timeline_duration
    )

    overlap_ratio = (
        clean_overlap / duration
        if duration > 0
        else 0.0
    )

    return {
        "raw_overlap_seconds": round(
            raw_overlap,
            2,
        ),

        "backchannel_overlap_removed_seconds": round(
            removed_overlap,
            2,
        ),

        "clean_overlap_seconds": round(
            clean_overlap,
            2,
        ),

        "clean_overlap_ratio": round(
            overlap_ratio,
            4,
        ),

        "clean_overlap_percent": round(
            100.0 * overlap_ratio,
            2,
        ),
    }


# ============================================================
# Summarize the exact same signed-offset distribution
# ============================================================

def summarize_independent_bc_signed_offsets(
    events,
):
    offsets = np.asarray(
        [
            event["signed_offset_seconds"]
            for event in events
        ],
        dtype=float,
    )

    num_negative = sum(
        event["event_type"]
        == "B_STARTS_SHORTLY_BEFORE_A_END"
        for event in events
    )

    num_positive = sum(
        event["event_type"]
        == "B_STARTS_AFTER_A_END"
        for event in events
    )

    if len(offsets) == 0:
        return {
            "signed_strict_offsets_seconds": [],
            "num_signed_strict_offsets": 0,

            "num_negative_preend_offsets": 0,
            "num_positive_postend_offsets": 0,

            "offset_mean_seconds": None,
            "offset_median_seconds": None,
            "offset_min_seconds": None,
            "offset_max_seconds": None,
            "offset_p75_seconds": None,
            "offset_p90_seconds": None,

            "num_offsets_above_1_5_seconds": 0,

            "percent_offsets_above_1_5_seconds": (
                None
            ),
        }

    num_above = int(
        np.sum(
            offsets
            > INDEPENDENT_BC_DELAY_THRESHOLD_SECONDS
        )
    )

    return {
        "signed_strict_offsets_seconds": (
            offsets.round(2).tolist()
        ),

        "num_signed_strict_offsets": int(
            len(offsets)
        ),

        "num_negative_preend_offsets": int(
            num_negative
        ),

        "num_positive_postend_offsets": int(
            num_positive
        ),

        "offset_mean_seconds": round(
            float(np.mean(offsets)),
            2,
        ),

        "offset_median_seconds": round(
            float(np.median(offsets)),
            2,
        ),

        "offset_min_seconds": round(
            float(np.min(offsets)),
            2,
        ),

        "offset_max_seconds": round(
            float(np.max(offsets)),
            2,
        ),

        "offset_p75_seconds": round(
            float(np.percentile(offsets, 75)),
            2,
        ),

        "offset_p90_seconds": round(
            float(np.percentile(offsets, 90)),
            2,
        ),

        "num_offsets_above_1_5_seconds": (
            num_above
        ),

        "percent_offsets_above_1_5_seconds": round(
            100.0
            * num_above
            / len(offsets),
            1,
        ),
    }


# ============================================================
# Compute all features for one observed case
# ============================================================

def compute_independent_bc_features(
    case,
):
    filtered = (
        independently_filter_case_backchannels(
            case
        )
    )

    overlap = (
        compute_independent_bc_overlap_features(
            filtered["original_turns_A"],
            filtered["original_turns_B"],

            filtered["filtered_turns_A"],
            filtered["filtered_turns_B"],

            timeline_duration=case[
                "analysis_duration"
            ],
        )
    )

    # Exact same strict signed-offset logic
    # as the previous duration-only experiment
    events = duration_only_signed_strict_events(
        filtered["filtered_turns_A"],
        filtered["filtered_turns_B"],

        max_post_delay_seconds=(
            INDEPENDENT_BC_MAX_POST_DELAY_SECONDS
        ),

        max_prestart_seconds=(
            INDEPENDENT_BC_MAX_PRESTART_SECONDS
        ),
    )

    distribution = (
        summarize_independent_bc_signed_offsets(
            events
        )
    )

    return {
        **overlap,
        **distribution,

        "backchannel_max_duration_seconds": (
            INDEPENDENT_BC_MAX_DURATION_SECONDS
        ),

        "backchannel_min_overlap_fraction": (
            INDEPENDENT_BC_MIN_OVERLAP_FRACTION
        ),

        "independent_filtering": True,

        "num_original_A_turns": filtered[
            "num_original_A_turns"
        ],

        "num_original_B_turns": filtered[
            "num_original_B_turns"
        ],

        "num_filtered_A_turns": filtered[
            "num_filtered_A_turns"
        ],

        "num_filtered_B_turns": filtered[
            "num_filtered_B_turns"
        ],

        "num_removed_A_backchannels": filtered[
            "num_removed_A_backchannels"
        ],

        "num_removed_B_backchannels": filtered[
            "num_removed_B_backchannels"
        ],

        "removed_A_backchannel_indices": filtered[
            "removed_A_backchannel_indices"
        ],

        "removed_B_backchannel_indices": filtered[
            "removed_B_backchannel_indices"
        ],

        "removed_A_backchannels": filtered[
            "removed_A_backchannels"
        ],

        "removed_B_backchannels": filtered[
            "removed_B_backchannels"
        ],

        "signed_strict_events": events,

        "filtered_turns_A": filtered[
            "filtered_turns_A"
        ],

        "filtered_turns_B": filtered[
            "filtered_turns_B"
        ],
    }

## 9. Attach temporal features and save feature-case caches

The exact same feature extractor is applied to:

- all 200 reference-profile cases;
- all 210 held-out cases.

The resulting feature-enriched case lists are saved as JSON files. These artifacts will later be reused by the consolidation pipeline rather than recomputed unnecessarily.

In [ ]:
def add_independent_bc_features(
    source_cases,
    description,
):
    output_cases = []

    for case in tqdm(
        source_cases,
        desc=description,
    ):
        new_case = dict(case)
        features = compute_independent_bc_features(case)

        new_case["independent_bc_temporal_features"] = features
        new_case["filtered_turns_A"] = features["filtered_turns_A"]
        new_case["filtered_turns_B"] = features["filtered_turns_B"]

        output_cases.append(new_case)

    return output_cases


reference_independent_bc_cases = add_independent_bc_features(
    reference_cases,
    "Reference temporal features",
)

inference_independent_bc_cases = add_independent_bc_features(
    inference_cases,
    "Inference temporal features",
)

REFERENCE_FEATURE_CASES_PATH.write_text(
    json.dumps(
        reference_independent_bc_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

INFERENCE_FEATURE_CASES_PATH.write_text(
    json.dumps(
        inference_independent_bc_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Reference feature cases:", len(reference_independent_bc_cases))
print("Inference feature cases:", len(inference_independent_bc_cases))

Reference temporal features:   0%|          | 0/200 [00:00<?, ?it/s]

Inference temporal features:   0%|          | 0/210 [00:00<?, ?it/s]

Reference feature cases: 200
Inference feature cases: 210


## 10. Compute the frozen base temporal reference statistics

The 200 reference cases are grouped into four separate profiles:

- `NORMAL`
- `LAG_1`
- `LAG_2`
- `LAG_3`

The profiles are not pooled.

For each profile, the notebook calculates the mean reference values for the signed-offset distribution, overlap, turn counts, and backchannel-filtering counts. The statistics are saved to:

`frozen_reference_base_statistics.json`

Assertions verify that each profile contains exactly 50 cases.

In [ ]:
def build_base_reference_row(case):
    features = case["independent_bc_temporal_features"]

    return {
        "case_id": case["case_id"],
        "conversation_id": case["conversation_id"],
        "reference_profile": case["reference_profile"],
        "num_original_A_turns": features["num_original_A_turns"],
        "num_original_B_turns": features["num_original_B_turns"],
        "num_removed_A_backchannels": features["num_removed_A_backchannels"],
        "num_removed_B_backchannels": features["num_removed_B_backchannels"],
        "num_filtered_A_turns": features["num_filtered_A_turns"],
        "num_filtered_B_turns": features["num_filtered_B_turns"],
        "num_offsets": features["num_signed_strict_offsets"],
        "num_negative": features["num_negative_preend_offsets"],
        "num_positive": features["num_positive_postend_offsets"],
        "offset_mean": features["offset_mean_seconds"],
        "offset_median": features["offset_median_seconds"],
        "offset_min": features["offset_min_seconds"],
        "offset_max": features["offset_max_seconds"],
        "offset_p75": features["offset_p75_seconds"],
        "offset_p90": features["offset_p90_seconds"],
        "percent_above_1_5": features["percent_offsets_above_1_5_seconds"],
        "clean_overlap_seconds": features["clean_overlap_seconds"],
        "clean_overlap_percent": features["clean_overlap_percent"],
    }


reference_base_df = pd.DataFrame(
    [
        build_base_reference_row(case)
        for case in reference_independent_bc_cases
    ]
)

PROFILE_ORDER = [
    "NORMAL",
    "LAG_1",
    "LAG_2",
    "LAG_3",
]

reference_base_stats = (
    reference_base_df
    .groupby("reference_profile")
    .mean(numeric_only=True)
    .reindex(PROFILE_ORDER)
)


def base_reference_value(profile, column):
    value = reference_base_stats.loc[profile, column]

    if pd.isna(value):
        return 0.0

    return float(value)


REFERENCE_BASE_STATS_PATH.write_text(
    json.dumps(
        reference_base_stats
        .reset_index()
        .to_dict(orient="records"),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("FROZEN BASE REFERENCE STATISTICS")

display(
    reference_base_stats[
        [
            "num_offsets",
            "offset_mean",
            "offset_median",
            "offset_max",
            "offset_p75",
            "offset_p90",
            "percent_above_1_5",
            "clean_overlap_seconds",
        ]
    ].round(3)
)

assert list(reference_base_stats.index) == PROFILE_ORDER
assert all(
    (reference_base_df["reference_profile"] == profile).sum() == 50
    for profile in PROFILE_ORDER
)

print("Saved frozen base statistics:", REFERENCE_BASE_STATS_PATH)

FROZEN BASE REFERENCE STATISTICS


,num_offsets,offset_mean,offset_median,offset_max,offset_p75,offset_p90,percent_above_1_5,clean_overlap_seconds
reference_profile,,,,,,,,
NORMAL,5.76,0.302,0.262,1.054,0.590,0.825,6.734,6.253
LAG_1,5.22,0.936,0.988,1.717,1.273,1.493,24.133,8.040
LAG_2,4.48,1.333,1.298,2.506,1.866,2.228,51.149,10.452
LAG_3,4.46,1.586,1.516,3.067,2.184,2.712,43.525,12.200


Saved frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json


## 11. Define the exact global B-correction search

This diagnostic searches for the hypothetical global temporal correction that would best align Participant B with Participant A.

The search is identical to the source notebook:

- correction range: `-6.0` to `+6.0` seconds;
- step: `0.1` seconds;
- bilateral A→B and B→A strict handoff events;
- score based on event coverage and proximity of offsets to zero.

A negative best correction means that B would need to move earlier, which is compatible with observed B lateness. The computation uses only filtered turns and does not use the gold label.

In [ ]:
# ============================================================
# DIAGNOSTIC FEATURE:
# Estimated global correction shift for Participant B
#
# Input:
#   independently backchannel-filtered A/B turns
#
# No gold label is used during feature computation.
# ============================================================

import numpy as np
import pandas as pd


# Search possible corrections applied to B.
#
# Negative:
#   B must move earlier.
#
# Positive:
#   B must move later.
ALIGNMENT_SHIFT_MIN_SECONDS = -6.0
ALIGNMENT_SHIFT_MAX_SECONDS = 6.0
ALIGNMENT_SHIFT_STEP_SECONDS = 0.10

# Controls how quickly large boundary errors lose score.
ALIGNMENT_OFFSET_SCALE_SECONDS = 1.50


def alignment_sorted_turns(turns):
    return sorted(
        [
            {
                "start": float(turn["start"]),
                "end": float(turn["end"]),
            }
            for turn in turns
            if (
                float(turn["end"])
                > float(turn["start"])
            )
        ],
        key=lambda turn: (
            turn["start"],
            turn["end"],
        ),
    )


def shift_turns_for_alignment(
    turns,
    shift_seconds,
    timeline_duration,
):
    """
    Apply a hypothetical global correction shift to B.

    Turns are clipped to the observed analysis window.
    """

    shift_seconds = float(
        shift_seconds
    )

    timeline_duration = float(
        timeline_duration
    )

    shifted = []

    for turn in alignment_sorted_turns(
        turns
    ):
        new_start = (
            float(turn["start"])
            + shift_seconds
        )

        new_end = (
            float(turn["end"])
            + shift_seconds
        )

        # Entire turn lies before or after the window.
        if new_end <= 0:
            continue

        if new_start >= timeline_duration:
            continue

        new_start = max(
            0.0,
            new_start,
        )

        new_end = min(
            timeline_duration,
            new_end,
        )

        if new_end <= new_start:
            continue

        shifted.append({
            "start": round(
                new_start,
                3,
            ),

            "end": round(
                new_end,
                3,
            ),
        })

    return shifted


def evaluate_bilateral_alignment_for_shift(
    turns_A,
    turns_B,
    shift_seconds,
    timeline_duration,
):
    """
    Evaluate one hypothetical correction shift of B.

    The same strict signed-offset extraction is used in
    both directions:

        A_end -> shifted B_start
        shifted B_end -> A_start

    The alignment score rewards:

    1. More valid bilateral strict events.
    2. Events whose signed offsets are close to zero.
    """

    turns_A = alignment_sorted_turns(
        turns_A
    )

    shifted_B = shift_turns_for_alignment(
        turns_B,
        shift_seconds=shift_seconds,
        timeline_duration=timeline_duration,
    )

    # A finishes -> B begins
    A_to_B_events = (
        duration_only_signed_strict_events(
            turns_A,
            shifted_B,

            max_post_delay_seconds=(
                INDEPENDENT_BC_MAX_POST_DELAY_SECONDS
            ),

            max_prestart_seconds=(
                INDEPENDENT_BC_MAX_PRESTART_SECONDS
            ),
        )
    )

    # B finishes -> A begins
    B_to_A_events = (
        duration_only_signed_strict_events(
            shifted_B,
            turns_A,

            max_post_delay_seconds=(
                INDEPENDENT_BC_MAX_POST_DELAY_SECONDS
            ),

            max_prestart_seconds=(
                INDEPENDENT_BC_MAX_PRESTART_SECONDS
            ),
        )
    )

    A_to_B_offsets = [
        float(
            event["signed_offset_seconds"]
        )
        for event in A_to_B_events
    ]

    B_to_A_offsets = [
        float(
            event["signed_offset_seconds"]
        )
        for event in B_to_A_events
    ]

    all_offsets = np.asarray(
        A_to_B_offsets
        + B_to_A_offsets,
        dtype=float,
    )

    num_events = int(
        len(all_offsets)
    )

    # Maximum possible evidence is approximately one
    # event per turn ending in both directions.
    possible_boundaries = max(
        1,
        len(turns_A)
        + len(shifted_B),
    )

    event_coverage = (
        num_events
        / possible_boundaries
    )

    if num_events == 0:
        return {
            "candidate_shift_seconds": round(
                float(shift_seconds),
                3,
            ),

            "alignment_score": 0.0,

            "num_bilateral_events": 0,
            "num_A_to_B_events": 0,
            "num_B_to_A_events": 0,

            "event_coverage": 0.0,

            "mean_abs_offset_seconds": None,
            "median_abs_offset_seconds": None,

            "mean_signed_offset_seconds": None,
            "median_signed_offset_seconds": None,

            "A_to_B_offsets": [],
            "B_to_A_offsets": [],

            "num_shifted_B_turns": len(
                shifted_B
            ),
        }

    absolute_offsets = np.abs(
        all_offsets
    )

    # Each event receives a score close to 1 when it lies
    # near zero, decreasing smoothly for larger offsets.
    boundary_closeness = float(
        np.mean(
            np.exp(
                -absolute_offsets
                / ALIGNMENT_OFFSET_SCALE_SECONDS
            )
        )
    )

    # A candidate needs both:
    #   good coverage
    #   offsets close to zero
    alignment_score = (
        event_coverage
        * boundary_closeness
    )

    return {
        "candidate_shift_seconds": round(
            float(shift_seconds),
            3,
        ),

        "alignment_score": round(
            float(alignment_score),
            6,
        ),

        "num_bilateral_events": num_events,

        "num_A_to_B_events": len(
            A_to_B_events
        ),

        "num_B_to_A_events": len(
            B_to_A_events
        ),

        "event_coverage": round(
            float(event_coverage),
            6,
        ),

        "mean_abs_offset_seconds": round(
            float(
                np.mean(
                    absolute_offsets
                )
            ),
            3,
        ),

        "median_abs_offset_seconds": round(
            float(
                np.median(
                    absolute_offsets
                )
            ),
            3,
        ),

        "mean_signed_offset_seconds": round(
            float(
                np.mean(
                    all_offsets
                )
            ),
            3,
        ),

        "median_signed_offset_seconds": round(
            float(
                np.median(
                    all_offsets
                )
            ),
            3,
        ),

        "A_to_B_offsets": [
            round(value, 3)
            for value in A_to_B_offsets
        ],

        "B_to_A_offsets": [
            round(value, 3)
            for value in B_to_A_offsets
        ],

        "num_shifted_B_turns": len(
            shifted_B
        ),
    }


def estimate_global_B_correction_shift(
    turns_A,
    turns_B,
    timeline_duration,
):
    """
    Search for the global B correction shift that produces
    the strongest bilateral boundary alignment.

    This function uses no label information.
    """

    candidate_shifts = np.round(
        np.arange(
            ALIGNMENT_SHIFT_MIN_SECONDS,
            (
                ALIGNMENT_SHIFT_MAX_SECONDS
                + ALIGNMENT_SHIFT_STEP_SECONDS / 2
            ),
            ALIGNMENT_SHIFT_STEP_SECONDS,
        ),
        3,
    )

    candidate_rows = [
        evaluate_bilateral_alignment_for_shift(
            turns_A=turns_A,
            turns_B=turns_B,

            shift_seconds=float(
                candidate_shift
            ),

            timeline_duration=(
                timeline_duration
            ),
        )
        for candidate_shift
        in candidate_shifts
    ]

    # Sort using:
    # 1. highest score
    # 2. highest event coverage
    # 3. lowest median absolute error
    # 4. smallest absolute correction as final tie-break
    def candidate_sort_key(row):
        median_abs = row[
            "median_abs_offset_seconds"
        ]

        if median_abs is None:
            median_abs = float("inf")

        return (
            -float(
                row["alignment_score"]
            ),

            -float(
                row["event_coverage"]
            ),

            float(median_abs),

            abs(
                float(
                    row[
                        "candidate_shift_seconds"
                    ]
                )
            ),
        )

    ranked_rows = sorted(
        candidate_rows,
        key=candidate_sort_key,
    )

    best = ranked_rows[0]

    zero_shift_rows = [
        row
        for row in candidate_rows
        if abs(
            float(
                row[
                    "candidate_shift_seconds"
                ]
            )
        ) < 1e-9
    ]

    if zero_shift_rows:
        zero_shift = zero_shift_rows[0]
    else:
        zero_shift = min(
            candidate_rows,
            key=lambda row: abs(
                float(
                    row[
                        "candidate_shift_seconds"
                    ]
                )
            ),
        )

    best_shift = float(
        best[
            "candidate_shift_seconds"
        ]
    )

    score_gain = (
        float(best["alignment_score"])
        - float(
            zero_shift[
                "alignment_score"
            ]
        )
    )

    # Since synthetic lag moves B later, a negative correction
    # estimates how late B appears to be.
    estimated_B_lateness = max(
        0.0,
        -best_shift,
    )

    return {
        "best_B_correction_shift_seconds": round(
            best_shift,
            3,
        ),

        "estimated_B_lateness_seconds": round(
            estimated_B_lateness,
            3,
        ),

        "best_alignment_score": float(
            best["alignment_score"]
        ),

        "zero_shift_alignment_score": float(
            zero_shift[
                "alignment_score"
            ]
        ),

        "alignment_score_gain_vs_zero": round(
            score_gain,
            6,
        ),

        "best_num_bilateral_events": int(
            best[
                "num_bilateral_events"
            ]
        ),

        "best_num_A_to_B_events": int(
            best[
                "num_A_to_B_events"
            ]
        ),

        "best_num_B_to_A_events": int(
            best[
                "num_B_to_A_events"
            ]
        ),

        "best_event_coverage": float(
            best[
                "event_coverage"
            ]
        ),

        "best_mean_abs_offset_seconds": (
            best[
                "mean_abs_offset_seconds"
            ]
        ),

        "best_median_abs_offset_seconds": (
            best[
                "median_abs_offset_seconds"
            ]
        ),

        "best_mean_signed_offset_seconds": (
            best[
                "mean_signed_offset_seconds"
            ]
        ),

        "best_median_signed_offset_seconds": (
            best[
                "median_signed_offset_seconds"
            ]
        ),

        "best_A_to_B_offsets": best[
            "A_to_B_offsets"
        ],

        "best_B_to_A_offsets": best[
            "B_to_A_offsets"
        ],

        "best_shift_at_search_boundary": bool(
            abs(
                best_shift
                - ALIGNMENT_SHIFT_MIN_SECONDS
            )
            < 1e-9
            or abs(
                best_shift
                - ALIGNMENT_SHIFT_MAX_SECONDS
            )
            < 1e-9
        ),

        # Keep all candidates for later inspection.
        "alignment_shift_curve": (
            candidate_rows
        ),
    }

## 12. Compute and save frozen global-shift reference statistics

Global-shift features are computed and cached for all reference and held-out feature cases.

The frozen reference statistics are calculated only from the 200 reference-profile cases. For each of the four profiles, the notebook saves:

- mean and median best B-correction shift;
- mean and median estimated B lateness;
- mean and median alignment-score gain versus zero shift;
- mean bilateral-event count;
- mean event coverage.

The statistics are saved to:

`frozen_reference_global_shift_statistics.json`

This is the end of Step 1. No model is loaded and no prediction is produced.

In [ ]:
MIXED_EXPERIMENT_VERSION = (
    "guided_independent_bc_signed_offsets_"
    "plus_global_shift_mixed_lag_v1"
)

MIXED_PROMPT_VERSION = (
    f"{MIXED_EXPERIMENT_VERSION}"
    f"_duration{INDEPENDENT_BC_MAX_DURATION_SECONDS}"
    f"_overlap{INDEPENDENT_BC_MIN_OVERLAP_FRACTION}"
    f"_threshold{INDEPENDENT_BC_DELAY_THRESHOLD_SECONDS}"
    f"_postmax{INDEPENDENT_BC_MAX_POST_DELAY_SECONDS}"
    f"_premax{INDEPENDENT_BC_MAX_PRESTART_SECONDS}"
    f"_shiftmin{ALIGNMENT_SHIFT_MIN_SECONDS}"
    f"_shiftmax{ALIGNMENT_SHIFT_MAX_SECONDS}"
    f"_shiftstep{ALIGNMENT_SHIFT_STEP_SECONDS}"
    f"_scale{ALIGNMENT_OFFSET_SCALE_SECONDS}"
    "_frozen50_heldout100"
)

GLOBAL_SHIFT_COMPACT_KEYS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "best_alignment_score",
    "zero_shift_alignment_score",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_num_A_to_B_events",
    "best_num_B_to_A_events",
    "best_event_coverage",
    "best_mean_abs_offset_seconds",
    "best_median_abs_offset_seconds",
    "best_mean_signed_offset_seconds",
    "best_median_signed_offset_seconds",
    "best_shift_at_search_boundary",
]

if GLOBAL_SHIFT_CACHE_PATH.exists():
    try:
        global_shift_feature_cache = json.loads(
            GLOBAL_SHIFT_CACHE_PATH.read_text(
                encoding="utf-8"
            )
        )

        if not isinstance(global_shift_feature_cache, dict):
            global_shift_feature_cache = {}

    except Exception as exc:
        print("Could not load global-shift cache:", exc)
        global_shift_feature_cache = {}
else:
    global_shift_feature_cache = {}

all_feature_cases = (
    reference_independent_bc_cases
    + inference_independent_bc_cases
)

for case in tqdm(
    all_feature_cases,
    desc="Global B-correction features",
):
    case_id = case["case_id"]

    if case_id in global_shift_feature_cache:
        continue

    full_shift_features = estimate_global_B_correction_shift(
        turns_A=case["filtered_turns_A"],
        turns_B=case["filtered_turns_B"],
        timeline_duration=case["analysis_duration"],
    )

    global_shift_feature_cache[case_id] = {
        key: full_shift_features[key]
        for key in GLOBAL_SHIFT_COMPACT_KEYS
    }

    GLOBAL_SHIFT_CACHE_PATH.write_text(
        json.dumps(
            global_shift_feature_cache,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def attach_global_shift_features(source_cases):
    output_cases = []

    for case in source_cases:
        new_case = dict(case)
        new_case["global_alignment_shift_features"] = (
            global_shift_feature_cache[case["case_id"]]
        )
        output_cases.append(new_case)

    return output_cases


reference_shift_cases = attach_global_shift_features(
    reference_independent_bc_cases
)

inference_shift_cases = attach_global_shift_features(
    inference_independent_bc_cases
)

reference_shift_rows = []

for case in reference_shift_cases:
    reference_shift_rows.append({
        "case_id": case["case_id"],
        "conversation_id": case["conversation_id"],
        "reference_profile": case["reference_profile"],
        **case["global_alignment_shift_features"],
    })

reference_shift_df = pd.DataFrame(reference_shift_rows)

SHIFT_REFERENCE_CORE_COLUMNS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
]

SHIFT_REFERENCE_RELIABILITY_COLUMNS = [
    "best_num_bilateral_events",
    "best_event_coverage",
]

reference_shift_stats = (
    reference_shift_df
    .groupby("reference_profile")
    .agg({
        **{
            column: ["mean", "median"]
            for column in SHIFT_REFERENCE_CORE_COLUMNS
        },
        **{
            column: ["mean"]
            for column in SHIFT_REFERENCE_RELIABILITY_COLUMNS
        },
    })
    .reindex(PROFILE_ORDER)
)


def shift_reference_value(
    profile,
    column,
    statistic="mean",
):
    value = reference_shift_stats.loc[
        profile,
        (column, statistic),
    ]

    if pd.isna(value):
        return 0.0

    return float(value)


shift_stats_records = []

for profile in PROFILE_ORDER:
    record = {
        "reference_profile": profile
    }

    for column in (
        SHIFT_REFERENCE_CORE_COLUMNS
        + SHIFT_REFERENCE_RELIABILITY_COLUMNS
    ):
        statistics = (
            ["mean", "median"]
            if column in SHIFT_REFERENCE_CORE_COLUMNS
            else ["mean"]
        )

        for statistic in statistics:
            record[f"{column}__{statistic}"] = shift_reference_value(
                profile,
                column,
                statistic,
            )

    shift_stats_records.append(record)

REFERENCE_SHIFT_STATS_PATH.write_text(
    json.dumps(
        shift_stats_records,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("FROZEN GLOBAL-SHIFT REFERENCE STATISTICS")
display(reference_shift_stats.round(3))

print("Mixed prompt version:", MIXED_PROMPT_VERSION)
print("Global-shift cache entries:", len(global_shift_feature_cache))
print("Saved frozen shift statistics:", REFERENCE_SHIFT_STATS_PATH)

Global B-correction features:   0%|          | 0/410 [00:00<?, ?it/s]

FROZEN GLOBAL-SHIFT REFERENCE STATISTICS


best_B_correction_shift_seconds         \
                                             mean median   
reference_profile                                          
NORMAL                                     -0.230   -0.0   
LAG_1                                      -0.680   -0.9   
LAG_2                                      -1.100   -1.9   
LAG_3                                      -2.034   -2.9   

                  estimated_B_lateness_seconds         \
                                          mean median   
reference_profile                                       
NORMAL                                   0.500    0.0   
LAG_1                                    1.084    0.9   
LAG_2                                    1.668    1.9   
LAG_3                                    2.464    2.9   

                  alignment_score_gain_vs_zero         \
                                          mean median   
reference_profile                                       
NORMAL                                   0.023  0.009   
LAG_1                                    0.105  0.095   
LAG_2                                    0.188  0.185   
LAG_3                                    0.207  0.179   

                  best_num_bilateral_events best_event_coverage  
                                       mean                mean  
reference_profile                                                
NORMAL                                11.94               0.595  
LAG_1                                 10.76               0.560  
LAG_2                                 10.78               0.562  
LAG_3                                 10.62               0.551

Mixed prompt version: guided_independent_bc_signed_offsets_plus_global_shift_mixed_lag_v1_duration1.0_overlap0.8_threshold1.5_postmax6.0_premax2.0_shiftmin-6.0_shiftmax6.0_shiftstep0.1_scale1.5_frozen50_heldout100
Global-shift cache entries: 410
Saved frozen shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json


## Step 1 outputs

After a successful run, the important reusable artifacts are:

### Split manifests

- `reference_50_conversations.json`
- `heldout_100_conversations.json`

### Constructed cases

- `reference_cases_normal_lag1_lag2_lag3.json`
- `inference_cases_100_normal_110_mixed_lag.json`

### Feature-enriched cases

- `reference_feature_cases.json`
- `inference_feature_cases.json`
- `global_shift_feature_cache.json`

### Frozen statistics

- `frozen_reference_base_statistics.json`
- `frozen_reference_global_shift_statistics.json`

The next consolidation step can load the same 100 held-out normal-conversation manifest and build the unified normal, wrong-partner, lag, and silent-partner cases without changing this frozen reference split.

# Step 2A — Construction of NORMAL and WRONG_PARTNER cases

In this step, we use exactly the 100 held-out normal conversations selected
during Step 1.

No new conversations are sampled and the participant orientation already stored
in `inference_selection_manifest` is preserved.

For every held-out conversation \(i\), two cases are constructed:

### NORMAL case

The original dyadic pair is retained:

\[
NORMAL_i = A_i + B_i
\]

where Participant A and Participant B come from the same source conversation.

### WRONG_PARTNER case

Participant A remains unchanged, while Participant B is replaced by Participant
B from another held-out conversation:

\[
WRONG\_PARTNER_i = A_i + B_j,\qquad j \neq i
\]

The foreign Participant B assignments are generated using a deterministic
derangement with random seed 42. A derangement is a permutation in which no
conversation is paired with itself.

This construction guarantees that:

- all 100 held-out conversations contribute one Participant A;
- every Participant A appears once in a NORMAL case and once in a
  WRONG_PARTNER case;
- every Participant B appears once as the true partner and once as a foreign
  wrong partner;
- all 100 wrong-partner cases contain participants from different source
  conversations;
- no single Participant A or Participant B is repeatedly reused across the
  entire anomalous set;
- the original fixed-participant generalisation bug cannot occur.

The case records contain internal construction metadata, participant metadata,
and the corresponding 120-second VAD turn lists. Gold labels and construction
information are retained only for dataset management and later evaluation.
They must not be included in the model-facing consolidation input.

After this step, the consolidation dataset contains:

- 100 NORMAL cases;
- 100 WRONG_PARTNER cases;
- 200 cases in total.

No model inference or semantic-summary generation is performed at this stage.

In [ ]:
# ============================================================
# STEP 2A
# BALANCED NORMAL / WRONG-PARTNER CASE CONSTRUCTION
#
# Source:
#   the exact 100 held-out conversations created in Step 1
#
# Output:
#   100 NORMAL cases
#   100 WRONG_PARTNER cases
# ============================================================

import json
import random
from pathlib import Path


WRONG_PARTNER_PAIRING_SEED = 42

CONSOLIDATION_NORMAL_WRONG_CASES_PATH = (
    OUT_DIR
    / "consolidation_cases_100_normal_100_wrong_partner.json"
)

WRONG_PARTNER_PAIRING_MANIFEST_PATH = (
    OUT_DIR
    / "wrong_partner_derangement_seed42.json"
)


# ------------------------------------------------------------
# Validate the held-out source manifest from Step 1
# ------------------------------------------------------------

assert "inference_selection_manifest" in globals(), (
    "inference_selection_manifest is missing. "
    "Run the Step 1 data-split cells first."
)

assert len(inference_selection_manifest) == 100, (
    "Expected exactly 100 held-out conversations, "
    f"but found {len(inference_selection_manifest)}."
)


required_record_fields = {
    "conversation_id",
    "participant_A",
    "participant_B",
    "turns_A_full_0_120",
    "turns_B_full_0_120",
}


for index, record in enumerate(
    inference_selection_manifest
):
    missing_fields = (
        required_record_fields
        - set(record.keys())
    )

    assert not missing_fields, (
        f"Held-out record {index} is missing fields: "
        f"{sorted(missing_fields)}"
    )


heldout_conversation_ids = [
    record["conversation_id"]
    for record in inference_selection_manifest
]


assert len(set(heldout_conversation_ids)) == 100, (
    "The held-out manifest does not contain "
    "100 unique conversations."
)


# The held-out set must remain disjoint from
# the 50 frozen reference conversations.
assert set(heldout_conversation_ids).isdisjoint(
    reference_ids
), (
    "Reference/held-out overlap detected."
)


print(
    "Held-out source conversations:",
    len(inference_selection_manifest),
)

print(
    "Unique held-out conversation IDs:",
    len(set(heldout_conversation_ids)),
)

print(
    "Reference/held-out overlap:",
    len(
        set(heldout_conversation_ids)
        & set(reference_ids)
    ),
)

Held-out source conversations: 100
Unique held-out conversation IDs: 100
Reference/held-out overlap: 0


In [ ]:
# ============================================================
# CREATE DETERMINISTIC WRONG-PARTNER DERANGEMENT
# ============================================================

def create_derangement(
    num_items,
    seed=42,
):
    """
    Create a reproducible permutation in which:

        permutation[i] != i

    for every source index i.
    """

    if num_items < 2:
        raise ValueError(
            "A derangement requires at least two items."
        )

    rng = random.Random(seed)

    permutation = list(
        range(num_items)
    )

    while True:
        rng.shuffle(permutation)

        if all(
            source_index
            != permutation[source_index]
            for source_index in range(num_items)
        ):
            return permutation.copy()


wrong_B_indices = create_derangement(
    num_items=len(
        inference_selection_manifest
    ),
    seed=WRONG_PARTNER_PAIRING_SEED,
)


assert len(wrong_B_indices) == 100

assert sorted(wrong_B_indices) == list(
    range(100)
)

assert all(
    source_index
    != wrong_B_indices[source_index]
    for source_index in range(100)
)


print(
    "Wrong-partner derangement created."
)

print(
    "Number of foreign-B assignments:",
    len(wrong_B_indices),
)

print(
    "Self-pairings:",
    sum(
        source_index
        == foreign_index
        for source_index, foreign_index
        in enumerate(wrong_B_indices)
    ),
)

Wrong-partner derangement created.
Number of foreign-B assignments: 100
Self-pairings: 0


In [ ]:
# ============================================================
# BUILD 100 NORMAL + 100 WRONG_PARTNER CASES
# ============================================================

consolidation_normal_cases = []
consolidation_wrong_partner_cases = []


for pair_index, base_record in enumerate(
    inference_selection_manifest
):

    foreign_B_index = wrong_B_indices[
        pair_index
    ]

    foreign_B_record = (
        inference_selection_manifest[
            foreign_B_index
        ]
    )


    # --------------------------------------------------------
    # Original participants from held-out conversation i
    # --------------------------------------------------------

    participant_A = (
        base_record["participant_A"]
    )

    true_participant_B = (
        base_record["participant_B"]
    )


    # --------------------------------------------------------
    # Foreign Participant B from held-out conversation j
    # --------------------------------------------------------

    wrong_participant_B = (
        foreign_B_record["participant_B"]
    )


    # --------------------------------------------------------
    # NORMAL_i = A_i + true B_i
    # --------------------------------------------------------

    normal_case = {
        "case_id": (
            f"consolidation_normal_"
            f"{pair_index:03d}"
        ),

        # Connects all later synthetic variants
        # derived from this held-out source.
        "source_group_id": (
            f"heldout_source_"
            f"{pair_index:03d}"
        ),

        "pair_index": pair_index,

        # Internal evaluation metadata.
        "gold_binary_label": "NORMAL",
        "gold_anomaly_type": "none",

        "case_variant": "normal",
        "pairing_type": "same_conversation",

        "participant_A": participant_A,
        "participant_B": true_participant_B,

        "A_source_conversation": (
            base_record["conversation_id"]
        ),

        "B_source_conversation": (
            base_record["conversation_id"]
        ),

        # Exact original 0–120 s VAD turns.
        "turns_A_0_120": (
            base_record[
                "turns_A_full_0_120"
            ]
        ),

        "turns_B_0_120": (
            base_record[
                "turns_B_full_0_120"
            ]
        ),

        "analysis_duration_seconds": (
            MAX_SECONDS
        ),

        "construction_seed": (
            WRONG_PARTNER_PAIRING_SEED
        ),
    }


    # --------------------------------------------------------
    # WRONG_PARTNER_i = A_i + foreign B_j
    # --------------------------------------------------------

    wrong_case = {
        "case_id": (
            f"consolidation_wrong_partner_"
            f"{pair_index:03d}"
        ),

        "source_group_id": (
            f"heldout_source_"
            f"{pair_index:03d}"
        ),

        "pair_index": pair_index,

        # Internal evaluation metadata.
        "gold_binary_label": "ANOMALOUS",
        "gold_anomaly_type": (
            "wrong_partner"
        ),

        "case_variant": "wrong_partner",
        "pairing_type": (
            "different_conversations"
        ),

        "participant_A": participant_A,
        "participant_B": wrong_participant_B,

        "A_source_conversation": (
            base_record["conversation_id"]
        ),

        "B_source_conversation": (
            foreign_B_record[
                "conversation_id"
            ]
        ),

        # A keeps the original turns from source i.
        "turns_A_0_120": (
            base_record[
                "turns_A_full_0_120"
            ]
        ),

        # B contributes the original turns
        # from foreign source j.
        "turns_B_0_120": (
            foreign_B_record[
                "turns_B_full_0_120"
            ]
        ),

        "analysis_duration_seconds": (
            MAX_SECONDS
        ),

        "construction_seed": (
            WRONG_PARTNER_PAIRING_SEED
        ),

        # Internal pairing audit metadata.
        "foreign_B_source_index": (
            foreign_B_index
        ),
    }


    consolidation_normal_cases.append(
        normal_case
    )

    consolidation_wrong_partner_cases.append(
        wrong_case
    )


consolidation_normal_wrong_cases = (
    consolidation_normal_cases
    + consolidation_wrong_partner_cases
)


print(
    "NORMAL cases:",
    len(consolidation_normal_cases),
)

print(
    "WRONG_PARTNER cases:",
    len(consolidation_wrong_partner_cases),
)

print(
    "Total current consolidation cases:",
    len(consolidation_normal_wrong_cases),
)

NORMAL cases: 100
WRONG_PARTNER cases: 100
Total current consolidation cases: 200


In [ ]:
# ============================================================
# GENERALISATION AND PAIRING AUDIT
# ============================================================

assert len(
    consolidation_normal_cases
) == 100

assert len(
    consolidation_wrong_partner_cases
) == 100

assert len(
    consolidation_normal_wrong_cases
) == 200


# ------------------------------------------------------------
# 1. All NORMAL pairs come from the same conversation
# ------------------------------------------------------------

assert all(
    case["A_source_conversation"]
    == case["B_source_conversation"]
    for case in consolidation_normal_cases
)


# ------------------------------------------------------------
# 2. All WRONG pairs come from different conversations
# ------------------------------------------------------------

assert all(
    case["A_source_conversation"]
    != case["B_source_conversation"]
    for case in consolidation_wrong_partner_cases
)


# ------------------------------------------------------------
# 3. Extract participant signatures
# ------------------------------------------------------------

normal_A_signatures = [
    (
        case["participant_A"][
            "conversation_id"
        ],
        case["participant_A"][
            "participant_id"
        ],
    )
    for case in consolidation_normal_cases
]


wrong_A_signatures = [
    (
        case["participant_A"][
            "conversation_id"
        ],
        case["participant_A"][
            "participant_id"
        ],
    )
    for case in consolidation_wrong_partner_cases
]


normal_B_signatures = [
    (
        case["participant_B"][
            "conversation_id"
        ],
        case["participant_B"][
            "participant_id"
        ],
    )
    for case in consolidation_normal_cases
]


wrong_B_signatures = [
    (
        case["participant_B"][
            "conversation_id"
        ],
        case["participant_B"][
            "participant_id"
        ],
    )
    for case in consolidation_wrong_partner_cases
]


# ------------------------------------------------------------
# 4. The same 100 A participants are used in NORMAL and WRONG
# ------------------------------------------------------------

assert (
    normal_A_signatures
    == wrong_A_signatures
)


assert len(
    set(normal_A_signatures)
) == 100


assert len(
    set(wrong_A_signatures)
) == 100


# ------------------------------------------------------------
# 5. Every B is used exactly once in each condition
# ------------------------------------------------------------

assert len(
    set(normal_B_signatures)
) == 100


assert len(
    set(wrong_B_signatures)
) == 100


# The wrong-partner B set must be exactly
# a permutation of the original B set.
assert set(
    normal_B_signatures
) == set(
    wrong_B_signatures
)


# ------------------------------------------------------------
# 6. No A is paired with its original B in WRONG cases
# ------------------------------------------------------------

assert all(
    normal_B_signatures[index]
    != wrong_B_signatures[index]
    for index in range(100)
)


# ------------------------------------------------------------
# 7. Every source group has one NORMAL and one WRONG case
# ------------------------------------------------------------

normal_source_groups = {
    case["source_group_id"]
    for case in consolidation_normal_cases
}


wrong_source_groups = {
    case["source_group_id"]
    for case in consolidation_wrong_partner_cases
}


assert len(
    normal_source_groups
) == 100

assert (
    normal_source_groups
    == wrong_source_groups
)


# ------------------------------------------------------------
# 8. All source conversations belong to the held-out set
# ------------------------------------------------------------

heldout_id_set = set(
    heldout_conversation_ids
)


assert all(
    case["A_source_conversation"]
    in heldout_id_set
    for case in consolidation_normal_wrong_cases
)


assert all(
    case["B_source_conversation"]
    in heldout_id_set
    for case in consolidation_normal_wrong_cases
)


# ------------------------------------------------------------
# 9. No reference conversation is used
# ------------------------------------------------------------

assert all(
    case["A_source_conversation"]
    not in reference_ids
    and case["B_source_conversation"]
    not in reference_ids
    for case in consolidation_normal_wrong_cases
)


# ------------------------------------------------------------
# 10. All case IDs are unique
# ------------------------------------------------------------

all_case_ids = [
    case["case_id"]
    for case in consolidation_normal_wrong_cases
]


assert len(
    all_case_ids
) == len(
    set(all_case_ids)
)


print("=" * 70)
print("NORMAL / WRONG-PARTNER AUDIT PASSED")
print("=" * 70)

print(
    "NORMAL cases:",
    len(consolidation_normal_cases),
)

print(
    "WRONG_PARTNER cases:",
    len(consolidation_wrong_partner_cases),
)

print(
    "Unique A participants in WRONG cases:",
    len(set(wrong_A_signatures)),
)

print(
    "Unique B participants in WRONG cases:",
    len(set(wrong_B_signatures)),
)

print(
    "Same-conversation WRONG pairs:",
    sum(
        case["A_source_conversation"]
        == case["B_source_conversation"]
        for case
        in consolidation_wrong_partner_cases
    ),
)

print(
    "Reference conversations used:",
    len(
        {
            case["A_source_conversation"]
            for case
            in consolidation_normal_wrong_cases
        }
        & reference_ids
    ),
)

print(
    "Current consolidation cases:",
    len(consolidation_normal_wrong_cases),
)

NORMAL / WRONG-PARTNER AUDIT PASSED
NORMAL cases: 100
WRONG_PARTNER cases: 100
Unique A participants in WRONG cases: 100
Unique B participants in WRONG cases: 100
Same-conversation WRONG pairs: 0
Reference conversations used: 0
Current consolidation cases: 200


In [ ]:
# ============================================================
# INSPECT EXAMPLES
# ============================================================

print("\nExample NORMAL / WRONG_PARTNER pairs")


for index in range(3):

    normal_case = (
        consolidation_normal_cases[
            index
        ]
    )

    wrong_case = (
        consolidation_wrong_partner_cases[
            index
        ]
    )

    print("\n" + "-" * 70)

    print(
        "Source group:",
        normal_case["source_group_id"],
    )

    print(
        "NORMAL:",
        normal_case["participant_A"][
            "conversation_id"
        ],
        normal_case["participant_A"][
            "participant_id"
        ],
        "+",
        normal_case["participant_B"][
            "conversation_id"
        ],
        normal_case["participant_B"][
            "participant_id"
        ],
    )

    print(
        "WRONG :",
        wrong_case["participant_A"][
            "conversation_id"
        ],
        wrong_case["participant_A"][
            "participant_id"
        ],
        "+",
        wrong_case["participant_B"][
            "conversation_id"
        ],
        wrong_case["participant_B"][
            "participant_id"
        ],
    )


Example NORMAL / WRONG_PARTNER pairs

----------------------------------------------------------------------
Source group: heldout_source_000
NORMAL: V03_S0148_I00000135 P1319 + V03_S0148_I00000135 P1274
WRONG : V03_S0148_I00000135 P1319 + V00_S2049_I00001111 P1304A

----------------------------------------------------------------------
Source group: heldout_source_001
NORMAL: V00_S2050_I00001124 P1307A + V00_S2050_I00001124 P1308A
WRONG : V00_S2050_I00001124 P1307A + V00_S2051_I00001001 P1308A

----------------------------------------------------------------------
Source group: heldout_source_002
NORMAL: V01_S0340_I00001104 P1682 + V01_S0340_I00001104 P1681
WRONG : V01_S0340_I00001104 P1682 + V03_S0148_I00000371 P1319


In [ ]:
# ============================================================
# SAVE CASE AND PAIRING MANIFESTS
# ============================================================

CONSOLIDATION_NORMAL_WRONG_CASES_PATH.write_text(
    json.dumps(
        consolidation_normal_wrong_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


wrong_partner_pairing_manifest = []


for source_index, foreign_B_index in enumerate(
    wrong_B_indices
):

    base_record = (
        inference_selection_manifest[
            source_index
        ]
    )

    foreign_record = (
        inference_selection_manifest[
            foreign_B_index
        ]
    )

    wrong_partner_pairing_manifest.append({
        "source_index": source_index,

        "source_group_id": (
            f"heldout_source_"
            f"{source_index:03d}"
        ),

        "A_conversation_id": (
            base_record["conversation_id"]
        ),

        "A_participant_id": (
            base_record[
                "participant_A"
            ]["participant_id"]
        ),

        "foreign_B_index": (
            foreign_B_index
        ),

        "foreign_B_conversation_id": (
            foreign_record[
                "conversation_id"
            ]
        ),

        "foreign_B_participant_id": (
            foreign_record[
                "participant_B"
            ]["participant_id"]
        ),
    })


WRONG_PARTNER_PAIRING_MANIFEST_PATH.write_text(
    json.dumps(
        wrong_partner_pairing_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print(
    "\nSaved cases:",
    CONSOLIDATION_NORMAL_WRONG_CASES_PATH,
)

print(
    "Saved pairing manifest:",
    WRONG_PARTNER_PAIRING_MANIFEST_PATH,
)


Saved cases: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_cases_100_normal_100_wrong_partner.json
Saved pairing manifest: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/wrong_partner_derangement_seed42.json


## Inspection of the current NORMAL / WRONG_PARTNER case database

At this stage, the dataset contains 200 assembled cases:

- 100 original NORMAL participant pairs;
- 100 deterministic WRONG_PARTNER pairs.

Each case currently stores:

- internal construction and evaluation metadata;
- Participant A and Participant B metadata;
- the source conversation of each participant;
- the merged VAD turns for both participants in the 0–120 second interval.

The VAD turns stored in `turns_A_0_120` and `turns_B_0_120` have already
been:

1. clipped to the first 120 seconds;
2. cleaned of invalid ranges;
3. merged when consecutive ranges are separated by at most 0.75 seconds.

However, the newly assembled NORMAL and WRONG_PARTNER cases have not yet passed
through the complete temporal interaction pipeline. They do not yet contain:

- independent backchannel filtering;
- filtered A/B turns;
- clean overlap features;
- signed strict A-end to B-start offsets;
- global B-correction features.

Those features must later be computed from the assembled pair itself. This is
particularly important for WRONG_PARTNER cases, because their A and B turns
come from different source conversations and therefore form a new interaction
timeline.

In [ ]:
# ============================================================
# INSPECT THE COMPLETE SCHEMA OF ONE CURRENT CASE
# ============================================================

import json
import pandas as pd


def describe_nested_schema(
    value,
    prefix="",
    max_list_examples=1,
):
    """
    Return a flat description of nested keys and value types.
    """

    rows = []

    if isinstance(value, dict):

        for key, child in value.items():

            full_key = (
                f"{prefix}.{key}"
                if prefix
                else key
            )

            rows.append({
                "field": full_key,
                "type": type(child).__name__,
                "length": (
                    len(child)
                    if isinstance(
                        child,
                        (dict, list, str),
                    )
                    else None
                ),
                "example": (
                    child
                    if not isinstance(
                        child,
                        (dict, list),
                    )
                    else None
                ),
            })

            rows.extend(
                describe_nested_schema(
                    child,
                    prefix=full_key,
                    max_list_examples=(
                        max_list_examples
                    ),
                )
            )

    elif isinstance(value, list):

        for index, child in enumerate(
            value[:max_list_examples]
        ):

            full_key = (
                f"{prefix}[{index}]"
            )

            rows.append({
                "field": full_key,
                "type": type(child).__name__,
                "length": (
                    len(child)
                    if isinstance(
                        child,
                        (dict, list, str),
                    )
                    else None
                ),
                "example": (
                    child
                    if not isinstance(
                        child,
                        (dict, list),
                    )
                    else None
                ),
            })

            rows.extend(
                describe_nested_schema(
                    child,
                    prefix=full_key,
                    max_list_examples=(
                        max_list_examples
                    ),
                )
            )

    return rows


example_case = (
    consolidation_normal_wrong_cases[0]
)


print(
    "Example case ID:",
    example_case["case_id"],
)

print(
    "Top-level keys:"
)

print(
    list(example_case.keys())
)


case_schema_df = pd.DataFrame(
    describe_nested_schema(
        example_case
    )
)


display(case_schema_df)

Example case ID: consolidation_normal_000
Top-level keys:
['case_id', 'source_group_id', 'pair_index', 'gold_binary_label', 'gold_anomaly_type', 'case_variant', 'pairing_type', 'participant_A', 'participant_B', 'A_source_conversation', 'B_source_conversation', 'turns_A_0_120', 'turns_B_0_120', 'analysis_duration_seconds', 'construction_seed']


,field,type,length,example
0,case_id,str,24.0,consolidation_normal_000
1,source_group_id,str,18.0,heldout_source_000
2,pair_index,int,NaN,0
3,gold_binary_label,str,6.0,NORMAL
4,gold_anomaly_type,str,4.0,none
5,case_variant,str,6.0,normal
6,pairing_type,str,17.0,same_conversation
7,participant_A,dict,4.0,None
8,participant_A.conversation_id,str,19.0,V03_S0148_I00000135
9,participant_A.participant_id,str,5.0,P1319


In [ ]:
# ============================================================
# PRINT ONE COMPLETE CASE
# ============================================================

print(
    json.dumps(
        example_case,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "case_id": "consolidation_normal_000",
  "source_group_id": "heldout_source_000",
  "pair_index": 0,
  "gold_binary_label": "NORMAL",
  "gold_anomaly_type": "none",
  "case_variant": "normal",
  "pairing_type": "same_conversation",
  "participant_A": {
    "conversation_id": "V03_S0148_I00000135",
    "participant_id": "P1319",
    "metadata_path": "/content/drive/MyDrive/seamless_download/data/V03_S0148_I00000135/P1319/V03_S0148_I00000135_P1319.json",
    "num_raw_vad_entries": 21
  },
  "participant_B": {
    "conversation_id": "V03_S0148_I00000135",
    "participant_id": "P1274",
    "metadata_path": "/content/drive/MyDrive/seamless_download/data/V03_S0148_I00000135/P1274/V03_S0148_I00000135_P1274.json",
    "num_raw_vad_entries": 31
  },
  "A_source_conversation": "V03_S0148_I00000135",
  "B_source_conversation": "V03_S0148_I00000135",
  "turns_A_0_120": [
    {
      "start": 9.89,
      "end": 10.78
    },
    {
      "start": 11.65,
      "end": 18.24
    },
    {
      "sta

# Step 2B — Temporal preprocessing of NORMAL and WRONG_PARTNER cases

The 200 NORMAL and WRONG_PARTNER cases already contain merged VAD speech turns
for both participants in the shared 0–120 second timeline.

The stored `turns_A_0_120` and `turns_B_0_120` have already undergone:

1. clipping to the first 120 seconds;
2. removal of invalid intervals;
3. merging of consecutive intervals separated by at most 0.75 seconds.

In this step, every assembled case is passed through the same deterministic
temporal preprocessing pipeline used in the lag experiment.

No temporal shift is applied to either participant.

For every case, the pipeline computes:

- independent operational backchannel filtering;
- filtered Participant A and Participant B turns;
- raw and filtered overlap;
- signed strict A-end-to-B-start events;
- the complete signed-offset distribution;
- mean, median, maximum, P75 and P90 offsets;
- the number and percentage of offsets above 1.5 seconds;
- the best global correction shift for Participant B;
- estimated Participant B lateness;
- alignment-score gain and reliability statistics.

For WRONG_PARTNER cases, these statistics are recomputed from the newly
assembled pair \(A_i + B_j\). They cannot be copied from the original source
conversations because the cross-participant temporal relationship has changed.

The preprocessing functions receive only:

- Participant A turns;
- Participant B turns;
- analysis duration.

They do not receive the gold label, anomaly type, pairing type, participant IDs,
or source-conversation IDs.

The frozen reference averages are not recomputed from these 200 cases.
The same statistics calculated exclusively from the 50 frozen reference
conversations are retained for later consolidation.

## Clean implementation and storage policy

The code below applies the temporal pipeline **exactly once** to each of the
100 NORMAL and 100 WRONG_PARTNER cases.

Input turns:

- `turns_A_0_120`
- `turns_B_0_120`

These turns have already been clipped to 0–120 seconds and merged with the
same `0.75 s` gap rule used in the lag experiment.

For each assembled pair, the code then applies the same lag-pipeline functions:

1. `compute_independent_bc_features(...)`
   - independent backchannel filtering;
   - final filtered A/B turns;
   - clean overlap;
   - signed strict offsets and their distribution features.

2. `estimate_global_B_correction_shift(...)`
   - global B-correction diagnostic features.

No synthetic temporal shift is applied to NORMAL or WRONG_PARTNER cases.

The final case JSON stores only:

- the metadata needed to identify and evaluate the case;
- the final filtered A/B turns;
- the exact current-case temporal features used by the lag prompt;
- the exact current-case global-shift features used by the lag prompt.

The frozen reference averages are stored once in a separate JSON file and
contain only the NORMAL, LAG +2 s and LAG +3 s averages that are used in the
main consolidation experiment.

In [ ]:
# ============================================================
# STEP 2B — CLEAN TEMPORAL OUTPUT CONFIGURATION
# ============================================================

import hashlib
import json
from pathlib import Path

from tqdm.auto import tqdm


CLEAN_NORMAL_WRONG_CASES_PATH = (
    OUT_DIR
    / "consolidation_normal_wrong_cases_with_lag_features.json"
)

CLEAN_FROZEN_REFERENCE_FEATURES_PATH = (
    OUT_DIR
    / "consolidation_frozen_lag_reference_features.json"
)

CLEAN_GLOBAL_SHIFT_CACHE_PATH = (
    OUT_DIR
    / "consolidation_normal_wrong_global_shift_cache_clean_v1.json"
)


# Exact current-case fields inserted into the lag prompt.
LAG_PROMPT_LOCAL_FEATURE_KEYS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",
    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",
    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",
    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


# Exact global-shift fields inserted into the lag prompt.
LAG_PROMPT_GLOBAL_FEATURE_KEYS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# Exact frozen base-reference averages inserted into the lag prompt.
LAG_PROMPT_FROZEN_BASE_KEYS = [
    "num_offsets",
    "offset_mean",
    "offset_median",
    "offset_max",
    "offset_p75",
    "offset_p90",
    "percent_above_1_5",
    "clean_overlap_seconds",
]


# Exact frozen global-shift averages inserted into the lag prompt.
LAG_PROMPT_FROZEN_GLOBAL_KEYS = [
    "best_B_correction_shift_seconds__mean",
    "best_B_correction_shift_seconds__median",
    "estimated_B_lateness_seconds__mean",
    "estimated_B_lateness_seconds__median",
    "alignment_score_gain_vs_zero__mean",
    "alignment_score_gain_vs_zero__median",
    "best_num_bilateral_events__mean",
    "best_event_coverage__mean",
]


MAIN_CONSOLIDATION_REFERENCE_PROFILES = [
    "NORMAL",
    "LAG_2",
    "LAG_3",
]


required_names = [
    "consolidation_normal_wrong_cases",
    "compute_independent_bc_features",
    "estimate_global_B_correction_shift",
    "REFERENCE_BASE_STATS_PATH",
    "REFERENCE_SHIFT_STATS_PATH",
    "inference_shift_cases",
]


missing_names = [
    name
    for name in required_names
    if name not in globals()
]


assert not missing_names, (
    "Run all previous Step 1 and Step 2A cells first. "
    f"Missing variables/functions: {missing_names}"
)

assert len(consolidation_normal_wrong_cases) == 200

print("Input assembled cases:", len(consolidation_normal_wrong_cases))
print("Final cases path:", CLEAN_NORMAL_WRONG_CASES_PATH)
print("Frozen references path:", CLEAN_FROZEN_REFERENCE_FEATURES_PATH)


Input assembled cases: 200
Final cases path: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_normal_wrong_cases_with_lag_features.json
Frozen references path: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_frozen_lag_reference_features.json


### Load only the frozen averages used by the lag prompt

The averages below come exclusively from the fixed 50-conversation reference
set. They are not recomputed from the 200 NORMAL/WRONG_PARTNER cases.

The `LAG_1` profile remains available in the original Step 1 artifacts, but it
is excluded from the main consolidation export.

In [ ]:
# ============================================================
# LOAD AND COMPACT THE FROZEN 50-CONVERSATION REFERENCES
# ============================================================

assert REFERENCE_BASE_STATS_PATH.exists(), (
    f"Missing frozen base statistics: {REFERENCE_BASE_STATS_PATH}"
)

assert REFERENCE_SHIFT_STATS_PATH.exists(), (
    f"Missing frozen global-shift statistics: {REFERENCE_SHIFT_STATS_PATH}"
)


all_base_reference_records = json.loads(
    REFERENCE_BASE_STATS_PATH.read_text(
        encoding="utf-8"
    )
)

all_global_reference_records = json.loads(
    REFERENCE_SHIFT_STATS_PATH.read_text(
        encoding="utf-8"
    )
)


base_record_by_profile = {
    record["reference_profile"]: record
    for record in all_base_reference_records
}

global_record_by_profile = {
    record["reference_profile"]: record
    for record in all_global_reference_records
}


frozen_lag_reference_features = {}


for profile in MAIN_CONSOLIDATION_REFERENCE_PROFILES:

    assert profile in base_record_by_profile
    assert profile in global_record_by_profile

    missing_base_keys = [
        key
        for key in LAG_PROMPT_FROZEN_BASE_KEYS
        if key not in base_record_by_profile[profile]
    ]

    missing_global_keys = [
        key
        for key in LAG_PROMPT_FROZEN_GLOBAL_KEYS
        if key not in global_record_by_profile[profile]
    ]

    assert not missing_base_keys, (
        f"{profile} missing frozen base fields: {missing_base_keys}"
    )

    assert not missing_global_keys, (
        f"{profile} missing frozen global fields: {missing_global_keys}"
    )

    frozen_lag_reference_features[profile] = {
        "base_temporal_averages": {
            key: base_record_by_profile[profile][key]
            for key in LAG_PROMPT_FROZEN_BASE_KEYS
        },

        "global_shift_averages": {
            key: global_record_by_profile[profile][key]
            for key in LAG_PROMPT_FROZEN_GLOBAL_KEYS
        },
    }


clean_frozen_reference_export = {
    "reference_source": (
        "50 frozen normal reference conversations from Step 1"
    ),

    "reference_conversations_used": 50,

    "reference_profiles": (
        frozen_lag_reference_features
    ),
}


CLEAN_FROZEN_REFERENCE_FEATURES_PATH.write_text(
    json.dumps(
        clean_frozen_reference_export,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print(
    json.dumps(
        clean_frozen_reference_export,
        indent=2,
        ensure_ascii=False,
    )
)

print(
    "\nSaved:",
    CLEAN_FROZEN_REFERENCE_FEATURES_PATH,
)


{
  "reference_source": "50 frozen normal reference conversations from Step 1",
  "reference_conversations_used": 50,
  "reference_profiles": {
    "NORMAL": {
      "base_temporal_averages": {
        "num_offsets": 5.76,
        "offset_mean": 0.3021276595744681,
        "offset_median": 0.2619148936170213,
        "offset_max": 1.054468085106383,
        "offset_p75": 0.5902127659574469,
        "offset_p90": 0.8253191489361702,
        "percent_above_1_5": 6.73404255319149,
        "clean_overlap_seconds": 6.2528
      },
      "global_shift_averages": {
        "best_B_correction_shift_seconds__mean": -0.23,
        "best_B_correction_shift_seconds__median": -0.0,
        "estimated_B_lateness_seconds__mean": 0.5,
        "estimated_B_lateness_seconds__median": 0.0,
        "alignment_score_gain_vs_zero__mean": 0.023376039999999997,
        "alignment_score_gain_vs_zero__median": 0.009105,
        "best_num_bilateral_events__mean": 11.94,
        "best_event_coverage__mean": 0.595

### Apply the exact VAD temporal preprocessing

For every assembled pair, the pipeline starts from its original merged
`turns_A_0_120` and `turns_B_0_120`.

The independent backchannel filter is therefore applied once to the assembled
pair. The global B-correction search then uses the resulting filtered turns.

For WRONG_PARTNER cases, the statistics are calculated from the new
\(A_i + B_j\) timeline and are not copied from either original conversation.

In [ ]:
# ============================================================
# LOAD A SMALL RESUMABLE CACHE FOR THE EXPENSIVE SHIFT SEARCH
# ============================================================

if CLEAN_GLOBAL_SHIFT_CACHE_PATH.exists():

    try:
        clean_global_shift_cache = json.loads(
            CLEAN_GLOBAL_SHIFT_CACHE_PATH.read_text(
                encoding="utf-8"
            )
        )

        if not isinstance(clean_global_shift_cache, dict):
            clean_global_shift_cache = {}

    except Exception as exc:

        print("Could not load clean shift cache:", exc)
        clean_global_shift_cache = {}

else:
    clean_global_shift_cache = {}


def temporal_input_fingerprint(
    filtered_turns_A,
    filtered_turns_B,
    analysis_duration,
):
    payload = {
        "filtered_turns_A": filtered_turns_A,
        "filtered_turns_B": filtered_turns_B,
        "analysis_duration": float(analysis_duration),
    }

    serialized = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


print(
    "Existing clean global-shift cache entries:",
    len(clean_global_shift_cache),
)


Existing clean global-shift cache entries: 200


In [ ]:
# ============================================================
# APPLY THE EXACT LAG TEMPORAL PIPELINE TO ALL 200 CASES
# AND IMMEDIATELY BUILD THE CLEAN CASE RECORDS
# ============================================================

clean_normal_wrong_cases = []


for case in tqdm(
    consolidation_normal_wrong_cases,
    desc="Exact VAD preprocessing for NORMAL / WRONG_PARTNER",
):

    # --------------------------------------------------------
    # Label-free temporal input.
    #
    # These are the merged 0–120 s turns created by Step 1.
    # No synthetic shift is applied here.
    # --------------------------------------------------------

    temporal_input = {
        "turns_A": case["turns_A_0_120"],
        "turns_B": case["turns_B_0_120"],
        "analysis_duration": float(
            case["analysis_duration_seconds"]
        ),
    }


    # --------------------------------------------------------
    # Exact independent backchannel filtering and exact
    # local feature extraction from the lag pipeline.
    # --------------------------------------------------------

    full_local_features = (
        compute_independent_bc_features(
            temporal_input
        )
    )

    filtered_turns_A = (
        full_local_features["filtered_turns_A"]
    )

    filtered_turns_B = (
        full_local_features["filtered_turns_B"]
    )


    # --------------------------------------------------------
    # Exact global B-correction search from the lag pipeline.
    # --------------------------------------------------------

    fingerprint = temporal_input_fingerprint(
        filtered_turns_A=filtered_turns_A,
        filtered_turns_B=filtered_turns_B,
        analysis_duration=(
            temporal_input["analysis_duration"]
        ),
    )

    cached_entry = clean_global_shift_cache.get(
        case["case_id"]
    )

    cache_is_valid = (
        isinstance(cached_entry, dict)
        and cached_entry.get("input_fingerprint") == fingerprint
        and isinstance(cached_entry.get("features"), dict)
        and set(LAG_PROMPT_GLOBAL_FEATURE_KEYS).issubset(
            cached_entry["features"].keys()
        )
    )


    if cache_is_valid:

        selected_global_features = {
            key: cached_entry["features"][key]
            for key in LAG_PROMPT_GLOBAL_FEATURE_KEYS
        }

    else:

        full_global_features = (
            estimate_global_B_correction_shift(
                turns_A=filtered_turns_A,
                turns_B=filtered_turns_B,
                timeline_duration=(
                    temporal_input["analysis_duration"]
                ),
            )
        )

        selected_global_features = {
            key: full_global_features[key]
            for key in LAG_PROMPT_GLOBAL_FEATURE_KEYS
        }

        clean_global_shift_cache[case["case_id"]] = {
            "input_fingerprint": fingerprint,
            "features": selected_global_features,
        }

        CLEAN_GLOBAL_SHIFT_CACHE_PATH.write_text(
            json.dumps(
                clean_global_shift_cache,
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )


    # --------------------------------------------------------
    # Keep only the local features actually inserted into
    # the lag prompt.
    # --------------------------------------------------------

    selected_local_features = {
        key: full_local_features[key]
        for key in LAG_PROMPT_LOCAL_FEATURE_KEYS
    }


    # --------------------------------------------------------
    # Clean final case record.
    #
    # It retains only:
    # - case/participant metadata required for later joining,
    # - final filtered turns,
    # - exact lag-prompt temporal features.
    # --------------------------------------------------------

    clean_case = {
        "case_id": case["case_id"],
        "source_group_id": case["source_group_id"],
        "pair_index": case["pair_index"],

        "gold_binary_label": (
            case["gold_binary_label"]
        ),

        "gold_anomaly_type": (
            case["gold_anomaly_type"]
        ),

        "case_variant": (
            case["case_variant"]
        ),

        "pairing_type": (
            case["pairing_type"]
        ),

        "participant_A": (
            case["participant_A"]
        ),

        "participant_B": (
            case["participant_B"]
        ),

        "A_source_conversation": (
            case["A_source_conversation"]
        ),

        "B_source_conversation": (
            case["B_source_conversation"]
        ),

        "analysis_duration_seconds": float(
            case["analysis_duration_seconds"]
        ),

        "participant_A_filtered_turns": (
            filtered_turns_A
        ),

        "participant_B_filtered_turns": (
            filtered_turns_B
        ),

        "local_temporal_features": (
            selected_local_features
        ),

        "global_shift_features": (
            selected_global_features
        ),
    }


    clean_normal_wrong_cases.append(
        clean_case
    )


CLEAN_NORMAL_WRONG_CASES_PATH.write_text(
    json.dumps(
        clean_normal_wrong_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print(
    "Processed and saved cases:",
    len(clean_normal_wrong_cases),
)

print(
    "Saved:",
    CLEAN_NORMAL_WRONG_CASES_PATH,
)

print(
    "Shift cache:",
    CLEAN_GLOBAL_SHIFT_CACHE_PATH,
)


Exact VAD preprocessing for NORMAL / WRONG_PARTNER:   0%|          | 0/200 [00:00<?, ?it/s]

Processed and saved cases: 200
Saved: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_normal_wrong_cases_with_lag_features.json
Shift cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_normal_wrong_global_shift_cache_clean_v1.json


### Strict verification

The following checks verify that:

- all 100 NORMAL and 100 WRONG_PARTNER cases are present;
- every case contains final filtered turns;
- every case contains exactly the lag-prompt feature fields and no additional
  temporal diagnostics;
- no synthetic shift was introduced;
- all 100 NORMAL cases reproduce the filtered turns and lag-prompt features
  already computed by the original held-out lag pipeline.

In [ ]:
# ============================================================
# STRICT CLEAN CASE AUDIT
# ============================================================

assert len(clean_normal_wrong_cases) == 200


clean_normal_cases = [
    case
    for case in clean_normal_wrong_cases
    if case["case_variant"] == "normal"
]

clean_wrong_partner_cases = [
    case
    for case in clean_normal_wrong_cases
    if case["case_variant"] == "wrong_partner"
]


assert len(clean_normal_cases) == 100
assert len(clean_wrong_partner_cases) == 100


expected_case_keys = {
    "case_id",
    "source_group_id",
    "pair_index",
    "gold_binary_label",
    "gold_anomaly_type",
    "case_variant",
    "pairing_type",
    "participant_A",
    "participant_B",
    "A_source_conversation",
    "B_source_conversation",
    "analysis_duration_seconds",
    "participant_A_filtered_turns",
    "participant_B_filtered_turns",
    "local_temporal_features",
    "global_shift_features",
}


for case in clean_normal_wrong_cases:

    assert set(case.keys()) == expected_case_keys

    assert isinstance(
        case["participant_A_filtered_turns"],
        list,
    )

    assert isinstance(
        case["participant_B_filtered_turns"],
        list,
    )

    assert set(
        case["local_temporal_features"].keys()
    ) == set(
        LAG_PROMPT_LOCAL_FEATURE_KEYS
    )

    assert set(
        case["global_shift_features"].keys()
    ) == set(
        LAG_PROMPT_GLOBAL_FEATURE_KEYS
    )

    assert (
        case["analysis_duration_seconds"]
        == 120.0
    )


assert all(
    case["A_source_conversation"]
    == case["B_source_conversation"]
    for case in clean_normal_cases
)


assert all(
    case["A_source_conversation"]
    != case["B_source_conversation"]
    for case in clean_wrong_partner_cases
)


assert len({
    (
        case["participant_A"]["conversation_id"],
        case["participant_A"]["participant_id"],
    )
    for case in clean_wrong_partner_cases
}) == 100


assert len({
    (
        case["participant_B"]["conversation_id"],
        case["participant_B"]["participant_id"],
    )
    for case in clean_wrong_partner_cases
}) == 100


print("=" * 76)
print("CLEAN NORMAL / WRONG-PARTNER TEMPORAL AUDIT PASSED")
print("=" * 76)

print("NORMAL cases:", len(clean_normal_cases))
print("WRONG_PARTNER cases:", len(clean_wrong_partner_cases))
print("Cases with filtered A/B turns:", len(clean_normal_wrong_cases))
print("Local lag-prompt features per case:", len(LAG_PROMPT_LOCAL_FEATURE_KEYS))
print("Global lag-prompt features per case:", len(LAG_PROMPT_GLOBAL_FEATURE_KEYS))
print(
    "Frozen reference profiles:",
    list(frozen_lag_reference_features.keys()),
)


CLEAN NORMAL / WRONG-PARTNER TEMPORAL AUDIT PASSED
NORMAL cases: 100
WRONG_PARTNER cases: 100
Cases with filtered A/B turns: 200
Local lag-prompt features per case: 11
Global lag-prompt features per case: 5
Frozen reference profiles: ['NORMAL', 'LAG_2', 'LAG_3']


In [ ]:
# ============================================================
# EXACT REPRODUCTION CHECK FOR ALL 100 NORMAL CASES
#
# The new clean records must reproduce the corresponding
# Step 1 held-out NORMAL records exactly for:
# - filtered turns,
# - local lag-prompt features,
# - global lag-prompt features.
# ============================================================

step1_normal_by_conversation = {
    case["conversation_id"]: case
    for case in inference_shift_cases
    if case["reference_profile"] == "NORMAL"
}


assert len(step1_normal_by_conversation) == 100


exact_filtered_turn_matches = 0
exact_local_feature_matches = 0
exact_global_feature_matches = 0


for clean_case in clean_normal_cases:

    conversation_id = (
        clean_case["A_source_conversation"]
    )

    original_step1_case = (
        step1_normal_by_conversation[
            conversation_id
        ]
    )


    if (
        clean_case["participant_A_filtered_turns"]
        == original_step1_case["filtered_turns_A"]
        and
        clean_case["participant_B_filtered_turns"]
        == original_step1_case["filtered_turns_B"]
    ):
        exact_filtered_turn_matches += 1


    original_local = (
        original_step1_case[
            "independent_bc_temporal_features"
        ]
    )

    expected_local = {
        key: original_local[key]
        for key in LAG_PROMPT_LOCAL_FEATURE_KEYS
    }


    if (
        clean_case["local_temporal_features"]
        == expected_local
    ):
        exact_local_feature_matches += 1


    original_global = (
        original_step1_case[
            "global_alignment_shift_features"
        ]
    )

    expected_global = {
        key: original_global[key]
        for key in LAG_PROMPT_GLOBAL_FEATURE_KEYS
    }


    if (
        clean_case["global_shift_features"]
        == expected_global
    ):
        exact_global_feature_matches += 1


assert exact_filtered_turn_matches == 100
assert exact_local_feature_matches == 100
assert exact_global_feature_matches == 100


print("=" * 76)
print("STEP-1 NORMAL REPRODUCTION CHECK PASSED")
print("=" * 76)

print(
    "Exact filtered-turn matches:",
    exact_filtered_turn_matches,
    "/ 100",
)

print(
    "Exact local-feature matches:",
    exact_local_feature_matches,
    "/ 100",
)

print(
    "Exact global-feature matches:",
    exact_global_feature_matches,
    "/ 100",
)


STEP-1 NORMAL REPRODUCTION CHECK PASSED
Exact filtered-turn matches: 100 / 100
Exact local-feature matches: 100 / 100
Exact global-feature matches: 100 / 100


## Inspect the current case database

Use the next function to inspect both the NORMAL and WRONG_PARTNER records that
belong to a selected `source_group_id`.

The displayed record shows exactly what has been saved for each case at this
stage:

- participant and source metadata;
- final independently backchannel-filtered turns;
- exact local lag-prompt features;
- exact global-shift lag-prompt features.

The frozen reference averages are displayed separately because they are shared
by every case and are not duplicated 200 times.

In [ ]:
# ============================================================
# INSPECT BOTH CASES OF ONE SOURCE GROUP
# ============================================================

import pandas as pd


def inspect_clean_source_group(
    source_index=0,
    show_full_json=True,
):
    source_group_id = (
        f"heldout_source_{int(source_index):03d}"
    )

    group_cases = [
        case
        for case in clean_normal_wrong_cases
        if case["source_group_id"] == source_group_id
    ]

    assert len(group_cases) == 2, (
        f"Expected 2 cases for {source_group_id}, "
        f"found {len(group_cases)}."
    )

    group_cases = sorted(
        group_cases,
        key=lambda case: (
            0
            if case["case_variant"] == "normal"
            else 1
        ),
    )


    print("=" * 90)
    print("SOURCE GROUP:", source_group_id)
    print("=" * 90)


    summary_rows = []

    for case in group_cases:

        summary_rows.append({
            "case_id": case["case_id"],
            "variant": case["case_variant"],
            "gold_binary_label": (
                case["gold_binary_label"]
            ),
            "A_conversation": (
                case["A_source_conversation"]
            ),
            "A_participant": (
                case["participant_A"]["participant_id"]
            ),
            "B_conversation": (
                case["B_source_conversation"]
            ),
            "B_participant": (
                case["participant_B"]["participant_id"]
            ),
            "num_filtered_A_turns": len(
                case["participant_A_filtered_turns"]
            ),
            "num_filtered_B_turns": len(
                case["participant_B_filtered_turns"]
            ),
            "num_signed_offsets": (
                case["local_temporal_features"][
                    "num_signed_strict_offsets"
                ]
            ),
        })


    display(
        pd.DataFrame(summary_rows)
    )


    for case in group_cases:

        print("\n" + "-" * 90)
        print(
            case["case_id"],
            "|",
            case["case_variant"],
        )

        print("\nParticipant A filtered turns:")
        display(
            pd.DataFrame(
                case[
                    "participant_A_filtered_turns"
                ]
            )
        )

        print("\nParticipant B filtered turns:")
        display(
            pd.DataFrame(
                case[
                    "participant_B_filtered_turns"
                ]
            )
        )

        print("\nLocal temporal features:")
        print(
            json.dumps(
                case["local_temporal_features"],
                indent=2,
                ensure_ascii=False,
            )
        )

        print("\nGlobal shift features:")
        print(
            json.dumps(
                case["global_shift_features"],
                indent=2,
                ensure_ascii=False,
            )
        )

        if show_full_json:
            print("\nComplete saved case record:")
            print(
                json.dumps(
                    case,
                    indent=2,
                    ensure_ascii=False,
                )
            )


    print("\n" + "=" * 90)
    print("SHARED FROZEN REFERENCE FEATURES")
    print("=" * 90)

    print(
        json.dumps(
            frozen_lag_reference_features,
            indent=2,
            ensure_ascii=False,
        )
    )


# Inspect the first source group.
inspect_clean_source_group(
    source_index=1,
    show_full_json=True,
)


SOURCE GROUP: heldout_source_001


,case_id,variant,gold_binary_label,A_conversation,A_participant,B_conversation,B_participant,num_filtered_A_turns,num_filtered_B_turns,num_signed_offsets
0,consolidation_normal_001,normal,NORMAL,V00_S2050_I00001124,P1307A,V00_S2050_I00001124,P1308A,4,9,4
1,consolidation_wrong_partner_001,wrong_partner,ANOMALOUS,V00_S2050_I00001124,P1307A,V00_S2051_I00001001,P1308A,5,8,3



------------------------------------------------------------------------------------------
consolidation_normal_001 | normal

Participant A filtered turns:


,start,end
0,21.35,23.81
1,38.95,42.17
2,48.51,55.45
3,68.64,110.11



Participant B filtered turns:


...


Local temporal features:
{
  "clean_overlap_seconds": 2.74,
  "clean_overlap_percent": 2.28,
  "signed_strict_offsets_seconds": [
    0.32,
    0.65,
    1.7,
    -0.99
  ],
  "num_signed_strict_offsets": 4,
  "offset_mean_seconds": 0.42,
  "offset_median_seconds": 0.48,
  "offset_max_seconds": 1.7,
  "offset_p75_seconds": 0.91,
  "offset_p90_seconds": 1.39,
  "num_offsets_above_1_5_seconds": 1,
  "percent_offsets_above_1_5_seconds": 25.0
}

Global shift features:
{
  "best_B_correction_shift_seconds": -0.6,
  "estimated_B_lateness_seconds": 0.6,
  "alignment_score_gain_vs_zero": 0.035408,
  "best_num_bilateral_events": 8,
  "best_event_coverage": 0.615385
}

Complete saved case record:
{
  "case_id": "consolidation_normal_001",
  "source_group_id": "heldout_source_001",
  "pair_index": 1,
  "gold_binary_label": "NORMAL",
  "gold_anomaly_type": "none",
  "case_variant": "normal",
  "pairing_type": "same_conversation",
  "participant_A": {
    "conversation_id": "V00_S2050_I00001124",


,start,end
0,21.35,23.81
1,30.98,31.68
2,38.95,42.17
3,48.51,55.45
4,68.64,110.11



Participant B filtered turns:


,start,end
0,0.37,3.76
1,22.26,26.73
2,28.98,29.52
3,45.36,46.77
4,48.31,50.45
5,55.44,62.19
6,63.54,69.23
7,97.20,99.34



Local temporal features:
{
  "clean_overlap_seconds": 6.23,
  "clean_overlap_percent": 5.19,
  "signed_strict_offsets_seconds": [
    -1.55,
    3.19,
    -0.01
  ],
  "num_signed_strict_offsets": 3,
  "offset_mean_seconds": 0.54,
  "offset_median_seconds": -0.01,
  "offset_max_seconds": 3.19,
  "offset_p75_seconds": 1.59,
  "offset_p90_seconds": 2.55,
  "num_offsets_above_1_5_seconds": 1,
  "percent_offsets_above_1_5_seconds": 33.3
}

Global shift features:
{
  "best_B_correction_shift_seconds": 1.4,
  "estimated_B_lateness_seconds": 0.0,
  "alignment_score_gain_vs_zero": 0.044272,
  "best_num_bilateral_events": 6,
  "best_event_coverage": 0.461538
}

Complete saved case record:
{
  "case_id": "consolidation_wrong_partner_001",
  "source_group_id": "heldout_source_001",
  "pair_index": 1,
  "gold_binary_label": "ANOMALOUS",
  "gold_anomaly_type": "wrong_partner",
  "case_variant": "wrong_partner",
  "pairing_type": "different_conversations",
  "participant_A": {
    "conversation_id"

## Outputs of the current notebook

After a successful run, the two final clean artifacts are:

### `consolidation_normal_wrong_cases_with_lag_features.json`

Contains 200 records:

- 100 NORMAL;
- 100 WRONG_PARTNER.

Each record contains participant metadata, final filtered turns and only the
exact current-case features used in the lag prompt.

### `consolidation_frozen_lag_reference_features.json`

Contains the shared frozen averages from the separate 50-conversation
reference set for:

- NORMAL;
- LAG +2 seconds;
- LAG +3 seconds.

The resumable global-shift cache is an implementation artifact only. It is not
a consolidation input.

# Step 3 — Construction of 50 LAG +2 s and 50 LAG +3 s cases

This step creates exactly 100 synthetic lag anomalies from the same 100
held-out normal conversations used in the previous consolidation steps.

Each held-out conversation contributes exactly one lag case:

- 50 conversations receive a +2 second shift;
- 50 conversations receive a +3 second shift.

The assignment is deterministic and reproducible.

For every source conversation:

1. The original merged Participant A turns are loaded from
   `inference_selection_manifest`.
2. The original merged Participant B turns are loaded from the same manifest.
3. The complete merged Participant B timeline is shifted later by either
   +2 or +3 seconds.
4. The shifted turns are clipped to the 0–120 second analysis window.
5. Independent backchannel filtering is applied to the assembled shifted pair.
6. Local offset and overlap features are calculated from the resulting
   filtered A/B turns.
7. The global Participant B correction-shift features are calculated from
   the same filtered turns.

The shift is therefore applied before backchannel filtering and before
temporal-feature extraction, exactly as in the lag pipeline.

The saved case records contain only:

- internal case and participant metadata;
- the applied synthetic lag amount;
- the final filtered Participant A and Participant B turns;
- the local temporal features used in the lag prompt;
- the global-shift features used in the lag prompt.

The frozen NORMAL, LAG +2 and LAG +3 reference averages remain those computed
from the separate 50-conversation reference set. They are not recomputed from
these held-out lag cases.

In [ ]:
# ============================================================
# STEP 3
# CREATE 50 LAG +2 s AND 50 LAG +3 s ASSIGNMENTS
# ============================================================

import json
import random
from pathlib import Path

from tqdm.auto import tqdm


CONSOLIDATION_LAG_ASSIGNMENT_SEED = (
    RANDOM_SEED + 3000
)


CONSOLIDATION_LAG_CASES_PATH = (
    OUT_DIR
    / "consolidation_lag_cases_50_lag2_50_lag3_with_features.json"
)


CONSOLIDATION_LAG_ASSIGNMENT_PATH = (
    OUT_DIR
    / "consolidation_lag_assignment_50_lag2_50_lag3.json"
)


# ============================================================
# EXACT CURRENT-CASE FEATURES USED IN THE LAG PROMPT
# ============================================================

LAG_CASE_LOCAL_FEATURE_KEYS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",

    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",

    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",

    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


LAG_CASE_GLOBAL_FEATURE_KEYS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# ============================================================
# VERIFY REQUIRED STEP-1 FUNCTIONS AND DATA
# ============================================================

required_names = [
    "inference_selection_manifest",
    "reference_ids",

    "shift_ranges",
    "clip_and_localize_ranges",

    "compute_independent_bc_features",
    "estimate_global_B_correction_shift",

    "MAX_SECONDS",
    "BOUNDARY_SAFE_MODE",
    "REFERENCE_LAG_SECONDS",
]


missing_names = [
    name
    for name in required_names
    if name not in globals()
]


assert not missing_names, (
    "Run the previous Step 1 cells first. "
    f"Missing variables/functions: {missing_names}"
)


assert len(
    inference_selection_manifest
) == 100


heldout_conversation_ids = [
    record["conversation_id"]
    for record in inference_selection_manifest
]


assert len(
    set(heldout_conversation_ids)
) == 100


assert set(
    heldout_conversation_ids
).isdisjoint(
    reference_ids
)


# ============================================================
# DETERMINISTIC 50 / 50 ASSIGNMENT
# ============================================================

assignment_rng = random.Random(
    CONSOLIDATION_LAG_ASSIGNMENT_SEED
)


assignment_order = list(
    range(100)
)


assignment_rng.shuffle(
    assignment_order
)


lag_2_indices = set(
    assignment_order[:50]
)


lag_3_indices = set(
    assignment_order[50:]
)


assert len(lag_2_indices) == 50
assert len(lag_3_indices) == 50
assert lag_2_indices.isdisjoint(
    lag_3_indices
)

assert (
    lag_2_indices
    | lag_3_indices
) == set(range(100))


lag_seconds_by_source_index = {
    source_index: (
        2.0
        if source_index in lag_2_indices
        else 3.0
    )
    for source_index in range(100)
}


lag_assignment_manifest = []


for source_index, record in enumerate(
    inference_selection_manifest
):

    lag_seconds = (
        lag_seconds_by_source_index[
            source_index
        ]
    )

    lag_assignment_manifest.append({
        "source_index": source_index,

        "source_group_id": (
            f"heldout_source_"
            f"{source_index:03d}"
        ),

        "conversation_id": (
            record["conversation_id"]
        ),

        "participant_A_id": (
            record[
                "participant_A"
            ]["participant_id"]
        ),

        "participant_B_id": (
            record[
                "participant_B"
            ]["participant_id"]
        ),

        "lag_seconds": lag_seconds,

        "case_variant": (
            f"lag_{int(lag_seconds)}sec"
        ),
    })


CONSOLIDATION_LAG_ASSIGNMENT_PATH.write_text(
    json.dumps(
        lag_assignment_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print(
    "Held-out conversations:",
    len(inference_selection_manifest),
)

print(
    "Assigned LAG +2 s:",
    sum(
        row["lag_seconds"] == 2.0
        for row in lag_assignment_manifest
    ),
)

print(
    "Assigned LAG +3 s:",
    sum(
        row["lag_seconds"] == 3.0
        for row in lag_assignment_manifest
    ),
)

print(
    "Saved assignment manifest:",
    CONSOLIDATION_LAG_ASSIGNMENT_PATH,
)

Held-out conversations: 100
Assigned LAG +2 s: 50
Assigned LAG +3 s: 50
Saved assignment manifest: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_lag_assignment_50_lag2_50_lag3.json


In [ ]:
# ============================================================
# BUILD AND PREPROCESS THE 100 LAG CASES
#
# Order:
#
# original merged turns
# -> shift merged Participant B turns
# -> clip/localize
# -> independent backchannel filtering
# -> local temporal features
# -> global B-correction features
# ============================================================

consolidation_lag_cases = []


for source_index, source_record in enumerate(
    tqdm(
        inference_selection_manifest,
        desc="Constructing 100 LAG cases",
    )
):

    lag_seconds = float(
        lag_seconds_by_source_index[
            source_index
        ]
    )


    # --------------------------------------------------------
    # 1. Original merged 0–120 second VAD turns
    #
    # These come directly from the held-out selection manifest.
    # They have already been:
    #
    # - clipped to 0–120 seconds;
    # - cleaned of invalid ranges;
    # - merged with max gap 0.75 seconds.
    #
    # They have NOT undergone backchannel filtering.
    # --------------------------------------------------------

    original_merged_turns_A = (
        source_record[
            "turns_A_full_0_120"
        ]
    )

    original_merged_turns_B = (
        source_record[
            "turns_B_full_0_120"
        ]
    )


    # --------------------------------------------------------
    # 2. Shift the complete merged Participant B timeline
    #
    # The shift occurs BEFORE backchannel filtering.
    # --------------------------------------------------------

    shifted_merged_turns_B = shift_ranges(
        original_merged_turns_B,
        shift_seconds=lag_seconds,
    )


    # --------------------------------------------------------
    # 3. Use the exact analysis-window logic of the lag pipeline
    # --------------------------------------------------------

    if BOUNDARY_SAFE_MODE:

        boundary_margin = max(
            REFERENCE_LAG_SECONDS
        )

        analysis_start = float(
            boundary_margin
        )

        analysis_end = float(
            MAX_SECONDS
            - boundary_margin
        )

    else:

        analysis_start = 0.0
        analysis_end = float(
            MAX_SECONDS
        )


    analysis_duration = (
        analysis_end
        - analysis_start
    )


    # Participant A is not shifted.
    prefilter_turns_A = (
        clip_and_localize_ranges(
            original_merged_turns_A,
            global_start=analysis_start,
            global_end=analysis_end,
        )
    )


    # Participant B has already been shifted.
    # Turns beyond 120 seconds are removed or clipped here.
    prefilter_shifted_turns_B = (
        clip_and_localize_ranges(
            shifted_merged_turns_B,
            global_start=analysis_start,
            global_end=analysis_end,
        )
    )


    # --------------------------------------------------------
    # 4. Minimal label-free input to the temporal pipeline
    # --------------------------------------------------------

    temporal_input = {
        "turns_A": prefilter_turns_A,
        "turns_B": prefilter_shifted_turns_B,
        "analysis_duration": float(
            analysis_duration
        ),
    }


    # --------------------------------------------------------
    # 5. Exact independent backchannel filtering and local
    #    temporal-feature extraction from the shifted pair
    # --------------------------------------------------------

    complete_local_features = (
        compute_independent_bc_features(
            temporal_input
        )
    )


    filtered_turns_A = (
        complete_local_features[
            "filtered_turns_A"
        ]
    )


    filtered_turns_B = (
        complete_local_features[
            "filtered_turns_B"
        ]
    )


    # --------------------------------------------------------
    # 6. Exact global B-correction search
    #
    # Input is the already filtered shifted pair.
    # This search does not permanently shift the saved turns.
    # --------------------------------------------------------

    complete_global_features = (
        estimate_global_B_correction_shift(
            turns_A=filtered_turns_A,
            turns_B=filtered_turns_B,
            timeline_duration=float(
                analysis_duration
            ),
        )
    )


    # --------------------------------------------------------
    # 7. Keep only the fields used by the lag reasoner
    # --------------------------------------------------------

    local_temporal_features = {
        key: complete_local_features[key]
        for key
        in LAG_CASE_LOCAL_FEATURE_KEYS
    }


    global_shift_features = {
        key: complete_global_features[key]
        for key
        in LAG_CASE_GLOBAL_FEATURE_KEYS
    }


    # --------------------------------------------------------
    # 8. Save the compact internal lag-case record
    # --------------------------------------------------------

    lag_case = {
        "case_id": (
            f"consolidation_lag_"
            f"{int(lag_seconds)}sec_"
            f"{source_index:03d}"
        ),

        "source_group_id": (
            f"heldout_source_"
            f"{source_index:03d}"
        ),

        "pair_index": source_index,

        "gold_binary_label": "ANOMALOUS",
        "gold_anomaly_type": "lag",

        "case_variant": (
            f"lag_{int(lag_seconds)}sec"
        ),

        "pairing_type": (
            "same_conversation_shifted_B"
        ),

        "lag_seconds": lag_seconds,

        "shifted_participant": "B",

        "participant_A": (
            source_record["participant_A"]
        ),

        "participant_B": (
            source_record["participant_B"]
        ),

        "A_source_conversation": (
            source_record["conversation_id"]
        ),

        "B_source_conversation": (
            source_record["conversation_id"]
        ),

        "analysis_duration_seconds": float(
            analysis_duration
        ),

        # Final turns after:
        # shift B -> clip -> pairwise backchannel filtering.
        "participant_A_filtered_turns": (
            filtered_turns_A
        ),

        "participant_B_filtered_turns": (
            filtered_turns_B
        ),

        # Exact current-case features used by the lag prompt.
        "local_temporal_features": (
            local_temporal_features
        ),

        "global_shift_features": (
            global_shift_features
        ),
    }


    consolidation_lag_cases.append(
        lag_case
    )


    # Incremental saving protects the completed cases
    # if the Colab runtime is interrupted.
    CONSOLIDATION_LAG_CASES_PATH.write_text(
        json.dumps(
            consolidation_lag_cases,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


print(
    "Constructed lag cases:",
    len(consolidation_lag_cases),
)

print(
    "Saved:",
    CONSOLIDATION_LAG_CASES_PATH,
)

Constructing 100 LAG cases:   0%|          | 0/100 [00:00<?, ?it/s]

Constructed lag cases: 100
Saved: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_lag_cases_50_lag2_50_lag3_with_features.json


In [ ]:
# ============================================================
# STRICT LAG CONSTRUCTION AND PREPROCESSING AUDIT
# ============================================================

assert len(
    consolidation_lag_cases
) == 100


lag_2_cases = [
    case
    for case in consolidation_lag_cases
    if case["lag_seconds"] == 2.0
]


lag_3_cases = [
    case
    for case in consolidation_lag_cases
    if case["lag_seconds"] == 3.0
]


assert len(lag_2_cases) == 50
assert len(lag_3_cases) == 50


# Every held-out source is used exactly once.
lag_source_group_ids = [
    case["source_group_id"]
    for case in consolidation_lag_cases
]


assert len(
    set(lag_source_group_ids)
) == 100


assert set(
    lag_source_group_ids
) == {
    f"heldout_source_{index:03d}"
    for index in range(100)
}


# All lag pairs retain their true conversation partner.
assert all(
    case["A_source_conversation"]
    == case["B_source_conversation"]
    for case in consolidation_lag_cases
)


# No reference conversation is used.
assert all(
    case["A_source_conversation"]
    not in reference_ids
    for case in consolidation_lag_cases
)


expected_saved_case_keys = {
    "case_id",
    "source_group_id",
    "pair_index",

    "gold_binary_label",
    "gold_anomaly_type",

    "case_variant",
    "pairing_type",

    "lag_seconds",
    "shifted_participant",

    "participant_A",
    "participant_B",

    "A_source_conversation",
    "B_source_conversation",

    "analysis_duration_seconds",

    "participant_A_filtered_turns",
    "participant_B_filtered_turns",

    "local_temporal_features",
    "global_shift_features",
}


num_exact_filter_reproductions = 0
num_exact_local_feature_reproductions = 0


for case in tqdm(
    consolidation_lag_cases,
    desc="Auditing lag preprocessing",
):

    assert set(case.keys()) == (
        expected_saved_case_keys
    )


    assert case["lag_seconds"] in {
        2.0,
        3.0,
    }


    assert case["shifted_participant"] == "B"


    assert set(
        case["local_temporal_features"]
    ) == set(
        LAG_CASE_LOCAL_FEATURE_KEYS
    )


    assert set(
        case["global_shift_features"]
    ) == set(
        LAG_CASE_GLOBAL_FEATURE_KEYS
    )


    source_index = int(
        case["pair_index"]
    )


    source_record = (
        inference_selection_manifest[
            source_index
        ]
    )


    # --------------------------------------------------------
    # Reconstruct the exact pre-filter input independently
    # --------------------------------------------------------

    reconstructed_shifted_B = shift_ranges(
        source_record[
            "turns_B_full_0_120"
        ],
        shift_seconds=float(
            case["lag_seconds"]
        ),
    )


    if BOUNDARY_SAFE_MODE:

        boundary_margin = max(
            REFERENCE_LAG_SECONDS
        )

        reconstructed_start = float(
            boundary_margin
        )

        reconstructed_end = float(
            MAX_SECONDS
            - boundary_margin
        )

    else:

        reconstructed_start = 0.0
        reconstructed_end = float(
            MAX_SECONDS
        )


    reconstructed_A_input = (
        clip_and_localize_ranges(
            source_record[
                "turns_A_full_0_120"
            ],
            global_start=reconstructed_start,
            global_end=reconstructed_end,
        )
    )


    reconstructed_shifted_B_input = (
        clip_and_localize_ranges(
            reconstructed_shifted_B,
            global_start=reconstructed_start,
            global_end=reconstructed_end,
        )
    )


    reconstructed_temporal_input = {
        "turns_A": (
            reconstructed_A_input
        ),

        "turns_B": (
            reconstructed_shifted_B_input
        ),

        "analysis_duration": float(
            reconstructed_end
            - reconstructed_start
        ),
    }


    # --------------------------------------------------------
    # Independently reproduce the filtering and local features
    # --------------------------------------------------------

    reproduced_local = (
        compute_independent_bc_features(
            reconstructed_temporal_input
        )
    )


    reproduced_filtered_A = (
        reproduced_local[
            "filtered_turns_A"
        ]
    )


    reproduced_filtered_B = (
        reproduced_local[
            "filtered_turns_B"
        ]
    )


    if (
        reproduced_filtered_A
        == case[
            "participant_A_filtered_turns"
        ]
        and reproduced_filtered_B
        == case[
            "participant_B_filtered_turns"
        ]
    ):
        num_exact_filter_reproductions += 1


    reproduced_minimal_local = {
        key: reproduced_local[key]
        for key in LAG_CASE_LOCAL_FEATURE_KEYS
    }


    if (
        reproduced_minimal_local
        == case["local_temporal_features"]
    ):
        num_exact_local_feature_reproductions += 1


assert num_exact_filter_reproductions == 100

assert (
    num_exact_local_feature_reproductions
    == 100
)


print("=" * 76)
print("LAG CONSTRUCTION AND PREPROCESSING AUDIT PASSED")
print("=" * 76)

print(
    "Total lag cases:",
    len(consolidation_lag_cases),
)

print(
    "LAG +2 s cases:",
    len(lag_2_cases),
)

print(
    "LAG +3 s cases:",
    len(lag_3_cases),
)

print(
    "Unique held-out source groups:",
    len(set(lag_source_group_ids)),
)

print(
    "Reference conversations used:",
    len({
        case["A_source_conversation"]
        for case in consolidation_lag_cases
    } & reference_ids),
)

print(
    "Exact filtered-turn reproductions:",
    num_exact_filter_reproductions,
    "/ 100",
)

print(
    "Exact local-feature reproductions:",
    num_exact_local_feature_reproductions,
    "/ 100",
)

print(
    "Cases with global-shift features:",
    sum(
        set(
            case["global_shift_features"]
        ) == set(
            LAG_CASE_GLOBAL_FEATURE_KEYS
        )
        for case in consolidation_lag_cases
    ),
)

Auditing lag preprocessing:   0%|          | 0/100 [00:00<?, ?it/s]

LAG CONSTRUCTION AND PREPROCESSING AUDIT PASSED
Total lag cases: 100
LAG +2 s cases: 50
LAG +3 s cases: 50
Unique held-out source groups: 100
Reference conversations used: 0
Exact filtered-turn reproductions: 100 / 100
Exact local-feature reproductions: 100 / 100
Cases with global-shift features: 100


In [ ]:
# ============================================================
# SUMMARY OF ALL 100 SAVED LAG CASES
# ============================================================

lag_case_summary_rows = []


for case in consolidation_lag_cases:

    local = case[
        "local_temporal_features"
    ]

    global_shift = case[
        "global_shift_features"
    ]


    lag_case_summary_rows.append({
        "case_id": case["case_id"],

        "source_group_id": (
            case["source_group_id"]
        ),

        "conversation_id": (
            case["A_source_conversation"]
        ),

        "participant_A_id": (
            case[
                "participant_A"
            ]["participant_id"]
        ),

        "participant_B_id": (
            case[
                "participant_B"
            ]["participant_id"]
        ),

        "lag_seconds": (
            case["lag_seconds"]
        ),

        "num_filtered_A_turns": len(
            case[
                "participant_A_filtered_turns"
            ]
        ),

        "num_filtered_B_turns": len(
            case[
                "participant_B_filtered_turns"
            ]
        ),

        "num_signed_offsets": (
            local[
                "num_signed_strict_offsets"
            ]
        ),

        "offset_mean_seconds": (
            local[
                "offset_mean_seconds"
            ]
        ),

        "offset_median_seconds": (
            local[
                "offset_median_seconds"
            ]
        ),

        "offset_p90_seconds": (
            local[
                "offset_p90_seconds"
            ]
        ),

        "clean_overlap_seconds": (
            local[
                "clean_overlap_seconds"
            ]
        ),

        "best_B_correction_shift": (
            global_shift[
                "best_B_correction_shift_seconds"
            ]
        ),

        "estimated_B_lateness": (
            global_shift[
                "estimated_B_lateness_seconds"
            ]
        ),

        "alignment_gain": (
            global_shift[
                "alignment_score_gain_vs_zero"
            ]
        ),

        "event_coverage": (
            global_shift[
                "best_event_coverage"
            ]
        ),
    })


lag_case_summary_df = (
    pd.DataFrame(
        lag_case_summary_rows
    )
    .sort_values(
        [
            "lag_seconds",
            "source_group_id",
        ]
    )
    .reset_index(drop=True)
)


display(
    lag_case_summary_df
)


print("\nCounts by lag magnitude:")

display(
    lag_case_summary_df[
        "lag_seconds"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "lag_seconds"
    )
    .reset_index(
        name="num_cases"
    )
)

,case_id,source_group_id,conversation_id,participant_A_id,participant_B_id,lag_seconds,num_filtered_A_turns,num_filtered_B_turns,num_signed_offsets,offset_mean_seconds,offset_median_seconds,offset_p90_seconds,clean_overlap_seconds,best_B_correction_shift,estimated_B_lateness,alignment_gain,event_coverage
0,consolidation_lag_2sec_000,heldout_source_000,V03_S0148_I00000135,P1319,P1274,2.0,9,12,2,3.72,3.72,4.17,0.00,2.7,0.0,0.069487,0.238095
1,consolidation_lag_2sec_001,heldout_source_001,V00_S2050_I00001124,P1307A,P1308A,2.0,4,9,4,2.42,2.48,3.39,8.11,-2.6,2.6,0.245195,0.615385
2,consolidation_lag_2sec_003,heldout_source_003,V03_S0180_I00000126,P1388,P1123,2.0,12,5,4,0.95,0.90,2.16,7.46,-1.0,1.0,0.090815,0.470588
3,consolidation_lag_2sec_007,heldout_source_007,V00_S2049_I00001108,P1304A,P1306A,2.0,16,22,11,0.51,0.47,2.00,17.10,-1.7,1.7,0.195699,0.684211
4,consolidation_lag_2sec_009,heldout_source_009,V03_S0148_I00000504,P1319,P1274,2.0,5,17,4,1.82,1.46,3.71,5.12,-0.3,0.3,0.006790,0.363636
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,consolidation_lag_3sec_095,heldout_source_095,V01_S0338_I00001109,P1352,P1679,3.0,9,7,3,0.39,0.38,1.18,7.60,-0.0,0.0,0.000000,0.312500
96,consolidation_lag_3sec_096,heldout_source_096,V01_S0563_I00001235,P1841,P1840,3.0,8,11,4,4.12,4.21,4.86,8.23,-1.8,1.8,0.218311,0.526316
97,consolidation_lag_3sec_097,heldout_source_097,V00_S2018_I00001016,P1274A,P1273A,3.0,11,10,5,0.85,0.38,2.12,20.55,-2.7,2.7,0.140602,0.476190
98,consolidation_lag_3sec_098,heldout_source_098,V03_S0148_I00000502,P1319,P1274,3.0,6,17,2,0.11,0.11,1.37,8.66,-3.0,3.0,0.293345,0.434783



Counts by lag magnitude:


,lag_seconds,num_cases
0,2.0,50
1,3.0,50


In [ ]:
# ============================================================
# INSPECT ONE COMPLETE LAG CASE
# ============================================================

def inspect_consolidation_lag_case(
    case_index,
    show_complete_saved_json=True,
):
    case = consolidation_lag_cases[
        int(case_index)
    ]


    source_record = (
        inference_selection_manifest[
            case["pair_index"]
        ]
    )


    original_A = (
        source_record[
            "turns_A_full_0_120"
        ]
    )


    original_B = (
        source_record[
            "turns_B_full_0_120"
        ]
    )


    shifted_B = shift_ranges(
        original_B,
        shift_seconds=float(
            case["lag_seconds"]
        ),
    )


    if BOUNDARY_SAFE_MODE:

        boundary_margin = max(
            REFERENCE_LAG_SECONDS
        )

        analysis_start = float(
            boundary_margin
        )

        analysis_end = float(
            MAX_SECONDS
            - boundary_margin
        )

    else:

        analysis_start = 0.0
        analysis_end = float(
            MAX_SECONDS
        )


    prefilter_A = (
        clip_and_localize_ranges(
            original_A,
            global_start=analysis_start,
            global_end=analysis_end,
        )
    )


    prefilter_shifted_B = (
        clip_and_localize_ranges(
            shifted_B,
            global_start=analysis_start,
            global_end=analysis_end,
        )
    )


    print("=" * 80)
    print("CASE:", case["case_id"])
    print("=" * 80)

    print(
        "Source group:",
        case["source_group_id"],
    )

    print(
        "Conversation:",
        case["A_source_conversation"],
    )

    print(
        "Participant A:",
        case["participant_A"]["participant_id"],
    )

    print(
        "Participant B:",
        case["participant_B"]["participant_id"],
    )

    print(
        "Applied B shift:",
        case["lag_seconds"],
        "seconds",
    )


    print("\n1. Original merged Participant A turns:")

    display(
        pd.DataFrame(
            prefilter_A
        )
    )


    print("\n2. Original merged Participant B turns:")

    display(
        pd.DataFrame(
            original_B
        )
    )


    print(
        "\n3. Shifted Participant B turns "
        "before backchannel filtering:"
    )

    display(
        pd.DataFrame(
            prefilter_shifted_B
        )
    )


    print(
        "\n4. Final filtered Participant A turns:"
    )

    display(
        pd.DataFrame(
            case[
                "participant_A_filtered_turns"
            ]
        )
    )


    print(
        "\n5. Final filtered Participant B turns:"
    )

    display(
        pd.DataFrame(
            case[
                "participant_B_filtered_turns"
            ]
        )
    )


    print("\n6. Local temporal features:")

    print(
        json.dumps(
            case[
                "local_temporal_features"
            ],
            indent=2,
            ensure_ascii=False,
        )
    )


    print("\n7. Global shift features:")

    print(
        json.dumps(
            case[
                "global_shift_features"
            ],
            indent=2,
            ensure_ascii=False,
        )
    )


    if show_complete_saved_json:

        print("\n8. Complete saved case record:")

        print(
            json.dumps(
                case,
                indent=2,
                ensure_ascii=False,
            )
        )


    return case

In [ ]:
example_lag_2_index = next(
    index
    for index, case in enumerate(
        consolidation_lag_cases
    )
    if case["lag_seconds"] == 2.0
)


example_lag_3_index = next(
    index
    for index, case in enumerate(
        consolidation_lag_cases
    )
    if case["lag_seconds"] == 3.0
)


inspect_consolidation_lag_case(
    example_lag_2_index
)


inspect_consolidation_lag_case(
    example_lag_3_index
)

CASE: consolidation_lag_2sec_000
Source group: heldout_source_000
Conversation: V03_S0148_I00000135
Participant A: P1319
Participant B: P1274
Applied B shift: 2.0 seconds

1. Original merged Participant A turns:


,start,end
0,9.89,10.78
1,11.65,18.24
2,25.38,27.71
3,32.83,33.76
4,35.65,44.83
5,68.67,75.01
6,75.81,76.67
7,78.75,84.67
8,85.57,90.49



2. Original merged Participant B turns:


,start,end
0,0.00,5.21
1,6.08,7.81
2,42.08,42.53
3,45.99,46.78
4,47.71,56.45
5,57.22,58.59
6,59.78,63.23
7,64.45,65.89
8,92.77,94.69
9,95.49,103.58



3. Shifted Participant B turns before backchannel filtering:


...

,start,end
0,0.32,1.02
1,7.78,9.82
2,45.31,53.44
3,54.21,55.65
4,57.70,65.82
5,67.43,71.68
6,72.90,77.44
7,78.91,90.49
8,117.41,120.00



5. Final filtered Participant B turns:


,start,end
0,5.18,9.85
1,13.02,13.78
2,15.32,30.07
3,33.98,36.69
4,37.69,39.29
5,40.09,46.65
6,48.57,50.97
7,57.37,57.97
8,75.96,77.75
9,82.62,84.05



6. Local temporal features:
{
  "clean_overlap_seconds": 10.73,
  "clean_overlap_percent": 8.94,
  "signed_strict_offsets_seconds": [
    4.16,
    1.72,
    -1.48,
    5.54
  ],
  "num_signed_strict_offsets": 4,
  "offset_mean_seconds": 2.49,
  "offset_median_seconds": 2.94,
  "offset_max_seconds": 5.54,
  "offset_p75_seconds": 4.5,
  "offset_p90_seconds": 5.13,
  "num_offsets_above_1_5_seconds": 3,
  "percent_offsets_above_1_5_seconds": 75.0
}

7. Global shift features:
{
  "best_B_correction_shift_seconds": -4.3,
  "estimated_B_lateness_seconds": 4.3,
  "alignment_score_gain_vs_zero": 0.16808,
  "best_num_bilateral_events": 12,
  "best_event_coverage": 0.545455
}

8. Complete saved case record:
{
  "case_id": "consolidation_lag_3sec_002",
  "source_group_id": "heldout_source_002",
  "pair_index": 2,
  "gold_binary_label": "ANOMALOUS",
  "gold_anomaly_type": "lag",
  "case_variant": "lag_3sec",
  "pairing_type": "same_conversation_shifted_B",
  "lag_seconds": 3.0,
  "shifted_partici

{'case_id': 'consolidation_lag_3sec_002',
 'source_group_id': 'heldout_source_002',
 'pair_index': 2,
 'gold_binary_label': 'ANOMALOUS',
 'gold_anomaly_type': 'lag',
 'case_variant': 'lag_3sec',
 'pairing_type': 'same_conversation_shifted_B',
 'lag_seconds': 3.0,
 'shifted_participant': 'B',
 'participant_A': {'conversation_id': 'V01_S0340_I00001104',
  'participant_id': 'P1682',
  'metadata_path': '/content/drive/MyDrive/seamless_download/data/V01_S0340_I00001104/P1682/V01_S0340_I00001104_P1682.json',
  'num_raw_vad_entries': 64},
 'participant_B': {'conversation_id': 'V01_S0340_I00001104',
  'participant_id': 'P1681',
  'metadata_path': '/content/drive/MyDrive/seamless_download/data/V01_S0340_I00001104/P1681/V01_S0340_I00001104_P1681.json',
  'num_raw_vad_entries': 91},
 'A_source_conversation': 'V01_S0340_I00001104',
 'B_source_conversation': 'V01_S0340_I00001104',
 'analysis_duration_seconds': 120.0,
 'participant_A_filtered_turns': [{'start': 0.32, 'end': 1.02},
  {'start': 7.78, 

# Step 4 — Construction of 100 SILENT_PARTNER cases

This step constructs exactly 100 synthetic silent-partner cases from the same
100 held-out Participant A sources used by all previous consolidation cases.

For every held-out source \(i\):

\[
SILENT\_PARTNER_i = A_i + S_j
\]

where \(S_j\) is a verified silent participant from:

`DATA_ROOT / "silent"`

A participant is considered eligible only when:

- the expected participant metadata JSON exists and can be read;
- the JSON explicitly contains `metadata:vad`;
- `metadata:vad` is a list of length zero;
- a participant video file exists for later semantic-summary extraction.

When at least 100 eligible silent participants are available, 100 different
participants are selected without replacement using a fixed random seed.
Therefore:

- every held-out Participant A is used exactly once;
- every selected silent Participant B is used exactly once;
- no single silent participant is repeatedly reused.

For each assembled pair:

1. Participant A starts from the original merged 0–120 second VAD turns stored
   in `inference_selection_manifest`.
2. Participant B starts from an empty VAD turn list.
3. The exact independent backchannel-filtering function of the lag pipeline is
   applied to the assembled pair.
4. The exact local temporal features are calculated from the filtered turns.
5. The exact global B-correction features are calculated from the same filtered
   turns.

No synthetic shift is applied.

The final record stores only the participant metadata, filtered A/B turns and
the exact current-case features used in the lag pipeline.

In [ ]:
# ============================================================
# STEP 4 — SILENT-PARTNER CONFIGURATION
# ============================================================

from collections import Counter
from pathlib import Path

import json
import random

import pandas as pd
from tqdm.auto import tqdm


SILENT_ROOT = DATA_ROOT / "silent"

NUM_SILENT_PARTNER_CASES = 100

SILENT_PAIRING_SEED = (
    RANDOM_SEED + 4000
)


CONSOLIDATION_SILENT_CASES_PATH = (
    OUT_DIR
    / "consolidation_silent_partner_cases_with_lag_features.json"
)


CONSOLIDATION_SILENT_ASSIGNMENT_PATH = (
    OUT_DIR
    / "consolidation_silent_partner_assignment.json"
)


CONSOLIDATION_SILENT_SCAN_PATH = (
    OUT_DIR
    / "consolidation_verified_silent_participants.json"
)


# ============================================================
# EXACT CURRENT-CASE FEATURES USED IN THE LAG PIPELINE
# ============================================================

SILENT_LOCAL_FEATURE_KEYS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",

    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",

    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",

    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


SILENT_GLOBAL_FEATURE_KEYS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# ============================================================
# VERIFY REQUIRED PREVIOUS PIPELINE OBJECTS
# ============================================================

required_names = [
    "inference_selection_manifest",
    "reference_ids",

    "compute_independent_bc_features",
    "estimate_global_B_correction_shift",

    "MAX_SECONDS",
]


missing_names = [
    name
    for name in required_names
    if name not in globals()
]


assert not missing_names, (
    "Run all previous consolidation cells first. "
    f"Missing variables/functions: {missing_names}"
)


assert SILENT_ROOT.exists(), (
    f"Silent folder not found: {SILENT_ROOT}"
)


assert len(
    inference_selection_manifest
) == 100


print("Silent root:", SILENT_ROOT)

print(
    "Held-out Participant A sources:",
    len(inference_selection_manifest),
)

Silent root: /content/drive/MyDrive/seamless_download/data/silent
Held-out Participant A sources: 100


In [ ]:
# ============================================================
# DISCOVER AND VERIFY SILENT PARTICIPANTS
# ============================================================

def find_silent_metadata_json(
    conversation_dir: Path,
    participant_dir: Path,
):
    """
    Find the participant metadata JSON.
    """

    expected_path = (
        participant_dir
        / (
            f"{conversation_dir.name}_"
            f"{participant_dir.name}.json"
        )
    )

    if expected_path.exists():
        return expected_path


    candidates = [
        path
        for path in sorted(
            participant_dir.glob("*.json")
        )
        if not path.name.endswith(
            ".metadata.json"
        )
    ]


    return (
        candidates[0]
        if candidates
        else None
    )


def find_silent_video(
    participant_dir: Path,
):
    """
    Find an available participant video.
    """

    candidates = []


    for extension in [
        "*.mp4",
        "*.mov",
        "*.mkv",
        "*.webm",
    ]:

        candidates.extend(
            participant_dir.glob(
                extension
            )
        )


    candidates = sorted(
        set(candidates)
    )


    return (
        candidates[0]
        if candidates
        else None
    )


silent_scan_records = []


for conversation_dir in sorted(
    path
    for path in SILENT_ROOT.iterdir()
    if path.is_dir()
):

    participant_dirs = sorted(
        path
        for path in conversation_dir.iterdir()
        if path.is_dir()
    )


    for participant_dir in participant_dirs:

        metadata_path = (
            find_silent_metadata_json(
                conversation_dir,
                participant_dir,
            )
        )


        video_path = find_silent_video(
            participant_dir
        )


        scan_record = {
            "conversation_id": (
                conversation_dir.name
            ),

            "participant_id": (
                participant_dir.name
            ),

            "participant_directory": str(
                participant_dir
            ),

            "metadata_path": (
                str(metadata_path)
                if metadata_path is not None
                else None
            ),

            "video_path": (
                str(video_path)
                if video_path is not None
                else None
            ),

            "metadata_exists": (
                metadata_path is not None
                and metadata_path.exists()
            ),

            "metadata_read_success": False,

            "vad_key_present": False,

            "vad_is_list": False,

            "num_raw_vad_entries": None,

            "verified_empty_vad": False,

            "video_exists": (
                video_path is not None
                and video_path.exists()
            ),

            "read_error": None,
        }


        if scan_record["metadata_exists"]:

            try:

                with metadata_path.open(
                    "r",
                    encoding="utf-8",
                ) as file:

                    metadata = json.load(
                        file
                    )


                vad_key_present = (
                    "metadata:vad"
                    in metadata
                )


                raw_vad = metadata.get(
                    "metadata:vad",
                    None,
                )


                vad_is_list = isinstance(
                    raw_vad,
                    list,
                )


                num_raw_vad_entries = (
                    len(raw_vad)
                    if vad_is_list
                    else None
                )


                verified_empty_vad = (
                    vad_key_present
                    and vad_is_list
                    and len(raw_vad) == 0
                )


                scan_record.update({
                    "metadata_read_success": True,

                    "vad_key_present": (
                        vad_key_present
                    ),

                    "vad_is_list": (
                        vad_is_list
                    ),

                    "num_raw_vad_entries": (
                        num_raw_vad_entries
                    ),

                    "verified_empty_vad": (
                        verified_empty_vad
                    ),
                })


            except Exception as exc:

                scan_record["read_error"] = (
                    repr(exc)
                )


        silent_scan_records.append(
            scan_record
        )


silent_scan_df = pd.DataFrame(
    silent_scan_records
)


print(
    "Participant directories scanned:",
    len(silent_scan_df),
)

print(
    "Readable metadata files:",
    int(
        silent_scan_df[
            "metadata_read_success"
        ].sum()
    ),
)

print(
    "Explicitly empty metadata:vad:",
    int(
        silent_scan_df[
            "verified_empty_vad"
        ].sum()
    ),
)

print(
    "Verified empty-VAD participants with video:",
    int(
        (
            silent_scan_df[
                "verified_empty_vad"
            ]
            & silent_scan_df[
                "video_exists"
            ]
        ).sum()
    ),
)


display(
    silent_scan_df
)

Participant directories scanned: 49
Readable metadata files: 49
Explicitly empty metadata:vad: 45
Verified empty-VAD participants with video: 43


,conversation_id,participant_id,participant_directory,metadata_path,video_path,metadata_exists,metadata_read_success,vad_key_present,vad_is_list,num_raw_vad_entries,verified_empty_vad,video_exists,read_error
0,V00_S2036_I00000716,P1293A,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,True,True,True,True,0,True,True,None
1,V00_S2036_I00000716,P1294A,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,True,True,True,True,0,True,True,None
2,V01_S0173_I00001223,P1318,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,True,True,True,True,0,True,True,None
3,V01_S0173_I00001223,P1319,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,True,True,True,True,1,False,True,None
4,V01_S0173_I00001224,P1318,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,True,True,True,True,0,True,True,None
5,V01_S0173_I00001224,P1319,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,True,True,True,True,0,True,True,None
6,V01_S0173_I00001225,P1318,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,True,True,True,True,0,True,True,None
7,V01_S0173_I00001225,P1319,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,True,True,True,True,0,True,True,None
8,V01_S0173_I00001226,P1318,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,True,True,True,True,0,True,True,None
9,V01_S0173_I00001226,P1319,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,/content/drive/MyDrive/seamless_download/data/...,True,True,True,True,0,True,True,None


In [ ]:
# ============================================================
# BUILD THE VERIFIED ELIGIBLE SILENT POOL
# ============================================================

eligible_silent_participants = [
    {
        "conversation_id": (
            record["conversation_id"]
        ),

        "participant_id": (
            record["participant_id"]
        ),

        "metadata_path": (
            record["metadata_path"]
        ),

        "video_path": (
            record["video_path"]
        ),

        "num_raw_vad_entries": 0,

        "verified_empty_vad": True,

        "is_silent_source": True,
    }

    for record in silent_scan_records

    if (
        record["metadata_read_success"]
        and record["verified_empty_vad"]
        and record["video_exists"]
    )
]


eligible_silent_signatures = [
    (
        participant[
            "conversation_id"
        ],
        participant[
            "participant_id"
        ],
    )

    for participant
    in eligible_silent_participants
]


assert len(
    eligible_silent_signatures
) == len(
    set(
        eligible_silent_signatures
    )
), (
    "Duplicate silent-participant records detected."
)


assert len(
    eligible_silent_participants
) > 0, (
    "No verified silent participants were found."
)


CONSOLIDATION_SILENT_SCAN_PATH.write_text(
    json.dumps(
        eligible_silent_participants,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print(
    "Eligible unique silent participants:",
    len(
        eligible_silent_participants
    ),
)

print(
    "Saved verified silent pool:",
    CONSOLIDATION_SILENT_SCAN_PATH,
)

Eligible unique silent participants: 43
Saved verified silent pool: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_verified_silent_participants.json


In [ ]:
# ============================================================
# DETERMINISTIC BALANCED REUSE OF SILENT PARTICIPANTS
# ============================================================

silent_pairing_rng = random.Random(
    SILENT_PAIRING_SEED
)


num_available_silent = len(
    eligible_silent_participants
)


base_repetitions = (
    NUM_SILENT_PARTNER_CASES
    // num_available_silent
)


remainder = (
    NUM_SILENT_PARTNER_CASES
    % num_available_silent
)


selected_silent_participants = []


# Every eligible silent participant appears the same
# base number of times.
for _ in range(
    base_repetitions
):

    repetition = list(
        eligible_silent_participants
    )

    silent_pairing_rng.shuffle(
        repetition
    )

    selected_silent_participants.extend(
        repetition
    )


# A subset appears one additional time.
if remainder > 0:

    additional_participants = (
        silent_pairing_rng.sample(
            eligible_silent_participants,
            k=remainder,
        )
    )

    selected_silent_participants.extend(
        additional_participants
    )


assert len(
    selected_silent_participants
) == NUM_SILENT_PARTNER_CASES


# ------------------------------------------------------------
# Shuffle assignments and avoid equal source conversation IDs
# ------------------------------------------------------------

for attempt in range(10000):

    silent_pairing_rng.shuffle(
        selected_silent_participants
    )


    valid_pairing = all(
        source_record[
            "conversation_id"
        ]
        != silent_participant[
            "conversation_id"
        ]

        for (
            source_record,
            silent_participant,
        )
        in zip(
            inference_selection_manifest,
            selected_silent_participants,
        )
    )


    if valid_pairing:
        break

else:

    raise RuntimeError(
        "Could not create silent assignments "
        "without same-conversation IDs."
    )


selected_silent_signatures = [
    (
        participant[
            "conversation_id"
        ],
        participant[
            "participant_id"
        ],
    )

    for participant
    in selected_silent_participants
]


silent_usage_counts = Counter(
    selected_silent_signatures
)


minimum_silent_usage = min(
    silent_usage_counts.values()
)


maximum_silent_usage = max(
    silent_usage_counts.values()
)


assert sum(
    silent_usage_counts.values()
) == NUM_SILENT_PARTNER_CASES


assert len(
    silent_usage_counts
) == num_available_silent


assert (
    maximum_silent_usage
    - minimum_silent_usage
) <= 1


print("=" * 72)
print("BALANCED SILENT ASSIGNMENT CREATED")
print("=" * 72)

print(
    "Total assignments:",
    len(
        selected_silent_participants
    ),
)

print(
    "Unique silent participants:",
    len(
        silent_usage_counts
    ),
)

print(
    "Minimum uses:",
    minimum_silent_usage,
)

print(
    "Maximum uses:",
    maximum_silent_usage,
)

print(
    "Usage distribution:",
    dict(
        sorted(
            Counter(
                silent_usage_counts.values()
            ).items()
        )
    ),
)

BALANCED SILENT ASSIGNMENT CREATED
Total assignments: 100
Unique silent participants: 43
Minimum uses: 2
Maximum uses: 3
Usage distribution: {2: 29, 3: 14}


In [ ]:
# ============================================================
# SAVE SILENT-PARTNER ASSIGNMENT MANIFEST
# ============================================================

silent_assignment_manifest = []


for source_index, (
    source_record,
    silent_participant,
) in enumerate(
    zip(
        inference_selection_manifest,
        selected_silent_participants,
    )
):

    silent_signature = (
        silent_participant[
            "conversation_id"
        ],
        silent_participant[
            "participant_id"
        ],
    )


    silent_assignment_manifest.append({
        "source_index": source_index,

        "source_group_id": (
            f"heldout_source_"
            f"{source_index:03d}"
        ),

        "participant_A_conversation_id": (
            source_record[
                "conversation_id"
            ]
        ),

        "participant_A_id": (
            source_record[
                "participant_A"
            ]["participant_id"]
        ),

        "silent_conversation_id": (
            silent_participant[
                "conversation_id"
            ]
        ),

        "silent_participant_id": (
            silent_participant[
                "participant_id"
            ]
        ),

        "silent_metadata_path": (
            silent_participant[
                "metadata_path"
            ]
        ),

        "silent_video_path": (
            silent_participant[
                "video_path"
            ]
        ),

        "verified_empty_vad": True,

        "silent_total_usage_count": (
            silent_usage_counts[
                silent_signature
            ]
        ),

        "pairing_seed": (
            SILENT_PAIRING_SEED
        ),

        "assignment_strategy": (
            "deterministic_balanced_reuse"
        ),
    })


CONSOLIDATION_SILENT_ASSIGNMENT_PATH.write_text(
    json.dumps(
        silent_assignment_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print(
    "Saved assignments:",
    len(
        silent_assignment_manifest
    ),
)

print(
    "Same-conversation IDs:",
    sum(
        record[
            "participant_A_conversation_id"
        ]
        == record[
            "silent_conversation_id"
        ]

        for record
        in silent_assignment_manifest
    ),
)

print(
    "Saved assignment manifest:",
    CONSOLIDATION_SILENT_ASSIGNMENT_PATH,
)

Saved assignments: 100
Same-conversation IDs: 0
Saved assignment manifest: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_silent_partner_assignment.json


In [ ]:
# ============================================================
# EMPTY-B TEMPORAL PIPELINE UNIT TEST
# ============================================================

probe_source = (
    inference_selection_manifest[0]
)


probe_temporal_input = {
    "turns_A": (
        probe_source[
            "turns_A_full_0_120"
        ]
    ),

    "turns_B": [],

    "analysis_duration": float(
        MAX_SECONDS
    ),
}


probe_local = (
    compute_independent_bc_features(
        probe_temporal_input
    )
)


probe_filtered_A = (
    probe_local[
        "filtered_turns_A"
    ]
)


probe_filtered_B = (
    probe_local[
        "filtered_turns_B"
    ]
)


assert isinstance(
    probe_filtered_A,
    list,
)


assert probe_filtered_B == []


assert (
    probe_local[
        "clean_overlap_seconds"
    ]
    == 0.0
)


assert (
    probe_local[
        "clean_overlap_percent"
    ]
    == 0.0
)


assert (
    probe_local[
        "signed_strict_offsets_seconds"
    ]
    == []
)


assert (
    probe_local[
        "num_signed_strict_offsets"
    ]
    == 0
)


probe_global = (
    estimate_global_B_correction_shift(
        turns_A=probe_filtered_A,
        turns_B=probe_filtered_B,
        timeline_duration=float(
            MAX_SECONDS
        ),
    )
)


for key, expected_value in {
    "best_B_correction_shift_seconds": 0.0,
    "estimated_B_lateness_seconds": 0.0,
    "alignment_score_gain_vs_zero": 0.0,
    "best_num_bilateral_events": 0,
    "best_event_coverage": 0.0,
}.items():

    assert (
        probe_global[key]
        == expected_value
    )


print("=" * 72)
print("EMPTY-B TEMPORAL PIPELINE TEST PASSED")
print("=" * 72)

print(
    "Filtered A turns:",
    len(probe_filtered_A),
)

print(
    "Filtered B turns:",
    len(probe_filtered_B),
)

EMPTY-B TEMPORAL PIPELINE TEST PASSED
Filtered A turns: 9
Filtered B turns: 0


In [ ]:
# ============================================================
# BUILD, PREPROCESS AND SAVE 100 SILENT-PARTNER CASES
# ============================================================

consolidation_silent_partner_cases = []


for source_index, (
    source_record,
    silent_participant,
) in enumerate(
    tqdm(
        zip(
            inference_selection_manifest,
            selected_silent_participants,
        ),
        total=NUM_SILENT_PARTNER_CASES,
        desc="Constructing SILENT_PARTNER cases",
    )
):

    # --------------------------------------------------------
    # Original merged Participant A turns
    #
    # They have already been clipped to 0–120 s and merged,
    # but have not yet undergone pairwise filtering for this
    # silent pairing.
    # --------------------------------------------------------

    original_merged_turns_A = (
        source_record[
            "turns_A_full_0_120"
        ]
    )


    # Verified silent Participant B.
    original_merged_turns_B = []


    # --------------------------------------------------------
    # Exact input to the lag temporal preprocessing
    # --------------------------------------------------------

    temporal_input = {
        "turns_A": (
            original_merged_turns_A
        ),

        "turns_B": (
            original_merged_turns_B
        ),

        "analysis_duration": float(
            MAX_SECONDS
        ),
    }


    # --------------------------------------------------------
    # Exact pairwise backchannel filtering and local features
    # --------------------------------------------------------

    complete_local_features = (
        compute_independent_bc_features(
            temporal_input
        )
    )


    filtered_turns_A = (
        complete_local_features[
            "filtered_turns_A"
        ]
    )


    filtered_turns_B = (
        complete_local_features[
            "filtered_turns_B"
        ]
    )


    # --------------------------------------------------------
    # Global-shift features from the final filtered turns
    # --------------------------------------------------------

    complete_global_features = (
        estimate_global_B_correction_shift(
            turns_A=filtered_turns_A,
            turns_B=filtered_turns_B,
            timeline_duration=float(
                MAX_SECONDS
            ),
        )
    )


    # --------------------------------------------------------
    # Keep only the exact features used in the lag pipeline
    # --------------------------------------------------------

    local_temporal_features = {
        key: complete_local_features[key]

        for key
        in SILENT_LOCAL_FEATURE_KEYS
    }


    global_shift_features = {
        key: complete_global_features[key]

        for key
        in SILENT_GLOBAL_FEATURE_KEYS
    }


    silent_signature = (
        silent_participant[
            "conversation_id"
        ],
        silent_participant[
            "participant_id"
        ],
    )


    # --------------------------------------------------------
    # Complete saved case record
    # --------------------------------------------------------

    silent_case = {
        "case_id": (
            f"consolidation_silent_partner_"
            f"{source_index:03d}"
        ),

        "source_group_id": (
            f"heldout_source_"
            f"{source_index:03d}"
        ),

        "pair_index": source_index,

        "gold_binary_label": (
            "ANOMALOUS"
        ),

        "gold_anomaly_type": (
            "silent_partner"
        ),

        "case_variant": (
            "silent_partner"
        ),

        "pairing_type": (
            "heldout_A_plus_verified_silent_B"
        ),

        "participant_A": (
            source_record[
                "participant_A"
            ]
        ),

        "participant_B": (
            silent_participant
        ),

        "A_source_conversation": (
            source_record[
                "conversation_id"
            ]
        ),

        "B_source_conversation": (
            silent_participant[
                "conversation_id"
            ]
        ),

        "silent_B_usage_count": (
            silent_usage_counts[
                silent_signature
            ]
        ),

        "analysis_duration_seconds": float(
            MAX_SECONDS
        ),

        # Final VAD turns after applying the exact
        # pairwise filtering of the lag pipeline.
        "participant_A_filtered_turns": (
            filtered_turns_A
        ),

        "participant_B_filtered_turns": (
            filtered_turns_B
        ),

        # Exact per-case local features from the
        # final filtered A and B turns.
        "local_temporal_features": (
            local_temporal_features
        ),

        # Exact global-shift features calculated
        # from the final filtered A and B turns.
        "global_shift_features": (
            global_shift_features
        ),
    }


    consolidation_silent_partner_cases.append(
        silent_case
    )


    # Incremental save.
    if (
        len(
            consolidation_silent_partner_cases
        )
        % 10
        == 0
    ):

        CONSOLIDATION_SILENT_CASES_PATH.write_text(
            json.dumps(
                consolidation_silent_partner_cases,
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )


# Final save.
CONSOLIDATION_SILENT_CASES_PATH.write_text(
    json.dumps(
        consolidation_silent_partner_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print(
    "Constructed SILENT_PARTNER cases:",
    len(
        consolidation_silent_partner_cases
    ),
)

print(
    "Saved complete cases:",
    CONSOLIDATION_SILENT_CASES_PATH,
)

Constructing SILENT_PARTNER cases:   0%|          | 0/100 [00:00<?, ?it/s]

Constructed SILENT_PARTNER cases: 100
Saved complete cases: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_silent_partner_cases_with_lag_features.json


In [ ]:
# ============================================================
# STRICT SILENT-PARTNER PREPROCESSING AND STORAGE AUDIT
# ============================================================

assert len(
    consolidation_silent_partner_cases
) == 100


expected_case_keys = {
    "case_id",
    "source_group_id",
    "pair_index",

    "gold_binary_label",
    "gold_anomaly_type",

    "case_variant",
    "pairing_type",

    "participant_A",
    "participant_B",

    "A_source_conversation",
    "B_source_conversation",

    "silent_B_usage_count",

    "analysis_duration_seconds",

    "participant_A_filtered_turns",
    "participant_B_filtered_turns",

    "local_temporal_features",
    "global_shift_features",
}


A_signatures = [
    (
        case[
            "participant_A"
        ]["conversation_id"],

        case[
            "participant_A"
        ]["participant_id"],
    )

    for case
    in consolidation_silent_partner_cases
]


B_signatures = [
    (
        case[
            "participant_B"
        ]["conversation_id"],

        case[
            "participant_B"
        ]["participant_id"],
    )

    for case
    in consolidation_silent_partner_cases
]


B_usage_audit = Counter(
    B_signatures
)


assert len(
    set(A_signatures)
) == 100


assert len(
    B_usage_audit
) == num_available_silent


assert sum(
    B_usage_audit.values()
) == 100


assert (
    max(
        B_usage_audit.values()
    )
    - min(
        B_usage_audit.values()
    )
) <= 1


exact_filtered_turn_matches = 0
exact_local_feature_matches = 0
exact_global_feature_matches = 0
verified_empty_vad_count = 0


for case in tqdm(
    consolidation_silent_partner_cases,
    desc="Auditing silent cases",
):

    assert set(
        case.keys()
    ) == expected_case_keys


    assert (
        case[
            "gold_binary_label"
        ]
        == "ANOMALOUS"
    )


    assert (
        case[
            "gold_anomaly_type"
        ]
        == "silent_partner"
    )


    assert isinstance(
        case[
            "participant_A_filtered_turns"
        ],
        list,
    )


    assert (
        case[
            "participant_B_filtered_turns"
        ]
        == []
    )


    assert set(
        case[
            "local_temporal_features"
        ].keys()
    ) == set(
        SILENT_LOCAL_FEATURE_KEYS
    )


    assert set(
        case[
            "global_shift_features"
        ].keys()
    ) == set(
        SILENT_GLOBAL_FEATURE_KEYS
    )


    # --------------------------------------------------------
    # Verify the silent metadata directly from disk
    # --------------------------------------------------------

    metadata_path = Path(
        case[
            "participant_B"
        ]["metadata_path"]
    )


    assert metadata_path.exists()


    with metadata_path.open(
        "r",
        encoding="utf-8",
    ) as file:

        metadata = json.load(
            file
        )


    assert (
        "metadata:vad"
        in metadata
    )


    assert isinstance(
        metadata["metadata:vad"],
        list,
    )


    assert len(
        metadata["metadata:vad"]
    ) == 0


    verified_empty_vad_count += 1


    # --------------------------------------------------------
    # Independently reproduce filtering and features
    # --------------------------------------------------------

    source_record = (
        inference_selection_manifest[
            case["pair_index"]
        ]
    )


    reproduced_input = {
        "turns_A": (
            source_record[
                "turns_A_full_0_120"
            ]
        ),

        "turns_B": [],

        "analysis_duration": float(
            MAX_SECONDS
        ),
    }


    reproduced_local = (
        compute_independent_bc_features(
            reproduced_input
        )
    )


    reproduced_filtered_A = (
        reproduced_local[
            "filtered_turns_A"
        ]
    )


    reproduced_filtered_B = (
        reproduced_local[
            "filtered_turns_B"
        ]
    )


    if (
        reproduced_filtered_A
        == case[
            "participant_A_filtered_turns"
        ]
        and reproduced_filtered_B
        == case[
            "participant_B_filtered_turns"
        ]
    ):

        exact_filtered_turn_matches += 1


    reproduced_selected_local = {
        key: reproduced_local[key]

        for key
        in SILENT_LOCAL_FEATURE_KEYS
    }


    if (
        reproduced_selected_local
        == case[
            "local_temporal_features"
        ]
    ):

        exact_local_feature_matches += 1


    reproduced_global = (
        estimate_global_B_correction_shift(
            turns_A=reproduced_filtered_A,
            turns_B=reproduced_filtered_B,
            timeline_duration=float(
                MAX_SECONDS
            ),
        )
    )


    reproduced_selected_global = {
        key: reproduced_global[key]

        for key
        in SILENT_GLOBAL_FEATURE_KEYS
    }


    if (
        reproduced_selected_global
        == case[
            "global_shift_features"
        ]
    ):

        exact_global_feature_matches += 1


    # --------------------------------------------------------
    # Expected empty-B behaviour
    # --------------------------------------------------------

    local = case[
        "local_temporal_features"
    ]


    global_shift = case[
        "global_shift_features"
    ]


    assert (
        local[
            "clean_overlap_seconds"
        ]
        == 0.0
    )


    assert (
        local[
            "clean_overlap_percent"
        ]
        == 0.0
    )


    assert (
        local[
            "signed_strict_offsets_seconds"
        ]
        == []
    )


    assert (
        local[
            "num_signed_strict_offsets"
        ]
        == 0
    )


    assert (
        local[
            "num_offsets_above_1_5_seconds"
        ]
        == 0
    )


    assert (
        global_shift[
            "best_B_correction_shift_seconds"
        ]
        == 0.0
    )


    assert (
        global_shift[
            "estimated_B_lateness_seconds"
        ]
        == 0.0
    )


    assert (
        global_shift[
            "alignment_score_gain_vs_zero"
        ]
        == 0.0
    )


    assert (
        global_shift[
            "best_num_bilateral_events"
        ]
        == 0
    )


    assert (
        global_shift[
            "best_event_coverage"
        ]
        == 0.0
    )


assert verified_empty_vad_count == 100
assert exact_filtered_turn_matches == 100
assert exact_local_feature_matches == 100
assert exact_global_feature_matches == 100


assert all(
    case[
        "A_source_conversation"
    ]
    not in reference_ids

    for case
    in consolidation_silent_partner_cases
)


print("=" * 76)
print("SILENT-PARTNER AUDIT PASSED")
print("=" * 76)

print(
    "Silent cases:",
    len(
        consolidation_silent_partner_cases
    ),
)

print(
    "Unique Participant A:",
    len(
        set(A_signatures)
    ),
)

print(
    "Unique silent Participant B:",
    len(
        B_usage_audit
    ),
)

print(
    "Silent B usage distribution:",
    dict(
        sorted(
            Counter(
                B_usage_audit.values()
            ).items()
        )
    ),
)

print(
    "Verified empty metadata:vad:",
    verified_empty_vad_count,
    "/ 100",
)

print(
    "Empty filtered B turn lists:",
    sum(
        case[
            "participant_B_filtered_turns"
        ]
        == []

        for case
        in consolidation_silent_partner_cases
    ),
    "/ 100",
)

print(
    "Exact filtered-turn reproductions:",
    exact_filtered_turn_matches,
    "/ 100",
)

print(
    "Exact local-feature reproductions:",
    exact_local_feature_matches,
    "/ 100",
)

print(
    "Exact global-feature reproductions:",
    exact_global_feature_matches,
    "/ 100",
)

Auditing silent cases:   0%|          | 0/100 [00:00<?, ?it/s]

SILENT-PARTNER AUDIT PASSED
Silent cases: 100
Unique Participant A: 100
Unique silent Participant B: 43
Silent B usage distribution: {2: 29, 3: 14}
Verified empty metadata:vad: 100 / 100
Empty filtered B turn lists: 100 / 100
Exact filtered-turn reproductions: 100 / 100
Exact local-feature reproductions: 100 / 100
Exact global-feature reproductions: 100 / 100


In [ ]:
# ============================================================
# INSPECT ONE COMPLETE SAVED SILENT CASE
# ============================================================

def inspect_silent_partner_case(
    case_index=0,
):

    case = (
        consolidation_silent_partner_cases[
            int(case_index)
        ]
    )


    print("=" * 88)
    print("CASE:", case["case_id"])
    print("=" * 88)


    print(
        "Source group:",
        case["source_group_id"],
    )


    print(
        "Participant A:",
        case["A_source_conversation"],
        case[
            "participant_A"
        ]["participant_id"],
    )


    print(
        "Silent Participant B:",
        case["B_source_conversation"],
        case[
            "participant_B"
        ]["participant_id"],
    )


    print(
        "Silent B usage count:",
        case["silent_B_usage_count"],
    )


    print(
        "\nParticipant A filtered turns:"
    )

    display(
        pd.DataFrame(
            case[
                "participant_A_filtered_turns"
            ]
        )
    )


    print(
        "\nParticipant B filtered turns:"
    )

    display(
        pd.DataFrame(
            case[
                "participant_B_filtered_turns"
            ]
        )
    )


    print(
        "\nLocal temporal features:"
    )

    print(
        json.dumps(
            case[
                "local_temporal_features"
            ],
            indent=2,
            ensure_ascii=False,
        )
    )


    print(
        "\nGlobal shift features:"
    )

    print(
        json.dumps(
            case[
                "global_shift_features"
            ],
            indent=2,
            ensure_ascii=False,
        )
    )


    print(
        "\nComplete saved case record:"
    )

    print(
        json.dumps(
            case,
            indent=2,
            ensure_ascii=False,
        )
    )


    return case


inspect_silent_partner_case(
    case_index=0
)

CASE: consolidation_silent_partner_000
Source group: heldout_source_000
Participant A: V03_S0148_I00000135 P1319
Silent Participant B: V01_S0173_I00001235 P1318
Silent B usage count: 3

Participant A filtered turns:


,start,end
0,9.89,10.78
1,11.65,18.24
2,25.38,27.71
3,32.83,33.76
4,35.65,44.83
5,68.67,75.01
6,75.81,76.67
7,78.75,84.67
8,85.57,90.49



Participant B filtered turns:


""



Local temporal features:
{
  "clean_overlap_seconds": 0.0,
  "clean_overlap_percent": 0.0,
  "signed_strict_offsets_seconds": [],
  "num_signed_strict_offsets": 0,
  "offset_mean_seconds": null,
  "offset_median_seconds": null,
  "offset_max_seconds": null,
  "offset_p75_seconds": null,
  "offset_p90_seconds": null,
  "num_offsets_above_1_5_seconds": 0,
  "percent_offsets_above_1_5_seconds": null
}

Global shift features:
{
  "best_B_correction_shift_seconds": -0.0,
  "estimated_B_lateness_seconds": 0.0,
  "alignment_score_gain_vs_zero": 0.0,
  "best_num_bilateral_events": 0,
  "best_event_coverage": 0.0
}

Complete saved case record:
{
  "case_id": "consolidation_silent_partner_000",
  "source_group_id": "heldout_source_000",
  "pair_index": 0,
  "gold_binary_label": "ANOMALOUS",
  "gold_anomaly_type": "silent_partner",
  "case_variant": "silent_partner",
  "pairing_type": "heldout_A_plus_verified_silent_B",
  "participant_A": {
    "conversation_id": "V03_S0148_I00000135",
    "part

{'case_id': 'consolidation_silent_partner_000',
 'source_group_id': 'heldout_source_000',
 'pair_index': 0,
 'gold_binary_label': 'ANOMALOUS',
 'gold_anomaly_type': 'silent_partner',
 'case_variant': 'silent_partner',
 'pairing_type': 'heldout_A_plus_verified_silent_B',
 'participant_A': {'conversation_id': 'V03_S0148_I00000135',
  'participant_id': 'P1319',
  'metadata_path': '/content/drive/MyDrive/seamless_download/data/V03_S0148_I00000135/P1319/V03_S0148_I00000135_P1319.json',
  'num_raw_vad_entries': 21},
 'participant_B': {'conversation_id': 'V01_S0173_I00001235',
  'participant_id': 'P1318',
  'metadata_path': '/content/drive/MyDrive/seamless_download/data/silent/V01_S0173_I00001235/P1318/V01_S0173_I00001235_P1318.json',
  'video_path': '/content/drive/MyDrive/seamless_download/data/silent/V01_S0173_I00001235/P1318/V01_S0173_I00001235_P1318.mp4',
  'num_raw_vad_entries': 0,
  'verified_empty_vad': True,
  'is_silent_source': True},
 'A_source_conversation': 'V03_S0148_I00000135'

# ΝΕΧΤ STEP, COARSE AND FOCUSED SUMMARIES

In [ ]:
# ============================================================
# STEP 5 — EXISTING SEMANTIC-SUMMARY COVERAGE
#
# Load:
# - 100 NORMAL
# - 100 WRONG_PARTNER
# - 100 LAG
# - 100 SILENT_PARTNER
#
# Load the old coarse/focused summary caches.
# No Qwen inference is performed in this step.
# ============================================================

from collections import Counter, defaultdict
from pathlib import Path

import copy
import json

import pandas as pd
from IPython.display import display


# ============================================================
# EXACT CACHE PATHS FROM THE OLD WRONG-PARTNER NOTEBOOK
# ============================================================

OLD_SEMANTIC_RUN_ROOT = (
    Path("/content/drive/MyDrive/qwen_strategy2")
    / "normal_vs_wrong_balanced_100_100_seed42_v1"
)

OLD_SEMANTIC_OUT_DIR = (
    OLD_SEMANTIC_RUN_ROOT
    / "outputs"
)


COARSE_SUMMARY_CACHE_PATH = (
    OLD_SEMANTIC_OUT_DIR
    / "participant_segment_analyses.json"
)


FOCUSED_SUMMARY_CACHE_PATH = (
    OLD_SEMANTIC_OUT_DIR
    / "focused_participant_semantic_summaries_ALL200_v1.json"
)


assert COARSE_SUMMARY_CACHE_PATH.exists(), (
    "Coarse summary cache not found: "
    f"{COARSE_SUMMARY_CACHE_PATH}"
)


assert FOCUSED_SUMMARY_CACHE_PATH.exists(), (
    "Focused summary cache not found: "
    f"{FOCUSED_SUMMARY_CACHE_PATH}"
)


# ============================================================
# CURRENT CONSOLIDATION CASE PATHS
# ============================================================

DEFAULT_NORMAL_WRONG_PATH = (
    OUT_DIR
    / "consolidation_normal_wrong_cases_with_lag_features.json"
)


DEFAULT_LAG_PATH = (
    OUT_DIR
    / "consolidation_lag_cases_50_lag2_50_lag3_with_features.json"
)


DEFAULT_SILENT_PATH = (
    OUT_DIR
    / "consolidation_silent_partner_cases_with_lag_features.json"
)


def load_json_list(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Case artifact not found: {path}"
        )

    data = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )

    if not isinstance(data, list):
        raise TypeError(
            f"Expected a JSON list in {path}"
        )

    return data


def load_cases_from_memory_or_disk(
    variable_names,
    path_candidates,
):
    """
    Prefer an already-created in-memory case list.
    Otherwise load the first existing artifact.
    """

    for variable_name in variable_names:

        value = globals().get(
            variable_name
        )

        if (
            isinstance(value, list)
            and len(value) > 0
        ):
            print(
                f"Loaded {len(value)} cases "
                f"from variable: {variable_name}"
            )

            return copy.deepcopy(value)


    for path in path_candidates:

        if path is None:
            continue

        path = Path(path)

        if path.exists():

            value = load_json_list(
                path
            )

            print(
                f"Loaded {len(value)} cases "
                f"from: {path}"
            )

            return value


    raise FileNotFoundError(
        "Could not find the required case list "
        "either in memory or on disk."
    )


normal_wrong_path_candidates = [
    globals().get(
        "CLEAN_NORMAL_WRONG_CASES_PATH"
    ),
    DEFAULT_NORMAL_WRONG_PATH,
]


lag_path_candidates = [
    globals().get(
        "CONSOLIDATION_LAG_CASES_PATH"
    ),
    DEFAULT_LAG_PATH,
]


silent_path_candidates = [
    globals().get(
        "CONSOLIDATION_SILENT_CASES_PATH"
    ),
    DEFAULT_SILENT_PATH,
]


current_normal_wrong_cases = (
    load_cases_from_memory_or_disk(
        variable_names=[
            "clean_normal_wrong_cases",
        ],
        path_candidates=(
            normal_wrong_path_candidates
        ),
    )
)


current_lag_cases = (
    load_cases_from_memory_or_disk(
        variable_names=[
            "consolidation_lag_cases",
        ],
        path_candidates=(
            lag_path_candidates
        ),
    )
)


current_silent_cases = (
    load_cases_from_memory_or_disk(
        variable_names=[
            "consolidation_silent_partner_cases",
        ],
        path_candidates=(
            silent_path_candidates
        ),
    )
)


current_normal_cases = [
    case
    for case in current_normal_wrong_cases
    if case["case_variant"] == "normal"
]


current_wrong_cases = [
    case
    for case in current_normal_wrong_cases
    if case["case_variant"] == "wrong_partner"
]


assert len(current_normal_cases) == 100
assert len(current_wrong_cases) == 100
assert len(current_lag_cases) == 100
assert len(current_silent_cases) == 100


print("=" * 72)
print("CURRENT CONSOLIDATION CASES LOADED")
print("=" * 72)

print("NORMAL:", len(current_normal_cases))
print("WRONG_PARTNER:", len(current_wrong_cases))
print("LAG:", len(current_lag_cases))
print("SILENT_PARTNER:", len(current_silent_cases))

print("\nCoarse cache:", COARSE_SUMMARY_CACHE_PATH)
print("Focused cache:", FOCUSED_SUMMARY_CACHE_PATH)

Loaded 200 cases from variable: clean_normal_wrong_cases
Loaded 100 cases from variable: consolidation_lag_cases
Loaded 100 cases from variable: consolidation_silent_partner_cases
CURRENT CONSOLIDATION CASES LOADED
NORMAL: 100
WRONG_PARTNER: 100
LAG: 100
SILENT_PARTNER: 100

Coarse cache: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/participant_segment_analyses.json
Focused cache: /content/drive/MyDrive/qwen_strategy2/normal_vs_wrong_balanced_100_100_seed42_v1/outputs/focused_participant_semantic_summaries_ALL200_v1.json


In [ ]:
# ============================================================
# LOAD AND INDEX THE EXISTING SUMMARY CACHES
# ============================================================

def load_json_dict(path):
    path = Path(path)

    data = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )

    if not isinstance(data, dict):
        raise TypeError(
            f"Expected a JSON object in {path}"
        )

    return data


coarse_summary_cache = load_json_dict(
    COARSE_SUMMARY_CACHE_PATH
)


focused_summary_cache = load_json_dict(
    FOCUSED_SUMMARY_CACHE_PATH
)


def build_summary_cache_index(
    cache,
):
    """
    Index by:

    conversation_id
    participant_id
    segment_idx
    """

    index = defaultdict(list)


    for cache_key, record in cache.items():

        if not isinstance(
            record,
            dict,
        ):
            continue


        try:

            identity = (
                str(
                    record[
                        "conversation_id"
                    ]
                ),

                str(
                    record[
                        "participant_id"
                    ]
                ),

                int(
                    record[
                        "segment_idx"
                    ]
                ),
            )

        except Exception:

            continue


        index[identity].append({
            "cache_key": cache_key,
            "record": record,
        })


    return index


COARSE_REQUIRED_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]


FOCUSED_REQUIRED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
]


def parsed_record_is_valid(
    record,
    required_fields,
):
    if not isinstance(record, dict):
        return False


    parsed = record.get(
        "parsed"
    )


    if not isinstance(
        parsed,
        dict,
    ):
        return False


    if parsed.get(
        "parse_error",
        False,
    ):
        return False


    return all(
        field in parsed
        for field in required_fields
    )


def latest_valid_summary_record(
    cache_index,
    conversation_id,
    participant_id,
    segment_idx,
    required_fields,
):
    identity = (
        str(conversation_id),
        str(participant_id),
        int(segment_idx),
    )


    matches = cache_index.get(
        identity,
        [],
    )


    valid_records = [
        match["record"]

        for match in matches

        if parsed_record_is_valid(
            match["record"],
            required_fields,
        )
    ]


    if not valid_records:
        return None


    return valid_records[-1]


coarse_cache_index = (
    build_summary_cache_index(
        coarse_summary_cache
    )
)


focused_cache_index = (
    build_summary_cache_index(
        focused_summary_cache
    )
)


print(
    "Raw coarse cache records:",
    len(coarse_summary_cache),
)

print(
    "Raw focused cache records:",
    len(focused_summary_cache),
)

print(
    "Indexed coarse identities:",
    len(coarse_cache_index),
)

print(
    "Indexed focused identities:",
    len(focused_cache_index),
)

Raw coarse cache records: 400
Raw focused cache records: 400
Indexed coarse identities: 400
Indexed focused identities: 400


In [ ]:
# ============================================================
# BUILD MODEL-FACING SUMMARY PROJECTIONS
# ============================================================

SEGMENT_NAMES = {
    0: "segment_0_0_to_60_seconds",
    1: "segment_1_60_to_120_seconds",
}


def safe_text(
    value,
    default="unclear",
):
    if value is None:
        return default

    value = str(value).strip()

    return (
        value
        if value
        else default
    )


def coarse_summary_projection(
    record,
):
    if record is None:
        return None


    parsed = record["parsed"]


    return {
        "speech_content_summary": (
            safe_text(
                parsed.get(
                    "speech_content_summary"
                )
            )
        ),

        "apparent_topic": (
            safe_text(
                parsed.get(
                    "apparent_topic"
                )
            )
        ),
    }


FOCUSED_OUTPUT_FIELDS = [
    "speaks",
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


def focused_summary_projection(
    record,
):
    if record is None:
        return None


    parsed = record["parsed"]


    result = {
        field: parsed.get(field)

        for field
        in FOCUSED_OUTPUT_FIELDS
    }


    for field in [
        "detailed_speech_summary",
        "main_topic",
        "summary_specificity",
        "unclear_content",
    ]:

        result[field] = safe_text(
            result.get(field)
        )


    for field in [
        "secondary_topics",
        "key_semantic_details",
    ]:

        if not isinstance(
            result.get(field),
            list,
        ):

            result[field] = []


    if not isinstance(
        result.get("speaks"),
        bool,
    ):

        result["speaks"] = None


    try:

        result["confidence"] = float(
            result.get(
                "confidence",
                0.0,
            )
        )

    except Exception:

        result["confidence"] = 0.0


    return result


def participant_signature(
    participant,
):
    return (
        str(
            participant[
                "conversation_id"
            ]
        ),

        str(
            participant[
                "participant_id"
            ]
        ),
    )


def case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith("lag_"):
        return "lag"


    return variant


def get_existing_segment_summaries(
    participant,
    segment_idx,
    source_kind="normal",
):
    """
    source_kind='normal':
        Search the old normal/wrong summary caches.

    source_kind='silent':
        Do not reuse the old normal/wrong cache.

    The old cache was not source-aware and was created only
    from the normal/wrong benchmark. Silent participants are
    therefore treated as missing until generated explicitly
    from the silent-folder videos.
    """

    conversation_id = (
        participant[
            "conversation_id"
        ]
    )


    participant_id = (
        participant[
            "participant_id"
        ]
    )


    if source_kind == "silent":

        coarse_record = None
        focused_record = None

    else:

        coarse_record = (
            latest_valid_summary_record(
                cache_index=(
                    coarse_cache_index
                ),

                conversation_id=(
                    conversation_id
                ),

                participant_id=(
                    participant_id
                ),

                segment_idx=(
                    segment_idx
                ),

                required_fields=(
                    COARSE_REQUIRED_FIELDS
                ),
            )
        )


        focused_record = (
            latest_valid_summary_record(
                cache_index=(
                    focused_cache_index
                ),

                conversation_id=(
                    conversation_id
                ),

                participant_id=(
                    participant_id
                ),

                segment_idx=(
                    segment_idx
                ),

                required_fields=(
                    FOCUSED_REQUIRED_FIELDS
                ),
            )
        )


    return {
        "coarse_summary": (
            coarse_summary_projection(
                coarse_record
            )
        ),

        "focused_summary": (
            focused_summary_projection(
                focused_record
            )
        ),
    }


semantic_coverage_rows = []


def enrich_case_from_existing_caches(
    case,
):
    """
    Enrich NORMAL, WRONG_PARTNER or SILENT_PARTNER.

    For SILENT_PARTNER:
    - A is a normal held-out participant.
    - B is a silent-folder participant and is forced missing.
    """

    enriched_case = copy.deepcopy(
        case
    )


    family = case_family(
        case
    )


    semantic_summaries = {}


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant = case[role]


        is_silent_B = (
            family == "silent_partner"
            and role == "participant_B"
        )


        source_kind = (
            "silent"
            if is_silent_B
            else "normal"
        )


        participant_segments = {}


        for segment_idx in [
            0,
            1,
        ]:

            segment_name = (
                SEGMENT_NAMES[
                    segment_idx
                ]
            )


            summaries = (
                get_existing_segment_summaries(
                    participant=participant,
                    segment_idx=segment_idx,
                    source_kind=source_kind,
                )
            )


            participant_segments[
                segment_name
            ] = summaries


            semantic_coverage_rows.append({
                "case_id": (
                    case["case_id"]
                ),

                "source_group_id": (
                    case["source_group_id"]
                ),

                "case_family": family,

                "case_variant": (
                    case[
                        "case_variant"
                    ]
                ),

                "participant_role": role,

                "source_kind": (
                    source_kind
                ),

                "conversation_id": (
                    participant[
                        "conversation_id"
                    ]
                ),

                "participant_id": (
                    participant[
                        "participant_id"
                    ]
                ),

                "segment_idx": (
                    segment_idx
                ),

                "segment_start_seconds": (
                    0
                    if segment_idx == 0
                    else 60
                ),

                "segment_end_seconds": (
                    60
                    if segment_idx == 0
                    else 120
                ),

                "coarse_available": (
                    summaries[
                        "coarse_summary"
                    ]
                    is not None
                ),

                "focused_available": (
                    summaries[
                        "focused_summary"
                    ]
                    is not None
                ),

                "summary_strategy": (
                    "existing_cache_lookup"
                    if source_kind == "normal"
                    else "silent_forced_missing"
                ),
            })


        semantic_summaries[
            role
        ] = participant_segments


    available_coarse = sum(
        segment[
            "coarse_summary"
        ]
        is not None

        for participant_segments
        in semantic_summaries.values()

        for segment
        in participant_segments.values()
    )


    available_focused = sum(
        segment[
            "focused_summary"
        ]
        is not None

        for participant_segments
        in semantic_summaries.values()

        for segment
        in participant_segments.values()
    )


    enriched_case[
        "semantic_summaries"
    ] = semantic_summaries


    enriched_case[
        "semantic_summary_coverage"
    ] = {
        "expected_coarse_summaries": 4,
        "available_coarse_summaries": (
            available_coarse
        ),
        "missing_coarse_summaries": (
            4 - available_coarse
        ),

        "expected_focused_summaries": 4,
        "available_focused_summaries": (
            available_focused
        ),
        "missing_focused_summaries": (
            4 - available_focused
        ),

        "all_summaries_available": (
            available_coarse == 4
            and available_focused == 4
        ),
    }


    enriched_case[
        "semantic_summary_source"
    ] = {
        "strategy": (
            "existing_cache_lookup"
        ),

        "coarse_cache": str(
            COARSE_SUMMARY_CACHE_PATH
        ),

        "focused_cache": str(
            FOCUSED_SUMMARY_CACHE_PATH
        ),

        "silent_B_cache_reuse_disabled": (
            family == "silent_partner"
        ),
    }


    return enriched_case


semantic_normal_cases = [
    enrich_case_from_existing_caches(
        case
    )
    for case
    in current_normal_cases
]


semantic_wrong_cases = [
    enrich_case_from_existing_caches(
        case
    )
    for case
    in current_wrong_cases
]


semantic_silent_cases = [
    enrich_case_from_existing_caches(
        case
    )
    for case
    in current_silent_cases
]


print(
    "Enriched NORMAL cases:",
    len(semantic_normal_cases),
)

print(
    "Enriched WRONG_PARTNER cases:",
    len(semantic_wrong_cases),
)

print(
    "Enriched SILENT_PARTNER cases:",
    len(semantic_silent_cases),
)

Enriched NORMAL cases: 100
Enriched WRONG_PARTNER cases: 100
Enriched SILENT_PARTNER cases: 100


In [ ]:
# ============================================================
# LAG CASES INHERIT EXACTLY THE NORMAL SEMANTIC SUMMARIES
#
# No additional cache lookup is performed.
# ============================================================

semantic_normal_by_group = {
    case["source_group_id"]: case

    for case
    in semantic_normal_cases
}


assert len(
    semantic_normal_by_group
) == 100


semantic_lag_cases = []


for lag_case in current_lag_cases:

    source_group_id = (
        lag_case[
            "source_group_id"
        ]
    )


    assert (
        source_group_id
        in semantic_normal_by_group
    )


    corresponding_normal = (
        semantic_normal_by_group[
            source_group_id
        ]
    )


    # Participant identities must be exactly the same.
    assert participant_signature(
        lag_case["participant_A"]
    ) == participant_signature(
        corresponding_normal[
            "participant_A"
        ]
    )


    assert participant_signature(
        lag_case["participant_B"]
    ) == participant_signature(
        corresponding_normal[
            "participant_B"
        ]
    )


    enriched_lag = copy.deepcopy(
        lag_case
    )


    enriched_lag[
        "semantic_summaries"
    ] = copy.deepcopy(
        corresponding_normal[
            "semantic_summaries"
        ]
    )


    enriched_lag[
        "semantic_summary_coverage"
    ] = copy.deepcopy(
        corresponding_normal[
            "semantic_summary_coverage"
        ]
    )


    enriched_lag[
        "semantic_summary_source"
    ] = {
        "strategy": (
            "inherited_exactly_from_normal_case"
        ),

        "normal_source_case_id": (
            corresponding_normal[
                "case_id"
            ]
        ),

        "reason": (
            "Synthetic temporal lag does not alter "
            "participant video semantics."
        ),
    }


    semantic_lag_cases.append(
        enriched_lag
    )


    # Add lag coverage rows for reporting only.
    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant = (
            lag_case[role]
        )


        for segment_idx in [
            0,
            1,
        ]:

            segment_name = (
                SEGMENT_NAMES[
                    segment_idx
                ]
            )


            summaries = (
                enriched_lag[
                    "semantic_summaries"
                ][role][segment_name]
            )


            semantic_coverage_rows.append({
                "case_id": (
                    lag_case[
                        "case_id"
                    ]
                ),

                "source_group_id": (
                    source_group_id
                ),

                "case_family": "lag",

                "case_variant": (
                    lag_case[
                        "case_variant"
                    ]
                ),

                "participant_role": role,

                "source_kind": "normal",

                "conversation_id": (
                    participant[
                        "conversation_id"
                    ]
                ),

                "participant_id": (
                    participant[
                        "participant_id"
                    ]
                ),

                "segment_idx": (
                    segment_idx
                ),

                "segment_start_seconds": (
                    0
                    if segment_idx == 0
                    else 60
                ),

                "segment_end_seconds": (
                    60
                    if segment_idx == 0
                    else 120
                ),

                "coarse_available": (
                    summaries[
                        "coarse_summary"
                    ]
                    is not None
                ),

                "focused_available": (
                    summaries[
                        "focused_summary"
                    ]
                    is not None
                ),

                "summary_strategy": (
                    "inherited_from_normal"
                ),
            })


assert len(
    semantic_lag_cases
) == 100


print(
    "LAG cases enriched by exact NORMAL inheritance:",
    len(semantic_lag_cases),
)

LAG cases enriched by exact NORMAL inheritance: 100


In [ ]:
# ============================================================
# BUILD COVERAGE REPORT
# ============================================================

semantic_coverage_df = (
    pd.DataFrame(
        semantic_coverage_rows
    )
    .sort_values(
        [
            "case_family",
            "case_id",
            "participant_role",
            "segment_idx",
        ]
    )
    .reset_index(drop=True)
)


coverage_summary_df = (
    semantic_coverage_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        participant_segment_rows=(
            "case_id",
            "size",
        ),

        coarse_available=(
            "coarse_available",
            "sum",
        ),

        focused_available=(
            "focused_available",
            "sum",
        ),
    )
)


coverage_summary_df[
    "coarse_missing"
] = (
    coverage_summary_df[
        "participant_segment_rows"
    ]
    - coverage_summary_df[
        "coarse_available"
    ]
)


coverage_summary_df[
    "focused_missing"
] = (
    coverage_summary_df[
        "participant_segment_rows"
    ]
    - coverage_summary_df[
        "focused_available"
    ]
)


print("=" * 80)
print("CASE-LEVEL SEMANTIC CACHE COVERAGE")
print("=" * 80)

display(
    coverage_summary_df
)


print(
    "\nDetailed availability combinations:"
)

display(
    semantic_coverage_df[
        [
            "case_family",
            "coarse_available",
            "focused_available",
        ]
    ]
    .value_counts()
    .reset_index(
        name="count"
    )
)


# ============================================================
# BUILD UNIQUE MISSING PARTICIPANT-SEGMENT TASKS
#
# NORMAL, WRONG and SILENT are checked directly.
# LAG is excluded because it inherits NORMAL summaries.
# ============================================================

participant_registry = {}

participant_usage = defaultdict(
    set
)


def register_case_participants(
    case,
):
    family = case_family(
        case
    )


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant = case[role]


        source_kind = (
            "silent"

            if (
                family == "silent_partner"
                and role == "participant_B"
            )

            else "normal"
        )


        registry_key = (
            source_kind,
            str(
                participant[
                    "conversation_id"
                ]
            ),
            str(
                participant[
                    "participant_id"
                ]
            ),
        )


        participant_registry[
            registry_key
        ] = participant


        participant_usage[
            registry_key
        ].add(
            family
        )


for case in (
    current_normal_cases
    + current_wrong_cases
    + current_silent_cases
):

    register_case_participants(
        case
    )


def resolve_participant_video_path(
    participant,
    source_kind,
):
    """
    Find the original full participant video.
    """

    for key in [
        "video_path",
        "mp4",
    ]:

        candidate = (
            participant.get(key)
        )

        if candidate:

            candidate = Path(
                candidate
            )

            if candidate.exists():
                return str(candidate)


    source_root = (
        SILENT_ROOT
        if source_kind == "silent"
        else DATA_ROOT
    )


    participant_dir = (
        source_root
        / str(
            participant[
                "conversation_id"
            ]
        )
        / str(
            participant[
                "participant_id"
            ]
        )
    )


    candidates = []


    for extension in [
        "*.mp4",
        "*.mov",
        "*.mkv",
        "*.webm",
    ]:

        candidates.extend(
            participant_dir.glob(
                extension
            )
        )


    candidates = sorted(
        set(candidates)
    )


    return (
        str(candidates[0])
        if candidates
        else None
    )


missing_summary_tasks = []


for (
    source_kind,
    conversation_id,
    participant_id,
), participant in sorted(
    participant_registry.items()
):

    for segment_idx in [
        0,
        1,
    ]:

        summaries = (
            get_existing_segment_summaries(
                participant=participant,
                segment_idx=segment_idx,
                source_kind=source_kind,
            )
        )


        missing_coarse = (
            summaries[
                "coarse_summary"
            ]
            is None
        )


        missing_focused = (
            summaries[
                "focused_summary"
            ]
            is None
        )


        if not (
            missing_coarse
            or missing_focused
        ):
            continue


        registry_key = (
            source_kind,
            conversation_id,
            participant_id,
        )


        missing_summary_tasks.append({
            "source_kind": (
                source_kind
            ),

            "conversation_id": (
                conversation_id
            ),

            "participant_id": (
                participant_id
            ),

            "segment_idx": (
                segment_idx
            ),

            "segment_start_seconds": (
                0
                if segment_idx == 0
                else 60
            ),

            "segment_duration_seconds": 60,

            "video_path": (
                resolve_participant_video_path(
                    participant=participant,
                    source_kind=source_kind,
                )
            ),

            "missing_coarse": (
                missing_coarse
            ),

            "missing_focused": (
                missing_focused
            ),

            "used_in_case_families": sorted(
                participant_usage[
                    registry_key
                ]
            ),
        })


print("=" * 80)
print("UNIQUE PARTICIPANT-SEGMENT GENERATION TASKS")
print("=" * 80)

print(
    "Unique participants requiring any new summary:",
    len({
        (
            task["source_kind"],
            task["conversation_id"],
            task["participant_id"],
        )

        for task
        in missing_summary_tasks
    }),
)

print(
    "Participant-segment tasks requiring anything:",
    len(
        missing_summary_tasks
    ),
)

print(
    "Missing coarse tasks:",
    sum(
        task["missing_coarse"]

        for task
        in missing_summary_tasks
    ),
)

print(
    "Missing focused tasks:",
    sum(
        task["missing_focused"]

        for task
        in missing_summary_tasks
    ),
)

print(
    "Silent participant-segment tasks:",
    sum(
        task["source_kind"] == "silent"

        for task
        in missing_summary_tasks
    ),
)

print(
    "Tasks without a resolvable video path:",
    sum(
        task["video_path"] is None

        for task
        in missing_summary_tasks
    ),
)


missing_summary_tasks_df = (
    pd.DataFrame(
        missing_summary_tasks
    )
)


display(
    missing_summary_tasks_df
)

CASE-LEVEL SEMANTIC CACHE COVERAGE


,case_family,participant_segment_rows,coarse_available,focused_available,coarse_missing,focused_missing
0,lag,400,212,212,188,188
1,normal,400,212,212,188,188
2,silent_partner,400,106,106,294,294
3,wrong_partner,400,212,212,188,188



Detailed availability combinations:


,case_family,coarse_available,focused_available,count
0,silent_partner,False,False,294
1,lag,True,True,212
2,wrong_partner,True,True,212
3,normal,True,True,212
4,normal,False,False,188
5,lag,False,False,188
6,wrong_partner,False,False,188
7,silent_partner,True,True,106


UNIQUE PARTICIPANT-SEGMENT GENERATION TASKS
Unique participants requiring any new summary: 137
Participant-segment tasks requiring anything: 274
Missing coarse tasks: 274
Missing focused tasks: 274
Silent participant-segment tasks: 86
Tasks without a resolvable video path: 0


,source_kind,conversation_id,participant_id,segment_idx,segment_start_seconds,segment_duration_seconds,video_path,missing_coarse,missing_focused,used_in_case_families
0,normal,V01_S0223_I00000137,P1505,0,0,60,/content/drive/MyDrive/seamless_download/data/...,True,True,"[normal, silent_partner, wrong_partner]"
1,normal,V01_S0223_I00000137,P1505,1,60,60,/content/drive/MyDrive/seamless_download/data/...,True,True,"[normal, silent_partner, wrong_partner]"
2,normal,V01_S0223_I00000137,P1506,0,0,60,/content/drive/MyDrive/seamless_download/data/...,True,True,"[normal, wrong_partner]"
3,normal,V01_S0223_I00000137,P1506,1,60,60,/content/drive/MyDrive/seamless_download/data/...,True,True,"[normal, wrong_partner]"
4,normal,V01_S0223_I00000138,P1505,0,0,60,/content/drive/MyDrive/seamless_download/data/...,True,True,"[normal, wrong_partner]"
...,...,...,...,...,...,...,...,...,...,...
269,silent,V01_S1545_I00000138,P2518,1,60,60,/content/drive/MyDrive/seamless_download/data/...,True,True,[silent_partner]
270,silent,V03_S1088_I00000498,P1420,0,0,60,/content/drive/MyDrive/seamless_download/data/...,True,True,[silent_partner]
271,silent,V03_S1088_I00000498,P1420,1,60,60,/content/drive/MyDrive/seamless_download/data/...,True,True,[silent_partner]
272,silent,V03_S1088_I00000498,P1766,0,0,60,/content/drive/MyDrive/seamless_download/data/...,True,True,[silent_partner]


In [ ]:
# ============================================================
# SAVE CASES ENRICHED WITH EXISTING SEMANTIC SUMMARIES
# ============================================================

ENRICHED_NORMAL_WRONG_PATH = (
    OUT_DIR
    / (
        "consolidation_normal_wrong_cases_with_"
        "temporal_and_existing_semantic_summaries.json"
    )
)


ENRICHED_LAG_PATH = (
    OUT_DIR
    / (
        "consolidation_lag_cases_with_"
        "temporal_and_existing_semantic_summaries.json"
    )
)


ENRICHED_SILENT_PATH = (
    OUT_DIR
    / (
        "consolidation_silent_partner_cases_with_"
        "temporal_and_existing_semantic_summaries.json"
    )
)


SEMANTIC_COVERAGE_CSV_PATH = (
    OUT_DIR
    / "consolidation_semantic_summary_coverage.csv"
)


MISSING_SUMMARY_TASKS_PATH = (
    OUT_DIR
    / "consolidation_missing_semantic_summary_tasks.json"
)


SEMANTIC_COVERAGE_MANIFEST_PATH = (
    OUT_DIR
    / "consolidation_semantic_summary_coverage_manifest.json"
)


enriched_normal_wrong_cases = (
    semantic_normal_cases
    + semantic_wrong_cases
)


ENRICHED_NORMAL_WRONG_PATH.write_text(
    json.dumps(
        enriched_normal_wrong_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


ENRICHED_LAG_PATH.write_text(
    json.dumps(
        semantic_lag_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


ENRICHED_SILENT_PATH.write_text(
    json.dumps(
        semantic_silent_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


semantic_coverage_df.to_csv(
    SEMANTIC_COVERAGE_CSV_PATH,
    index=False,
)


MISSING_SUMMARY_TASKS_PATH.write_text(
    json.dumps(
        missing_summary_tasks,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


semantic_coverage_manifest = {
    "old_summary_caches": {
        "coarse": str(
            COARSE_SUMMARY_CACHE_PATH
        ),

        "focused": str(
            FOCUSED_SUMMARY_CACHE_PATH
        ),
    },

    "segments": {
        "segment_0": [
            0,
            60,
        ],

        "segment_1": [
            60,
            120,
        ],
    },

    "case_counts": {
        "normal": len(
            semantic_normal_cases
        ),

        "wrong_partner": len(
            semantic_wrong_cases
        ),

        "lag": len(
            semantic_lag_cases
        ),

        "silent_partner": len(
            semantic_silent_cases
        ),
    },

    "lag_summary_strategy": (
        "Inherited exactly from the corresponding "
        "NORMAL source-group case."
    ),

    "silent_summary_strategy": (
        "Old normal/wrong cache reuse is disabled "
        "for silent Participant B. Silent summaries "
        "must be generated from silent-folder videos."
    ),

    "missing_unique_participant_segment_tasks": (
        len(
            missing_summary_tasks
        )
    ),

    "missing_coarse_tasks": (
        sum(
            task["missing_coarse"]

            for task
            in missing_summary_tasks
        )
    ),

    "missing_focused_tasks": (
        sum(
            task["missing_focused"]

            for task
            in missing_summary_tasks
        )
    ),

    "output_files": {
        "normal_wrong_enriched": str(
            ENRICHED_NORMAL_WRONG_PATH
        ),

        "lag_enriched": str(
            ENRICHED_LAG_PATH
        ),

        "silent_enriched": str(
            ENRICHED_SILENT_PATH
        ),

        "coverage_csv": str(
            SEMANTIC_COVERAGE_CSV_PATH
        ),

        "missing_tasks": str(
            MISSING_SUMMARY_TASKS_PATH
        ),
    },
}


SEMANTIC_COVERAGE_MANIFEST_PATH.write_text(
    json.dumps(
        semantic_coverage_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 80)
print("EXISTING SEMANTIC SUMMARIES SAVED INTO CASES")
print("=" * 80)

print(
    "NORMAL + WRONG:",
    ENRICHED_NORMAL_WRONG_PATH,
)

print(
    "LAG:",
    ENRICHED_LAG_PATH,
)

print(
    "SILENT:",
    ENRICHED_SILENT_PATH,
)

print(
    "Coverage CSV:",
    SEMANTIC_COVERAGE_CSV_PATH,
)

print(
    "Missing generation tasks:",
    MISSING_SUMMARY_TASKS_PATH,
)

print(
    "Coverage manifest:",
    SEMANTIC_COVERAGE_MANIFEST_PATH,
)

EXISTING SEMANTIC SUMMARIES SAVED INTO CASES
NORMAL + WRONG: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_normal_wrong_cases_with_temporal_and_existing_semantic_summaries.json
LAG: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_lag_cases_with_temporal_and_existing_semantic_summaries.json
SILENT: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_silent_partner_cases_with_temporal_and_existing_semantic_summaries.json
Coverage CSV: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_semantic_summary_coverage.csv
Missing generation tasks: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_missing_semantic_summary_tasks.json
Coverage manifest: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_semantic_summary_coverage_manifest.json


In [ ]:
# ============================================================
# STRICT SEMANTIC ENRICHMENT AUDIT
# ============================================================

assert len(
    enriched_normal_wrong_cases
) == 200


assert len(
    semantic_lag_cases
) == 100


assert len(
    semantic_silent_cases
) == 100


for case in (
    enriched_normal_wrong_cases
    + semantic_lag_cases
    + semantic_silent_cases
):

    assert (
        "semantic_summaries"
        in case
    )


    assert (
        "semantic_summary_coverage"
        in case
    )


    assert set(
        case[
            "semantic_summaries"
        ]
    ) == {
        "participant_A",
        "participant_B",
    }


    for role in [
        "participant_A",
        "participant_B",
    ]:

        assert set(
            case[
                "semantic_summaries"
            ][role]
        ) == set(
            SEGMENT_NAMES.values()
        )


# ------------------------------------------------------------
# Every LAG case must have summaries exactly equal to NORMAL
# ------------------------------------------------------------

for lag_case in semantic_lag_cases:

    normal_case = (
        semantic_normal_by_group[
            lag_case[
                "source_group_id"
            ]
        ]
    )


    assert (
        lag_case[
            "semantic_summaries"
        ]
        == normal_case[
            "semantic_summaries"
        ]
    )


# ------------------------------------------------------------
# Silent B summaries must currently be absent
# ------------------------------------------------------------

for silent_case in semantic_silent_cases:

    for segment_name in (
        SEGMENT_NAMES.values()
    ):

        silent_segment = (
            silent_case[
                "semantic_summaries"
            ][
                "participant_B"
            ][
                segment_name
            ]
        )


        assert (
            silent_segment[
                "coarse_summary"
            ]
            is None
        )


        assert (
            silent_segment[
                "focused_summary"
            ]
            is None
        )


# ------------------------------------------------------------
# No duplicate generation tasks
# ------------------------------------------------------------

missing_task_signatures = [
    (
        task[
            "source_kind"
        ],

        task[
            "conversation_id"
        ],

        task[
            "participant_id"
        ],

        task[
            "segment_idx"
        ],
    )

    for task
    in missing_summary_tasks
]


assert len(
    missing_task_signatures
) == len(
    set(
        missing_task_signatures
    )
)


print("=" * 80)
print("SEMANTIC COVERAGE AUDIT PASSED")
print("=" * 80)

print(
    "NORMAL cases with semantic fields:",
    len(semantic_normal_cases),
)

print(
    "WRONG cases with semantic fields:",
    len(semantic_wrong_cases),
)

print(
    "LAG cases inheriting NORMAL summaries:",
    len(semantic_lag_cases),
)

print(
    "SILENT cases with explicit missing B summaries:",
    len(semantic_silent_cases),
)

print(
    "Unique missing participant-segment tasks:",
    len(missing_summary_tasks),
)

print(
    "Missing coarse generations:",
    sum(
        task["missing_coarse"]

        for task
        in missing_summary_tasks
    ),
)

print(
    "Missing focused generations:",
    sum(
        task["missing_focused"]

        for task
        in missing_summary_tasks
    ),
)

SEMANTIC COVERAGE AUDIT PASSED
NORMAL cases with semantic fields: 100
WRONG cases with semantic fields: 100
LAG cases inheriting NORMAL summaries: 100
SILENT cases with explicit missing B summaries: 100
Unique missing participant-segment tasks: 274
Missing coarse generations: 274
Missing focused generations: 274


In [ ]:
missing_tasks_df = pd.DataFrame(
    missing_summary_tasks
)

print(
    "Unique participant-segment tasks:",
    len(missing_tasks_df)
)

print(
    "Missing coarse generations:",
    int(
        missing_tasks_df[
            "missing_coarse"
        ].sum()
    )
)

print(
    "Missing focused generations:",
    int(
        missing_tasks_df[
            "missing_focused"
        ].sum()
    )
)

print("\nBreakdown by source:")

display(
    missing_tasks_df
    .groupby(
        "source_kind",
        as_index=False,
    )
    .agg(
        unique_participant_segment_tasks=(
            "segment_idx",
            "size",
        ),

        missing_coarse=(
            "missing_coarse",
            "sum",
        ),

        missing_focused=(
            "missing_focused",
            "sum",
        ),
    )
)

Unique participant-segment tasks: 274
Missing coarse generations: 274
Missing focused generations: 274

Breakdown by source:


,source_kind,unique_participant_segment_tasks,missing_coarse,missing_focused
0,normal,188,188,188
1,silent,86,86,86


In [ ]:
# ============================================================
# CASE-LEVEL COVERAGE BY PARTICIPANT ROLE
# ============================================================

role_coverage = (
    semantic_coverage_df
    .groupby(
        [
            "case_family",
            "participant_role",
        ],
        as_index=False,
    )
    .agg(
        participant_segment_rows=(
            "case_id",
            "size",
        ),

        coarse_available=(
            "coarse_available",
            "sum",
        ),

        focused_available=(
            "focused_available",
            "sum",
        ),
    )
)


role_coverage[
    "coarse_missing"
] = (
    role_coverage[
        "participant_segment_rows"
    ]
    - role_coverage[
        "coarse_available"
    ]
)


role_coverage[
    "focused_missing"
] = (
    role_coverage[
        "participant_segment_rows"
    ]
    - role_coverage[
        "focused_available"
    ]
)


display(
    role_coverage
)

,case_family,participant_role,participant_segment_rows,coarse_available,focused_available,coarse_missing,focused_missing
0,lag,participant_A,200,106,106,94,94
1,lag,participant_B,200,106,106,94,94
2,normal,participant_A,200,106,106,94,94
3,normal,participant_B,200,106,106,94,94
4,silent_partner,participant_A,200,106,106,94,94
5,silent_partner,participant_B,200,0,0,200,200
6,wrong_partner,participant_A,200,106,106,94,94
7,wrong_partner,participant_B,200,106,106,94,94


In [ ]:
# ============================================================
# PARTICIPANT-LEVEL COVERAGE
# ============================================================

participant_coverage = (
    semantic_coverage_df
    .groupby(
        [
            "case_family",
            "participant_role",
            "source_kind",
            "conversation_id",
            "participant_id",
        ],
        as_index=False,
    )
    .agg(
        coarse_segments_available=(
            "coarse_available",
            "sum",
        ),

        focused_segments_available=(
            "focused_available",
            "sum",
        ),
    )
)


def coverage_status(
    num_segments,
):
    if num_segments == 2:
        return "both_segments_available"

    if num_segments == 1:
        return "one_segment_available"

    return "no_segments_available"


participant_coverage[
    "coarse_status"
] = (
    participant_coverage[
        "coarse_segments_available"
    ].apply(
        coverage_status
    )
)


participant_coverage[
    "focused_status"
] = (
    participant_coverage[
        "focused_segments_available"
    ].apply(
        coverage_status
    )
)


print("NORMAL coarse coverage:")

display(
    participant_coverage[
        participant_coverage[
            "case_family"
        ]
        == "normal"
    ][
        [
            "participant_role",
            "coarse_status",
        ]
    ]
    .value_counts()
    .reset_index(
        name="num_participants"
    )
)


print("SILENT_PARTNER coarse coverage:")

display(
    participant_coverage[
        participant_coverage[
            "case_family"
        ]
        == "silent_partner"
    ][
        [
            "participant_role",
            "source_kind",
            "coarse_status",
        ]
    ]
    .value_counts()
    .reset_index(
        name="num_participants"
    )
)

NORMAL coarse coverage:


,participant_role,coarse_status,num_participants
0,participant_A,both_segments_available,53
1,participant_B,both_segments_available,53
2,participant_A,no_segments_available,47
3,participant_B,no_segments_available,47


SILENT_PARTNER coarse coverage:


,participant_role,source_kind,coarse_status,num_participants
0,participant_A,normal,both_segments_available,53
1,participant_A,normal,no_segments_available,47
2,participant_B,silent,no_segments_available,43


In [ ]:
# ============================================================
# INSPECT CASES WITH EXISTING COARSE + FOCUSED SUMMARIES
# ============================================================

import json
import pandas as pd
from IPython.display import display


SEMANTIC_CASES_BY_FAMILY = {
    "normal": semantic_normal_cases,
    "wrong_partner": semantic_wrong_cases,
    "lag": semantic_lag_cases,
    "silent_partner": semantic_silent_cases,
}


SEGMENT_ORDER = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]


def segment_has_coarse_and_focused(
    segment_record,
):
    """
    True only when this specific participant-segment
    has both summary types.
    """

    return (
        isinstance(segment_record, dict)
        and segment_record.get(
            "coarse_summary"
        ) is not None
        and segment_record.get(
            "focused_summary"
        ) is not None
    )


def case_semantic_status(
    case,
):
    """
    Count semantic-summary availability for one case.

    Each case expects:
        2 participants × 2 segments
        = 4 coarse summaries
        = 4 focused summaries
    """

    coarse_available = 0
    focused_available = 0
    both_available = 0


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_segments = (
            case[
                "semantic_summaries"
            ][role]
        )


        for segment_name in SEGMENT_ORDER:

            segment_record = (
                participant_segments[
                    segment_name
                ]
            )


            coarse_exists = (
                segment_record.get(
                    "coarse_summary"
                )
                is not None
            )


            focused_exists = (
                segment_record.get(
                    "focused_summary"
                )
                is not None
            )


            coarse_available += int(
                coarse_exists
            )


            focused_available += int(
                focused_exists
            )


            both_available += int(
                coarse_exists
                and focused_exists
            )


    return {
        "coarse_available": (
            coarse_available
        ),

        "focused_available": (
            focused_available
        ),

        "participant_segments_with_both": (
            both_available
        ),

        "complete_case": (
            coarse_available == 4
            and focused_available == 4
        ),

        "has_at_least_one_complete_segment": (
            both_available > 0
        ),
    }


# ============================================================
# BUILD INSPECTION TABLE
# ============================================================

semantic_case_inspection_rows = []


for case_family, cases in (
    SEMANTIC_CASES_BY_FAMILY.items()
):

    for case_index, case in enumerate(
        cases
    ):

        status = case_semantic_status(
            case
        )


        semantic_case_inspection_rows.append({
            "case_family": (
                case_family
            ),

            "case_index": (
                case_index
            ),

            "case_id": (
                case["case_id"]
            ),

            "source_group_id": (
                case["source_group_id"]
            ),

            "participant_A": (
                case[
                    "participant_A"
                ]["participant_id"]
            ),

            "participant_B": (
                case[
                    "participant_B"
                ]["participant_id"]
            ),

            "coarse_available": (
                status[
                    "coarse_available"
                ]
            ),

            "focused_available": (
                status[
                    "focused_available"
                ]
            ),

            "segments_with_both": (
                status[
                    "participant_segments_with_both"
                ]
            ),

            "complete_case": (
                status[
                    "complete_case"
                ]
            ),

            "has_any_complete_segment": (
                status[
                    "has_at_least_one_complete_segment"
                ]
            ),
        })


semantic_case_inspection_df = (
    pd.DataFrame(
        semantic_case_inspection_rows
    )
    .sort_values(
        [
            "case_family",
            "complete_case",
            "segments_with_both",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


print("=" * 80)
print("CASES WITH ALL 4 COARSE AND ALL 4 FOCUSED SUMMARIES")
print("=" * 80)


display(
    semantic_case_inspection_df[
        semantic_case_inspection_df[
            "complete_case"
        ]
    ]
)


print("\nComplete-case counts by family:")


display(
    semantic_case_inspection_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        complete_cases=(
            "complete_case",
            "sum",
        ),

        cases_with_any_complete_segment=(
            "has_any_complete_segment",
            "sum",
        ),
    )
)

CASES WITH ALL 4 COARSE AND ALL 4 FOCUSED SUMMARIES


,case_family,case_index,case_id,source_group_id,participant_A,participant_B,coarse_available,focused_available,segments_with_both,complete_case,has_any_complete_segment
0,lag,1,consolidation_lag_2sec_001,heldout_source_001,P1307A,P1308A,4,4,4,True,True
1,lag,4,consolidation_lag_3sec_004,heldout_source_004,P1305A,P1304A,4,4,4,True,True
2,lag,6,consolidation_lag_3sec_006,heldout_source_006,P1293A,P1294A,4,4,4,True,True
3,lag,7,consolidation_lag_2sec_007,heldout_source_007,P1304A,P1306A,4,4,4,True,True
4,lag,8,consolidation_lag_3sec_008,heldout_source_008,P1274A,P1272A,4,4,4,True,True
...,...,...,...,...,...,...,...,...,...,...,...
320,wrong_partner,78,consolidation_wrong_partner_078,heldout_source_078,P1304A,P1306A,4,4,4,True,True
321,wrong_partner,79,consolidation_wrong_partner_079,heldout_source_079,P1308A,P1309A,4,4,4,True,True
322,wrong_partner,81,consolidation_wrong_partner_081,heldout_source_081,P1274A,P1305A,4,4,4,True,True
323,wrong_partner,83,consolidation_wrong_partner_083,heldout_source_083,P1309A,P1309A,4,4,4,True,True



Complete-case counts by family:


,case_family,total_cases,complete_cases,cases_with_any_complete_segment
0,lag,100,53,53
1,normal,100,53,53
2,silent_partner,100,0,53
3,wrong_partner,100,25,81


In [ ]:
# ============================================================
# PRETTY INSPECTION OF ONE SEMANTIC CASE
# ============================================================

def inspect_semantic_case(
    case_family="normal",
    case_index=None,
    complete_only=True,
    candidate_rank=0,
    show_temporal_information=False,
):
    """
    Inspect one case and print its existing coarse and
    focused summaries for both participants and both segments.

    Parameters
    ----------
    case_family:
        normal, wrong_partner, lag, silent_partner

    case_index:
        Direct index inside the corresponding case list.
        When None, the function automatically selects a case.

    complete_only:
        When True, automatically select only a case that has
        all 4 coarse and all 4 focused summaries.

    candidate_rank:
        Select the first, second, third, etc. matching case.

    show_temporal_information:
        Also print filtered VAD turns and temporal features.
    """

    assert case_family in (
        SEMANTIC_CASES_BY_FAMILY
    ), (
        f"Unknown case family: {case_family}"
    )


    cases = (
        SEMANTIC_CASES_BY_FAMILY[
            case_family
        ]
    )


    if case_index is None:

        family_table = (
            semantic_case_inspection_df[
                semantic_case_inspection_df[
                    "case_family"
                ]
                == case_family
            ]
        )


        if complete_only:

            candidates = (
                family_table[
                    family_table[
                        "complete_case"
                    ]
                ]
            )

        else:

            candidates = (
                family_table[
                    family_table[
                        "has_any_complete_segment"
                    ]
                ]
            )


        if len(candidates) == 0:

            raise ValueError(
                f"No matching {case_family} cases found. "
                f"complete_only={complete_only}"
            )


        if candidate_rank >= len(
            candidates
        ):

            raise IndexError(
                f"candidate_rank={candidate_rank}, "
                f"but only {len(candidates)} "
                "matching cases exist."
            )


        selected_row = (
            candidates.iloc[
                candidate_rank
            ]
        )


        case_index = int(
            selected_row[
                "case_index"
            ]
        )


    case = cases[
        int(case_index)
    ]


    status = case_semantic_status(
        case
    )


    print("=" * 100)
    print("CASE:", case["case_id"])
    print("=" * 100)

    print(
        "Case family:",
        case_family,
    )

    print(
        "Case variant:",
        case.get(
            "case_variant"
        ),
    )

    print(
        "Source group:",
        case[
            "source_group_id"
        ],
    )

    print(
        "Gold label:",
        case.get(
            "gold_binary_label"
        ),
    )

    print(
        "Anomaly type:",
        case.get(
            "gold_anomaly_type"
        ),
    )

    print(
        "\nSemantic coverage:",
        status,
    )


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant = case[role]


        print("\n" + "#" * 100)

        print(
            role.upper(),
        )

        print("#" * 100)

        print(
            "Conversation:",
            participant[
                "conversation_id"
            ],
        )

        print(
            "Participant:",
            participant[
                "participant_id"
            ],
        )


        for segment_idx, segment_name in enumerate(
            SEGMENT_ORDER
        ):

            segment_start = (
                0
                if segment_idx == 0
                else 60
            )

            segment_end = (
                60
                if segment_idx == 0
                else 120
            )


            segment_record = (
                case[
                    "semantic_summaries"
                ][role][segment_name]
            )


            coarse_summary = (
                segment_record.get(
                    "coarse_summary"
                )
            )


            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )


            print("\n" + "-" * 100)

            print(
                f"SEGMENT {segment_idx}: "
                f"{segment_start}–{segment_end} sec"
            )

            print("-" * 100)


            print("\nCOARSE SUMMARY:")

            if coarse_summary is None:

                print("MISSING")

            else:

                print(
                    json.dumps(
                        coarse_summary,
                        indent=2,
                        ensure_ascii=False,
                    )
                )


            print("\nFOCUSED SUMMARY:")

            if focused_summary is None:

                print("MISSING")

            else:

                print(
                    json.dumps(
                        focused_summary,
                        indent=2,
                        ensure_ascii=False,
                    )
                )


    if show_temporal_information:

        print("\n" + "=" * 100)
        print("FILTERED VAD TURNS")
        print("=" * 100)


        print(
            "\nParticipant A filtered turns:"
        )

        display(
            pd.DataFrame(
                case[
                    "participant_A_filtered_turns"
                ]
            )
        )


        print(
            "\nParticipant B filtered turns:"
        )

        display(
            pd.DataFrame(
                case[
                    "participant_B_filtered_turns"
                ]
            )
        )


        print(
            "\nLocal temporal features:"
        )

        print(
            json.dumps(
                case[
                    "local_temporal_features"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\nGlobal shift features:"
        )

        print(
            json.dumps(
                case[
                    "global_shift_features"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


    return case

In [ ]:
inspect_semantic_case(
    case_family="normal",
    complete_only=True,
    candidate_rank=0,
)

CASE: consolidation_normal_001
Case family: normal
Case variant: normal
Source group: heldout_source_001
Gold label: NORMAL
Anomaly type: none

Semantic coverage: {'coarse_available': 4, 'focused_available': 4, 'participant_segments_with_both': 4, 'complete_case': True, 'has_at_least_one_complete_segment': True}

####################################################################################################
PARTICIPANT_A
####################################################################################################
Conversation: V00_S2050_I00001124
Participant: P1307A

----------------------------------------------------------------------------------------------------
SEGMENT 0: 0–60 sec
----------------------------------------------------------------------------------------------------

COARSE SUMMARY:
{
  "speech_content_summary": "The woman discusses the length of breaks and their impact on career advancement in a company.",
  "apparent_topic": "Workplace break policies an

{'case_id': 'consolidation_normal_001',
 'source_group_id': 'heldout_source_001',
 'pair_index': 1,
 'gold_binary_label': 'NORMAL',
 'gold_anomaly_type': 'none',
 'case_variant': 'normal',
 'pairing_type': 'same_conversation',
 'participant_A': {'conversation_id': 'V00_S2050_I00001124',
  'participant_id': 'P1307A',
  'metadata_path': '/content/drive/MyDrive/seamless_download/data/V00_S2050_I00001124/P1307A/V00_S2050_I00001124_P1307A.json',
  'num_raw_vad_entries': 31},
 'participant_B': {'conversation_id': 'V00_S2050_I00001124',
  'participant_id': 'P1308A',
  'metadata_path': '/content/drive/MyDrive/seamless_download/data/V00_S2050_I00001124/P1308A/V00_S2050_I00001124_P1308A.json',
  'num_raw_vad_entries': 55},
 'A_source_conversation': 'V00_S2050_I00001124',
 'B_source_conversation': 'V00_S2050_I00001124',
 'analysis_duration_seconds': 120.0,
 'participant_A_filtered_turns': [{'start': 21.35, 'end': 23.81},
  {'start': 38.95, 'end': 42.17},
  {'start': 48.51, 'end': 55.45},
  {'star

In [ ]:
inspect_semantic_case(
    case_family="wrong_partner",
    complete_only=True,
    candidate_rank=0,
)

CASE: consolidation_wrong_partner_001
Case family: wrong_partner
Case variant: wrong_partner
Source group: heldout_source_001
Gold label: ANOMALOUS
Anomaly type: wrong_partner

Semantic coverage: {'coarse_available': 4, 'focused_available': 4, 'participant_segments_with_both': 4, 'complete_case': True, 'has_at_least_one_complete_segment': True}

####################################################################################################
PARTICIPANT_A
####################################################################################################
Conversation: V00_S2050_I00001124
Participant: P1307A

----------------------------------------------------------------------------------------------------
SEGMENT 0: 0–60 sec
----------------------------------------------------------------------------------------------------

COARSE SUMMARY:
{
  "speech_content_summary": "The woman discusses the length of breaks and their impact on career advancement in a company.",
  "apparent_top

{'case_id': 'consolidation_wrong_partner_001',
 'source_group_id': 'heldout_source_001',
 'pair_index': 1,
 'gold_binary_label': 'ANOMALOUS',
 'gold_anomaly_type': 'wrong_partner',
 'case_variant': 'wrong_partner',
 'pairing_type': 'different_conversations',
 'participant_A': {'conversation_id': 'V00_S2050_I00001124',
  'participant_id': 'P1307A',
  'metadata_path': '/content/drive/MyDrive/seamless_download/data/V00_S2050_I00001124/P1307A/V00_S2050_I00001124_P1307A.json',
  'num_raw_vad_entries': 31},
 'participant_B': {'conversation_id': 'V00_S2051_I00001001',
  'participant_id': 'P1308A',
  'metadata_path': '/content/drive/MyDrive/seamless_download/data/V00_S2051_I00001001/P1308A/V00_S2051_I00001001_P1308A.json',
  'num_raw_vad_entries': 37},
 'A_source_conversation': 'V00_S2050_I00001124',
 'B_source_conversation': 'V00_S2051_I00001001',
 'analysis_duration_seconds': 120.0,
 'participant_A_filtered_turns': [{'start': 21.35, 'end': 23.81},
  {'start': 30.98, 'end': 31.68},
  {'start'

In [ ]:
inspect_semantic_case(
    case_family="lag",
    complete_only=True,
    candidate_rank=0,
    show_temporal_information=True,
)

CASE: consolidation_lag_2sec_001
Case family: lag
Case variant: lag_2sec
Source group: heldout_source_001
Gold label: ANOMALOUS
Anomaly type: lag

Semantic coverage: {'coarse_available': 4, 'focused_available': 4, 'participant_segments_with_both': 4, 'complete_case': True, 'has_at_least_one_complete_segment': True}

####################################################################################################
PARTICIPANT_A
####################################################################################################
Conversation: V00_S2050_I00001124
Participant: P1307A

----------------------------------------------------------------------------------------------------
SEGMENT 0: 0–60 sec
----------------------------------------------------------------------------------------------------

COARSE SUMMARY:
{
  "speech_content_summary": "The woman discusses the length of breaks and their impact on career advancement in a company.",
  "apparent_topic": "Workplace break policies

,start,end
0,21.35,23.81
1,38.95,42.17
2,48.51,55.45
3,68.64,110.11



Participant B filtered turns:


,start,end
0,2.00,7.02
1,8.05,21.81
2,26.13,29.93
3,30.93,41.55
4,44.82,51.66
5,59.15,59.47
6,60.82,62.22
7,63.06,70.54
8,111.12,120.00



Local temporal features:
{
  "clean_overlap_seconds": 8.11,
  "clean_overlap_percent": 6.76,
  "signed_strict_offsets_seconds": [
    2.32,
    2.65,
    3.7,
    1.01
  ],
  "num_signed_strict_offsets": 4,
  "offset_mean_seconds": 2.42,
  "offset_median_seconds": 2.48,
  "offset_max_seconds": 3.7,
  "offset_p75_seconds": 2.91,
  "offset_p90_seconds": 3.39,
  "num_offsets_above_1_5_seconds": 3,
  "percent_offsets_above_1_5_seconds": 75.0
}

Global shift features:
{
  "best_B_correction_shift_seconds": -2.6,
  "estimated_B_lateness_seconds": 2.6,
  "alignment_score_gain_vs_zero": 0.245195,
  "best_num_bilateral_events": 8,
  "best_event_coverage": 0.615385
}


{'case_id': 'consolidation_lag_2sec_001',
 'source_group_id': 'heldout_source_001',
 'pair_index': 1,
 'gold_binary_label': 'ANOMALOUS',
 'gold_anomaly_type': 'lag',
 'case_variant': 'lag_2sec',
 'pairing_type': 'same_conversation_shifted_B',
 'lag_seconds': 2.0,
 'shifted_participant': 'B',
 'participant_A': {'conversation_id': 'V00_S2050_I00001124',
  'participant_id': 'P1307A',
  'metadata_path': '/content/drive/MyDrive/seamless_download/data/V00_S2050_I00001124/P1307A/V00_S2050_I00001124_P1307A.json',
  'num_raw_vad_entries': 31},
 'participant_B': {'conversation_id': 'V00_S2050_I00001124',
  'participant_id': 'P1308A',
  'metadata_path': '/content/drive/MyDrive/seamless_download/data/V00_S2050_I00001124/P1308A/V00_S2050_I00001124_P1308A.json',
  'num_raw_vad_entries': 55},
 'A_source_conversation': 'V00_S2050_I00001124',
 'B_source_conversation': 'V00_S2050_I00001124',
 'analysis_duration_seconds': 120.0,
 'participant_A_filtered_turns': [{'start': 21.35, 'end': 23.81},
  {'start'

In [ ]:
# ============================================================
# COLLECT ALL UNIQUE PARTICIPANTS WITH MISSING
# COARSE AND/OR FOCUSED SUMMARIES
#
# IMPORTANT:
# - NORMAL and WRONG_PARTNER are checked.
# - SILENT_PARTNER is checked.
# - LAG is excluded because it uses exactly the same
#   participants and summaries as NORMAL.
# - A summary is considered missing only when it is None.
# - Existing low-quality/unclear summaries are kept as available.
# ============================================================

from collections import defaultdict
from pathlib import Path

import json
import pandas as pd


SEGMENT_DEFINITIONS = {
    "segment_0_0_to_60_seconds": {
        "segment_idx": 0,
        "segment_start_seconds": 0,
        "segment_end_seconds": 60,
    },

    "segment_1_60_to_120_seconds": {
        "segment_idx": 1,
        "segment_start_seconds": 60,
        "segment_end_seconds": 120,
    },
}


# LAG is intentionally excluded because it inherits
# exactly the corresponding NORMAL summaries.
CASES_TO_CHECK_FOR_MISSING_SUMMARIES = (
    semantic_normal_cases
    + semantic_wrong_cases
    + semantic_silent_cases
)


def get_case_family(case):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()

    if variant == "normal":
        return "normal"

    if variant == "wrong_partner":
        return "wrong_partner"

    if variant == "silent_partner":
        return "silent_partner"

    return variant


def resolve_video_path(
    participant,
    source_kind,
):
    """
    Resolve the original participant video path.

    source_kind:
        normal -> DATA_ROOT / conversation / participant
        silent -> SILENT_ROOT / conversation / participant
    """

    for key in [
        "video_path",
        "mp4",
    ]:
        value = participant.get(key)

        if value:
            candidate = Path(value)

            if candidate.exists():
                return str(candidate)


    root = (
        SILENT_ROOT
        if source_kind == "silent"
        else DATA_ROOT
    )


    participant_dir = (
        Path(root)
        / str(
            participant["conversation_id"]
        )
        / str(
            participant["participant_id"]
        )
    )


    candidates = []

    for pattern in [
        "*.mp4",
        "*.mov",
        "*.mkv",
        "*.webm",
    ]:
        candidates.extend(
            participant_dir.glob(pattern)
        )


    candidates = sorted(
        set(candidates)
    )


    return (
        str(candidates[0])
        if candidates
        else None
    )


# ============================================================
# UNIQUE PARTICIPANT REGISTRY
#
# Identity:
# (
#     source_kind,
#     conversation_id,
#     participant_id
# )
#
# source_kind is included so silent-folder participants
# cannot be confused with normal-folder participants.
# ============================================================

participant_missing_registry = {}


for case in CASES_TO_CHECK_FOR_MISSING_SUMMARIES:

    family = get_case_family(case)


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant = case[role]


        source_kind = (
            "silent"
            if (
                family == "silent_partner"
                and role == "participant_B"
            )
            else "normal"
        )


        conversation_id = str(
            participant["conversation_id"]
        )

        participant_id = str(
            participant["participant_id"]
        )


        participant_key = (
            source_kind,
            conversation_id,
            participant_id,
        )


        if participant_key not in participant_missing_registry:

            participant_missing_registry[
                participant_key
            ] = {
                "source_kind": source_kind,

                "conversation_id": (
                    conversation_id
                ),

                "participant_id": (
                    participant_id
                ),

                "metadata_path": (
                    participant.get(
                        "metadata_path"
                    )
                ),

                "video_path": (
                    resolve_video_path(
                        participant=participant,
                        source_kind=source_kind,
                    )
                ),

                "used_in_case_families": set(),

                "used_in_roles": set(),

                "used_in_case_ids": set(),

                "segments": {},
            }


        registry_record = (
            participant_missing_registry[
                participant_key
            ]
        )


        registry_record[
            "used_in_case_families"
        ].add(
            family
        )


        registry_record[
            "used_in_roles"
        ].add(
            role
        )


        registry_record[
            "used_in_case_ids"
        ].add(
            case["case_id"]
        )


        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        for (
            segment_name,
            segment_definition,
        ) in SEGMENT_DEFINITIONS.items():

            segment_record = (
                participant_semantics[
                    segment_name
                ]
            )


            coarse_missing = (
                segment_record.get(
                    "coarse_summary"
                )
                is None
            )


            focused_missing = (
                segment_record.get(
                    "focused_summary"
                )
                is None
            )


            if segment_name not in registry_record[
                "segments"
            ]:

                registry_record[
                    "segments"
                ][segment_name] = {
                    **segment_definition,

                    "coarse_missing": (
                        coarse_missing
                    ),

                    "focused_missing": (
                        focused_missing
                    ),
                }

            else:
                # The same participant may appear in NORMAL,
                # WRONG_PARTNER and SILENT_PARTNER cases.
                # Missing status must remain consistent.
                existing_segment = (
                    registry_record[
                        "segments"
                    ][segment_name]
                )

                assert (
                    existing_segment[
                        "coarse_missing"
                    ]
                    == coarse_missing
                ), (
                    "Inconsistent coarse-summary status for "
                    f"{participant_key}, {segment_name}"
                )

                assert (
                    existing_segment[
                        "focused_missing"
                    ]
                    == focused_missing
                ), (
                    "Inconsistent focused-summary status for "
                    f"{participant_key}, {segment_name}"
                )


# ============================================================
# KEEP ONLY PARTICIPANTS MISSING AT LEAST ONE SUMMARY
# ============================================================

participants_missing_summaries = []


for participant_key, record in sorted(
    participant_missing_registry.items()
):

    missing_segments = []


    for segment_name in SEGMENT_DEFINITIONS:

        segment_record = (
            record[
                "segments"
            ][segment_name]
        )


        if (
            segment_record[
                "coarse_missing"
            ]
            or segment_record[
                "focused_missing"
            ]
        ):

            missing_segments.append({
                "segment_name": (
                    segment_name
                ),

                **segment_record,
            })


    if not missing_segments:
        continue


    participants_missing_summaries.append({
        "source_kind": (
            record["source_kind"]
        ),

        "conversation_id": (
            record["conversation_id"]
        ),

        "participant_id": (
            record["participant_id"]
        ),

        "metadata_path": (
            record["metadata_path"]
        ),

        "video_path": (
            record["video_path"]
        ),

        "used_in_case_families": sorted(
            record[
                "used_in_case_families"
            ]
        ),

        "used_in_roles": sorted(
            record[
                "used_in_roles"
            ]
        ),

        "num_case_appearances": len(
            record[
                "used_in_case_ids"
            ]
        ),

        "missing_segments": (
            missing_segments
        ),

        "num_missing_coarse_segments": sum(
            segment[
                "coarse_missing"
            ]
            for segment in missing_segments
        ),

        "num_missing_focused_segments": sum(
            segment[
                "focused_missing"
            ]
            for segment in missing_segments
        ),
    })


# ============================================================
# CREATE UNIQUE PARTICIPANT-SEGMENT GENERATION TASKS
# ============================================================

missing_summary_generation_tasks = []


for participant in participants_missing_summaries:

    for segment in participant[
        "missing_segments"
    ]:

        missing_summary_generation_tasks.append({
            "source_kind": (
                participant[
                    "source_kind"
                ]
            ),

            "conversation_id": (
                participant[
                    "conversation_id"
                ]
            ),

            "participant_id": (
                participant[
                    "participant_id"
                ]
            ),

            "metadata_path": (
                participant[
                    "metadata_path"
                ]
            ),

            "video_path": (
                participant[
                    "video_path"
                ]
            ),

            "segment_name": (
                segment[
                    "segment_name"
                ]
            ),

            "segment_idx": (
                segment[
                    "segment_idx"
                ]
            ),

            "segment_start_seconds": (
                segment[
                    "segment_start_seconds"
                ]
            ),

            "segment_end_seconds": (
                segment[
                    "segment_end_seconds"
                ]
            ),

            "segment_duration_seconds": (
                segment[
                    "segment_end_seconds"
                ]
                - segment[
                    "segment_start_seconds"
                ]
            ),

            "generate_coarse": (
                segment[
                    "coarse_missing"
                ]
            ),

            "generate_focused": (
                segment[
                    "focused_missing"
                ]
            ),

            "used_in_case_families": (
                participant[
                    "used_in_case_families"
                ]
            ),
        })


# ============================================================
# STRICT DEDUPLICATION AUDIT
# ============================================================

participant_signatures = [
    (
        participant[
            "source_kind"
        ],

        participant[
            "conversation_id"
        ],

        participant[
            "participant_id"
        ],
    )

    for participant
    in participants_missing_summaries
]


task_signatures = [
    (
        task[
            "source_kind"
        ],

        task[
            "conversation_id"
        ],

        task[
            "participant_id"
        ],

        task[
            "segment_idx"
        ],
    )

    for task
    in missing_summary_generation_tasks
]


assert len(
    participant_signatures
) == len(
    set(participant_signatures)
)


assert len(
    task_signatures
) == len(
    set(task_signatures)
)


# ============================================================
# SAVE RESULTS
# ============================================================

MISSING_PARTICIPANTS_PATH = (
    OUT_DIR
    / "participants_missing_coarse_or_focused_summaries.json"
)


MISSING_GENERATION_TASKS_PATH = (
    OUT_DIR
    / "participant_segments_missing_coarse_or_focused_summaries.json"
)


MISSING_PARTICIPANTS_PATH.write_text(
    json.dumps(
        participants_missing_summaries,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


MISSING_GENERATION_TASKS_PATH.write_text(
    json.dumps(
        missing_summary_generation_tasks,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ============================================================
# DISPLAY SUMMARY
# ============================================================

print("=" * 80)
print("MISSING SEMANTIC SUMMARY COLLECTION COMPLETE")
print("=" * 80)

print(
    "Unique participants missing anything:",
    len(
        participants_missing_summaries
    ),
)

print(
    "Unique participant-segment tasks:",
    len(
        missing_summary_generation_tasks
    ),
)

print(
    "Coarse summaries to generate:",
    sum(
        task["generate_coarse"]

        for task
        in missing_summary_generation_tasks
    ),
)

print(
    "Focused summaries to generate:",
    sum(
        task["generate_focused"]

        for task
        in missing_summary_generation_tasks
    ),
)

print(
    "Tasks without video path:",
    sum(
        task["video_path"] is None

        for task
        in missing_summary_generation_tasks
    ),
)


missing_participants_df = pd.DataFrame([
    {
        "source_kind": (
            participant[
                "source_kind"
            ]
        ),

        "conversation_id": (
            participant[
                "conversation_id"
            ]
        ),

        "participant_id": (
            participant[
                "participant_id"
            ]
        ),

        "missing_coarse_segments": (
            participant[
                "num_missing_coarse_segments"
            ]
        ),

        "missing_focused_segments": (
            participant[
                "num_missing_focused_segments"
            ]
        ),

        "used_in_case_families": ", ".join(
            participant[
                "used_in_case_families"
            ]
        ),

        "video_found": (
            participant[
                "video_path"
            ]
            is not None
        ),
    }

    for participant
    in participants_missing_summaries
])


display(
    missing_participants_df
)


print(
    "\nSaved missing participants:",
    MISSING_PARTICIPANTS_PATH,
)

print(
    "Saved generation tasks:",
    MISSING_GENERATION_TASKS_PATH,
)

MISSING SEMANTIC SUMMARY COLLECTION COMPLETE
Unique participants missing anything: 137
Unique participant-segment tasks: 274
Coarse summaries to generate: 274
Focused summaries to generate: 274
Tasks without video path: 0


,source_kind,conversation_id,participant_id,missing_coarse_segments,missing_focused_segments,used_in_case_families,video_found
0,normal,V01_S0223_I00000137,P1505,2,2,"normal, silent_partner, wrong_partner",True
1,normal,V01_S0223_I00000137,P1506,2,2,"normal, wrong_partner",True
2,normal,V01_S0223_I00000138,P1505,2,2,"normal, wrong_partner",True
3,normal,V01_S0223_I00000138,P1506,2,2,"normal, silent_partner, wrong_partner",True
4,normal,V01_S0223_I00000307,P1505,2,2,"normal, wrong_partner",True
...,...,...,...,...,...,...,...
132,silent,V01_S0307_I00001235,P1633,2,2,silent_partner,True
133,silent,V01_S0337_I00001104,P1678,2,2,silent_partner,True
134,silent,V01_S1545_I00000138,P2518,2,2,silent_partner,True
135,silent,V03_S1088_I00000498,P1420,2,2,silent_partner,True



Saved missing participants: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/participants_missing_coarse_or_focused_summaries.json
Saved generation tasks: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/participant_segments_missing_coarse_or_focused_summaries.json


# Produce the coarse and focused summaries for the participants that we dont already have this information

In [ ]:
# ============================================================
# INSTALL EXACT DEPENDENCIES USED BY THE ORIGINAL PIPELINE
# ============================================================

!pip uninstall -y transformers
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes sentencepiece
!pip install -U qwen-omni-utils decord ffmpeg-python

!apt-get update -qq
!apt-get install -y -qq ffmpeg

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 145.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 145.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 80.1 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
# ============================================================
# LOAD AND VERIFY THE MISSING SUMMARY TASKS
# ============================================================

from pathlib import Path
from tqdm.auto import tqdm

import json
import os
import re
import subprocess


assert "OUT_DIR" in globals(), (
    "OUT_DIR is missing. Run the previous consolidation cells."
)


MISSING_GENERATION_TASKS_PATH = (
    OUT_DIR
    / "participant_segments_missing_coarse_or_focused_summaries.json"
)


if (
    "missing_summary_generation_tasks"
    not in globals()
):

    assert MISSING_GENERATION_TASKS_PATH.exists(), (
        "Missing task file: "
        f"{MISSING_GENERATION_TASKS_PATH}"
    )

    missing_summary_generation_tasks = json.loads(
        MISSING_GENERATION_TASKS_PATH.read_text(
            encoding="utf-8"
        )
    )


assert isinstance(
    missing_summary_generation_tasks,
    list,
)


# ------------------------------------------------------------
# Keep coarse and focused tasks separate.
# Coarse will be completed first.
# ------------------------------------------------------------

missing_coarse_generation_tasks = [
    task
    for task in missing_summary_generation_tasks
    if task["generate_coarse"]
]


missing_focused_generation_tasks = [
    task
    for task in missing_summary_generation_tasks
    if task["generate_focused"]
]


participant_signatures = {
    (
        task["source_kind"],
        task["conversation_id"],
        task["participant_id"],
    )
    for task in missing_summary_generation_tasks
}


all_task_signatures = [
    (
        task["source_kind"],
        task["conversation_id"],
        task["participant_id"],
        int(task["segment_idx"]),
    )
    for task in missing_summary_generation_tasks
]


coarse_task_signatures = [
    (
        task["source_kind"],
        task["conversation_id"],
        task["participant_id"],
        int(task["segment_idx"]),
    )
    for task in missing_coarse_generation_tasks
]


focused_task_signatures = [
    (
        task["source_kind"],
        task["conversation_id"],
        task["participant_id"],
        int(task["segment_idx"]),
    )
    for task in missing_focused_generation_tasks
]


assert len(
    all_task_signatures
) == len(
    set(all_task_signatures)
)


assert len(
    coarse_task_signatures
) == len(
    set(coarse_task_signatures)
)


assert len(
    focused_task_signatures
) == len(
    set(focused_task_signatures)
)


assert len(participant_signatures) == 137
assert len(missing_summary_generation_tasks) == 274
assert len(missing_coarse_generation_tasks) == 274
assert len(missing_focused_generation_tasks) == 274


for task in missing_summary_generation_tasks:

    assert int(
        task["segment_idx"]
    ) in {
        0,
        1,
    }

    assert int(
        task["segment_start_seconds"]
    ) in {
        0,
        60,
    }

    assert int(
        task["segment_duration_seconds"]
    ) == 60

    assert task["video_path"] is not None

    assert Path(
        task["video_path"]
    ).exists(), (
        "Video not found: "
        f'{task["video_path"]}'
    )


print("=" * 80)
print("MISSING SUMMARY TASK AUDIT PASSED")
print("=" * 80)

print(
    "Unique participants:",
    len(participant_signatures),
)

print(
    "Participant-segment tasks:",
    len(
        missing_summary_generation_tasks
    ),
)

print(
    "Coarse tasks:",
    len(
        missing_coarse_generation_tasks
    ),
)

print(
    "Focused tasks:",
    len(
        missing_focused_generation_tasks
    ),
)

MISSING SUMMARY TASK AUDIT PASSED
Unique participants: 137
Participant-segment tasks: 274
Coarse tasks: 274
Focused tasks: 274


In [ ]:
# ============================================================
# OUTPUT AND PREPROCESSING PATHS
# ============================================================

MISSING_SUMMARY_RUN_DIR = (
    OUT_DIR
    / "missing_semantic_summary_generation"
)


MISSING_SUMMARY_PREP_DIR = (
    MISSING_SUMMARY_RUN_DIR
    / "preprocessed_segments_60s"
)


MISSING_COARSE_RAW_CACHE_PATH = (
    MISSING_SUMMARY_RUN_DIR
    / "missing_coarse_raw_generation_records.json"
)


MISSING_COARSE_SUMMARIES_PATH = (
    MISSING_SUMMARY_RUN_DIR
    / "missing_coarse_summaries_clean.json"
)


MISSING_FOCUSED_RAW_CACHE_PATH = (
    MISSING_SUMMARY_RUN_DIR
    / "missing_focused_raw_generation_records.json"
)


MISSING_FOCUSED_SUMMARIES_PATH = (
    MISSING_SUMMARY_RUN_DIR
    / "missing_focused_summaries_clean_without_speaks.json"
)


MISSING_SUMMARY_RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


MISSING_SUMMARY_PREP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Run directory:",
    MISSING_SUMMARY_RUN_DIR,
)

print(
    "Segment directory:",
    MISSING_SUMMARY_PREP_DIR,
)

print(
    "Coarse clean summaries:",
    MISSING_COARSE_SUMMARIES_PATH,
)

print(
    "Focused clean summaries:",
    MISSING_FOCUSED_SUMMARIES_PATH,
)

Run directory: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation
Segment directory: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/preprocessed_segments_60s
Coarse clean summaries: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/missing_coarse_summaries_clean.json
Focused clean summaries: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/missing_focused_summaries_clean_without_speaks.json


In [ ]:
# ============================================================
# LOAD QWEN/QWEN2.5-OMNI-7B THINKER
# ============================================================

import torch

from transformers import (
    Qwen2_5OmniThinkerForConditionalGeneration,
    Qwen2_5OmniProcessor,
)

from qwen_omni_utils import process_mm_info


MODEL_ID = "Qwen/Qwen2.5-Omni-7B"


model = (
    Qwen2_5OmniThinkerForConditionalGeneration
    .from_pretrained(
        MODEL_ID,
        torch_dtype="auto",
        device_map="auto",
    )
)


processor = (
    Qwen2_5OmniProcessor
    .from_pretrained(
        MODEL_ID
    )
)


print(
    "Loaded:",
    MODEL_ID,
)

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/233k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-Omni-7B


In [ ]:
# ============================================================
# EXACT JSON EXTRACTION LOGIC
# ============================================================

def extract_json_from_text(
    text: str,
):
    text = str(
        text
    ).strip()


    text = re.sub(
        r"^```(?:json)?",
        "",
        text,
        flags=re.IGNORECASE,
    )


    text = re.sub(
        r"```$",
        "",
        text,
    ).strip()


    try:
        return json.loads(
            text
        )

    except Exception:
        pass


    start = text.find(
        "{"
    )


    if start == -1:

        return {
            "parse_error": True,
            "raw_output": text,
        }


    depth = 0
    in_string = False
    escaped = False


    for index in range(
        start,
        len(text),
    ):

        char = text[index]


        if in_string:

            if escaped:
                escaped = False

            elif char == "\\":
                escaped = True

            elif char == '"':
                in_string = False

            continue


        if char == '"':
            in_string = True

        elif char == "{":
            depth += 1

        elif char == "}":

            depth -= 1

            if depth == 0:

                candidate = text[
                    start:index + 1
                ]


                try:
                    return json.loads(
                        candidate
                    )

                except Exception:

                    return {
                        "parse_error": True,
                        "raw_output": text,
                        "json_candidate": candidate,
                    }


    return {
        "parse_error": True,
        "raw_output": text,
    }


# ============================================================
# EXACT QWEN VIDEO + AUDIO INFERENCE LOGIC
# ============================================================

def qwen_video_text(
    video_path: str,
    prompt: str,
    max_new_tokens: int = 500,
):

    messages = [
        {
            "role": "system",

            "content": [
                {
                    "type": "text",

                    "text": (
                        "You are Qwen, a virtual human developed "
                        "by the Qwen Team, Alibaba Group, capable "
                        "of perceiving auditory and visual inputs, "
                        "as well as generating text and speech."
                    ),
                }
            ],
        },

        {
            "role": "user",

            "content": [
                {
                    "type": "video",
                    "video": video_path,
                    "fps": 7.0,
                },

                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        },
    ]


    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )


    audios, images, videos = process_mm_info(
        messages,
        use_audio_in_video=True,
    )


    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=True,
    ).to(
        model.device
    )

    input_token_count = int(
        inputs["input_ids"].shape[-1]
    )

    print(
        "Input tokens:",
        input_token_count,
    )

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )


    generated_ids = output_ids[
        :,
        inputs["input_ids"].shape[1]:,
    ]


    return processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

In [ ]:
# ============================================================
# EXACT 60-SECOND SEGMENT PREPROCESSING
# ============================================================

SEGMENTS = {
    0: (
        0,
        60,
    ),

    1: (
        60,
        60,
    ),
}


def safe_name(
    value: str,
):
    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(value),
    )


def segmented_path(
    task,
    segment_idx,
):
    source_kind = safe_name(
        task["source_kind"]
    )

    conversation = safe_name(
        task["conversation_id"]
    )

    participant = safe_name(
        task["participant_id"]
    )


    output_dir = (
        MISSING_SUMMARY_PREP_DIR
        / source_kind
        / conversation
    )


    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    return (
        output_dir
        / (
            f"{conversation}_"
            f"{participant}_"
            f"seg{int(segment_idx)}_60s.mp4"
        )
    )


def video_has_audio(
    path,
):
    command = [
        "ffprobe",
        "-v",
        "error",

        "-select_streams",
        "a",

        "-show_entries",
        "stream=index",

        "-of",
        "csv=p=0",

        str(path),
    ]


    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )


    return bool(
        result.stdout.strip()
    )


def preprocess_video_segment(
    input_path,
    output_path,
    start,
    seconds=60,
    fps=30,
):
    output_path = Path(
        output_path
    )


    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    if (
        output_path.exists()
        and output_path.stat().st_size > 0
        and video_has_audio(output_path)
    ):

        return str(
            output_path
        )


    output_path.unlink(
        missing_ok=True
    )


    command = [
        "ffmpeg",
        "-y",

        "-ss",
        str(start),

        "-i",
        str(input_path),

        "-t",
        str(seconds),

        "-vf",
        (
            f"scale=360:-2,"
            f"fps={fps},"
            "format=yuv420p"
        ),

        "-c:v",
        "libx264",

        "-crf",
        "23",

        "-preset",
        "veryfast",

        "-c:a",
        "aac",

        "-ar",
        "16000",

        "-ac",
        "1",

        str(output_path),
    ]


    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )


    if (
        result.returncode != 0
        or not output_path.exists()
        or output_path.stat().st_size == 0
        or not video_has_audio(output_path)
    ):

        output_path.unlink(
            missing_ok=True
        )


        raise RuntimeError(
            "Could not create segment from "
            f"{input_path}, start={start}.\n"
            f"{result.stderr[-2000:]}"
        )


    return str(
        output_path
    )


def get_or_create_segment(
    task,
):
    segment_idx = int(
        task["segment_idx"]
    )


    start, duration = (
        SEGMENTS[
            segment_idx
        ]
    )


    path = segmented_path(
        task,
        segment_idx,
    )


    if (
        not path.exists()
        or path.stat().st_size == 0
        or not video_has_audio(path)
    ):

        preprocess_video_segment(
            input_path=(
                task["video_path"]
            ),

            output_path=path,

            start=start,

            seconds=duration,

            fps=30,
        )


    return {
        "segment_idx": segment_idx,
        "start": start,
        "duration": duration,
        "path": str(path),
    }

In [ ]:
# ============================================================
# EXACT ORIGINAL COARSE PARTICIPANT PROMPT
# ============================================================

PARTICIPANT_ANALYSIS_PROMPT_TEMPLATE = """
You are analyzing ONE participant video from a dyadic conversation.

Use both visible behavior and embedded audio.

Main task:
Extract compact evidence that can later be compared with another participant
to decide whether they belong to the same conversation.

Return ONLY valid JSON. Do not include markdown or text outside JSON.

Use exactly this schema:
{{
  "participant_id": "{participant_id}",
  "speaks": true,
  "speaking_amount": "none / low / moderate / high",
  "speech_content_summary": "brief summary of what the participant seems to talk about; use unclear if not understandable",
  "apparent_topic": "short topic label or unclear",
  "asks_questions": true,
  "answers_or_responds": true,
  "visual_engagement": "low / medium / high / unclear",
  "listening_or_reacting": true,
  "waiting_for_other_person": true,
  "interaction_style": "active / passive / mostly listening / unclear",
  "short_description": "one short sentence",
  "confidence": 0.0
}}
""".strip()

In [ ]:
# ============================================================
# COARSE CACHE AND CLEAN PROJECTION
# ============================================================

COARSE_SUMMARY_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]


def load_json_dict(
    path,
):
    path = Path(
        path
    )


    if not path.exists():
        return {}


    data = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


    if not isinstance(
        data,
        dict,
    ):

        raise TypeError(
            f"Expected JSON object in {path}"
        )


    return data


missing_coarse_raw_cache = (
    load_json_dict(
        MISSING_COARSE_RAW_CACHE_PATH
    )
)


missing_coarse_clean_cache = (
    load_json_dict(
        MISSING_COARSE_SUMMARIES_PATH
    )
)


def coarse_cache_key(
    task,
    segment,
):
    return (
        f'{task["source_kind"]}::'
        f'{task["conversation_id"]}::'
        f'{task["participant_id"]}::'
        f'seg{segment["segment_idx"]}::'
        f'{segment["path"]}'
    )


def build_coarse_projection(
    parsed_output,
):
    if not isinstance(
        parsed_output,
        dict,
    ):
        return None


    if parsed_output.get(
        "parse_error",
        False,
    ):
        return None


    if not all(
        field in parsed_output
        for field in COARSE_SUMMARY_FIELDS
    ):
        return None


    return {
        field: parsed_output[field]
        for field in COARSE_SUMMARY_FIELDS
    }


def save_coarse_caches():
    MISSING_COARSE_RAW_CACHE_PATH.write_text(
        json.dumps(
            missing_coarse_raw_cache,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    MISSING_COARSE_SUMMARIES_PATH.write_text(
        json.dumps(
            missing_coarse_clean_cache,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


print(
    "Existing raw coarse records:",
    len(
        missing_coarse_raw_cache
    ),
)

print(
    "Existing clean coarse summaries:",
    len(
        missing_coarse_clean_cache
    ),
)

Existing raw coarse records: 273
Existing clean coarse summaries: 273


In [ ]:
# ============================================================
# GENERATE ONE COARSE SUMMARY
# ============================================================

def generate_missing_coarse_summary(
    task,
    force_rerun=False,
):
    segment = get_or_create_segment(
        task
    )


    key = coarse_cache_key(
        task,
        segment,
    )


    if (
        not force_rerun
        and key in missing_coarse_raw_cache
    ):

        return missing_coarse_raw_cache[
            key
        ]


    prompt = (
        PARTICIPANT_ANALYSIS_PROMPT_TEMPLATE
        .format(
            participant_id=(
                f'{task["participant_id"]}'
                f'_seg{segment["segment_idx"]}'
            )
        )
    )


    raw_output = qwen_video_text(
        video_path=segment["path"],
        prompt=prompt,
        max_new_tokens=300,
    )


    parsed_output = extract_json_from_text(
        raw_output
    )


    coarse_summary = (
        build_coarse_projection(
            parsed_output
        )
    )


    raw_record = {
        "source_kind": (
            task["source_kind"]
        ),

        "conversation_id": (
            task["conversation_id"]
        ),

        "participant_id": (
            task["participant_id"]
        ),

        "segment_idx": int(
            segment["segment_idx"]
        ),

        "start": segment["start"],
        "duration": segment["duration"],

        "video": segment["path"],

        "raw_output": raw_output,
        "parsed": parsed_output,

        "coarse_summary": (
            coarse_summary
        ),
    }


    missing_coarse_raw_cache[
        key
    ] = raw_record


    if coarse_summary is not None:

        missing_coarse_clean_cache[
            key
        ] = {
            "source_kind": (
                task["source_kind"]
            ),

            "conversation_id": (
                task["conversation_id"]
            ),

            "participant_id": (
                task["participant_id"]
            ),

            "segment_idx": int(
                segment["segment_idx"]
            ),

            "segment_start_seconds": (
                segment["start"]
            ),

            "segment_duration_seconds": (
                segment["duration"]
            ),

            "coarse_summary": (
                coarse_summary
            ),
        }


    save_coarse_caches()


    return raw_record


# ============================================================
# RUN ALL COARSE TASKS FIRST — SKIP FAILED VIDEO SEGMENTS
# ============================================================

skipped_coarse_tasks = []


for task in tqdm(
    missing_coarse_generation_tasks,
    desc="Generating missing coarse summaries",
):

    try:

        _ = generate_missing_coarse_summary(
            task=task,
            force_rerun=False,
        )

    except RuntimeError as exc:

        # Skip only ffmpeg/source-video segment failures.
        # Do not hide CUDA, model or other RuntimeErrors.
        if (
            "Could not create segment from"
            not in str(exc)
        ):
            raise


        skipped_record = {
            "source_kind": (
                task["source_kind"]
            ),

            "conversation_id": (
                task["conversation_id"]
            ),

            "participant_id": (
                task["participant_id"]
            ),

            "segment_idx": int(
                task["segment_idx"]
            ),

            "segment_start_seconds": (
                task[
                    "segment_start_seconds"
                ]
            ),

            "segment_duration_seconds": (
                task[
                    "segment_duration_seconds"
                ]
            ),

            "video_path": (
                task["video_path"]
            ),

            "error": str(exc),
        }


        skipped_coarse_tasks.append(
            skipped_record
        )


        print(
            "\nSKIPPING:",
            task["conversation_id"],
            task["participant_id"],
            f'segment={task["segment_idx"]}',
        )


        continue


save_coarse_caches()


# ============================================================
# SAVE SKIPPED COARSE TASKS
# ============================================================

SKIPPED_COARSE_TASKS_PATH = (
    MISSING_SUMMARY_RUN_DIR
    / "skipped_coarse_summary_tasks.json"
)


SKIPPED_COARSE_TASKS_PATH.write_text(
    json.dumps(
        skipped_coarse_tasks,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ============================================================
# COARSE GENERATION AUDIT
#
# Build expected keys only for segments that can actually
# be created. Invalid/short segments are skipped again.
# ============================================================

coarse_expected_keys = []

audit_skipped_signatures = set()


for task in missing_coarse_generation_tasks:

    try:

        segment = get_or_create_segment(
            task
        )

    except RuntimeError as exc:

        # Skip only ffmpeg/source-video segment failures.
        # Do not hide other RuntimeErrors.
        if (
            "Could not create segment from"
            not in str(exc)
        ):
            raise


        signature = (
            task["source_kind"],
            task["conversation_id"],
            task["participant_id"],
            int(task["segment_idx"]),
        )


        audit_skipped_signatures.add(
            signature
        )


        continue


    coarse_expected_keys.append(
        coarse_cache_key(
            task,
            segment,
        )
    )


coarse_raw_completed = sum(
    key in missing_coarse_raw_cache
    for key in coarse_expected_keys
)


coarse_clean_completed = sum(
    key in missing_coarse_clean_cache
    for key in coarse_expected_keys
)


coarse_parse_failures = [
    missing_coarse_raw_cache[key]

    for key in coarse_expected_keys

    if (
        key in missing_coarse_raw_cache
        and missing_coarse_raw_cache[key].get(
            "coarse_summary"
        )
        is None
    )
]


# All runnable tasks must be completed.
assert coarse_raw_completed == len(
    coarse_expected_keys
)


# Runnable + skipped must cover all original tasks.
assert (
    len(coarse_expected_keys)
    + len(audit_skipped_signatures)
) == len(
    missing_coarse_generation_tasks
)


print("=" * 80)
print("COARSE GENERATION COMPLETE")
print("=" * 80)

print(
    "Originally requested coarse tasks:",
    len(
        missing_coarse_generation_tasks
    ),
)

print(
    "Runnable coarse tasks:",
    len(
        coarse_expected_keys
    ),
)

print(
    "Skipped invalid/short segments:",
    len(
        audit_skipped_signatures
    ),
)

print(
    "Raw generations completed:",
    coarse_raw_completed,
)

print(
    "Clean coarse summaries:",
    coarse_clean_completed,
)

print(
    "Parse/schema failures:",
    len(
        coarse_parse_failures
    ),
)

print(
    "Raw checkpoint:",
    MISSING_COARSE_RAW_CACHE_PATH,
)

print(
    "Clean coarse summaries:",
    MISSING_COARSE_SUMMARIES_PATH,
)

print(
    "Skipped-task manifest:",
    SKIPPED_COARSE_TASKS_PATH,
)

Generating missing coarse summaries:   0%|          | 0/274 [00:00<?, ?it/s]


SKIPPING: V01_S1545_I00000138 P2518 segment=1
COARSE GENERATION COMPLETE
Originally requested coarse tasks: 274
Runnable coarse tasks: 273
Skipped invalid/short segments: 1
Raw generations completed: 273
Clean coarse summaries: 273
Parse/schema failures: 0
Raw checkpoint: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/missing_coarse_raw_generation_records.json
Clean coarse summaries: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/missing_coarse_summaries_clean.json
Skipped-task manifest: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/skipped_coarse_summary_tasks.json


# CREATE FOCUSED SUMMARIES

In [ ]:
# ============================================================
# EXACT ORIGINAL FOCUSED SEMANTIC PROMPT
# ============================================================

FOCUSED_SUMMARY_VERSION = (
    "focused_participant_semantic_summary_v1"
)


FOCUSED_SEMANTIC_SUMMARY_PROMPT_TEMPLATE = """
You are analyzing ONE 60-second participant video from a dyadic conversation.

Use the embedded audio as the primary source of information.
Use visible behavior only when it helps interpret what the participant is saying.

Your task is to produce an accurate, specific semantic description of what
this participant says during this segment.

This output will later be compared with the semantic description of another
participant to determine whether they plausibly belong to the same conversation.

Therefore:

- Describe the concrete content of the speech, not merely a broad category.
- Preserve important people, objects, events, activities, opinions, decisions,
  questions, problems, examples, places, products, and contextual details.
- Explain what the participant says clearly enough that another model can
  compare it with another participant's speech.
- If the participant discusses multiple subjects, include each important one.
- Distinguish clearly between what is understandable and what is uncertain.
- Do not replace specific content with vague phrases such as:
  "personal experiences",
  "personal preferences",
  "general discussion",
  "daily life",
  "various topics",
  "opinions",
  or "lifestyle".
- Do not infer that two broad personal subjects are the same topic.
- Do not invent details that are not supported by the audio.
- Do not decide whether this is a normal or wrong-partner conversation.
- Analyze only the participant in this video.
- If no audible speech is present, set speaks=false and state that the semantic
  content is unavailable.

Return ONLY valid JSON. Do not include markdown or text outside JSON.

Use exactly this schema:

{{
  "participant_id": "{participant_id}",
  "speaks": true,
  "detailed_speech_summary": "specific and informative description of what the participant says",
  "main_topic": "specific primary subject or unclear",
  "secondary_topics": [
    "specific additional subject"
  ],
  "key_semantic_details": [
    "specific person, object, event, opinion, question, decision, action, place, product, or claim"
  ],
  "summary_specificity": "high / medium / low",
  "unclear_content": "brief description of anything that could not be understood, or none",
  "confidence": 0.0
}}
""".strip()

In [ ]:
# ============================================================
# FOCUSED CACHE AND CLEAN PROJECTION WITHOUT "speaks"
# ============================================================

FOCUSED_SUMMARY_FIELDS_WITHOUT_SPEAKS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


missing_focused_raw_cache = (
    load_json_dict(
        MISSING_FOCUSED_RAW_CACHE_PATH
    )
)


missing_focused_clean_cache = (
    load_json_dict(
        MISSING_FOCUSED_SUMMARIES_PATH
    )
)


def focused_cache_key(
    task,
    segment,
):
    return (
        f"{FOCUSED_SUMMARY_VERSION}::"
        f'{task["source_kind"]}::'
        f'{task["conversation_id"]}::'
        f'{task["participant_id"]}::'
        f'seg{segment["segment_idx"]}::'
        f'{segment["path"]}'
    )


def build_focused_projection_without_speaks(
    parsed_output,
):
    if not isinstance(
        parsed_output,
        dict,
    ):
        return None


    if parsed_output.get(
        "parse_error",
        False,
    ):
        return None


    if not all(
        field in parsed_output
        for field
        in FOCUSED_SUMMARY_FIELDS_WITHOUT_SPEAKS
    ):
        return None


    # "speaks" is deliberately not copied.
    return {
        field: parsed_output[field]
        for field
        in FOCUSED_SUMMARY_FIELDS_WITHOUT_SPEAKS
    }


def save_focused_caches():
    MISSING_FOCUSED_RAW_CACHE_PATH.write_text(
        json.dumps(
            missing_focused_raw_cache,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    MISSING_FOCUSED_SUMMARIES_PATH.write_text(
        json.dumps(
            missing_focused_clean_cache,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


print(
    "Existing raw focused records:",
    len(
        missing_focused_raw_cache
    ),
)

print(
    "Existing clean focused summaries:",
    len(
        missing_focused_clean_cache
    ),
)

Existing raw focused records: 273
Existing clean focused summaries: 265


In [ ]:
# ============================================================
# GENERATE ONE FOCUSED SUMMARY
# ============================================================

def generate_missing_focused_summary(
    task,
    force_rerun=False,
):
    segment = get_or_create_segment(
        task
    )


    key = focused_cache_key(
        task,
        segment,
    )


    if (
        not force_rerun
        and key in missing_focused_raw_cache
    ):

        return missing_focused_raw_cache[
            key
        ]


    prompt = (
        FOCUSED_SEMANTIC_SUMMARY_PROMPT_TEMPLATE
        .format(
            participant_id=(
                f'{task["participant_id"]}'
                f'_seg{segment["segment_idx"]}'
            )
        )
    )


    raw_output = qwen_video_text(
        video_path=segment["path"],
        prompt=prompt,
        max_new_tokens=500,
    )


    parsed_output = extract_json_from_text(
        raw_output
    )


    focused_summary = (
        build_focused_projection_without_speaks(
            parsed_output
        )
    )


    raw_record = {
        "experiment_version": (
            FOCUSED_SUMMARY_VERSION
        ),

        "source_kind": (
            task["source_kind"]
        ),

        "conversation_id": (
            task["conversation_id"]
        ),

        "participant_id": (
            task["participant_id"]
        ),

        "segment_idx": int(
            segment["segment_idx"]
        ),

        "start": segment["start"],
        "duration": segment["duration"],

        "video": segment["path"],

        "raw_output": raw_output,
        "parsed": parsed_output,

        "focused_summary": (
            focused_summary
        ),
    }


    missing_focused_raw_cache[
        key
    ] = raw_record


    if focused_summary is not None:

        missing_focused_clean_cache[
            key
        ] = {
            "source_kind": (
                task["source_kind"]
            ),

            "conversation_id": (
                task["conversation_id"]
            ),

            "participant_id": (
                task["participant_id"]
            ),

            "segment_idx": int(
                segment["segment_idx"]
            ),

            "segment_start_seconds": (
                segment["start"]
            ),

            "segment_duration_seconds": (
                segment["duration"]
            ),

            # Does not contain "speaks".
            "focused_summary": (
                focused_summary
            ),
        }


    save_focused_caches()


    return raw_record


# ============================================================
# RUN FOCUSED TASKS AFTER COARSE HAS FINISHED
# ============================================================

# All runnable coarse tasks must already be completed.
assert all(
    key in missing_coarse_raw_cache

    for key in coarse_expected_keys
), (
    "Not all runnable coarse tasks have completed."
)


# ============================================================
# RUN ALL FOCUSED TASKS — SKIP FAILED VIDEO SEGMENTS
# ============================================================

skipped_focused_tasks = []


for task in tqdm(
    missing_focused_generation_tasks,
    desc="Generating missing focused summaries",
):

    try:

        _ = generate_missing_focused_summary(
            task=task,
            force_rerun=False,
        )

    except RuntimeError as exc:

        # Skip only ffmpeg/source-video segment failures.
        # Do not hide CUDA, model or other RuntimeErrors.
        if (
            "Could not create segment from"
            not in str(exc)
        ):
            raise


        skipped_record = {
            "source_kind": (
                task["source_kind"]
            ),

            "conversation_id": (
                task["conversation_id"]
            ),

            "participant_id": (
                task["participant_id"]
            ),

            "segment_idx": int(
                task["segment_idx"]
            ),

            "segment_start_seconds": (
                task[
                    "segment_start_seconds"
                ]
            ),

            "segment_duration_seconds": (
                task[
                    "segment_duration_seconds"
                ]
            ),

            "video_path": (
                task["video_path"]
            ),

            "error": str(exc),
        }


        skipped_focused_tasks.append(
            skipped_record
        )


        print(
            "\nSKIPPING:",
            task["conversation_id"],
            task["participant_id"],
            f'segment={task["segment_idx"]}',
        )


        continue


save_focused_caches()


# ============================================================
# SAVE SKIPPED FOCUSED TASKS
# ============================================================

SKIPPED_FOCUSED_TASKS_PATH = (
    MISSING_SUMMARY_RUN_DIR
    / "skipped_focused_summary_tasks.json"
)


SKIPPED_FOCUSED_TASKS_PATH.write_text(
    json.dumps(
        skipped_focused_tasks,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ============================================================
# FOCUSED GENERATION AUDIT
#
# Build expected keys only for segments that can actually
# be created. Invalid/short segments are skipped again.
# ============================================================

focused_expected_keys = []

focused_audit_skipped_signatures = set()


for task in missing_focused_generation_tasks:

    try:

        segment = get_or_create_segment(
            task
        )

    except RuntimeError as exc:

        # Skip only ffmpeg/source-video segment failures.
        # Do not hide other RuntimeErrors.
        if (
            "Could not create segment from"
            not in str(exc)
        ):
            raise


        signature = (
            task["source_kind"],
            task["conversation_id"],
            task["participant_id"],
            int(task["segment_idx"]),
        )


        focused_audit_skipped_signatures.add(
            signature
        )


        continue


    focused_expected_keys.append(
        focused_cache_key(
            task,
            segment,
        )
    )


focused_raw_completed = sum(
    key in missing_focused_raw_cache
    for key in focused_expected_keys
)


focused_clean_completed = sum(
    key in missing_focused_clean_cache
    for key in focused_expected_keys
)


focused_parse_failures = [
    missing_focused_raw_cache[key]

    for key in focused_expected_keys

    if (
        key in missing_focused_raw_cache
        and missing_focused_raw_cache[key].get(
            "focused_summary"
        )
        is None
    )
]


# All runnable focused tasks must be completed.
assert focused_raw_completed == len(
    focused_expected_keys
)


# Runnable + skipped must cover all original tasks.
assert (
    len(focused_expected_keys)
    + len(focused_audit_skipped_signatures)
) == len(
    missing_focused_generation_tasks
)


# Strictly verify that "speaks" was not retained
# in any clean focused summary.
for record in missing_focused_clean_cache.values():

    assert (
        "speaks"
        not in record["focused_summary"]
    )


print("=" * 80)
print("FOCUSED GENERATION COMPLETE")
print("=" * 80)

print(
    "Originally requested focused tasks:",
    len(
        missing_focused_generation_tasks
    ),
)

print(
    "Runnable focused tasks:",
    len(
        focused_expected_keys
    ),
)

print(
    "Skipped invalid/short segments:",
    len(
        focused_audit_skipped_signatures
    ),
)

print(
    "Raw generations completed:",
    focused_raw_completed,
)

print(
    "Clean focused summaries:",
    focused_clean_completed,
)

print(
    "Parse/schema failures:",
    len(
        focused_parse_failures
    ),
)

print(
    "Raw checkpoint:",
    MISSING_FOCUSED_RAW_CACHE_PATH,
)

print(
    "Clean focused summaries without speaks:",
    MISSING_FOCUSED_SUMMARIES_PATH,
)

print(
    "Skipped-task manifest:",
    SKIPPED_FOCUSED_TASKS_PATH,
)

Generating missing focused summaries:   0%|          | 0/274 [00:00<?, ?it/s]


SKIPPING: V01_S1545_I00000138 P2518 segment=1
FOCUSED GENERATION COMPLETE
Originally requested focused tasks: 274
Runnable focused tasks: 273
Skipped invalid/short segments: 1
Raw generations completed: 273
Clean focused summaries: 265
Parse/schema failures: 8
Raw checkpoint: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/missing_focused_raw_generation_records.json
Clean focused summaries without speaks: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/missing_focused_summaries_clean_without_speaks.json
Skipped-task manifest: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/skipped_focused_summary_tasks.json


# Fill the Participants of the Dataset with the coarse and focused summaries i just created

In [ ]:
# ============================================================
# LOAD AND INDEX THE NEWLY GENERATED COARSE + FOCUSED SUMMARIES
# ============================================================

from pathlib import Path

import copy
import json
import pandas as pd


required_names = [
    "OUT_DIR",

    "semantic_normal_cases",
    "semantic_wrong_cases",
    "semantic_silent_cases",

    "MISSING_COARSE_SUMMARIES_PATH",
    "MISSING_FOCUSED_SUMMARIES_PATH",
]


missing_names = [
    name
    for name in required_names
    if name not in globals()
]


assert not missing_names, (
    "Run the previous semantic and generation cells first. "
    f"Missing: {missing_names}"
)


assert Path(
    MISSING_COARSE_SUMMARIES_PATH
).exists(), (
    "Generated coarse summary file not found: "
    f"{MISSING_COARSE_SUMMARIES_PATH}"
)


assert Path(
    MISSING_FOCUSED_SUMMARIES_PATH
).exists(), (
    "Generated focused summary file not found: "
    f"{MISSING_FOCUSED_SUMMARIES_PATH}"
)


def load_generated_summary_file(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )

    assert isinstance(
        data,
        dict,
    ), (
        f"Expected a JSON dictionary in: {path}"
    )

    return data


generated_coarse_records = (
    load_generated_summary_file(
        MISSING_COARSE_SUMMARIES_PATH
    )
)


generated_focused_records = (
    load_generated_summary_file(
        MISSING_FOCUSED_SUMMARIES_PATH
    )
)


def build_generated_summary_index(
    records,
    summary_field,
):
    """
    Build an exact index using:

        source_kind
        conversation_id
        participant_id
        segment_idx
    """

    index = {}


    for cache_key, record in records.items():

        assert isinstance(
            record,
            dict,
        )


        identity = (
            str(
                record["source_kind"]
            ),

            str(
                record["conversation_id"]
            ),

            str(
                record["participant_id"]
            ),

            int(
                record["segment_idx"]
            ),
        )


        summary = record.get(
            summary_field
        )


        assert summary is not None, (
            f"Missing {summary_field} in clean record: "
            f"{cache_key}"
        )


        assert isinstance(
            summary,
            dict,
        )


        if identity in index:

            assert index[
                identity
            ] == summary, (
                "Conflicting generated summaries found for "
                f"{identity}"
            )

        else:

            index[
                identity
            ] = copy.deepcopy(
                summary
            )


    return index


generated_coarse_index = (
    build_generated_summary_index(
        records=generated_coarse_records,
        summary_field="coarse_summary",
    )
)


generated_focused_index = (
    build_generated_summary_index(
        records=generated_focused_records,
        summary_field="focused_summary",
    )
)


assert len(
    generated_coarse_index
) == 273


assert len(
    generated_focused_index
) == 265


for summary in generated_coarse_index.values():

    assert set(
        summary.keys()
    ) == {
        "speech_content_summary",
        "apparent_topic",
    }


for summary in generated_focused_index.values():

    assert "speaks" not in summary


print("=" * 80)
print("GENERATED SUMMARY INDEXES LOADED")
print("=" * 80)

print(
    "Generated coarse summaries:",
    len(
        generated_coarse_index
    ),
)

print(
    "Generated focused summaries:",
    len(
        generated_focused_index
    ),
)

GENERATED SUMMARY INDEXES LOADED
Generated coarse summaries: 273
Generated focused summaries: 265


In [ ]:
# ============================================================
# FILL ONLY MISSING SUMMARIES IN NORMAL, WRONG AND SILENT CASES
# ============================================================

FINAL_SEGMENT_NAMES = {
    0: "segment_0_0_to_60_seconds",
    1: "segment_1_60_to_120_seconds",
}


SEMANTIC_ONLY_FIELDS = {
    "semantic_summaries",
    "semantic_summary_coverage",
    "semantic_summary_source",
}


def direct_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"

    if variant == "wrong_partner":
        return "wrong_partner"

    if variant == "silent_partner":
        return "silent_partner"

    if variant.startswith(
        "lag_"
    ):
        return "lag"

    return variant


def participant_identity(
    participant,
):
    return (
        str(
            participant[
                "conversation_id"
            ]
        ),

        str(
            participant[
                "participant_id"
            ]
        ),
    )


def source_kind_for_case_role(
    case,
    role,
):
    family = direct_case_family(
        case
    )


    if (
        family == "silent_partner"
        and role == "participant_B"
    ):
        return "silent"


    return "normal"


def recalculate_case_semantic_coverage(
    case,
):
    available_coarse = 0
    available_focused = 0


    for role in [
        "participant_A",
        "participant_B",
    ]:

        for segment_name in (
            FINAL_SEGMENT_NAMES.values()
        ):

            segment_record = (
                case[
                    "semantic_summaries"
                ][role][segment_name]
            )


            available_coarse += int(
                segment_record.get(
                    "coarse_summary"
                )
                is not None
            )


            available_focused += int(
                segment_record.get(
                    "focused_summary"
                )
                is not None
            )


    case[
        "semantic_summary_coverage"
    ] = {
        "expected_coarse_summaries": 4,

        "available_coarse_summaries": (
            available_coarse
        ),

        "missing_coarse_summaries": (
            4 - available_coarse
        ),

        "expected_focused_summaries": 4,

        "available_focused_summaries": (
            available_focused
        ),

        "missing_focused_summaries": (
            4 - available_focused
        ),

        "all_summaries_available": (
            available_coarse == 4
            and available_focused == 4
        ),
    }


def remove_semantic_fields_for_comparison(
    case,
):
    return {
        key: value

        for key, value in case.items()

        if key not in SEMANTIC_ONLY_FIELDS
    }


def fill_missing_generated_summaries(
    cases,
    family_name,
):
    """
    Fill only None summaries.

    Existing non-None summaries are never overwritten.
    """

    updated_cases = copy.deepcopy(
        cases
    )


    insertion_records = []


    for case in updated_cases:

        assert (
            "semantic_summaries"
            in case
        )


        for role in [
            "participant_A",
            "participant_B",
        ]:

            participant = case[role]


            source_kind = (
                source_kind_for_case_role(
                    case=case,
                    role=role,
                )
            )


            conversation_id = str(
                participant[
                    "conversation_id"
                ]
            )


            participant_id = str(
                participant[
                    "participant_id"
                ]
            )


            for segment_idx, segment_name in (
                FINAL_SEGMENT_NAMES.items()
            ):

                segment_record = (
                    case[
                        "semantic_summaries"
                    ][role][segment_name]
                )


                identity = (
                    source_kind,
                    conversation_id,
                    participant_id,
                    int(segment_idx),
                )


                # --------------------------------------------
                # Fill missing coarse summary
                # --------------------------------------------

                if (
                    segment_record.get(
                        "coarse_summary"
                    )
                    is None
                ):

                    generated_coarse = (
                        generated_coarse_index.get(
                            identity
                        )
                    )


                    if generated_coarse is not None:

                        segment_record[
                            "coarse_summary"
                        ] = copy.deepcopy(
                            generated_coarse
                        )


                        insertion_records.append({
                            "case_family": family_name,

                            "case_id": (
                                case["case_id"]
                            ),

                            "participant_role": role,

                            "source_kind": (
                                source_kind
                            ),

                            "conversation_id": (
                                conversation_id
                            ),

                            "participant_id": (
                                participant_id
                            ),

                            "segment_idx": (
                                int(segment_idx)
                            ),

                            "summary_type": (
                                "coarse"
                            ),

                            "identity": identity,
                        })


                # --------------------------------------------
                # Fill missing focused summary
                # --------------------------------------------

                if (
                    segment_record.get(
                        "focused_summary"
                    )
                    is None
                ):

                    generated_focused = (
                        generated_focused_index.get(
                            identity
                        )
                    )


                    if generated_focused is not None:

                        segment_record[
                            "focused_summary"
                        ] = copy.deepcopy(
                            generated_focused
                        )


                        insertion_records.append({
                            "case_family": family_name,

                            "case_id": (
                                case["case_id"]
                            ),

                            "participant_role": role,

                            "source_kind": (
                                source_kind
                            ),

                            "conversation_id": (
                                conversation_id
                            ),

                            "participant_id": (
                                participant_id
                            ),

                            "segment_idx": (
                                int(segment_idx)
                            ),

                            "summary_type": (
                                "focused"
                            ),

                            "identity": identity,
                        })


        recalculate_case_semantic_coverage(
            case
        )


        case[
            "semantic_summary_source"
        ] = {
            "strategy": (
                "existing_cache_plus_generated_missing_fill"
            ),

            "existing_summaries_preserved": True,

            "generated_coarse_cache": str(
                MISSING_COARSE_SUMMARIES_PATH
            ),

            "generated_focused_cache": str(
                MISSING_FOCUSED_SUMMARIES_PATH
            ),

            "generated_focused_summaries_include_speaks": (
                False
            ),
        }


    return (
        updated_cases,
        insertion_records,
    )


(
    completed_semantic_normal_cases,
    normal_insertion_records,
) = fill_missing_generated_summaries(
    cases=semantic_normal_cases,
    family_name="normal",
)


(
    completed_semantic_wrong_cases,
    wrong_insertion_records,
) = fill_missing_generated_summaries(
    cases=semantic_wrong_cases,
    family_name="wrong_partner",
)


(
    completed_semantic_silent_cases,
    silent_insertion_records,
) = fill_missing_generated_summaries(
    cases=semantic_silent_cases,
    family_name="silent_partner",
)


all_direct_insertion_records = (
    normal_insertion_records
    + wrong_insertion_records
    + silent_insertion_records
)


print("=" * 80)
print("DIRECT CASE SUMMARY FILL COMPLETE")
print("=" * 80)

print(
    "NORMAL summary slots filled:",
    len(
        normal_insertion_records
    ),
)

print(
    "WRONG summary slots filled:",
    len(
        wrong_insertion_records
    ),
)

print(
    "SILENT summary slots filled:",
    len(
        silent_insertion_records
    ),
)

print(
    "Total case-level summary slots filled:",
    len(
        all_direct_insertion_records
    ),
)

DIRECT CASE SUMMARY FILL COMPLETE
NORMAL summary slots filled: 370
WRONG summary slots filled: 370
SILENT summary slots filled: 573
Total case-level summary slots filled: 1313


In [ ]:
# ============================================================
# REBUILD LAG SUMMARIES FROM THE UPDATED NORMAL CASES
#
# LAG does not perform any independent semantic lookup.
# It inherits exactly the corresponding updated NORMAL
# participant summaries.
# ============================================================

completed_normal_by_group = {
    case[
        "source_group_id"
    ]: case

    for case
    in completed_semantic_normal_cases
}


assert len(
    completed_normal_by_group
) == 100


if (
    "current_lag_cases"
    in globals()
    and isinstance(
        current_lag_cases,
        list,
    )
    and len(
        current_lag_cases
    ) == 100
):

    lag_base_cases = (
        current_lag_cases
    )

else:

    assert (
        "semantic_lag_cases"
        in globals()
    )

    lag_base_cases = (
        semantic_lag_cases
    )


completed_semantic_lag_cases = []


for lag_case in lag_base_cases:

    source_group_id = (
        lag_case[
            "source_group_id"
        ]
    )


    assert (
        source_group_id
        in completed_normal_by_group
    )


    corresponding_normal = (
        completed_normal_by_group[
            source_group_id
        ]
    )


    assert participant_identity(
        lag_case[
            "participant_A"
        ]
    ) == participant_identity(
        corresponding_normal[
            "participant_A"
        ]
    )


    assert participant_identity(
        lag_case[
            "participant_B"
        ]
    ) == participant_identity(
        corresponding_normal[
            "participant_B"
        ]
    )


    completed_lag_case = copy.deepcopy(
        lag_case
    )


    completed_lag_case[
        "semantic_summaries"
    ] = copy.deepcopy(
        corresponding_normal[
            "semantic_summaries"
        ]
    )


    completed_lag_case[
        "semantic_summary_coverage"
    ] = copy.deepcopy(
        corresponding_normal[
            "semantic_summary_coverage"
        ]
    )


    completed_lag_case[
        "semantic_summary_source"
    ] = {
        "strategy": (
            "inherited_exactly_from_completed_normal_case"
        ),

        "normal_source_case_id": (
            corresponding_normal[
                "case_id"
            ]
        ),

        "reason": (
            "Synthetic temporal lag changes VAD timing "
            "but not participant video semantics."
        ),
    }


    completed_semantic_lag_cases.append(
        completed_lag_case
    )


assert len(
    completed_semantic_lag_cases
) == 100


print(
    "Completed LAG cases:",
    len(
        completed_semantic_lag_cases
    ),
)

Completed LAG cases: 100


In [ ]:
# ============================================================
# STRICT AUDIT, REMAINING GAPS AND FINAL DATABASE SAVE
# ============================================================

# ------------------------------------------------------------
# 1. Basic case counts
# ------------------------------------------------------------

assert len(
    completed_semantic_normal_cases
) == 100


assert len(
    completed_semantic_wrong_cases
) == 100


assert len(
    completed_semantic_lag_cases
) == 100


assert len(
    completed_semantic_silent_cases
) == 100


# ------------------------------------------------------------
# 2. Verify that no temporal/case information changed
# ------------------------------------------------------------

for original_case, completed_case in zip(
    semantic_normal_cases,
    completed_semantic_normal_cases,
):

    assert (
        remove_semantic_fields_for_comparison(
            original_case
        )
        ==
        remove_semantic_fields_for_comparison(
            completed_case
        )
    )


for original_case, completed_case in zip(
    semantic_wrong_cases,
    completed_semantic_wrong_cases,
):

    assert (
        remove_semantic_fields_for_comparison(
            original_case
        )
        ==
        remove_semantic_fields_for_comparison(
            completed_case
        )
    )


for original_case, completed_case in zip(
    semantic_silent_cases,
    completed_semantic_silent_cases,
):

    assert (
        remove_semantic_fields_for_comparison(
            original_case
        )
        ==
        remove_semantic_fields_for_comparison(
            completed_case
        )
    )


for original_case, completed_case in zip(
    lag_base_cases,
    completed_semantic_lag_cases,
):

    assert (
        remove_semantic_fields_for_comparison(
            original_case
        )
        ==
        remove_semantic_fields_for_comparison(
            completed_case
        )
    )


# ------------------------------------------------------------
# 3. Verify every inserted summary against its exact cache
# ------------------------------------------------------------

completed_cases_by_id = {
    case[
        "case_id"
    ]: case

    for case in (
        completed_semantic_normal_cases
        + completed_semantic_wrong_cases
        + completed_semantic_silent_cases
    )
}


for insertion in all_direct_insertion_records:

    case = completed_cases_by_id[
        insertion[
            "case_id"
        ]
    ]


    segment_name = (
        FINAL_SEGMENT_NAMES[
            insertion[
                "segment_idx"
            ]
        ]
    )


    stored_summary = (
        case[
            "semantic_summaries"
        ][
            insertion[
                "participant_role"
            ]
        ][
            segment_name
        ][
            (
                "coarse_summary"
                if insertion[
                    "summary_type"
                ] == "coarse"
                else "focused_summary"
            )
        ]
    )


    if insertion[
        "summary_type"
    ] == "coarse":

        expected_summary = (
            generated_coarse_index[
                insertion[
                    "identity"
                ]
            ]
        )

    else:

        expected_summary = (
            generated_focused_index[
                insertion[
                    "identity"
                ]
            ]
        )


    assert stored_summary == expected_summary


# ------------------------------------------------------------
# 4. Every LAG must exactly equal its corresponding NORMAL
# ------------------------------------------------------------

for lag_case in completed_semantic_lag_cases:

    corresponding_normal = (
        completed_normal_by_group[
            lag_case[
                "source_group_id"
            ]
        ]
    )


    assert (
        lag_case[
            "semantic_summaries"
        ]
        ==
        corresponding_normal[
            "semantic_summaries"
        ]
    )


    assert (
        lag_case[
            "semantic_summary_coverage"
        ]
        ==
        corresponding_normal[
            "semantic_summary_coverage"
        ]
    )


# ------------------------------------------------------------
# 5. Build the remaining unique missing-summary manifest
#
# LAG is excluded because it duplicates NORMAL semantics.
# ------------------------------------------------------------

remaining_missing_registry = {}


direct_completed_cases = (
    completed_semantic_normal_cases
    + completed_semantic_wrong_cases
    + completed_semantic_silent_cases
)


for case in direct_completed_cases:

    family = direct_case_family(
        case
    )


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant = case[role]


        source_kind = (
            source_kind_for_case_role(
                case=case,
                role=role,
            )
        )


        conversation_id = str(
            participant[
                "conversation_id"
            ]
        )


        participant_id = str(
            participant[
                "participant_id"
            ]
        )


        for segment_idx, segment_name in (
            FINAL_SEGMENT_NAMES.items()
        ):

            segment_record = (
                case[
                    "semantic_summaries"
                ][role][segment_name]
            )


            coarse_missing = (
                segment_record.get(
                    "coarse_summary"
                )
                is None
            )


            focused_missing = (
                segment_record.get(
                    "focused_summary"
                )
                is None
            )


            if not (
                coarse_missing
                or focused_missing
            ):
                continue


            identity = (
                source_kind,
                conversation_id,
                participant_id,
                int(segment_idx),
            )


            if identity not in remaining_missing_registry:

                remaining_missing_registry[
                    identity
                ] = {
                    "source_kind": (
                        source_kind
                    ),

                    "conversation_id": (
                        conversation_id
                    ),

                    "participant_id": (
                        participant_id
                    ),

                    "segment_idx": (
                        int(segment_idx)
                    ),

                    "segment_start_seconds": (
                        0
                        if segment_idx == 0
                        else 60
                    ),

                    "segment_end_seconds": (
                        60
                        if segment_idx == 0
                        else 120
                    ),

                    "coarse_missing": (
                        coarse_missing
                    ),

                    "focused_missing": (
                        focused_missing
                    ),

                    "appears_in_case_families": {
                        family
                    },

                    "appears_in_case_ids": {
                        case[
                            "case_id"
                        ]
                    },
                }

            else:

                existing = (
                    remaining_missing_registry[
                        identity
                    ]
                )


                assert (
                    existing[
                        "coarse_missing"
                    ]
                    == coarse_missing
                )


                assert (
                    existing[
                        "focused_missing"
                    ]
                    == focused_missing
                )


                existing[
                    "appears_in_case_families"
                ].add(
                    family
                )


                existing[
                    "appears_in_case_ids"
                ].add(
                    case[
                        "case_id"
                    ]
                )


remaining_missing_summaries = []


for identity, record in sorted(
    remaining_missing_registry.items()
):

    remaining_missing_summaries.append({
        **{
            key: value

            for key, value in record.items()

            if key not in {
                "appears_in_case_families",
                "appears_in_case_ids",
            }
        },

        "appears_in_case_families": sorted(
            record[
                "appears_in_case_families"
            ]
        ),

        "num_case_appearances": len(
            record[
                "appears_in_case_ids"
            ]
        ),
    })


# ------------------------------------------------------------
# 6. Final coverage report
# ------------------------------------------------------------

final_coverage_rows = []


FINAL_CASES_BY_FAMILY = {
    "normal": (
        completed_semantic_normal_cases
    ),

    "wrong_partner": (
        completed_semantic_wrong_cases
    ),

    "lag": (
        completed_semantic_lag_cases
    ),

    "silent_partner": (
        completed_semantic_silent_cases
    ),
}


for family, cases in (
    FINAL_CASES_BY_FAMILY.items()
):

    for case in cases:

        coverage = (
            case[
                "semantic_summary_coverage"
            ]
        )


        final_coverage_rows.append({
            "case_family": family,

            "case_id": (
                case["case_id"]
            ),

            "available_coarse_summaries": (
                coverage[
                    "available_coarse_summaries"
                ]
            ),

            "missing_coarse_summaries": (
                coverage[
                    "missing_coarse_summaries"
                ]
            ),

            "available_focused_summaries": (
                coverage[
                    "available_focused_summaries"
                ]
            ),

            "missing_focused_summaries": (
                coverage[
                    "missing_focused_summaries"
                ]
            ),

            "all_summaries_available": (
                coverage[
                    "all_summaries_available"
                ]
            ),
        })


final_coverage_df = pd.DataFrame(
    final_coverage_rows
)


final_coverage_summary_df = (
    final_coverage_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        complete_cases=(
            "all_summaries_available",
            "sum",
        ),

        available_coarse_summaries=(
            "available_coarse_summaries",
            "sum",
        ),

        missing_coarse_summaries=(
            "missing_coarse_summaries",
            "sum",
        ),

        available_focused_summaries=(
            "available_focused_summaries",
            "sum",
        ),

        missing_focused_summaries=(
            "missing_focused_summaries",
            "sum",
        ),
    )
)


# ------------------------------------------------------------
# 7. Create final 400-case database
# ------------------------------------------------------------

completed_normal_wrong_cases = (
    completed_semantic_normal_cases
    + completed_semantic_wrong_cases
)


completed_all_400_cases = (
    completed_semantic_normal_cases
    + completed_semantic_wrong_cases
    + completed_semantic_lag_cases
    + completed_semantic_silent_cases
)


assert len(
    completed_all_400_cases
) == 400


final_case_ids = [
    case[
        "case_id"
    ]

    for case
    in completed_all_400_cases
]


assert len(
    final_case_ids
) == len(
    set(
        final_case_ids
    )
)


assert sum(
    case[
        "gold_binary_label"
    ] == "NORMAL"

    for case
    in completed_all_400_cases
) == 100


assert sum(
    case[
        "gold_binary_label"
    ] == "ANOMALOUS"

    for case
    in completed_all_400_cases
) == 300


# ------------------------------------------------------------
# 8. Output paths
# ------------------------------------------------------------

FINAL_NORMAL_WRONG_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_normal_wrong_cases_with_"
        "temporal_and_completed_semantic_summaries.json"
    )
)


FINAL_LAG_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_lag_cases_with_"
        "temporal_and_completed_semantic_summaries.json"
    )
)


FINAL_SILENT_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_silent_partner_cases_with_"
        "temporal_and_completed_semantic_summaries.json"
    )
)


FINAL_ALL_400_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_all_400_cases_with_"
        "temporal_and_semantic_summaries.json"
    )
)


FINAL_REMAINING_MISSING_PATH = (
    OUT_DIR
    / (
        "consolidation_remaining_missing_semantic_"
        "summaries_after_generation.json"
    )
)


FINAL_SEMANTIC_COVERAGE_CSV_PATH = (
    OUT_DIR
    / (
        "consolidation_final_semantic_coverage_"
        "after_generation.csv"
    )
)


# ------------------------------------------------------------
# 9. Save everything
# ------------------------------------------------------------

FINAL_NORMAL_WRONG_DATABASE_PATH.write_text(
    json.dumps(
        completed_normal_wrong_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


FINAL_LAG_DATABASE_PATH.write_text(
    json.dumps(
        completed_semantic_lag_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


FINAL_SILENT_DATABASE_PATH.write_text(
    json.dumps(
        completed_semantic_silent_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


FINAL_ALL_400_DATABASE_PATH.write_text(
    json.dumps(
        completed_all_400_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


FINAL_REMAINING_MISSING_PATH.write_text(
    json.dumps(
        remaining_missing_summaries,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


final_coverage_df.to_csv(
    FINAL_SEMANTIC_COVERAGE_CSV_PATH,
    index=False,
)


print("=" * 88)
print("FINAL SEMANTIC DATABASE ENRICHMENT AUDIT PASSED")
print("=" * 88)

display(
    final_coverage_summary_df
)


print(
    "\nUnique remaining participant-segment gaps:",
    len(
        remaining_missing_summaries
    ),
)

print(
    "Remaining coarse gaps:",
    sum(
        record[
            "coarse_missing"
        ]

        for record
        in remaining_missing_summaries
    ),
)

print(
    "Remaining focused gaps:",
    sum(
        record[
            "focused_missing"
        ]

        for record
        in remaining_missing_summaries
    ),
)


print(
    "\nNORMAL + WRONG database:",
    FINAL_NORMAL_WRONG_DATABASE_PATH,
)

print(
    "LAG database:",
    FINAL_LAG_DATABASE_PATH,
)

print(
    "SILENT database:",
    FINAL_SILENT_DATABASE_PATH,
)

print(
    "Combined 400-case database:",
    FINAL_ALL_400_DATABASE_PATH,
)

print(
    "Remaining gaps:",
    FINAL_REMAINING_MISSING_PATH,
)

print(
    "Final coverage CSV:",
    FINAL_SEMANTIC_COVERAGE_CSV_PATH,
)

FINAL SEMANTIC DATABASE ENRICHMENT AUDIT PASSED


,case_family,total_cases,complete_cases,available_coarse_summaries,missing_coarse_summaries,available_focused_summaries,missing_focused_summaries
0,lag,100,94,400,0,394,6
1,normal,100,94,400,0,394,6
2,silent_partner,100,91,397,3,388,12
3,wrong_partner,100,94,400,0,394,6



Unique remaining participant-segment gaps: 9
Remaining coarse gaps: 1
Remaining focused gaps: 9

NORMAL + WRONG database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_normal_wrong_cases_with_temporal_and_completed_semantic_summaries.json
LAG database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_lag_cases_with_temporal_and_completed_semantic_summaries.json
SILENT database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_silent_partner_cases_with_temporal_and_completed_semantic_summaries.json
Combined 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Remaining gaps: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_remaining_missing_semantic_summaries_after_generation.json
Final coverage CSV: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_la

# Replace the problematic silent partner

In [ ]:
# ============================================================
# REPLACE THE SHORT/INCOMPLETE SILENT PARTICIPANT
#
# Target:
#   V01_S1545_I00000138 / P2518
#
# Procedure:
# 1. Verify that the target really has an explicit empty
#    metadata:vad list.
# 2. Find another real silent participant already used in the
#    silent database with:
#       - explicit empty metadata:vad
#       - complete coarse summaries for both segments
#       - complete focused summaries for both segments
#       - no duplicate A/B pairing after replacement
# 3. Replace the target Participant B in every affected sample.
# 4. Preserve every temporal feature exactly.
# 5. Save the repaired silent and combined 400-case databases.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict

import copy
import json
import shutil
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

TARGET_SILENT_CONVERSATION_ID = (
    "V01_S1545_I00000138"
)

TARGET_SILENT_PARTICIPANT_ID = (
    "P2518"
)


if "SILENT_ROOT" not in globals():

    assert "DATA_ROOT" in globals()

    SILENT_ROOT = (
        Path(DATA_ROOT)
        / "silent"
    )


SILENT_ROOT = Path(
    SILENT_ROOT
)


assert SILENT_ROOT.exists(), (
    f"Silent root not found: {SILENT_ROOT}"
)


assert (
    "completed_semantic_silent_cases"
    in globals()
), (
    "Run the semantic database enrichment cell first."
)


assert (
    "completed_semantic_normal_cases"
    in globals()
)


assert (
    "completed_semantic_wrong_cases"
    in globals()
)


assert (
    "completed_semantic_lag_cases"
    in globals()
)


SEGMENT_NAMES = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def participant_identity(
    participant,
):
    return (
        str(
            participant[
                "conversation_id"
            ]
        ),

        str(
            participant[
                "participant_id"
            ]
        ),
    )


def find_silent_metadata_path(
    participant,
):
    """
    Resolve the participant metadata from the dedicated
    silent directory.

    The metadata_path already stored in the case is checked
    first, followed by the expected silent-folder path.
    """

    conversation_id = str(
        participant[
            "conversation_id"
        ]
    )

    participant_id = str(
        participant[
            "participant_id"
        ]
    )


    candidates = []


    stored_metadata_path = (
        participant.get(
            "metadata_path"
        )
    )


    if stored_metadata_path:

        candidates.append(
            Path(
                stored_metadata_path
            )
        )


    participant_dir = (
        SILENT_ROOT
        / conversation_id
        / participant_id
    )


    expected_path = (
        participant_dir
        / (
            f"{conversation_id}_"
            f"{participant_id}.json"
        )
    )


    candidates.append(
        expected_path
    )


    if participant_dir.exists():

        candidates.extend([
            path

            for path in sorted(
                participant_dir.glob(
                    "*.json"
                )
            )

            if not path.name.endswith(
                ".metadata.json"
            )
        ])


    checked_paths = set()


    for candidate in candidates:

        candidate = Path(
            candidate
        )


        candidate_string = str(
            candidate
        )


        if candidate_string in checked_paths:
            continue


        checked_paths.add(
            candidate_string
        )


        if (
            candidate.exists()
            and candidate.is_file()
        ):
            return candidate


    raise FileNotFoundError(
        "Could not locate silent metadata for "
        f"{conversation_id}/{participant_id}"
    )


def load_and_verify_empty_silent_metadata(
    participant,
    label,
):
    """
    Strict definition of silent:

    - metadata JSON is readable;
    - metadata:vad exists;
    - metadata:vad is a list;
    - metadata:vad has length zero.
    """

    metadata_path = (
        find_silent_metadata_path(
            participant
        )
    )


    metadata = json.loads(
        metadata_path.read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        metadata,
        dict,
    ), (
        f"{label}: metadata is not a JSON object."
    )


    assert (
        "metadata:vad"
        in metadata
    ), (
        f"{label}: explicit metadata:vad field is missing."
    )


    vad = metadata[
        "metadata:vad"
    ]


    assert isinstance(
        vad,
        list,
    ), (
        f"{label}: metadata:vad is not a list."
    )


    assert len(
        vad
    ) == 0, (
        f"{label}: metadata:vad is not empty. "
        f"Found {len(vad)} entries."
    )


    if (
        participant.get(
            "num_raw_vad_entries"
        )
        is not None
    ):

        assert int(
            participant[
                "num_raw_vad_entries"
            ]
        ) == 0, (
            f"{label}: stored num_raw_vad_entries "
            "is not zero."
        )


    return (
        metadata_path,
        metadata,
    )


def participant_semantics_are_complete(
    participant_semantics,
):
    for segment_name in SEGMENT_NAMES:

        if (
            segment_name
            not in participant_semantics
        ):
            return False


        segment_record = (
            participant_semantics[
                segment_name
            ]
        )


        if (
            segment_record.get(
                "coarse_summary"
            )
            is None
        ):
            return False


        if (
            segment_record.get(
                "focused_summary"
            )
            is None
        ):
            return False


    return True


def recalculate_semantic_coverage(
    case,
):
    available_coarse = 0
    available_focused = 0


    for role in [
        "participant_A",
        "participant_B",
    ]:

        for segment_name in SEGMENT_NAMES:

            segment_record = (
                case[
                    "semantic_summaries"
                ][role][segment_name]
            )


            available_coarse += int(
                segment_record.get(
                    "coarse_summary"
                )
                is not None
            )


            available_focused += int(
                segment_record.get(
                    "focused_summary"
                )
                is not None
            )


    case[
        "semantic_summary_coverage"
    ] = {
        "expected_coarse_summaries": 4,

        "available_coarse_summaries": (
            available_coarse
        ),

        "missing_coarse_summaries": (
            4 - available_coarse
        ),

        "expected_focused_summaries": 4,

        "available_focused_summaries": (
            available_focused
        ),

        "missing_focused_summaries": (
            4 - available_focused
        ),

        "all_summaries_available": (
            available_coarse == 4
            and available_focused == 4
        ),
    }


def case_without_replaceable_fields(
    case,
):
    """
    Everything outside these fields must stay byte-for-byte
    equivalent as a Python object.

    In particular, temporal turns and temporal features
    must not change.
    """

    replaceable_fields = {
        "participant_B",
        "B_source_conversation",
        "semantic_summaries",
        "semantic_summary_coverage",
    }


    return {
        key: value

        for key, value in case.items()

        if key not in replaceable_fields
    }


# ============================================================
# COPY THE CURRENT SILENT DATABASE
# ============================================================

repaired_semantic_silent_cases = (
    copy.deepcopy(
        completed_semantic_silent_cases
    )
)


assert len(
    repaired_semantic_silent_cases
) == 100


for case in repaired_semantic_silent_cases:

    assert str(
        case.get(
            "case_variant"
        )
    ).lower() == "silent_partner"


# ============================================================
# LOCATE EVERY SAMPLE CONTAINING THE TARGET
# ============================================================

target_identity = (
    TARGET_SILENT_CONVERSATION_ID,
    TARGET_SILENT_PARTICIPANT_ID,
)


affected_case_indexes = [
    index

    for index, case in enumerate(
        repaired_semantic_silent_cases
    )

    if participant_identity(
        case[
            "participant_B"
        ]
    ) == target_identity
]


assert affected_case_indexes, (
    "No silent-partner cases were found with target "
    f"{target_identity}. The replacement may already "
    "have been performed."
)


affected_case_ids = [
    repaired_semantic_silent_cases[
        index
    ][
        "case_id"
    ]

    for index in affected_case_indexes
]


print("=" * 88)
print("TARGET SILENT PARTICIPANT")
print("=" * 88)

print(
    "Target identity:",
    target_identity,
)

print(
    "Affected silent samples:",
    len(
        affected_case_indexes
    ),
)

print(
    "Affected case IDs:",
    affected_case_ids,
)


# ============================================================
# STRICTLY VERIFY THAT THE TARGET IS REALLY SILENT
# ============================================================

target_participant_records = [
    repaired_semantic_silent_cases[
        index
    ][
        "participant_B"
    ]

    for index in affected_case_indexes
]


for participant_record in (
    target_participant_records
):

    assert participant_identity(
        participant_record
    ) == target_identity


target_metadata_path, target_metadata = (
    load_and_verify_empty_silent_metadata(
        participant=(
            target_participant_records[0]
        ),

        label="Target silent participant",
    )
)


# All affected cases must already have an empty B timeline.
for index in affected_case_indexes:

    case = (
        repaired_semantic_silent_cases[
            index
        ]
    )


    assert (
        case[
            "participant_B_filtered_turns"
        ]
        == []
    ), (
        f'Target case {case["case_id"]} does not '
        "have empty Participant B filtered turns."
    )


print(
    "Target metadata path:",
    target_metadata_path,
)

print(
    "Target metadata:vad entries:",
    len(
        target_metadata[
            "metadata:vad"
        ]
    ),
)

print(
    "TARGET VERIFIED AS SILENT: metadata:vad == []"
)


# ============================================================
# BUILD A REGISTRY OF OTHER SILENT PARTICIPANTS
# ============================================================

silent_occurrences_by_identity = (
    defaultdict(
        list
    )
)


for case in repaired_semantic_silent_cases:

    identity = participant_identity(
        case[
            "participant_B"
        ]
    )


    if identity == target_identity:
        continue


    silent_occurrences_by_identity[
        identity
    ].append(
        case
    )


# Current number of sample appearances for each silent participant.
silent_usage_counts = Counter(
    participant_identity(
        case[
            "participant_B"
        ]
    )

    for case in repaired_semantic_silent_cases
)


# Existing A/B combinations, excluding the cases that will change.
existing_unaffected_pairs = set()


for index, case in enumerate(
    repaired_semantic_silent_cases
):

    if index in affected_case_indexes:
        continue


    A_identity = participant_identity(
        case[
            "participant_A"
        ]
    )


    B_identity = participant_identity(
        case[
            "participant_B"
        ]
    )


    existing_unaffected_pairs.add(
        (
            A_identity,
            B_identity,
        )
    )


affected_A_identities = [
    participant_identity(
        repaired_semantic_silent_cases[
            index
        ][
            "participant_A"
        ]
    )

    for index in affected_case_indexes
]


# ============================================================
# FIND VALID COMPLETE REPLACEMENT CANDIDATES
# ============================================================

replacement_candidates = []

candidate_rejections = []


for (
    candidate_identity,
    candidate_cases,
) in sorted(
    silent_occurrences_by_identity.items()
):

    exemplar_case = (
        candidate_cases[0]
    )


    candidate_participant = (
        exemplar_case[
            "participant_B"
        ]
    )


    candidate_semantics = (
        exemplar_case[
            "semantic_summaries"
        ][
            "participant_B"
        ]
    )


    # Candidate must have both coarse and focused summaries
    # for both 60-second segments.
    if not participant_semantics_are_complete(
        candidate_semantics
    ):

        candidate_rejections.append({
            "identity": candidate_identity,
            "reason": (
                "incomplete_coarse_or_focused_summaries"
            ),
        })

        continue


    # All appearances of this silent participant must carry
    # exactly the same semantic summaries.
    semantics_are_consistent = all(
        case[
            "semantic_summaries"
        ][
            "participant_B"
        ] == candidate_semantics

        for case in candidate_cases
    )


    if not semantics_are_consistent:

        candidate_rejections.append({
            "identity": candidate_identity,
            "reason": (
                "inconsistent_semantics_across_cases"
            ),
        })

        continue


    # Every existing sample using the candidate must already
    # have empty B filtered turns.
    B_turns_are_empty = all(
        case[
            "participant_B_filtered_turns"
        ] == []

        for case in candidate_cases
    )


    if not B_turns_are_empty:

        candidate_rejections.append({
            "identity": candidate_identity,
            "reason": (
                "non_empty_B_filtered_turns"
            ),
        })

        continue


    # Verify the real source metadata.
    try:

        (
            candidate_metadata_path,
            candidate_metadata,
        ) = load_and_verify_empty_silent_metadata(
            participant=(
                candidate_participant
            ),

            label=(
                "Replacement candidate "
                f"{candidate_identity}"
            ),
        )

    except Exception as exc:

        candidate_rejections.append({
            "identity": candidate_identity,
            "reason": (
                "metadata_not_explicitly_empty"
            ),
            "error": str(exc),
        })

        continue


    # Do not create a duplicate normal-A + silent-B sample.
    would_create_duplicate_pair = any(
        (
            A_identity,
            candidate_identity,
        ) in existing_unaffected_pairs

        for A_identity
        in affected_A_identities
    )


    if would_create_duplicate_pair:

        candidate_rejections.append({
            "identity": candidate_identity,
            "reason": (
                "would_create_duplicate_A_B_pair"
            ),
        })

        continue


    normalized_candidate_participant = (
        copy.deepcopy(
            candidate_participant
        )
    )


    normalized_candidate_participant[
        "metadata_path"
    ] = str(
        candidate_metadata_path
    )


    normalized_candidate_participant[
        "num_raw_vad_entries"
    ] = 0


    replacement_candidates.append({
        "identity": (
            candidate_identity
        ),

        "participant": (
            normalized_candidate_participant
        ),

        "semantic_summaries": (
            copy.deepcopy(
                candidate_semantics
            )
        ),

        "metadata_path": str(
            candidate_metadata_path
        ),

        "current_usage_count": int(
            silent_usage_counts[
                candidate_identity
            ]
        ),
    })


assert replacement_candidates, (
    "No valid complete silent replacement participant "
    "was found."
)


# Pick deterministically:
# 1. least-used silent participant;
# 2. conversation ID;
# 3. participant ID.
replacement_candidates = sorted(
    replacement_candidates,
    key=lambda record: (
        record[
            "current_usage_count"
        ],

        record[
            "identity"
        ][0],

        record[
            "identity"
        ][1],
    ),
)


selected_replacement = (
    replacement_candidates[0]
)


replacement_identity = (
    selected_replacement[
        "identity"
    ]
)


replacement_participant = (
    selected_replacement[
        "participant"
    ]
)


replacement_semantics = (
    selected_replacement[
        "semantic_summaries"
    ]
)


assert replacement_identity != target_identity


assert participant_semantics_are_complete(
    replacement_semantics
)


print("\n" + "=" * 88)
print("SELECTED REPLACEMENT SILENT PARTICIPANT")
print("=" * 88)

print(
    "Replacement identity:",
    replacement_identity,
)

print(
    "Replacement metadata path:",
    selected_replacement[
        "metadata_path"
    ],
)

print(
    "Previous sample appearances:",
    selected_replacement[
        "current_usage_count"
    ],
)

print(
    "Replacement metadata:vad entries: 0"
)

print(
    "Replacement has complete coarse/focused summaries: True"
)


# ============================================================
# REPLACE THE TARGET IN EVERY AFFECTED SILENT SAMPLE
# ============================================================

replacement_audit_records = []


for index in affected_case_indexes:

    case = (
        repaired_semantic_silent_cases[
            index
        ]
    )


    before_case = copy.deepcopy(
        case
    )


    assert participant_identity(
        before_case[
            "participant_B"
        ]
    ) == target_identity


    assert (
        before_case[
            "participant_B_filtered_turns"
        ]
        == []
    )


    # Replace Participant B identity and metadata.
    case[
        "participant_B"
    ] = copy.deepcopy(
        replacement_participant
    )


    # Replace the source-conversation pointer.
    case[
        "B_source_conversation"
    ] = replacement_identity[0]


    # Replace only Participant B semantic summaries.
    case[
        "semantic_summaries"
    ][
        "participant_B"
    ] = copy.deepcopy(
        replacement_semantics
    )


    # Recalculate semantic availability.
    recalculate_semantic_coverage(
        case
    )


    # --------------------------------------------------------
    # Strict temporal/case invariance audit
    # --------------------------------------------------------

    assert (
        case_without_replaceable_fields(
            before_case
        )
        ==
        case_without_replaceable_fields(
            case
        )
    ), (
        "A field outside Participant B identity/source/"
        "semantics/coverage changed unexpectedly."
    )


    assert (
        case[
            "participant_A_filtered_turns"
        ]
        ==
        before_case[
            "participant_A_filtered_turns"
        ]
    )


    assert (
        case[
            "participant_B_filtered_turns"
        ]
        ==
        before_case[
            "participant_B_filtered_turns"
        ]
        == []
    )


    assert (
        case[
            "local_temporal_features"
        ]
        ==
        before_case[
            "local_temporal_features"
        ]
    )


    assert (
        case[
            "global_shift_features"
        ]
        ==
        before_case[
            "global_shift_features"
        ]
    )


    assert participant_identity(
        case[
            "participant_B"
        ]
    ) == replacement_identity


    assert (
        case[
            "semantic_summaries"
        ][
            "participant_B"
        ]
        == replacement_semantics
    )


    assert participant_semantics_are_complete(
        case[
            "semantic_summaries"
        ][
            "participant_B"
        ]
    )


    replacement_audit_records.append({
        "case_id": (
            case["case_id"]
        ),

        "participant_A": (
            participant_identity(
                case[
                    "participant_A"
                ]
            )
        ),

        "old_participant_B": (
            target_identity
        ),

        "new_participant_B": (
            replacement_identity
        ),

        "temporal_features_preserved": (
            True
        ),

        "new_B_semantics_complete": (
            True
        ),
    })


# ============================================================
# POST-REPLACEMENT AUDITS
# ============================================================

assert not any(
    participant_identity(
        case[
            "participant_B"
        ]
    ) == target_identity

    for case in repaired_semantic_silent_cases
)


assert sum(
    participant_identity(
        case[
            "participant_B"
        ]
    ) == replacement_identity

    for case in repaired_semantic_silent_cases
) == (
    selected_replacement[
        "current_usage_count"
    ]
    + len(
        affected_case_indexes
    )
)


assert len(
    repaired_semantic_silent_cases
) == 100


assert len({
    case[
        "case_id"
    ]

    for case
    in repaired_semantic_silent_cases
}) == 100


# Check that the replacement did not create duplicate A/B pairs.
repaired_A_B_pairs = [
    (
        participant_identity(
            case[
                "participant_A"
            ]
        ),

        participant_identity(
            case[
                "participant_B"
            ]
        ),
    )

    for case in repaired_semantic_silent_cases
]


assert len(
    repaired_A_B_pairs
) == len(
    set(
        repaired_A_B_pairs
    )
), (
    "Replacement created duplicate silent-partner A/B samples."
)


# ============================================================
# UPDATE THE FINAL IN-MEMORY DATABASES
# ============================================================

completed_semantic_silent_cases = (
    repaired_semantic_silent_cases
)


completed_all_400_cases = (
    completed_semantic_normal_cases
    + completed_semantic_wrong_cases
    + completed_semantic_lag_cases
    + completed_semantic_silent_cases
)


assert len(
    completed_all_400_cases
) == 400


assert len({
    case[
        "case_id"
    ]

    for case
    in completed_all_400_cases
}) == 400


# ============================================================
# REBUILD FINAL COVERAGE
# ============================================================

coverage_rows_after_replacement = []


cases_by_family_after_replacement = {
    "normal": (
        completed_semantic_normal_cases
    ),

    "wrong_partner": (
        completed_semantic_wrong_cases
    ),

    "lag": (
        completed_semantic_lag_cases
    ),

    "silent_partner": (
        completed_semantic_silent_cases
    ),
}


for family, cases in (
    cases_by_family_after_replacement.items()
):

    for case in cases:

        coverage = (
            case[
                "semantic_summary_coverage"
            ]
        )


        coverage_rows_after_replacement.append({
            "case_family": (
                family
            ),

            "case_id": (
                case["case_id"]
            ),

            "available_coarse_summaries": (
                coverage[
                    "available_coarse_summaries"
                ]
            ),

            "missing_coarse_summaries": (
                coverage[
                    "missing_coarse_summaries"
                ]
            ),

            "available_focused_summaries": (
                coverage[
                    "available_focused_summaries"
                ]
            ),

            "missing_focused_summaries": (
                coverage[
                    "missing_focused_summaries"
                ]
            ),

            "all_summaries_available": (
                coverage[
                    "all_summaries_available"
                ]
            ),
        })


final_coverage_df = pd.DataFrame(
    coverage_rows_after_replacement
)


final_coverage_summary_df = (
    final_coverage_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        complete_cases=(
            "all_summaries_available",
            "sum",
        ),

        available_coarse_summaries=(
            "available_coarse_summaries",
            "sum",
        ),

        missing_coarse_summaries=(
            "missing_coarse_summaries",
            "sum",
        ),

        available_focused_summaries=(
            "available_focused_summaries",
            "sum",
        ),

        missing_focused_summaries=(
            "missing_focused_summaries",
            "sum",
        ),
    )
)


# The three missing coarse slots caused by the short target
# participant must now be fixed.
silent_coverage_row = (
    final_coverage_summary_df[
        final_coverage_summary_df[
            "case_family"
        ] == "silent_partner"
    ]
    .iloc[0]
)


assert int(
    silent_coverage_row[
        "missing_coarse_summaries"
    ]
) == 0


# ============================================================
# REBUILD THE REMAINING UNIQUE MISSING-SUMMARY MANIFEST
#
# LAG is excluded because it duplicates NORMAL semantics.
# ============================================================

remaining_gap_registry = {}


direct_cases_after_replacement = (
    completed_semantic_normal_cases
    + completed_semantic_wrong_cases
    + completed_semantic_silent_cases
)


for case in direct_cases_after_replacement:

    family = str(
        case[
            "case_variant"
        ]
    ).lower()


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant = case[role]


        source_kind = (
            "silent"

            if (
                family == "silent_partner"
                and role == "participant_B"
            )

            else "normal"
        )


        conversation_id = str(
            participant[
                "conversation_id"
            ]
        )


        participant_id = str(
            participant[
                "participant_id"
            ]
        )


        for (
            segment_idx,
            segment_name,
        ) in enumerate(
            SEGMENT_NAMES
        ):

            segment_record = (
                case[
                    "semantic_summaries"
                ][role][segment_name]
            )


            coarse_missing = (
                segment_record.get(
                    "coarse_summary"
                )
                is None
            )


            focused_missing = (
                segment_record.get(
                    "focused_summary"
                )
                is None
            )


            if not (
                coarse_missing
                or focused_missing
            ):
                continue


            identity = (
                source_kind,
                conversation_id,
                participant_id,
                int(segment_idx),
            )


            if identity not in remaining_gap_registry:

                remaining_gap_registry[
                    identity
                ] = {
                    "source_kind": (
                        source_kind
                    ),

                    "conversation_id": (
                        conversation_id
                    ),

                    "participant_id": (
                        participant_id
                    ),

                    "segment_idx": (
                        int(segment_idx)
                    ),

                    "segment_start_seconds": (
                        0
                        if segment_idx == 0
                        else 60
                    ),

                    "segment_end_seconds": (
                        60
                        if segment_idx == 0
                        else 120
                    ),

                    "coarse_missing": (
                        coarse_missing
                    ),

                    "focused_missing": (
                        focused_missing
                    ),

                    "appears_in_case_families": set(),

                    "appears_in_case_ids": set(),
                }


            remaining_gap_registry[
                identity
            ][
                "appears_in_case_families"
            ].add(
                family
            )


            remaining_gap_registry[
                identity
            ][
                "appears_in_case_ids"
            ].add(
                case[
                    "case_id"
                ]
            )


remaining_missing_summaries = []


for identity, record in sorted(
    remaining_gap_registry.items()
):

    remaining_missing_summaries.append({
        "source_kind": (
            record[
                "source_kind"
            ]
        ),

        "conversation_id": (
            record[
                "conversation_id"
            ]
        ),

        "participant_id": (
            record[
                "participant_id"
            ]
        ),

        "segment_idx": (
            record[
                "segment_idx"
            ]
        ),

        "segment_start_seconds": (
            record[
                "segment_start_seconds"
            ]
        ),

        "segment_end_seconds": (
            record[
                "segment_end_seconds"
            ]
        ),

        "coarse_missing": (
            record[
                "coarse_missing"
            ]
        ),

        "focused_missing": (
            record[
                "focused_missing"
            ]
        ),

        "appears_in_case_families": sorted(
            record[
                "appears_in_case_families"
            ]
        ),

        "num_case_appearances": len(
            record[
                "appears_in_case_ids"
            ]
        ),
    })


# ============================================================
# OUTPUT PATHS
# ============================================================

if (
    "FINAL_SILENT_DATABASE_PATH"
    not in globals()
):

    FINAL_SILENT_DATABASE_PATH = (
        OUT_DIR
        / (
            "consolidation_silent_partner_cases_with_"
            "temporal_and_completed_semantic_summaries.json"
        )
    )


if (
    "FINAL_ALL_400_DATABASE_PATH"
    not in globals()
):

    FINAL_ALL_400_DATABASE_PATH = (
        OUT_DIR
        / (
            "consolidation_all_400_cases_with_"
            "temporal_and_semantic_summaries.json"
        )
    )


if (
    "FINAL_REMAINING_MISSING_PATH"
    not in globals()
):

    FINAL_REMAINING_MISSING_PATH = (
        OUT_DIR
        / (
            "consolidation_remaining_missing_semantic_"
            "summaries_after_generation.json"
        )
    )


if (
    "FINAL_SEMANTIC_COVERAGE_CSV_PATH"
    not in globals()
):

    FINAL_SEMANTIC_COVERAGE_CSV_PATH = (
        OUT_DIR
        / (
            "consolidation_final_semantic_coverage_"
            "after_generation.csv"
        )
    )


SILENT_REPLACEMENT_MANIFEST_PATH = (
    OUT_DIR
    / (
        "silent_participant_replacement_"
        "V01_S1545_I00000138_P2518.json"
    )
)


# ============================================================
# BACK UP THE CURRENT FINAL FILES BEFORE OVERWRITING
# ============================================================

def create_backup_once(
    path,
):
    path = Path(
        path
    )


    backup_path = path.with_name(
        path.stem
        + (
            ".before_replacing_"
            "V01_S1545_I00000138_P2518"
        )
        + path.suffix
    )


    if (
        path.exists()
        and not backup_path.exists()
    ):

        shutil.copy2(
            path,
            backup_path,
        )


    return backup_path


silent_backup_path = (
    create_backup_once(
        FINAL_SILENT_DATABASE_PATH
    )
)


combined_backup_path = (
    create_backup_once(
        FINAL_ALL_400_DATABASE_PATH
    )
)


# ============================================================
# SAVE REPAIRED DATABASES
# ============================================================

Path(
    FINAL_SILENT_DATABASE_PATH
).write_text(
    json.dumps(
        completed_semantic_silent_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


Path(
    FINAL_ALL_400_DATABASE_PATH
).write_text(
    json.dumps(
        completed_all_400_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


Path(
    FINAL_REMAINING_MISSING_PATH
).write_text(
    json.dumps(
        remaining_missing_summaries,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


final_coverage_df.to_csv(
    FINAL_SEMANTIC_COVERAGE_CSV_PATH,
    index=False,
)


replacement_manifest = {
    "target_participant": {
        "conversation_id": (
            TARGET_SILENT_CONVERSATION_ID
        ),

        "participant_id": (
            TARGET_SILENT_PARTICIPANT_ID
        ),

        "metadata_path": str(
            target_metadata_path
        ),

        "metadata_vad_entries": 0,
    },

    "replacement_participant": {
        "conversation_id": (
            replacement_identity[0]
        ),

        "participant_id": (
            replacement_identity[1]
        ),

        "metadata_path": (
            selected_replacement[
                "metadata_path"
            ]
        ),

        "metadata_vad_entries": 0,

        "previous_sample_appearances": (
            selected_replacement[
                "current_usage_count"
            ]
        ),

        "new_sample_appearances": (
            selected_replacement[
                "current_usage_count"
            ]
            + len(
                affected_case_indexes
            )
        ),
    },

    "num_replaced_cases": len(
        affected_case_indexes
    ),

    "affected_cases": (
        replacement_audit_records
    ),

    "temporal_features_recomputed": False,

    "temporal_features_preserved_exactly": True,

    "replacement_reason": (
        "Original silent participant video did not contain "
        "the complete 60-120 second segment required for "
        "semantic-summary generation."
    ),
}


SILENT_REPLACEMENT_MANIFEST_PATH.write_text(
    json.dumps(
        replacement_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ============================================================
# FINAL REPORT
# ============================================================

print("\n" + "=" * 88)
print("SILENT PARTICIPANT REPLACEMENT AUDIT PASSED")
print("=" * 88)

print(
    "Verified target silent participant:",
    target_identity,
)

print(
    "Selected replacement:",
    replacement_identity,
)

print(
    "Replaced samples:",
    len(
        affected_case_indexes
    ),
)

print(
    "Temporal turns/features changed:",
    False,
)

print(
    "Target still present in database:",
    False,
)


print("\nFINAL COVERAGE AFTER REPLACEMENT")

display(
    final_coverage_summary_df
)


print(
    "\nRemaining unique semantic gaps:",
    len(
        remaining_missing_summaries
    ),
)

print(
    "Remaining coarse gaps:",
    sum(
        record[
            "coarse_missing"
        ]

        for record
        in remaining_missing_summaries
    ),
)

print(
    "Remaining focused gaps:",
    sum(
        record[
            "focused_missing"
        ]

        for record
        in remaining_missing_summaries
    ),
)


print(
    "\nRepaired silent database:",
    FINAL_SILENT_DATABASE_PATH,
)

print(
    "Repaired combined 400-case database:",
    FINAL_ALL_400_DATABASE_PATH,
)

print(
    "Replacement manifest:",
    SILENT_REPLACEMENT_MANIFEST_PATH,
)

print(
    "Silent database backup:",
    silent_backup_path,
)

print(
    "Combined database backup:",
    combined_backup_path,
)

TARGET SILENT PARTICIPANT
Target identity: ('V01_S1545_I00000138', 'P2518')
Affected silent samples: 3
Affected case IDs: ['consolidation_silent_partner_036', 'consolidation_silent_partner_042', 'consolidation_silent_partner_070']
Target metadata path: /content/drive/MyDrive/seamless_download/data/silent/V01_S1545_I00000138/P2518/V01_S1545_I00000138_P2518.json
Target metadata:vad entries: 0
TARGET VERIFIED AS SILENT: metadata:vad == []

SELECTED REPLACEMENT SILENT PARTICIPANT
Replacement identity: ('V00_S2036_I00000716', 'P1293A')
Replacement metadata path: /content/drive/MyDrive/seamless_download/data/silent/V00_S2036_I00000716/P1293A/V00_S2036_I00000716_P1293A.json
Previous sample appearances: 2
Replacement metadata:vad entries: 0
Replacement has complete coarse/focused summaries: True

SILENT PARTICIPANT REPLACEMENT AUDIT PASSED
Verified target silent participant: ('V01_S1545_I00000138', 'P2518')
Selected replacement: ('V00_S2036_I00000716', 'P1293A')
Replaced samples: 3
Temporal tu

,case_family,total_cases,complete_cases,available_coarse_summaries,missing_coarse_summaries,available_focused_summaries,missing_focused_summaries
0,lag,100,94,400,0,394,6
1,normal,100,94,400,0,394,6
2,silent_partner,100,94,400,0,394,6
3,wrong_partner,100,94,400,0,394,6



Remaining unique semantic gaps: 7
Remaining coarse gaps: 0
Remaining focused gaps: 7

Repaired silent database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_silent_partner_cases_with_temporal_and_completed_semantic_summaries.json
Repaired combined 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Replacement manifest: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/silent_participant_replacement_V01_S1545_I00000138_P2518.json
Silent database backup: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_silent_partner_cases_with_temporal_and_completed_semantic_summaries.before_replacing_V01_S1545_I00000138_P2518.json
Combined database backup: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.before_replacing_V01_S1545_I0

# Correct the missing focused summaries

In [ ]:
# ============================================================
# DIAGNOSE FOCUSED SUMMARY FAILURES
#
# This cell:
# - does not rerun Qwen
# - does not modify the caches
# - identifies parse errors versus schema errors
# - prints the first failed raw output
# ============================================================

import json
import pandas as pd


FOCUSED_REQUIRED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


focused_failure_records = []


for cache_key, record in (
    missing_focused_raw_cache.items()
):

    focused_summary = record.get(
        "focused_summary"
    )


    if focused_summary is not None:
        continue


    parsed = record.get(
        "parsed"
    )


    if not isinstance(
        parsed,
        dict,
    ):

        failure_type = (
            "parsed_output_not_dict"
        )

        missing_fields = (
            FOCUSED_REQUIRED_FIELDS
        )

    elif parsed.get(
        "parse_error",
        False,
    ):

        failure_type = (
            "json_parse_error"
        )

        missing_fields = (
            FOCUSED_REQUIRED_FIELDS
        )

    else:

        missing_fields = [
            field

            for field
            in FOCUSED_REQUIRED_FIELDS

            if field not in parsed
        ]


        if missing_fields:

            failure_type = (
                "valid_json_but_missing_fields"
            )

        else:

            failure_type = (
                "projection_failure_for_other_reason"
            )


    raw_output = str(
        record.get(
            "raw_output",
            "",
        )
    )


    focused_failure_records.append({
        "cache_key": (
            cache_key
        ),

        "source_kind": (
            record["source_kind"]
        ),

        "conversation_id": (
            record["conversation_id"]
        ),

        "participant_id": (
            record["participant_id"]
        ),

        "segment_idx": int(
            record["segment_idx"]
        ),

        "failure_type": (
            failure_type
        ),

        "missing_fields": (
            missing_fields
        ),

        "raw_output_characters": len(
            raw_output
        ),

        "raw_output_ends_with_brace": (
            raw_output
            .rstrip()
            .endswith("}")
        ),

        "raw_record": (
            record
        ),
    })


print("=" * 88)
print("FOCUSED FAILURE DIAGNOSIS")
print("=" * 88)

print(
    "Failed focused generations:",
    len(
        focused_failure_records
    ),
)


focused_failure_df = pd.DataFrame([
    {
        key: value

        for key, value
        in record.items()

        if key != "raw_record"
    }

    for record
    in focused_failure_records
])


display(
    focused_failure_df[
        [
            "source_kind",
            "conversation_id",
            "participant_id",
            "segment_idx",
            "failure_type",
            "missing_fields",
            "raw_output_characters",
            "raw_output_ends_with_brace",
        ]
    ]
)


assert focused_failure_records, (
    "No failed focused generations found."
)


first_focused_failure = (
    focused_failure_records[0]
)


first_failed_raw_record = (
    first_focused_failure[
        "raw_record"
    ]
)


print("\n" + "=" * 88)
print("FIRST FAILED FOCUSED GENERATION")
print("=" * 88)

print(
    "Source kind:",
    first_focused_failure[
        "source_kind"
    ],
)

print(
    "Conversation:",
    first_focused_failure[
        "conversation_id"
    ],
)

print(
    "Participant:",
    first_focused_failure[
        "participant_id"
    ],
)

print(
    "Segment:",
    first_focused_failure[
        "segment_idx"
    ],
)

print(
    "Failure type:",
    first_focused_failure[
        "failure_type"
    ],
)

print(
    "Missing fields:",
    first_focused_failure[
        "missing_fields"
    ],
)


print("\n" + "-" * 88)
print("RAW MODEL OUTPUT")
print("-" * 88)

print(
    first_failed_raw_record.get(
        "raw_output"
    )
)


print("\n" + "-" * 88)
print("CURRENT PARSED OUTPUT")
print("-" * 88)

print(
    json.dumps(
        first_failed_raw_record.get(
            "parsed"
        ),
        indent=2,
        ensure_ascii=False,
    )
)

FOCUSED FAILURE DIAGNOSIS
Failed focused generations: 8


,source_kind,conversation_id,participant_id,segment_idx,failure_type,missing_fields,raw_output_characters,raw_output_ends_with_brace
0,normal,V01_S0340_I00001106,P1681,1,json_parse_error,"[detailed_speech_summary, main_topic, secondar...",2397,False
1,normal,V01_S0340_I00001111,P1681,1,json_parse_error,"[detailed_speech_summary, main_topic, secondar...",2447,False
2,normal,V01_S0563_I00001226,P1841,0,json_parse_error,"[detailed_speech_summary, main_topic, secondar...",2314,False
3,normal,V01_S1545_I00000632,P2519,1,json_parse_error,"[detailed_speech_summary, main_topic, secondar...",2204,False
4,normal,V03_S0148_I00000377,P1319,0,json_parse_error,"[detailed_speech_summary, main_topic, secondar...",1796,False
5,normal,V03_S0148_I00000502,P1319,0,json_parse_error,"[detailed_speech_summary, main_topic, secondar...",1937,False
6,silent,V01_S0307_I00001228,P1633,1,json_parse_error,"[detailed_speech_summary, main_topic, secondar...",2628,False
7,silent,V01_S1545_I00000138,P2518,0,json_parse_error,"[detailed_speech_summary, main_topic, secondar...",2221,False



FIRST FAILED FOCUSED GENERATION
Source kind: normal
Conversation: V01_S0340_I00001106
Participant: P1681
Segment: 1
Failure type: json_parse_error
Missing fields: ['detailed_speech_summary', 'main_topic', 'secondary_topics', 'key_semantic_details', 'summary_specificity', 'unclear_content', 'confidence']

----------------------------------------------------------------------------------------
RAW MODEL OUTPUT
----------------------------------------------------------------------------------------
{
  "participant_id": "P1681_seg1",
  "speaks": true,
  "detailed_speech_summary": "The man is talking about his family, mentioning his parents, siblings, and nieces. He also mentions his mother's side of the family, including his aunt and her children. He talks about his father's side of the family, mentioning his uncle and his children. He also mentions his father's sister and her children. He talks about his father's brother and his children. He mentions his father's sister and her children

In [ ]:
# ============================================================
# RETRY ONE FAILED FOCUSED SUMMARY
#
# Target:
#   normal / V01_S0340_I00001106 / P1681 / segment 1
#
# Changes only for this retry:
#   max_new_tokens=500
#   repetition_penalty=1.1
#   stricter concise-output instruction
#
# The caches are updated only if the new output is valid.
# ============================================================

import copy
import json
import torch


TARGET_SOURCE_KIND = "normal"
TARGET_CONVERSATION_ID = "V01_S0340_I00001106"
TARGET_PARTICIPANT_ID = "P1681"
TARGET_SEGMENT_IDX = 1


# ============================================================
# RETRY-SPECIFIC QWEN INFERENCE FUNCTION
# ============================================================

def qwen_video_text_retry(
    video_path: str,
    prompt: str,
    max_new_tokens: int = 500,
    repetition_penalty: float = 1.1,
):

    messages = [
        {
            "role": "system",

            "content": [
                {
                    "type": "text",

                    "text": (
                        "You are Qwen, a virtual human developed "
                        "by the Qwen Team, Alibaba Group, capable "
                        "of perceiving auditory and visual inputs, "
                        "as well as generating text and speech."
                    ),
                }
            ],
        },

        {
            "role": "user",

            "content": [
                {
                    "type": "video",
                    "video": video_path,
                    "fps": 7.0,
                },

                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        },
    ]


    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )


    audios, images, videos = process_mm_info(
        messages,
        use_audio_in_video=True,
    )


    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=True,
    ).to(
        model.device
    )


    print(
        "Input tokens:",
        int(
            inputs["input_ids"].shape[-1]
        ),
    )


    with torch.no_grad():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            repetition_penalty=(
                repetition_penalty
            ),
        )


    generated_ids = output_ids[
        :,
        inputs["input_ids"].shape[1]:,
    ]


    return processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]


# ============================================================
# FIND THE EXACT FAILED TASK
# ============================================================

matching_tasks = [
    task

    for task in missing_focused_generation_tasks

    if (
        str(
            task["source_kind"]
        )
        == TARGET_SOURCE_KIND

        and str(
            task["conversation_id"]
        )
        == TARGET_CONVERSATION_ID

        and str(
            task["participant_id"]
        )
        == TARGET_PARTICIPANT_ID

        and int(
            task["segment_idx"]
        )
        == TARGET_SEGMENT_IDX
    )
]


assert len(
    matching_tasks
) == 1, (
    "Expected exactly one matching focused task, "
    f"but found {len(matching_tasks)}."
)


retry_task = matching_tasks[0]


# ============================================================
# LOAD THE EXISTING SEGMENT AND CACHE KEY
# ============================================================

retry_segment = get_or_create_segment(
    retry_task
)


retry_key = focused_cache_key(
    retry_task,
    retry_segment,
)


old_failed_raw_record = copy.deepcopy(
    missing_focused_raw_cache.get(
        retry_key
    )
)


old_clean_record = copy.deepcopy(
    missing_focused_clean_cache.get(
        retry_key
    )
)


print("=" * 88)
print("RETRYING ONE FAILED FOCUSED SUMMARY")
print("=" * 88)

print(
    "Source kind:",
    retry_task["source_kind"],
)

print(
    "Conversation:",
    retry_task["conversation_id"],
)

print(
    "Participant:",
    retry_task["participant_id"],
)

print(
    "Segment index:",
    retry_segment["segment_idx"],
)

print(
    "Video:",
    retry_segment["path"],
)

print(
    "Old raw record exists:",
    retry_key in missing_focused_raw_cache,
)

print(
    "Old clean record exists:",
    retry_key in missing_focused_clean_cache,
)


# ============================================================
# BUILD THE ORIGINAL PROMPT
# ============================================================

original_retry_prompt = (
    FOCUSED_SEMANTIC_SUMMARY_PROMPT_TEMPLATE
    .format(
        participant_id=(
            f'{retry_task["participant_id"]}'
            f'_seg{retry_segment["segment_idx"]}'
        )
    )
)


# ============================================================
# ADD RETRY-SPECIFIC CONCISENESS INSTRUCTIONS
# ============================================================

retry_prompt = (
    original_retry_prompt
    + """

IMPORTANT OUTPUT CONSTRAINTS FOR THIS RETRY:

- The value of "detailed_speech_summary" must contain no more than 80 words.
- Use no more than 4 short sentences in "detailed_speech_summary".
- Do not repeat the same person, fact, relationship, topic, or sentence.
- Keep "secondary_topics" and "key_semantic_details" concise.
- Produce every required JSON field.
- Return one complete valid JSON object.
- Finish and close the JSON object before ending the response.
"""
)


# ============================================================
# RUN THE RETRY
# ============================================================

retry_raw_output = qwen_video_text_retry(
    video_path=retry_segment["path"],
    prompt=retry_prompt,
    max_new_tokens=500,
    repetition_penalty=1.1,
)


# Use the robust parser if it exists.
if "extract_json_from_text_v2" in globals():

    retry_parsed_output = (
        extract_json_from_text_v2(
            retry_raw_output
        )
    )

    parser_used = (
        "extract_json_from_text_v2"
    )

else:

    retry_parsed_output = (
        extract_json_from_text(
            retry_raw_output
        )
    )

    parser_used = (
        "extract_json_from_text"
    )


retry_focused_summary = (
    build_focused_projection_without_speaks(
        retry_parsed_output
    )
)


retry_succeeded = (
    retry_focused_summary
    is not None
)


# ============================================================
# DISPLAY THE RESULT
# ============================================================

print("\n" + "=" * 88)
print("RETRY RESULT")
print("=" * 88)

print(
    "Parser used:",
    parser_used,
)

print(
    "Maximum new tokens:",
    500,
)

print(
    "Repetition penalty:",
    1.1,
)

print(
    "Raw output characters:",
    len(
        retry_raw_output
    ),
)

print(
    "Raw output ends with closing brace:",
    retry_raw_output
    .rstrip()
    .endswith("}"),
)

print(
    "Valid focused summary:",
    retry_succeeded,
)


print("\n" + "-" * 88)
print("NEW RAW OUTPUT")
print("-" * 88)

print(
    retry_raw_output
)


print("\n" + "-" * 88)
print("NEW PARSED OUTPUT")
print("-" * 88)

print(
    json.dumps(
        retry_parsed_output,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "-" * 88)
print("NEW CLEAN FOCUSED SUMMARY")
print("-" * 88)

print(
    json.dumps(
        retry_focused_summary,
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# SAVE ONLY IF THE RETRY SUCCEEDED
# ============================================================

if retry_succeeded:

    new_raw_record = {
        "experiment_version": (
            FOCUSED_SUMMARY_VERSION
        ),

        "source_kind": (
            retry_task["source_kind"]
        ),

        "conversation_id": (
            retry_task["conversation_id"]
        ),

        "participant_id": (
            retry_task["participant_id"]
        ),

        "segment_idx": int(
            retry_segment["segment_idx"]
        ),

        "start": retry_segment["start"],

        "duration": (
            retry_segment["duration"]
        ),

        "video": retry_segment["path"],

        "raw_output": (
            retry_raw_output
        ),

        "parsed": (
            retry_parsed_output
        ),

        "focused_summary": (
            retry_focused_summary
        ),

        "retry_information": {
            "retry_reason": (
                "Original generation entered a repetitive "
                "loop and was truncated."
            ),

            "original_max_new_tokens": 500,

            "retry_max_new_tokens": 500,

            "repetition_penalty": 1.1,

            "additional_prompt_constraints": {
                "detailed_summary_max_words": 80,
                "detailed_summary_max_sentences": 4,
                "repetition_forbidden": True,
                "complete_json_required": True,
            },

            "parser_used": (
                parser_used
            ),
        },
    }


    new_clean_record = {
        "source_kind": (
            retry_task["source_kind"]
        ),

        "conversation_id": (
            retry_task["conversation_id"]
        ),

        "participant_id": (
            retry_task["participant_id"]
        ),

        "segment_idx": int(
            retry_segment["segment_idx"]
        ),

        "segment_start_seconds": (
            retry_segment["start"]
        ),

        "segment_duration_seconds": (
            retry_segment["duration"]
        ),

        "focused_summary": (
            retry_focused_summary
        ),
    }


    assert (
        "speaks"
        not in new_clean_record[
            "focused_summary"
        ]
    )


    missing_focused_raw_cache[
        retry_key
    ] = new_raw_record


    missing_focused_clean_cache[
        retry_key
    ] = new_clean_record


    save_focused_caches()


    assert (
        retry_key
        in missing_focused_clean_cache
    )


    print("\n" + "=" * 88)
    print("RETRY SUCCEEDED AND CACHES WERE UPDATED")
    print("=" * 88)

    print(
        "Updated raw focused cache:",
        MISSING_FOCUSED_RAW_CACHE_PATH,
    )

    print(
        "Updated clean focused cache:",
        MISSING_FOCUSED_SUMMARIES_PATH,
    )


else:

    # Verify that the failed retry changed nothing.
    assert (
        missing_focused_raw_cache.get(
            retry_key
        )
        == old_failed_raw_record
    )


    assert (
        missing_focused_clean_cache.get(
            retry_key
        )
        == old_clean_record
    )


    print("\n" + "=" * 88)
    print("RETRY FAILED — CACHES WERE NOT MODIFIED")
    print("=" * 88)

    print(
        "The new output was still incomplete "
        "or did not satisfy the focused schema."
    )

RETRYING ONE FAILED FOCUSED SUMMARY
Source kind: normal
Conversation: V01_S0340_I00001106
Participant: P1681
Segment index: 1
Video: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/preprocessed_segments_60s/normal/V01_S0340_I00001106/V01_S0340_I00001106_P1681_seg1_60s.mp4
Old raw record exists: True
Old clean record exists: False


/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:172: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Input tokens: 29009

RETRY RESULT
Parser used: extract_json_from_text
Maximum new tokens: 500
Repetition penalty: 1.1
Raw output characters: 791
Raw output ends with closing brace: False
Valid focused summary: True

----------------------------------------------------------------------------------------
NEW RAW OUTPUT
----------------------------------------------------------------------------------------
```json
{
  "participant_id": "P1681_seg1",
  "speaks": true,
  "detailed_speech_summary": "The man talks about his family, mentioning his wife and children. He also mentions his parents and expresses gratitude for their support.",
  "main_topic": "Family relationships and appreciation",
  "secondary_topics": [],
  "key_semantic_details": [
    "wife",
    "children",
    "parents",
    "gratitude"
  ],
  "summary_specificity": "high",
  "unclear_content": "",
  "confidence": 1.0
}
```
: 

: 

: 

: 
: 

: 
: 

: 
: 
: 
: 
: 
: 
: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 



In [ ]:
# ============================================================
# RETRY ALL REMAINING MISSING FOCUSED SUMMARIES
#
# This cell retries only participant-segments that:
#   - belong to missing_focused_generation_tasks
#   - do not already have a clean focused summary
#   - have an available video segment
#
# Retry settings:
#   max_new_tokens=500
#   repetition_penalty=1.1
#   concise and non-repetitive output instructions
#
# Successful retries update the focused caches immediately.
# Failed retries do not overwrite the previous cache records.
# ============================================================

from pathlib import Path

import copy
import json
import torch

from tqdm.auto import tqdm


# ============================================================
# RETRY CONFIGURATION
# ============================================================

RETRY_MAX_NEW_TOKENS = 500
RETRY_REPETITION_PENALTY = 1.1


RETRY_PROMPT_SUFFIX = """

IMPORTANT OUTPUT CONSTRAINTS FOR THIS RETRY:

- The value of "detailed_speech_summary" must contain no more than 80 words.
- Use no more than 4 short sentences in "detailed_speech_summary".
- Do not repeat the same person, fact, relationship, topic, phrase, or sentence.
- Keep "secondary_topics" concise.
- Keep "key_semantic_details" concise.
- Produce every required JSON field.
- Return one complete valid JSON object.
- Finish and close the JSON object before ending the response.
"""


FOCUSED_RETRY_LOG_PATH = (
    Path(
        MISSING_FOCUSED_RAW_CACHE_PATH
    ).parent
    / "focused_missing_summary_retry_attempts.json"
)


# ============================================================
# RETRY-SPECIFIC QWEN INFERENCE
# ============================================================

def qwen_video_text_focused_retry(
    video_path: str,
    prompt: str,
    max_new_tokens: int = 500,
    repetition_penalty: float = 1.1,
):

    messages = [
        {
            "role": "system",

            "content": [
                {
                    "type": "text",

                    "text": (
                        "You are Qwen, a virtual human developed "
                        "by the Qwen Team, Alibaba Group, capable "
                        "of perceiving auditory and visual inputs, "
                        "as well as generating text and speech."
                    ),
                }
            ],
        },

        {
            "role": "user",

            "content": [
                {
                    "type": "video",
                    "video": video_path,
                    "fps": 7.0,
                },

                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        },
    ]


    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )


    audios, images, videos = process_mm_info(
        messages,
        use_audio_in_video=True,
    )


    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=True,
    ).to(
        model.device
    )


    input_token_count = int(
        inputs["input_ids"].shape[-1]
    )


    print(
        "Input tokens:",
        input_token_count,
    )


    with torch.no_grad():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            repetition_penalty=(
                repetition_penalty
            ),
        )


    generated_ids = output_ids[
        :,
        inputs["input_ids"].shape[1]:,
    ]


    decoded_output = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]


    return (
        decoded_output,
        input_token_count,
    )


# ============================================================
# LOAD EXISTING RETRY LOG
# ============================================================

if FOCUSED_RETRY_LOG_PATH.exists():

    focused_retry_log = json.loads(
        FOCUSED_RETRY_LOG_PATH.read_text(
            encoding="utf-8"
        )
    )


    if not isinstance(
        focused_retry_log,
        dict,
    ):

        raise TypeError(
            "Expected the focused retry log "
            "to contain a JSON object."
        )

else:

    focused_retry_log = {}


def save_focused_retry_log():

    FOCUSED_RETRY_LOG_PATH.write_text(
        json.dumps(
            focused_retry_log,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


# ============================================================
# IDENTIFY ONLY THE REMAINING MISSING FOCUSED TASKS
# ============================================================

remaining_focused_retry_tasks = []

unavailable_focused_retry_tasks = []


for task in missing_focused_generation_tasks:

    try:

        segment = get_or_create_segment(
            task
        )

    except RuntimeError as exc:

        # Skip only unavailable/invalid source-video segments.
        # Other RuntimeErrors must still stop execution.
        if (
            "Could not create segment from"
            not in str(exc)
        ):
            raise


        unavailable_focused_retry_tasks.append({
            "source_kind": (
                task["source_kind"]
            ),

            "conversation_id": (
                task["conversation_id"]
            ),

            "participant_id": (
                task["participant_id"]
            ),

            "segment_idx": int(
                task["segment_idx"]
            ),

            "video_path": (
                task["video_path"]
            ),

            "error": str(exc),
        })


        continue


    key = focused_cache_key(
        task,
        segment,
    )


    # Already fixed or originally successful.
    if key in missing_focused_clean_cache:
        continue


    remaining_focused_retry_tasks.append({
        "task": task,
        "segment": segment,
        "key": key,
    })


print("=" * 88)
print("REMAINING FOCUSED SUMMARY RETRIES")
print("=" * 88)

print(
    "Remaining runnable focused retries:",
    len(
        remaining_focused_retry_tasks
    ),
)

print(
    "Unavailable/short segments skipped:",
    len(
        unavailable_focused_retry_tasks
    ),
)


if unavailable_focused_retry_tasks:

    print(
        "\nSkipped unavailable segments:"
    )


    for record in (
        unavailable_focused_retry_tasks
    ):

        print(
            "-",
            record["source_kind"],
            record["conversation_id"],
            record["participant_id"],
            f'segment={record["segment_idx"]}',
        )


# ============================================================
# RUN ALL REMAINING RETRIES
# ============================================================

focused_retry_successes = []

focused_retry_failures = []


for retry_item in tqdm(
    remaining_focused_retry_tasks,
    desc="Retrying missing focused summaries",
):

    task = retry_item[
        "task"
    ]

    segment = retry_item[
        "segment"
    ]

    key = retry_item[
        "key"
    ]


    # Extra safeguard:
    # do not rerun a task that became available during this run.
    if key in missing_focused_clean_cache:
        continue


    print("\n" + "=" * 88)
    print("FOCUSED RETRY")
    print("=" * 88)

    print(
        "Source kind:",
        task["source_kind"],
    )

    print(
        "Conversation:",
        task["conversation_id"],
    )

    print(
        "Participant:",
        task["participant_id"],
    )

    print(
        "Segment:",
        segment["segment_idx"],
    )


    original_prompt = (
        FOCUSED_SEMANTIC_SUMMARY_PROMPT_TEMPLATE
        .format(
            participant_id=(
                f'{task["participant_id"]}'
                f'_seg{segment["segment_idx"]}'
            )
        )
    )


    retry_prompt = (
        original_prompt
        + RETRY_PROMPT_SUFFIX
    )


    retry_raw_output, input_token_count = (
        qwen_video_text_focused_retry(
            video_path=segment["path"],
            prompt=retry_prompt,
            max_new_tokens=(
                RETRY_MAX_NEW_TOKENS
            ),
            repetition_penalty=(
                RETRY_REPETITION_PENALTY
            ),
        )
    )


    if (
        "extract_json_from_text_v2"
        in globals()
    ):

        retry_parsed_output = (
            extract_json_from_text_v2(
                retry_raw_output
            )
        )

        parser_used = (
            "extract_json_from_text_v2"
        )

    else:

        retry_parsed_output = (
            extract_json_from_text(
                retry_raw_output
            )
        )

        parser_used = (
            "extract_json_from_text"
        )


    retry_focused_summary = (
        build_focused_projection_without_speaks(
            retry_parsed_output
        )
    )


    retry_succeeded = (
        retry_focused_summary
        is not None
    )


    retry_attempt_record = {
        "source_kind": (
            task["source_kind"]
        ),

        "conversation_id": (
            task["conversation_id"]
        ),

        "participant_id": (
            task["participant_id"]
        ),

        "segment_idx": int(
            segment["segment_idx"]
        ),

        "segment_start_seconds": (
            segment["start"]
        ),

        "segment_duration_seconds": (
            segment["duration"]
        ),

        "video": (
            segment["path"]
        ),

        "input_tokens": (
            input_token_count
        ),

        "max_new_tokens": (
            RETRY_MAX_NEW_TOKENS
        ),

        "repetition_penalty": (
            RETRY_REPETITION_PENALTY
        ),

        "parser_used": (
            parser_used
        ),

        "raw_output": (
            retry_raw_output
        ),

        "raw_output_characters": len(
            retry_raw_output
        ),

        "raw_output_ends_with_closing_brace": (
            retry_raw_output
            .rstrip()
            .endswith("}")
        ),

        "parsed": (
            retry_parsed_output
        ),

        "focused_summary": (
            retry_focused_summary
        ),

        "retry_succeeded": (
            retry_succeeded
        ),
    }


    focused_retry_log[
        key
    ] = retry_attempt_record


    # Save the diagnostic record after every retry.
    save_focused_retry_log()


    print(
        "Raw output ends with closing brace:",
        retry_raw_output
        .rstrip()
        .endswith("}"),
    )

    print(
        "Valid focused summary:",
        retry_succeeded,
    )


    # ========================================================
    # SAVE ONLY SUCCESSFUL RETRIES
    # ========================================================

    if retry_succeeded:

        new_raw_record = {
            "experiment_version": (
                FOCUSED_SUMMARY_VERSION
            ),

            "source_kind": (
                task["source_kind"]
            ),

            "conversation_id": (
                task["conversation_id"]
            ),

            "participant_id": (
                task["participant_id"]
            ),

            "segment_idx": int(
                segment["segment_idx"]
            ),

            "start": (
                segment["start"]
            ),

            "duration": (
                segment["duration"]
            ),

            "video": (
                segment["path"]
            ),

            "raw_output": (
                retry_raw_output
            ),

            "parsed": (
                retry_parsed_output
            ),

            "focused_summary": (
                retry_focused_summary
            ),

            "retry_information": {
                "retry_reason": (
                    "Original focused generation was "
                    "missing or invalid."
                ),

                "retry_max_new_tokens": (
                    RETRY_MAX_NEW_TOKENS
                ),

                "repetition_penalty": (
                    RETRY_REPETITION_PENALTY
                ),

                "detailed_summary_max_words": 80,

                "detailed_summary_max_sentences": 4,

                "parser_used": (
                    parser_used
                ),
            },
        }


        new_clean_record = {
            "source_kind": (
                task["source_kind"]
            ),

            "conversation_id": (
                task["conversation_id"]
            ),

            "participant_id": (
                task["participant_id"]
            ),

            "segment_idx": int(
                segment["segment_idx"]
            ),

            "segment_start_seconds": (
                segment["start"]
            ),

            "segment_duration_seconds": (
                segment["duration"]
            ),

            "focused_summary": (
                retry_focused_summary
            ),
        }


        assert (
            "speaks"
            not in new_clean_record[
                "focused_summary"
            ]
        )


        missing_focused_raw_cache[
            key
        ] = new_raw_record


        missing_focused_clean_cache[
            key
        ] = new_clean_record


        # Checkpoint after every successful retry.
        save_focused_caches()


        focused_retry_successes.append({
            "source_kind": (
                task["source_kind"]
            ),

            "conversation_id": (
                task["conversation_id"]
            ),

            "participant_id": (
                task["participant_id"]
            ),

            "segment_idx": int(
                segment["segment_idx"]
            ),

            "cache_key": key,
        })


        print(
            "SAVED: valid focused summary"
        )


    else:

        focused_retry_failures.append({
            "source_kind": (
                task["source_kind"]
            ),

            "conversation_id": (
                task["conversation_id"]
            ),

            "participant_id": (
                task["participant_id"]
            ),

            "segment_idx": int(
                segment["segment_idx"]
            ),

            "cache_key": key,

            "raw_output_ends_with_closing_brace": (
                retry_raw_output
                .rstrip()
                .endswith("}")
            ),
        })


        print(
            "NOT SAVED: retry remained invalid"
        )


# Final checkpoint.
save_focused_caches()
save_focused_retry_log()


# ============================================================
# FINAL RETRY AUDIT
# ============================================================

still_missing_runnable_focused = []


for retry_item in (
    remaining_focused_retry_tasks
):

    key = retry_item[
        "key"
    ]


    if (
        key
        not in missing_focused_clean_cache
    ):

        task = retry_item[
            "task"
        ]

        segment = retry_item[
            "segment"
        ]


        still_missing_runnable_focused.append({
            "source_kind": (
                task["source_kind"]
            ),

            "conversation_id": (
                task["conversation_id"]
            ),

            "participant_id": (
                task["participant_id"]
            ),

            "segment_idx": int(
                segment["segment_idx"]
            ),

            "cache_key": key,
        })


for record in (
    missing_focused_clean_cache.values()
):

    assert (
        "speaks"
        not in record[
            "focused_summary"
        ]
    )


print("\n" + "=" * 88)
print("ALL REMAINING FOCUSED RETRIES COMPLETE")
print("=" * 88)

print(
    "Runnable retries attempted:",
    len(
        remaining_focused_retry_tasks
    ),
)

print(
    "Successful retries:",
    len(
        focused_retry_successes
    ),
)

print(
    "Failed retries:",
    len(
        focused_retry_failures
    ),
)

print(
    "Unavailable/short segments skipped:",
    len(
        unavailable_focused_retry_tasks
    ),
)

print(
    "Runnable focused summaries still missing:",
    len(
        still_missing_runnable_focused
    ),
)

print(
    "Total clean focused summaries now:",
    len(
        missing_focused_clean_cache
    ),
)

print(
    "Updated raw focused cache:",
    MISSING_FOCUSED_RAW_CACHE_PATH,
)

print(
    "Updated clean focused cache:",
    MISSING_FOCUSED_SUMMARIES_PATH,
)

print(
    "Retry diagnostic log:",
    FOCUSED_RETRY_LOG_PATH,
)


if focused_retry_failures:

    print(
        "\nRetries that remained invalid:"
    )


    for record in focused_retry_failures:

        print(
            "-",
            record["source_kind"],
            record["conversation_id"],
            record["participant_id"],
            f'segment={record["segment_idx"]}',
        )

REMAINING FOCUSED SUMMARY RETRIES
Remaining runnable focused retries: 7
Unavailable/short segments skipped: 1

Skipped unavailable segments:
- silent V01_S1545_I00000138 P2518 segment=1


Retrying missing focused summaries:   0%|          | 0/7 [00:00<?, ?it/s]


FOCUSED RETRY
Source kind: normal
Conversation: V01_S0340_I00001111
Participant: P1681
Segment: 1


/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:172: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Input tokens: 29014
Raw output ends with closing brace: False
Valid focused summary: True
SAVED: valid focused summary

FOCUSED RETRY
Source kind: normal
Conversation: V01_S0563_I00001226
Participant: P1841
Segment: 0


...

Input tokens: 32164
Raw output ends with closing brace: False
Valid focused summary: True
SAVED: valid focused summary

FOCUSED RETRY
Source kind: silent
Conversation: V01_S0307_I00001228
Participant: P1633
Segment: 1


/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:172: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Input tokens: 29014
Raw output ends with closing brace: False
Valid focused summary: True
SAVED: valid focused summary

FOCUSED RETRY
Source kind: silent
Conversation: V01_S1545_I00000138
Participant: P2518
Segment: 0


/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:172: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Input tokens: 24284
Raw output ends with closing brace: False
Valid focused summary: True
SAVED: valid focused summary

ALL REMAINING FOCUSED RETRIES COMPLETE
Runnable retries attempted: 7
Successful retries: 7
Failed retries: 0
Unavailable/short segments skipped: 1
Runnable focused summaries still missing: 0
Total clean focused summaries now: 273
Updated raw focused cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/missing_focused_raw_generation_records.json
Updated clean focused cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/missing_focused_summaries_clean_without_speaks.json
Retry diagnostic log: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/missing_semantic_summary_generation/focused_missing_summary_retry_attempts.json


In [ ]:
# ============================================================
# FILL THE FINAL FOCUSED-SUMMARY GAPS AFTER SUCCESSFUL RETRIES
#
# This cell:
# 1. Reloads the updated clean focused-summary cache.
# 2. Fills only focused_summary == None slots.
# 3. Never overwrites an existing focused summary.
# 4. Updates NORMAL, WRONG_PARTNER and SILENT_PARTNER.
# 5. Re-inherits updated NORMAL semantics into LAG cases.
# 6. Recalculates final coverage.
# 7. Saves the updated final databases.
# ============================================================

from pathlib import Path
from collections import defaultdict

import copy
import json
import shutil
import pandas as pd


# ============================================================
# REQUIRED PATHS
# ============================================================

assert (
    "MISSING_FOCUSED_SUMMARIES_PATH"
    in globals()
), (
    "MISSING_FOCUSED_SUMMARIES_PATH is not defined."
)


assert Path(
    MISSING_FOCUSED_SUMMARIES_PATH
).exists(), (
    "Updated clean focused-summary cache not found: "
    f"{MISSING_FOCUSED_SUMMARIES_PATH}"
)


assert (
    "OUT_DIR"
    in globals()
), (
    "OUT_DIR is not defined."
)


OUT_DIR = Path(
    OUT_DIR
)


# ============================================================
# FINAL OUTPUT PATHS
# ============================================================

if (
    "FINAL_NORMAL_WRONG_DATABASE_PATH"
    not in globals()
):

    FINAL_NORMAL_WRONG_DATABASE_PATH = (
        OUT_DIR
        / (
            "consolidation_normal_wrong_cases_with_"
            "temporal_and_completed_semantic_summaries.json"
        )
    )


if (
    "FINAL_LAG_DATABASE_PATH"
    not in globals()
):

    FINAL_LAG_DATABASE_PATH = (
        OUT_DIR
        / (
            "consolidation_lag_cases_with_"
            "temporal_and_completed_semantic_summaries.json"
        )
    )


if (
    "FINAL_SILENT_DATABASE_PATH"
    not in globals()
):

    FINAL_SILENT_DATABASE_PATH = (
        OUT_DIR
        / (
            "consolidation_silent_partner_cases_with_"
            "temporal_and_completed_semantic_summaries.json"
        )
    )


if (
    "FINAL_ALL_400_DATABASE_PATH"
    not in globals()
):

    FINAL_ALL_400_DATABASE_PATH = (
        OUT_DIR
        / (
            "consolidation_all_400_cases_with_"
            "temporal_and_semantic_summaries.json"
        )
    )


if (
    "FINAL_REMAINING_MISSING_PATH"
    not in globals()
):

    FINAL_REMAINING_MISSING_PATH = (
        OUT_DIR
        / (
            "consolidation_remaining_missing_semantic_"
            "summaries_after_generation.json"
        )
    )


if (
    "FINAL_SEMANTIC_COVERAGE_CSV_PATH"
    not in globals()
):

    FINAL_SEMANTIC_COVERAGE_CSV_PATH = (
        OUT_DIR
        / (
            "consolidation_final_semantic_coverage_"
            "after_generation.csv"
        )
    )


# ============================================================
# GENERAL HELPERS
# ============================================================

SEGMENT_NAMES_BY_INDEX = {
    0: "segment_0_0_to_60_seconds",
    1: "segment_1_60_to_120_seconds",
}


SEMANTIC_FIELDS = {
    "semantic_summaries",
    "semantic_summary_coverage",
    "semantic_summary_source",
}


def load_json_list(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        data,
        list,
    ), (
        f"Expected JSON list in {path}"
    )


    return data


def load_json_dict(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        data,
        dict,
    ), (
        f"Expected JSON object in {path}"
    )


    return data


def case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        f"Unknown case variant: {variant}"
    )


def participant_identity(
    participant,
):
    return (
        str(
            participant[
                "conversation_id"
            ]
        ),

        str(
            participant[
                "participant_id"
            ]
        ),
    )


def source_kind_for_role(
    case,
    role,
):
    if (
        case_family(
            case
        ) == "silent_partner"

        and role == "participant_B"
    ):

        return "silent"


    return "normal"


def case_without_semantic_fields(
    case,
):
    return {
        key: value

        for key, value in case.items()

        if key not in SEMANTIC_FIELDS
    }


def recalculate_case_semantic_coverage(
    case,
):
    available_coarse = 0
    available_focused = 0


    for role in [
        "participant_A",
        "participant_B",
    ]:

        for segment_name in (
            SEGMENT_NAMES_BY_INDEX.values()
        ):

            segment_record = (
                case[
                    "semantic_summaries"
                ][role][segment_name]
            )


            available_coarse += int(
                segment_record.get(
                    "coarse_summary"
                )
                is not None
            )


            available_focused += int(
                segment_record.get(
                    "focused_summary"
                )
                is not None
            )


    case[
        "semantic_summary_coverage"
    ] = {
        "expected_coarse_summaries": 4,

        "available_coarse_summaries": (
            available_coarse
        ),

        "missing_coarse_summaries": (
            4 - available_coarse
        ),

        "expected_focused_summaries": 4,

        "available_focused_summaries": (
            available_focused
        ),

        "missing_focused_summaries": (
            4 - available_focused
        ),

        "all_summaries_available": (
            available_coarse == 4
            and available_focused == 4
        ),
    }


# ============================================================
# LOAD THE CURRENT FINAL DATABASE STATE
#
# Prefer current in-memory objects.
# Otherwise load the already-saved final 400-case database.
#
# This preserves the previous silent-participant replacement.
# ============================================================

required_completed_variables = [
    "completed_semantic_normal_cases",
    "completed_semantic_wrong_cases",
    "completed_semantic_lag_cases",
    "completed_semantic_silent_cases",
]


all_completed_variables_available = all(
    name in globals()

    for name
    in required_completed_variables
)


if all_completed_variables_available:

    current_normal_cases = copy.deepcopy(
        completed_semantic_normal_cases
    )


    current_wrong_cases = copy.deepcopy(
        completed_semantic_wrong_cases
    )


    current_lag_cases = copy.deepcopy(
        completed_semantic_lag_cases
    )


    current_silent_cases = copy.deepcopy(
        completed_semantic_silent_cases
    )


    loaded_database_source = (
        "current in-memory completed cases"
    )


else:

    assert Path(
        FINAL_ALL_400_DATABASE_PATH
    ).exists(), (
        "Neither the completed in-memory variables nor the "
        "saved final 400-case database are available."
    )


    saved_all_cases = load_json_list(
        FINAL_ALL_400_DATABASE_PATH
    )


    current_normal_cases = [
        case

        for case in saved_all_cases

        if case_family(
            case
        ) == "normal"
    ]


    current_wrong_cases = [
        case

        for case in saved_all_cases

        if case_family(
            case
        ) == "wrong_partner"
    ]


    current_lag_cases = [
        case

        for case in saved_all_cases

        if case_family(
            case
        ) == "lag"
    ]


    current_silent_cases = [
        case

        for case in saved_all_cases

        if case_family(
            case
        ) == "silent_partner"
    ]


    loaded_database_source = str(
        FINAL_ALL_400_DATABASE_PATH
    )


assert len(
    current_normal_cases
) == 100


assert len(
    current_wrong_cases
) == 100


assert len(
    current_lag_cases
) == 100


assert len(
    current_silent_cases
) == 100


print("=" * 88)
print("CURRENT FINAL DATABASE LOADED")
print("=" * 88)

print(
    "Database source:",
    loaded_database_source,
)

print(
    "NORMAL cases:",
    len(
        current_normal_cases
    ),
)

print(
    "WRONG cases:",
    len(
        current_wrong_cases
    ),
)

print(
    "LAG cases:",
    len(
        current_lag_cases
    ),
)

print(
    "SILENT cases:",
    len(
        current_silent_cases
    ),
)


# ============================================================
# RELOAD THE UPDATED CLEAN FOCUSED CACHE
#
# This is required because the retry cells updated the file
# after the original generated_focused_index was created.
# ============================================================

updated_focused_clean_records = (
    load_json_dict(
        MISSING_FOCUSED_SUMMARIES_PATH
    )
)


updated_focused_index = {}


for cache_key, record in (
    updated_focused_clean_records.items()
):

    identity = (
        str(
            record[
                "source_kind"
            ]
        ),

        str(
            record[
                "conversation_id"
            ]
        ),

        str(
            record[
                "participant_id"
            ]
        ),

        int(
            record[
                "segment_idx"
            ]
        ),
    )


    focused_summary = record.get(
        "focused_summary"
    )


    assert isinstance(
        focused_summary,
        dict,
    ), (
        "Invalid focused summary in clean cache: "
        f"{cache_key}"
    )


    assert (
        "speaks"
        not in focused_summary
    ), (
        "Clean focused summary unexpectedly contains "
        f"'speaks': {identity}"
    )


    if identity in updated_focused_index:

        assert (
            updated_focused_index[
                identity
            ]
            == focused_summary
        ), (
            "Conflicting focused summaries found for "
            f"{identity}"
        )

    else:

        updated_focused_index[
            identity
        ] = copy.deepcopy(
            focused_summary
        )


print("\n" + "=" * 88)
print("UPDATED FOCUSED CACHE LOADED")
print("=" * 88)

print(
    "Clean focused cache records:",
    len(
        updated_focused_clean_records
    ),
)

print(
    "Unique focused participant-segments:",
    len(
        updated_focused_index
    ),
)


# ============================================================
# FILL ONLY THE MISSING FOCUSED SUMMARIES
# ============================================================

def fill_only_missing_focused_summaries(
    cases,
    family_name,
):
    updated_cases = copy.deepcopy(
        cases
    )


    insertion_records = []


    for original_case, updated_case in zip(
        cases,
        updated_cases,
    ):

        assert (
            case_without_semantic_fields(
                original_case
            )
            ==
            case_without_semantic_fields(
                updated_case
            )
        )


        for role in [
            "participant_A",
            "participant_B",
        ]:

            participant = (
                updated_case[
                    role
                ]
            )


            source_kind = (
                source_kind_for_role(
                    updated_case,
                    role,
                )
            )


            conversation_id = str(
                participant[
                    "conversation_id"
                ]
            )


            participant_id = str(
                participant[
                    "participant_id"
                ]
            )


            for (
                segment_idx,
                segment_name,
            ) in (
                SEGMENT_NAMES_BY_INDEX.items()
            ):

                original_segment_record = (
                    original_case[
                        "semantic_summaries"
                    ][role][segment_name]
                )


                updated_segment_record = (
                    updated_case[
                        "semantic_summaries"
                    ][role][segment_name]
                )


                existing_focused_summary = (
                    original_segment_record.get(
                        "focused_summary"
                    )
                )


                # Never overwrite an already existing summary.
                if existing_focused_summary is not None:

                    assert (
                        updated_segment_record.get(
                            "focused_summary"
                        )
                        ==
                        existing_focused_summary
                    )

                    continue


                identity = (
                    source_kind,
                    conversation_id,
                    participant_id,
                    int(segment_idx),
                )


                new_focused_summary = (
                    updated_focused_index.get(
                        identity
                    )
                )


                if new_focused_summary is None:
                    continue


                updated_segment_record[
                    "focused_summary"
                ] = copy.deepcopy(
                    new_focused_summary
                )


                insertion_records.append({
                    "case_family": (
                        family_name
                    ),

                    "case_id": (
                        updated_case[
                            "case_id"
                        ]
                    ),

                    "participant_role": (
                        role
                    ),

                    "source_kind": (
                        source_kind
                    ),

                    "conversation_id": (
                        conversation_id
                    ),

                    "participant_id": (
                        participant_id
                    ),

                    "segment_idx": (
                        int(
                            segment_idx
                        )
                    ),

                    "identity": (
                        identity
                    ),
                })


        recalculate_case_semantic_coverage(
            updated_case
        )


        assert (
            case_without_semantic_fields(
                original_case
            )
            ==
            case_without_semantic_fields(
                updated_case
            )
        )


    return (
        updated_cases,
        insertion_records,
    )


(
    final_normal_cases,
    normal_focused_insertions,
) = fill_only_missing_focused_summaries(
    cases=current_normal_cases,
    family_name="normal",
)


(
    final_wrong_cases,
    wrong_focused_insertions,
) = fill_only_missing_focused_summaries(
    cases=current_wrong_cases,
    family_name="wrong_partner",
)


(
    final_silent_cases,
    silent_focused_insertions,
) = fill_only_missing_focused_summaries(
    cases=current_silent_cases,
    family_name="silent_partner",
)


all_direct_focused_insertions = (
    normal_focused_insertions
    + wrong_focused_insertions
    + silent_focused_insertions
)


print("\n" + "=" * 88)
print("MISSING FOCUSED SUMMARY FILL COMPLETE")
print("=" * 88)

print(
    "NORMAL focused slots filled:",
    len(
        normal_focused_insertions
    ),
)

print(
    "WRONG focused slots filled:",
    len(
        wrong_focused_insertions
    ),
)

print(
    "SILENT focused slots filled:",
    len(
        silent_focused_insertions
    ),
)

print(
    "Total direct focused slots filled:",
    len(
        all_direct_focused_insertions
    ),
)


# ============================================================
# VERIFY EVERY INSERTION AGAINST THE UPDATED CACHE
# ============================================================

direct_cases_by_id = {
    case[
        "case_id"
    ]: case

    for case in (
        final_normal_cases
        + final_wrong_cases
        + final_silent_cases
    )
}


for insertion in (
    all_direct_focused_insertions
):

    case = direct_cases_by_id[
        insertion[
            "case_id"
        ]
    ]


    segment_name = (
        SEGMENT_NAMES_BY_INDEX[
            insertion[
                "segment_idx"
            ]
        ]
    )


    stored_summary = (
        case[
            "semantic_summaries"
        ][
            insertion[
                "participant_role"
            ]
        ][
            segment_name
        ][
            "focused_summary"
        ]
    )


    expected_summary = (
        updated_focused_index[
            insertion[
                "identity"
            ]
        ]
    )


    assert (
        stored_summary
        ==
        expected_summary
    )


# ============================================================
# RE-INHERIT UPDATED NORMAL SEMANTICS INTO LAG
# ============================================================

final_normal_by_group = {
    case[
        "source_group_id"
    ]: case

    for case in final_normal_cases
}


assert len(
    final_normal_by_group
) == 100


final_lag_cases = []


for original_lag_case in (
    current_lag_cases
):

    source_group_id = (
        original_lag_case[
            "source_group_id"
        ]
    )


    assert (
        source_group_id
        in final_normal_by_group
    )


    matching_normal_case = (
        final_normal_by_group[
            source_group_id
        ]
    )


    assert (
        participant_identity(
            original_lag_case[
                "participant_A"
            ]
        )
        ==
        participant_identity(
            matching_normal_case[
                "participant_A"
            ]
        )
    )


    assert (
        participant_identity(
            original_lag_case[
                "participant_B"
            ]
        )
        ==
        participant_identity(
            matching_normal_case[
                "participant_B"
            ]
        )
    )


    updated_lag_case = copy.deepcopy(
        original_lag_case
    )


    updated_lag_case[
        "semantic_summaries"
    ] = copy.deepcopy(
        matching_normal_case[
            "semantic_summaries"
        ]
    )


    updated_lag_case[
        "semantic_summary_coverage"
    ] = copy.deepcopy(
        matching_normal_case[
            "semantic_summary_coverage"
        ]
    )


    # No temporal or case-definition field may change.
    assert (
        case_without_semantic_fields(
            original_lag_case
        )
        ==
        case_without_semantic_fields(
            updated_lag_case
        )
    )


    final_lag_cases.append(
        updated_lag_case
    )


assert len(
    final_lag_cases
) == 100


for lag_case in final_lag_cases:

    matching_normal_case = (
        final_normal_by_group[
            lag_case[
                "source_group_id"
            ]
        ]
    )


    assert (
        lag_case[
            "semantic_summaries"
        ]
        ==
        matching_normal_case[
            "semantic_summaries"
        ]
    )


# ============================================================
# FINAL CASE COUNTS
# ============================================================

assert len(
    final_normal_cases
) == 100


assert len(
    final_wrong_cases
) == 100


assert len(
    final_lag_cases
) == 100


assert len(
    final_silent_cases
) == 100


final_all_400_cases = (
    final_normal_cases
    + final_wrong_cases
    + final_lag_cases
    + final_silent_cases
)


assert len(
    final_all_400_cases
) == 400


final_case_ids = [
    case[
        "case_id"
    ]

    for case
    in final_all_400_cases
]


assert len(
    final_case_ids
) == len(
    set(
        final_case_ids
    )
)


# ============================================================
# BUILD FINAL COVERAGE TABLE
# ============================================================

final_coverage_rows = []


final_cases_by_family = {
    "normal": (
        final_normal_cases
    ),

    "wrong_partner": (
        final_wrong_cases
    ),

    "lag": (
        final_lag_cases
    ),

    "silent_partner": (
        final_silent_cases
    ),
}


for family_name, cases in (
    final_cases_by_family.items()
):

    for case in cases:

        # Recalculate once more for strict consistency.
        recalculate_case_semantic_coverage(
            case
        )


        coverage = (
            case[
                "semantic_summary_coverage"
            ]
        )


        final_coverage_rows.append({
            "case_family": (
                family_name
            ),

            "case_id": (
                case[
                    "case_id"
                ]
            ),

            "available_coarse_summaries": (
                coverage[
                    "available_coarse_summaries"
                ]
            ),

            "missing_coarse_summaries": (
                coverage[
                    "missing_coarse_summaries"
                ]
            ),

            "available_focused_summaries": (
                coverage[
                    "available_focused_summaries"
                ]
            ),

            "missing_focused_summaries": (
                coverage[
                    "missing_focused_summaries"
                ]
            ),

            "all_summaries_available": (
                coverage[
                    "all_summaries_available"
                ]
            ),
        })


final_coverage_df = pd.DataFrame(
    final_coverage_rows
)


final_coverage_summary_df = (
    final_coverage_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        complete_cases=(
            "all_summaries_available",
            "sum",
        ),

        available_coarse_summaries=(
            "available_coarse_summaries",
            "sum",
        ),

        missing_coarse_summaries=(
            "missing_coarse_summaries",
            "sum",
        ),

        available_focused_summaries=(
            "available_focused_summaries",
            "sum",
        ),

        missing_focused_summaries=(
            "missing_focused_summaries",
            "sum",
        ),
    )
)


# ============================================================
# BUILD THE REMAINING UNIQUE-GAP MANIFEST
#
# LAG is excluded because it exactly duplicates NORMAL
# participant semantics.
# ============================================================

remaining_gap_registry = {}


direct_final_cases = (
    final_normal_cases
    + final_wrong_cases
    + final_silent_cases
)


for case in direct_final_cases:

    family_name = case_family(
        case
    )


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant = (
            case[
                role
            ]
        )


        source_kind = (
            source_kind_for_role(
                case,
                role,
            )
        )


        conversation_id = str(
            participant[
                "conversation_id"
            ]
        )


        participant_id = str(
            participant[
                "participant_id"
            ]
        )


        for (
            segment_idx,
            segment_name,
        ) in (
            SEGMENT_NAMES_BY_INDEX.items()
        ):

            segment_record = (
                case[
                    "semantic_summaries"
                ][role][segment_name]
            )


            coarse_missing = (
                segment_record.get(
                    "coarse_summary"
                )
                is None
            )


            focused_missing = (
                segment_record.get(
                    "focused_summary"
                )
                is None
            )


            if not (
                coarse_missing
                or focused_missing
            ):

                continue


            identity = (
                source_kind,
                conversation_id,
                participant_id,
                int(
                    segment_idx
                ),
            )


            if identity not in (
                remaining_gap_registry
            ):

                remaining_gap_registry[
                    identity
                ] = {
                    "source_kind": (
                        source_kind
                    ),

                    "conversation_id": (
                        conversation_id
                    ),

                    "participant_id": (
                        participant_id
                    ),

                    "segment_idx": int(
                        segment_idx
                    ),

                    "segment_start_seconds": (
                        0
                        if segment_idx == 0
                        else 60
                    ),

                    "segment_end_seconds": (
                        60
                        if segment_idx == 0
                        else 120
                    ),

                    "coarse_missing": (
                        coarse_missing
                    ),

                    "focused_missing": (
                        focused_missing
                    ),

                    "appears_in_case_families": set(),

                    "appears_in_case_ids": set(),
                }


            remaining_gap_registry[
                identity
            ][
                "appears_in_case_families"
            ].add(
                family_name
            )


            remaining_gap_registry[
                identity
            ][
                "appears_in_case_ids"
            ].add(
                case[
                    "case_id"
                ]
            )


remaining_missing_summaries = []


for identity, record in sorted(
    remaining_gap_registry.items()
):

    remaining_missing_summaries.append({
        "source_kind": (
            record[
                "source_kind"
            ]
        ),

        "conversation_id": (
            record[
                "conversation_id"
            ]
        ),

        "participant_id": (
            record[
                "participant_id"
            ]
        ),

        "segment_idx": (
            record[
                "segment_idx"
            ]
        ),

        "segment_start_seconds": (
            record[
                "segment_start_seconds"
            ]
        ),

        "segment_end_seconds": (
            record[
                "segment_end_seconds"
            ]
        ),

        "coarse_missing": (
            record[
                "coarse_missing"
            ]
        ),

        "focused_missing": (
            record[
                "focused_missing"
            ]
        ),

        "appears_in_case_families": sorted(
            record[
                "appears_in_case_families"
            ]
        ),

        "num_case_appearances": len(
            record[
                "appears_in_case_ids"
            ]
        ),
    })


# ============================================================
# BACK UP CURRENT FINAL FILES BEFORE OVERWRITING
# ============================================================

def create_backup_once(
    path,
):
    path = Path(
        path
    )


    backup_path = path.with_name(
        path.stem
        + ".before_final_focused_gap_fill"
        + path.suffix
    )


    if (
        path.exists()
        and not backup_path.exists()
    ):

        shutil.copy2(
            path,
            backup_path,
        )


    return backup_path


normal_wrong_backup = create_backup_once(
    FINAL_NORMAL_WRONG_DATABASE_PATH
)


lag_backup = create_backup_once(
    FINAL_LAG_DATABASE_PATH
)


silent_backup = create_backup_once(
    FINAL_SILENT_DATABASE_PATH
)


all_400_backup = create_backup_once(
    FINAL_ALL_400_DATABASE_PATH
)


# ============================================================
# SAVE UPDATED FINAL DATABASES
# ============================================================

final_normal_wrong_cases = (
    final_normal_cases
    + final_wrong_cases
)


Path(
    FINAL_NORMAL_WRONG_DATABASE_PATH
).write_text(
    json.dumps(
        final_normal_wrong_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


Path(
    FINAL_LAG_DATABASE_PATH
).write_text(
    json.dumps(
        final_lag_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


Path(
    FINAL_SILENT_DATABASE_PATH
).write_text(
    json.dumps(
        final_silent_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


Path(
    FINAL_ALL_400_DATABASE_PATH
).write_text(
    json.dumps(
        final_all_400_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


Path(
    FINAL_REMAINING_MISSING_PATH
).write_text(
    json.dumps(
        remaining_missing_summaries,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


final_coverage_df.to_csv(
    FINAL_SEMANTIC_COVERAGE_CSV_PATH,
    index=False,
)


# ============================================================
# UPDATE THE MAIN IN-MEMORY VARIABLES
# ============================================================

completed_semantic_normal_cases = (
    final_normal_cases
)


completed_semantic_wrong_cases = (
    final_wrong_cases
)


completed_semantic_lag_cases = (
    final_lag_cases
)


completed_semantic_silent_cases = (
    final_silent_cases
)


completed_all_400_cases = (
    final_all_400_cases
)


# ============================================================
# FINAL REPORT
# ============================================================

total_missing_coarse = int(
    final_coverage_summary_df[
        "missing_coarse_summaries"
    ].sum()
)


total_missing_focused = int(
    final_coverage_summary_df[
        "missing_focused_summaries"
    ].sum()
)


total_complete_cases = int(
    final_coverage_summary_df[
        "complete_cases"
    ].sum()
)


print("\n" + "=" * 88)
print("FINAL FOCUSED-GAP DATABASE ENRICHMENT COMPLETE")
print("=" * 88)

print(
    "Direct focused summary slots inserted:",
    len(
        all_direct_focused_insertions
    ),
)

print(
    "Complete cases:",
    f"{total_complete_cases}/400",
)

print(
    "Total missing coarse summaries:",
    total_missing_coarse,
)

print(
    "Total missing focused summaries:",
    total_missing_focused,
)


print("\nFINAL COVERAGE AFTER FOCUSED RETRIES")

display(
    final_coverage_summary_df
)


print(
    "\nRemaining unique participant-segment gaps:",
    len(
        remaining_missing_summaries
    ),
)

print(
    "Remaining unique coarse gaps:",
    sum(
        record[
            "coarse_missing"
        ]

        for record
        in remaining_missing_summaries
    ),
)

print(
    "Remaining unique focused gaps:",
    sum(
        record[
            "focused_missing"
        ]

        for record
        in remaining_missing_summaries
    ),
)


print(
    "\nUpdated NORMAL + WRONG database:",
    FINAL_NORMAL_WRONG_DATABASE_PATH,
)

print(
    "Updated LAG database:",
    FINAL_LAG_DATABASE_PATH,
)

print(
    "Updated SILENT database:",
    FINAL_SILENT_DATABASE_PATH,
)

print(
    "Updated combined 400-case database:",
    FINAL_ALL_400_DATABASE_PATH,
)

print(
    "Updated remaining-gap manifest:",
    FINAL_REMAINING_MISSING_PATH,
)

print(
    "Updated coverage CSV:",
    FINAL_SEMANTIC_COVERAGE_CSV_PATH,
)


if (
    total_missing_coarse == 0
    and total_missing_focused == 0
):

    print("\n" + "=" * 88)
    print("ALL 400 CASES NOW HAVE COMPLETE COARSE + FOCUSED SUMMARIES")
    print("=" * 88)

else:

    print("\nWARNING: Some semantic-summary gaps still remain.")

CURRENT FINAL DATABASE LOADED
Database source: current in-memory completed cases
NORMAL cases: 100
WRONG cases: 100
LAG cases: 100
SILENT cases: 100

UPDATED FOCUSED CACHE LOADED
Clean focused cache records: 273
Unique focused participant-segments: 273

MISSING FOCUSED SUMMARY FILL COMPLETE
NORMAL focused slots filled: 6
WRONG focused slots filled: 6
SILENT focused slots filled: 6
Total direct focused slots filled: 18

FINAL FOCUSED-GAP DATABASE ENRICHMENT COMPLETE
Direct focused summary slots inserted: 18
Complete cases: 400/400
Total missing coarse summaries: 0
Total missing focused summaries: 0

FINAL COVERAGE AFTER FOCUSED RETRIES


,case_family,total_cases,complete_cases,available_coarse_summaries,missing_coarse_summaries,available_focused_summaries,missing_focused_summaries
0,lag,100,100,400,0,400,0
1,normal,100,100,400,0,400,0
2,silent_partner,100,100,400,0,400,0
3,wrong_partner,100,100,400,0,400,0



Remaining unique participant-segment gaps: 0
Remaining unique coarse gaps: 0
Remaining unique focused gaps: 0

Updated NORMAL + WRONG database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_normal_wrong_cases_with_temporal_and_completed_semantic_summaries.json
Updated LAG database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_lag_cases_with_temporal_and_completed_semantic_summaries.json
Updated SILENT database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_silent_partner_cases_with_temporal_and_completed_semantic_summaries.json
Updated combined 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Updated remaining-gap manifest: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_remaining_missing_semantic_summaries_after_generation.json
Updated covera

# Προσθηκη field speaks = True or Speaks = False

In [ ]:
# ============================================================
# ADD PARTICIPANT-LEVEL "speaks" FIELD TO ALL 400 CASES
#
# Rule:
#   participant_A["speaks"] = True
#       if participant_A_filtered_turns is not empty
#
#   participant_B["speaks"] = True
#       if participant_B_filtered_turns is not empty
#
# Otherwise:
#   speaks = False
#
# The field is added inside each participant JSON object.
# No temporal or semantic information is changed.
# ============================================================

from pathlib import Path

import copy
import json
import shutil
import pandas as pd


# ============================================================
# REQUIRED FINAL DATABASE PATH
# ============================================================

assert (
    "FINAL_ALL_400_DATABASE_PATH"
    in globals()
), (
    "FINAL_ALL_400_DATABASE_PATH is not defined. "
    "Run the final database enrichment cell first."
)


FINAL_ALL_400_DATABASE_PATH = Path(
    FINAL_ALL_400_DATABASE_PATH
)


assert FINAL_ALL_400_DATABASE_PATH.exists(), (
    "Final 400-case database not found: "
    f"{FINAL_ALL_400_DATABASE_PATH}"
)


# ============================================================
# LOAD THE CURRENT FINAL DATABASE
#
# Prefer the current in-memory final database.
# Otherwise reload the saved JSON.
# ============================================================

if (
    "completed_all_400_cases"
    in globals()
    and isinstance(
        completed_all_400_cases,
        list,
    )
    and len(
        completed_all_400_cases
    ) == 400
):

    current_all_400_cases = copy.deepcopy(
        completed_all_400_cases
    )

    database_source = (
        "completed_all_400_cases in memory"
    )

else:

    current_all_400_cases = json.loads(
        FINAL_ALL_400_DATABASE_PATH.read_text(
            encoding="utf-8"
        )
    )

    database_source = str(
        FINAL_ALL_400_DATABASE_PATH
    )


assert isinstance(
    current_all_400_cases,
    list,
)


assert len(
    current_all_400_cases
) == 400


# ============================================================
# HELPERS
# ============================================================

def get_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        f"Unknown case variant: {variant}"
    )


def remove_participant_speaks_fields(
    case,
):
    """
    Return a copy of the case without the newly added
    participant-level speaks fields.

    Used to verify that no other information changed.
    """

    cleaned_case = copy.deepcopy(
        case
    )


    for role in [
        "participant_A",
        "participant_B",
    ]:

        if (
            role in cleaned_case
            and isinstance(
                cleaned_case[role],
                dict,
            )
        ):

            cleaned_case[
                role
            ].pop(
                "speaks",
                None,
            )


    return cleaned_case


# ============================================================
# ADD THE SPEAKS FIELD
# ============================================================

cases_with_participant_speaks = copy.deepcopy(
    current_all_400_cases
)


speaks_audit_rows = []


for original_case, updated_case in zip(
    current_all_400_cases,
    cases_with_participant_speaks,
):

    assert (
        "participant_A"
        in updated_case
    )


    assert (
        "participant_B"
        in updated_case
    )


    assert (
        "participant_A_filtered_turns"
        in updated_case
    )


    assert (
        "participant_B_filtered_turns"
        in updated_case
    )


    A_turns = updated_case[
        "participant_A_filtered_turns"
    ]


    B_turns = updated_case[
        "participant_B_filtered_turns"
    ]


    assert isinstance(
        A_turns,
        list,
    ), (
        "participant_A_filtered_turns must be a list "
        f'in case {updated_case["case_id"]}'
    )


    assert isinstance(
        B_turns,
        list,
    ), (
        "participant_B_filtered_turns must be a list "
        f'in case {updated_case["case_id"]}'
    )


    A_speaks = (
        len(
            A_turns
        ) > 0
    )


    B_speaks = (
        len(
            B_turns
        ) > 0
    )


    updated_case[
        "participant_A"
    ][
        "speaks"
    ] = bool(
        A_speaks
    )


    updated_case[
        "participant_B"
    ][
        "speaks"
    ] = bool(
        B_speaks
    )


    # --------------------------------------------------------
    # Strict rule verification
    # --------------------------------------------------------

    assert (
        updated_case[
            "participant_A"
        ][
            "speaks"
        ]
        ==
        (
            len(
                updated_case[
                    "participant_A_filtered_turns"
                ]
            ) > 0
        )
    )


    assert (
        updated_case[
            "participant_B"
        ][
            "speaks"
        ]
        ==
        (
            len(
                updated_case[
                    "participant_B_filtered_turns"
                ]
            ) > 0
        )
    )


    # --------------------------------------------------------
    # Verify that no other case information changed
    # --------------------------------------------------------

    assert (
        remove_participant_speaks_fields(
            original_case
        )
        ==
        remove_participant_speaks_fields(
            updated_case
        )
    ), (
        "A field other than participant-level speaks "
        f'changed in case {updated_case["case_id"]}'
    )


    speaks_audit_rows.append({
        "case_family": (
            get_case_family(
                updated_case
            )
        ),

        "case_id": (
            updated_case[
                "case_id"
            ]
        ),

        "participant_A_speaks": (
            updated_case[
                "participant_A"
            ][
                "speaks"
            ]
        ),

        "participant_A_num_turns": len(
            updated_case[
                "participant_A_filtered_turns"
            ]
        ),

        "participant_B_speaks": (
            updated_case[
                "participant_B"
            ][
                "speaks"
            ]
        ),

        "participant_B_num_turns": len(
            updated_case[
                "participant_B_filtered_turns"
            ]
        ),
    })


# ============================================================
# GENERAL DATABASE AUDITS
# ============================================================

assert len(
    cases_with_participant_speaks
) == 400


case_ids = [
    case[
        "case_id"
    ]

    for case
    in cases_with_participant_speaks
]


assert len(
    case_ids
) == len(
    set(
        case_ids
    )
)


for case in cases_with_participant_speaks:

    assert isinstance(
        case[
            "participant_A"
        ][
            "speaks"
        ],
        bool,
    )


    assert isinstance(
        case[
            "participant_B"
        ][
            "speaks"
        ],
        bool,
    )


# Silent-partner B must always have empty turns
# and therefore speaks=False.
silent_cases = [
    case

    for case in cases_with_participant_speaks

    if get_case_family(
        case
    ) == "silent_partner"
]


assert len(
    silent_cases
) == 100


for case in silent_cases:

    assert (
        case[
            "participant_B_filtered_turns"
        ]
        == []
    )


    assert (
        case[
            "participant_B"
        ][
            "speaks"
        ]
        is False
    )


# ============================================================
# BUILD AUDIT TABLE
# ============================================================

speaks_audit_df = pd.DataFrame(
    speaks_audit_rows
)


speaks_summary_df = (
    speaks_audit_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        participant_A_speaks_true=(
            "participant_A_speaks",
            "sum",
        ),

        participant_B_speaks_true=(
            "participant_B_speaks",
            "sum",
        ),
    )
)


speaks_summary_df[
    "participant_A_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_A_speaks_true"
    ]
)


speaks_summary_df[
    "participant_B_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_B_speaks_true"
    ]
)


speaks_summary_df = speaks_summary_df[
    [
        "case_family",
        "total_cases",

        "participant_A_speaks_true",
        "participant_A_speaks_false",

        "participant_B_speaks_true",
        "participant_B_speaks_false",
    ]
]


# ============================================================
# SPLIT THE UPDATED DATABASE BY CASE FAMILY
# ============================================================

updated_normal_cases = [
    case

    for case in cases_with_participant_speaks

    if get_case_family(
        case
    ) == "normal"
]


updated_wrong_cases = [
    case

    for case in cases_with_participant_speaks

    if get_case_family(
        case
    ) == "wrong_partner"
]


updated_lag_cases = [
    case

    for case in cases_with_participant_speaks

    if get_case_family(
        case
    ) == "lag"
]


updated_silent_cases = [
    case

    for case in cases_with_participant_speaks

    if get_case_family(
        case
    ) == "silent_partner"
]


assert len(
    updated_normal_cases
) == 100


assert len(
    updated_wrong_cases
) == 100


assert len(
    updated_lag_cases
) == 100


assert len(
    updated_silent_cases
) == 100


# ============================================================
# BACK UP THE CURRENT FINAL FILES
# ============================================================

def create_speaks_backup_once(
    path,
):
    path = Path(
        path
    )


    backup_path = path.with_name(
        path.stem
        + ".before_participant_speaks"
        + path.suffix
    )


    if (
        path.exists()
        and not backup_path.exists()
    ):

        shutil.copy2(
            path,
            backup_path,
        )


    return backup_path


all_400_backup_path = (
    create_speaks_backup_once(
        FINAL_ALL_400_DATABASE_PATH
    )
)


if (
    "FINAL_NORMAL_WRONG_DATABASE_PATH"
    not in globals()
):

    FINAL_NORMAL_WRONG_DATABASE_PATH = (
        Path(
            FINAL_ALL_400_DATABASE_PATH
        ).parent
        / (
            "consolidation_normal_wrong_cases_with_"
            "temporal_and_completed_semantic_summaries.json"
        )
    )


if (
    "FINAL_LAG_DATABASE_PATH"
    not in globals()
):

    FINAL_LAG_DATABASE_PATH = (
        Path(
            FINAL_ALL_400_DATABASE_PATH
        ).parent
        / (
            "consolidation_lag_cases_with_"
            "temporal_and_completed_semantic_summaries.json"
        )
    )


if (
    "FINAL_SILENT_DATABASE_PATH"
    not in globals()
):

    FINAL_SILENT_DATABASE_PATH = (
        Path(
            FINAL_ALL_400_DATABASE_PATH
        ).parent
        / (
            "consolidation_silent_partner_cases_with_"
            "temporal_and_completed_semantic_summaries.json"
        )
    )


FINAL_NORMAL_WRONG_DATABASE_PATH = Path(
    FINAL_NORMAL_WRONG_DATABASE_PATH
)


FINAL_LAG_DATABASE_PATH = Path(
    FINAL_LAG_DATABASE_PATH
)


FINAL_SILENT_DATABASE_PATH = Path(
    FINAL_SILENT_DATABASE_PATH
)


normal_wrong_backup_path = (
    create_speaks_backup_once(
        FINAL_NORMAL_WRONG_DATABASE_PATH
    )
)


lag_backup_path = (
    create_speaks_backup_once(
        FINAL_LAG_DATABASE_PATH
    )
)


silent_backup_path = (
    create_speaks_backup_once(
        FINAL_SILENT_DATABASE_PATH
    )
)


# ============================================================
# SAVE THE UPDATED DATABASES
# ============================================================

updated_normal_wrong_cases = (
    updated_normal_cases
    + updated_wrong_cases
)


FINAL_NORMAL_WRONG_DATABASE_PATH.write_text(
    json.dumps(
        updated_normal_wrong_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


FINAL_LAG_DATABASE_PATH.write_text(
    json.dumps(
        updated_lag_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


FINAL_SILENT_DATABASE_PATH.write_text(
    json.dumps(
        updated_silent_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


FINAL_ALL_400_DATABASE_PATH.write_text(
    json.dumps(
        cases_with_participant_speaks,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


PARTICIPANT_SPEAKS_AUDIT_CSV_PATH = (
    FINAL_ALL_400_DATABASE_PATH.parent
    / "participant_speaks_audit_all_400_cases.csv"
)


speaks_audit_df.to_csv(
    PARTICIPANT_SPEAKS_AUDIT_CSV_PATH,
    index=False,
)


# ============================================================
# RELOAD AND VERIFY THE SAVED DATABASE
# ============================================================

reloaded_cases_with_speaks = json.loads(
    FINAL_ALL_400_DATABASE_PATH.read_text(
        encoding="utf-8"
    )
)


assert len(
    reloaded_cases_with_speaks
) == 400


for case in reloaded_cases_with_speaks:

    assert (
        case[
            "participant_A"
        ][
            "speaks"
        ]
        ==
        (
            len(
                case[
                    "participant_A_filtered_turns"
                ]
            ) > 0
        )
    )


    assert (
        case[
            "participant_B"
        ][
            "speaks"
        ]
        ==
        (
            len(
                case[
                    "participant_B_filtered_turns"
                ]
            ) > 0
        )
    )


# ============================================================
# UPDATE THE MAIN IN-MEMORY VARIABLES
# ============================================================

completed_semantic_normal_cases = (
    updated_normal_cases
)


completed_semantic_wrong_cases = (
    updated_wrong_cases
)


completed_semantic_lag_cases = (
    updated_lag_cases
)


completed_semantic_silent_cases = (
    updated_silent_cases
)


completed_all_400_cases = (
    cases_with_participant_speaks
)


# ============================================================
# FINAL REPORT
# ============================================================

print("=" * 88)
print("PARTICIPANT-LEVEL SPEAKS FIELD ADDED SUCCESSFULLY")
print("=" * 88)

print(
    "Database source:",
    database_source,
)

print(
    "Updated cases:",
    len(
        cases_with_participant_speaks
    ),
)


print("\nSPEAKS COVERAGE BY CASE FAMILY")

display(
    speaks_summary_df
)


print(
    "\nSilent Participant B speaks=True:",
    sum(
        case[
            "participant_B"
        ][
            "speaks"
        ]

        for case
        in updated_silent_cases
    ),
)

print(
    "Silent Participant B speaks=False:",
    sum(
        not case[
            "participant_B"
        ][
            "speaks"
        ]

        for case
        in updated_silent_cases
    ),
)


print(
    "\nUpdated combined 400-case database:",
    FINAL_ALL_400_DATABASE_PATH,
)

print(
    "Updated NORMAL + WRONG database:",
    FINAL_NORMAL_WRONG_DATABASE_PATH,
)

print(
    "Updated LAG database:",
    FINAL_LAG_DATABASE_PATH,
)

print(
    "Updated SILENT database:",
    FINAL_SILENT_DATABASE_PATH,
)

print(
    "Participant speaks audit CSV:",
    PARTICIPANT_SPEAKS_AUDIT_CSV_PATH,
)


print(
    "\nCombined database backup:",
    all_400_backup_path,
)

print(
    "NORMAL + WRONG backup:",
    normal_wrong_backup_path,
)

print(
    "LAG backup:",
    lag_backup_path,
)

print(
    "SILENT backup:",
    silent_backup_path,
)

PARTICIPANT-LEVEL SPEAKS FIELD ADDED SUCCESSFULLY
Database source: completed_all_400_cases in memory
Updated cases: 400

SPEAKS COVERAGE BY CASE FAMILY


,case_family,total_cases,participant_A_speaks_true,participant_A_speaks_false,participant_B_speaks_true,participant_B_speaks_false
0,lag,100,100,0,100,0
1,normal,100,100,0,100,0
2,silent_partner,100,100,0,0,100
3,wrong_partner,100,100,0,100,0



Silent Participant B speaks=True: 0
Silent Participant B speaks=False: 100

Updated combined 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Updated NORMAL + WRONG database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_normal_wrong_cases_with_temporal_and_completed_semantic_summaries.json
Updated LAG database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_lag_cases_with_temporal_and_completed_semantic_summaries.json
Updated SILENT database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_silent_partner_cases_with_temporal_and_completed_semantic_summaries.json
Participant speaks audit CSV: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/participant_speaks_audit_all_400_cases.csv

Combined database backup: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_

In [ ]:
# ============================================================
# INTERACTIVE INSPECTION OF ALL 400 DATABASE SAMPLES
#
# Allows inspection of:
#   - NORMAL
#   - WRONG_PARTNER
#   - LAG
#   - SILENT_PARTNER
#
# Each sample can display:
#   - case metadata
#   - Participant A / Participant B information
#   - speaks fields
#   - filtered turns
#   - local temporal features
#   - global shift features
#   - coarse summaries
#   - focused summaries
#   - complete raw JSON
# ============================================================

from pathlib import Path

import json
import pandas as pd
import ipywidgets as widgets

from IPython.display import (
    display,
    clear_output,
)


# ============================================================
# ENABLE COLAB WIDGETS
# ============================================================

try:

    from google.colab import output

    output.enable_custom_widget_manager()

except Exception:

    pass


# ============================================================
# LOAD THE FINAL 400-CASE DATABASE
#
# Prefer the current in-memory object.
# Otherwise reload the saved final JSON.
# ============================================================

if (
    "completed_all_400_cases"
    in globals()
    and isinstance(
        completed_all_400_cases,
        list,
    )
    and len(
        completed_all_400_cases
    ) == 400
):

    inspection_cases = (
        completed_all_400_cases
    )

    inspection_database_source = (
        "completed_all_400_cases in memory"
    )

else:

    assert (
        "FINAL_ALL_400_DATABASE_PATH"
        in globals()
    ), (
        "FINAL_ALL_400_DATABASE_PATH is not defined."
    )


    FINAL_ALL_400_DATABASE_PATH = Path(
        FINAL_ALL_400_DATABASE_PATH
    )


    assert (
        FINAL_ALL_400_DATABASE_PATH.exists()
    ), (
        "Final database not found: "
        f"{FINAL_ALL_400_DATABASE_PATH}"
    )


    inspection_cases = json.loads(
        FINAL_ALL_400_DATABASE_PATH.read_text(
            encoding="utf-8"
        )
    )


    inspection_database_source = str(
        FINAL_ALL_400_DATABASE_PATH
    )


assert isinstance(
    inspection_cases,
    list,
)


assert len(
    inspection_cases
) == 400


# ============================================================
# CASE-FAMILY NORMALIZATION
# ============================================================

def inspection_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        "Unknown case variant: "
        f"{case.get('case_variant')}"
    )


# ============================================================
# SPLIT DATABASE BY FAMILY
# ============================================================

inspection_cases_by_family = {
    "normal": [],
    "wrong_partner": [],
    "lag": [],
    "silent_partner": [],
}


for case in inspection_cases:

    family = inspection_case_family(
        case
    )


    inspection_cases_by_family[
        family
    ].append(
        case
    )


for family in inspection_cases_by_family:

    inspection_cases_by_family[
        family
    ] = sorted(
        inspection_cases_by_family[
            family
        ],
        key=lambda case: str(
            case.get(
                "case_id",
                "",
            )
        ),
    )


assert len(
    inspection_cases_by_family[
        "normal"
    ]
) == 100


assert len(
    inspection_cases_by_family[
        "wrong_partner"
    ]
) == 100


assert len(
    inspection_cases_by_family[
        "lag"
    ]
) == 100


assert len(
    inspection_cases_by_family[
        "silent_partner"
    ]
) == 100


# ============================================================
# HELPER: DISPLAY ONE PARTICIPANT
# ============================================================

def participant_inspection_row(
    case,
    role,
):
    participant = case.get(
        role,
        {},
    )


    turns_key = (
        f"{role}_filtered_turns"
    )


    turns = case.get(
        turns_key,
        [],
    )


    return {
        "role": role,

        "conversation_id": (
            participant.get(
                "conversation_id"
            )
        ),

        "participant_id": (
            participant.get(
                "participant_id"
            )
        ),

        "speaks": (
            participant.get(
                "speaks"
            )
        ),

        "num_filtered_turns": (
            len(turns)
            if isinstance(
                turns,
                list,
            )
            else None
        ),

        "metadata_path": (
            participant.get(
                "metadata_path"
            )
        ),

        "video_path": (
            participant.get(
                "video_path",
                participant.get(
                    "video"
                ),
            )
        ),
    }


# ============================================================
# HELPER: DISPLAY SEMANTIC SUMMARIES
# ============================================================

def display_participant_semantics(
    case,
    role,
):
    semantic_summaries = case.get(
        "semantic_summaries",
        {},
    )


    participant_semantics = (
        semantic_summaries.get(
            role,
            {},
        )
    )


    print(
        "\n" + "-" * 88
    )

    print(
        f"{role.upper()} SEMANTIC SUMMARIES"
    )

    print(
        "-" * 88
    )


    if not participant_semantics:

        print(
            "No semantic summaries found."
        )

        return


    for (
        segment_name,
        segment_record,
    ) in participant_semantics.items():

        print(
            f"\nSEGMENT: {segment_name}"
        )


        coarse_summary = (
            segment_record.get(
                "coarse_summary"
            )
        )


        focused_summary = (
            segment_record.get(
                "focused_summary"
            )
        )


        print(
            "\nCoarse summary:"
        )


        print(
            json.dumps(
                coarse_summary,
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\nFocused summary:"
        )


        print(
            json.dumps(
                focused_summary,
                indent=2,
                ensure_ascii=False,
            )
        )


# ============================================================
# MAIN NON-INTERACTIVE INSPECTION FUNCTION
# ============================================================

def inspect_database_sample(
    family=None,
    sample_index=None,
    case_id=None,
    show_turns=True,
    show_temporal_features=True,
    show_semantics=True,
    show_full_json=False,
):
    """
    Inspect one database sample.

    Usage examples:

        inspect_database_sample(
            family="normal",
            sample_index=0,
        )

        inspect_database_sample(
            family="wrong_partner",
            sample_index=25,
        )

        inspect_database_sample(
            case_id="...",
        )
    """

    if case_id is not None:

        matching_cases = [
            case

            for case in inspection_cases

            if str(
                case.get(
                    "case_id"
                )
            ) == str(
                case_id
            )
        ]


        assert len(
            matching_cases
        ) == 1, (
            "Expected exactly one case with case_id "
            f"{case_id}, found {len(matching_cases)}."
        )


        case = matching_cases[0]

        family = inspection_case_family(
            case
        )


        family_cases = (
            inspection_cases_by_family[
                family
            ]
        )


        sample_index = next(
            index

            for index, family_case
            in enumerate(
                family_cases
            )

            if family_case is case
        )


    else:

        assert family in (
            inspection_cases_by_family
        ), (
            "family must be one of: "
            "normal, wrong_partner, lag, silent_partner"
        )


        family_cases = (
            inspection_cases_by_family[
                family
            ]
        )


        assert sample_index is not None


        assert (
            0 <= int(
                sample_index
            ) < len(
                family_cases
            )
        )


        sample_index = int(
            sample_index
        )


        case = family_cases[
            sample_index
        ]


    # ========================================================
    # HEADER
    # ========================================================

    print("=" * 88)
    print("DATABASE SAMPLE INSPECTION")
    print("=" * 88)

    print(
        "Database source:",
        inspection_database_source,
    )

    print(
        "Family:",
        family,
    )

    print(
        "Family sample index:",
        f"{sample_index + 1}/"
        f"{len(inspection_cases_by_family[family])}",
    )

    print(
        "case_id:",
        case.get(
            "case_id"
        ),
    )

    print(
        "source_group_id:",
        case.get(
            "source_group_id"
        ),
    )

    print(
        "case_variant:",
        case.get(
            "case_variant"
        ),
    )

    print(
        "gold_binary_label:",
        case.get(
            "gold_binary_label"
        ),
    )

    print(
        "gold_anomaly_type:",
        case.get(
            "gold_anomaly_type"
        ),
    )

    print(
        "A source conversation:",
        case.get(
            "A_source_conversation"
        ),
    )

    print(
        "B source conversation:",
        case.get(
            "B_source_conversation"
        ),
    )


    # ========================================================
    # PARTICIPANTS
    # ========================================================

    participant_df = pd.DataFrame([
        participant_inspection_row(
            case,
            "participant_A",
        ),

        participant_inspection_row(
            case,
            "participant_B",
        ),
    ])


    print(
        "\nPARTICIPANTS"
    )


    display(
        participant_df
    )


    # ========================================================
    # SEMANTIC COVERAGE
    # ========================================================

    print(
        "\nSEMANTIC SUMMARY COVERAGE"
    )


    print(
        json.dumps(
            case.get(
                "semantic_summary_coverage"
            ),
            indent=2,
            ensure_ascii=False,
        )
    )


    # ========================================================
    # TURNS
    # ========================================================

    if show_turns:

        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT A FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_A_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT B FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_B_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    # ========================================================
    # TEMPORAL FEATURES
    # ========================================================

    if show_temporal_features:

        print(
            "\n" + "=" * 88
        )

        print(
            "LOCAL TEMPORAL FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "local_temporal_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "GLOBAL SHIFT FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "global_shift_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    # ========================================================
    # SEMANTIC SUMMARIES
    # ========================================================

    if show_semantics:

        display_participant_semantics(
            case,
            "participant_A",
        )


        display_participant_semantics(
            case,
            "participant_B",
        )


    # ========================================================
    # FULL RAW JSON
    # ========================================================

    if show_full_json:

        print(
            "\n" + "=" * 88
        )

        print(
            "COMPLETE SAMPLE JSON"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case,
                indent=2,
                ensure_ascii=False,
            )
        )


    return case


# ============================================================
# INTERACTIVE WIDGETS
# ============================================================

family_display_names = {
    "normal": "NORMAL",
    "wrong_partner": "WRONG_PARTNER",
    "lag": "LAG",
    "silent_partner": "SILENT_PARTNER",
}


family_dropdown = widgets.Dropdown(
    options=[
        (
            family_display_names[
                family
            ],
            family,
        )

        for family in [
            "normal",
            "wrong_partner",
            "lag",
            "silent_partner",
        ]
    ],

    value="normal",

    description="Family:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="450px"
    ),
)


case_dropdown = widgets.Dropdown(
    description="Sample:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="850px"
    ),
)


show_turns_checkbox = widgets.Checkbox(
    value=True,
    description="Show filtered turns",
    indent=False,
)


show_temporal_checkbox = widgets.Checkbox(
    value=True,
    description="Show temporal features",
    indent=False,
)


show_semantics_checkbox = widgets.Checkbox(
    value=True,
    description="Show semantic summaries",
    indent=False,
)


show_full_json_checkbox = widgets.Checkbox(
    value=False,
    description="Show complete JSON",
    indent=False,
)


inspection_output = widgets.Output()


def update_case_dropdown(
    *args,
):
    family = family_dropdown.value


    family_cases = (
        inspection_cases_by_family[
            family
        ]
    )


    case_dropdown.options = [
        (
            (
                f"{index + 1:03d}/100 | "
                f'{case.get("case_id")} | '
                f'{case.get("case_variant")}'
            ),

            index,
        )

        for index, case
        in enumerate(
            family_cases
        )
    ]


    case_dropdown.value = 0


def render_selected_case(
    *args,
):
    family = family_dropdown.value


    sample_index = (
        case_dropdown.value
    )


    if sample_index is None:
        return


    with inspection_output:

        clear_output(
            wait=True
        )


        inspect_database_sample(
            family=family,
            sample_index=sample_index,

            show_turns=(
                show_turns_checkbox.value
            ),

            show_temporal_features=(
                show_temporal_checkbox.value
            ),

            show_semantics=(
                show_semantics_checkbox.value
            ),

            show_full_json=(
                show_full_json_checkbox.value
            ),
        )


family_dropdown.observe(
    update_case_dropdown,
    names="value",
)


family_dropdown.observe(
    render_selected_case,
    names="value",
)


case_dropdown.observe(
    render_selected_case,
    names="value",
)


show_turns_checkbox.observe(
    render_selected_case,
    names="value",
)


show_temporal_checkbox.observe(
    render_selected_case,
    names="value",
)


show_semantics_checkbox.observe(
    render_selected_case,
    names="value",
)


show_full_json_checkbox.observe(
    render_selected_case,
    names="value",
)


update_case_dropdown()


controls = widgets.VBox([
    family_dropdown,
    case_dropdown,

    widgets.HBox([
        show_turns_checkbox,
        show_temporal_checkbox,
    ]),

    widgets.HBox([
        show_semantics_checkbox,
        show_full_json_checkbox,
    ]),
])


print("=" * 88)
print("INTERACTIVE DATABASE INSPECTOR READY")
print("=" * 88)

print(
    "Database source:",
    inspection_database_source,
)

print(
    "NORMAL samples:",
    len(
        inspection_cases_by_family[
            "normal"
        ]
    ),
)

print(
    "WRONG_PARTNER samples:",
    len(
        inspection_cases_by_family[
            "wrong_partner"
        ]
    ),
)

print(
    "LAG samples:",
    len(
        inspection_cases_by_family[
            "lag"
        ]
    ),
)

print(
    "SILENT_PARTNER samples:",
    len(
        inspection_cases_by_family[
            "silent_partner"
        ]
    ),
)


display(
    controls,
    inspection_output,
)


render_selected_case()

INTERACTIVE DATABASE INSPECTOR READY
Database source: completed_all_400_cases in memory
NORMAL samples: 100
WRONG_PARTNER samples: 100
LAG samples: 100
SILENT_PARTNER samples: 100


Output()

In [ ]:
# ============================================================
# VERIFY FINAL DATABASE COMPOSITION
# ============================================================

import json
from collections import Counter
from pathlib import Path


FINAL_DATABASE_PATH = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/"
    "consolidation_all_400_cases_with_temporal_and_semantic_summaries.json"
)


final_cases = json.loads(
    FINAL_DATABASE_PATH.read_text(
        encoding="utf-8"
    )
)


print(
    "Total cases:",
    len(final_cases),
)


print(
    "\nCase variants:"
)

print(
    Counter(
        case["case_variant"]
        for case in final_cases
    )
)


lag_cases = [
    case
    for case in final_cases
    if str(
        case["case_variant"]
    ).lower().startswith("lag")
]


print(
    "\nTotal LAG cases:",
    len(lag_cases),
)


print(
    "LAG variants:"
)

print(
    Counter(
        case["case_variant"]
        for case in lag_cases
    )
)

Total cases: 400

Case variants:
Counter({'normal': 100, 'wrong_partner': 100, 'silent_partner': 100, 'lag_2sec': 50, 'lag_3sec': 50})

Total LAG cases: 100
LAG variants:
Counter({'lag_2sec': 50, 'lag_3sec': 50})


In [ ]:
# ============================================================
# REMOVE LEGACY "speaks" FIELD FROM ALL FOCUSED SUMMARIES
#
# Scope:
#   case["semantic_summaries"]
#       ["participant_A" / "participant_B"]
#       [segment]
#       ["focused_summary"]
#       ["speaks"]   <-- remove only this legacy field
#
# Preserve:
#   case["participant_A"]["speaks"]
#   case["participant_B"]["speaks"]
#
# The participant-level speaks field must continue to follow:
#   speaks = len(participant_X_filtered_turns) > 0
# ============================================================

from pathlib import Path

import copy
import json
import shutil
import pandas as pd


# ============================================================
# DATABASE PATHS
# ============================================================

FINAL_ALL_400_DATABASE_PATH = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/"
    "consolidation_all_400_cases_with_temporal_and_semantic_summaries.json"
)


FINAL_NORMAL_WRONG_DATABASE_PATH = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/"
    "consolidation_normal_wrong_cases_with_temporal_and_completed_semantic_summaries.json"
)


FINAL_LAG_DATABASE_PATH = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/"
    "consolidation_lag_cases_with_temporal_and_completed_semantic_summaries.json"
)


FINAL_SILENT_DATABASE_PATH = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/"
    "consolidation_silent_partner_cases_with_temporal_and_completed_semantic_summaries.json"
)


assert FINAL_ALL_400_DATABASE_PATH.exists(), (
    "Final 400-case database not found: "
    f"{FINAL_ALL_400_DATABASE_PATH}"
)


# ============================================================
# LOAD THE CURRENT FINAL DATABASE
# ============================================================

all_400_cases_before_cleanup = json.loads(
    FINAL_ALL_400_DATABASE_PATH.read_text(
        encoding="utf-8"
    )
)


assert isinstance(
    all_400_cases_before_cleanup,
    list,
)


assert len(
    all_400_cases_before_cleanup
) == 400


# ============================================================
# HELPERS
# ============================================================

def focused_cleanup_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        "Unknown case variant: "
        f"{case.get('case_variant')}"
    )


def remove_focused_speaks_for_comparison(
    case,
):
    """
    Return a copy in which every legacy focused-summary
    'speaks' field is removed.

    This is used to prove that no other information changed.
    """

    cleaned_case = copy.deepcopy(
        case
    )


    semantic_summaries = (
        cleaned_case.get(
            "semantic_summaries",
            {},
        )
    )


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_semantics = (
            semantic_summaries.get(
                role,
                {},
            )
        )


        if not isinstance(
            participant_semantics,
            dict,
        ):
            continue


        for segment_record in (
            participant_semantics.values()
        ):

            if not isinstance(
                segment_record,
                dict,
            ):
                continue


            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )


            if isinstance(
                focused_summary,
                dict,
            ):

                focused_summary.pop(
                    "speaks",
                    None,
                )


    return cleaned_case


def count_focused_speaks_fields(
    cases,
):
    count = 0


    for case in cases:

        semantic_summaries = (
            case.get(
                "semantic_summaries",
                {},
            )
        )


        for role in [
            "participant_A",
            "participant_B",
        ]:

            participant_semantics = (
                semantic_summaries.get(
                    role,
                    {},
                )
            )


            if not isinstance(
                participant_semantics,
                dict,
            ):
                continue


            for segment_record in (
                participant_semantics.values()
            ):

                if not isinstance(
                    segment_record,
                    dict,
                ):
                    continue


                focused_summary = (
                    segment_record.get(
                        "focused_summary"
                    )
                )


                if (
                    isinstance(
                        focused_summary,
                        dict,
                    )
                    and "speaks" in focused_summary
                ):

                    count += 1


    return count


# ============================================================
# COUNT LEGACY FIELDS BEFORE CLEANUP
# ============================================================

legacy_focused_speaks_before = (
    count_focused_speaks_fields(
        all_400_cases_before_cleanup
    )
)


print("=" * 88)
print("FOCUSED SUMMARY SPEAKS AUDIT — BEFORE CLEANUP")
print("=" * 88)

print(
    "Total cases:",
    len(
        all_400_cases_before_cleanup
    ),
)

print(
    "Legacy focused-summary speaks fields found:",
    legacy_focused_speaks_before,
)


# ============================================================
# REMOVE ONLY THE LEGACY FOCUSED-SUMMARY SPEAKS FIELDS
# ============================================================

all_400_cases_after_cleanup = (
    copy.deepcopy(
        all_400_cases_before_cleanup
    )
)


focused_speaks_removal_records = []


for original_case, updated_case in zip(
    all_400_cases_before_cleanup,
    all_400_cases_after_cleanup,
):

    family = focused_cleanup_case_family(
        updated_case
    )


    # Save the new VAD-derived participant-level speaks values
    # before touching focused summaries.
    participant_level_speaks_before = {
        "participant_A": (
            original_case[
                "participant_A"
            ].get(
                "speaks"
            )
        ),

        "participant_B": (
            original_case[
                "participant_B"
            ].get(
                "speaks"
            )
        ),
    }


    semantic_summaries = updated_case.get(
        "semantic_summaries",
        {},
    )


    assert isinstance(
        semantic_summaries,
        dict,
    ), (
        "semantic_summaries must be a dictionary in case "
        f'{updated_case["case_id"]}'
    )


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_semantics = (
            semantic_summaries.get(
                role,
                {},
            )
        )


        assert isinstance(
            participant_semantics,
            dict,
        ), (
            f"{role} semantic summaries must be a dictionary "
            f'in case {updated_case["case_id"]}'
        )


        for (
            segment_name,
            segment_record,
        ) in participant_semantics.items():

            assert isinstance(
                segment_record,
                dict,
            )


            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )


            assert isinstance(
                focused_summary,
                dict,
            ), (
                "Expected a complete focused summary for "
                f'{updated_case["case_id"]} / '
                f"{role} / {segment_name}"
            )


            if "speaks" in focused_summary:

                removed_value = (
                    focused_summary[
                        "speaks"
                    ]
                )


                del focused_summary[
                    "speaks"
                ]


                focused_speaks_removal_records.append({
                    "case_family": (
                        family
                    ),

                    "case_id": (
                        updated_case[
                            "case_id"
                        ]
                    ),

                    "participant_role": (
                        role
                    ),

                    "conversation_id": (
                        updated_case[
                            role
                        ][
                            "conversation_id"
                        ]
                    ),

                    "participant_id": (
                        updated_case[
                            role
                        ][
                            "participant_id"
                        ]
                    ),

                    "segment_name": (
                        segment_name
                    ),

                    "removed_focused_speaks_value": (
                        removed_value
                    ),
                })


    # --------------------------------------------------------
    # Verify participant-level VAD-derived speaks was preserved
    # --------------------------------------------------------

    assert (
        updated_case[
            "participant_A"
        ].get(
            "speaks"
        )
        ==
        participant_level_speaks_before[
            "participant_A"
        ]
    )


    assert (
        updated_case[
            "participant_B"
        ].get(
            "speaks"
        )
        ==
        participant_level_speaks_before[
            "participant_B"
        ]
    )


    # --------------------------------------------------------
    # Verify participant-level speaks still matches VAD turns
    # --------------------------------------------------------

    assert (
        updated_case[
            "participant_A"
        ][
            "speaks"
        ]
        ==
        (
            len(
                updated_case[
                    "participant_A_filtered_turns"
                ]
            ) > 0
        )
    ), (
        "Participant A speaks does not match filtered turns "
        f'in case {updated_case["case_id"]}'
    )


    assert (
        updated_case[
            "participant_B"
        ][
            "speaks"
        ]
        ==
        (
            len(
                updated_case[
                    "participant_B_filtered_turns"
                ]
            ) > 0
        )
    ), (
        "Participant B speaks does not match filtered turns "
        f'in case {updated_case["case_id"]}'
    )


    # --------------------------------------------------------
    # Strictly verify that nothing else changed
    # --------------------------------------------------------

    assert (
        remove_focused_speaks_for_comparison(
            original_case
        )
        ==
        remove_focused_speaks_for_comparison(
            updated_case
        )
    ), (
        "A field other than focused-summary speaks changed "
        f'in case {updated_case["case_id"]}'
    )


# ============================================================
# STRICT POST-CLEANUP AUDITS
# ============================================================

assert len(
    all_400_cases_after_cleanup
) == 400


legacy_focused_speaks_after = (
    count_focused_speaks_fields(
        all_400_cases_after_cleanup
    )
)


assert legacy_focused_speaks_after == 0, (
    "Some focused summaries still contain speaks."
)


for case in all_400_cases_after_cleanup:

    for role, turns_key in [
        (
            "participant_A",
            "participant_A_filtered_turns",
        ),
        (
            "participant_B",
            "participant_B_filtered_turns",
        ),
    ]:

        assert (
            "speaks"
            in case[role]
        ), (
            f"Participant-level speaks is missing for "
            f'{case["case_id"]} / {role}'
        )


        assert isinstance(
            case[
                role
            ][
                "speaks"
            ],
            bool,
        )


        assert (
            case[
                role
            ][
                "speaks"
            ]
            ==
            (
                len(
                    case[
                        turns_key
                    ]
                ) > 0
            )
        )


# ============================================================
# SPLIT THE CLEANED DATABASE BY FAMILY
# ============================================================

cleaned_normal_cases = [
    case

    for case in all_400_cases_after_cleanup

    if focused_cleanup_case_family(
        case
    ) == "normal"
]


cleaned_wrong_cases = [
    case

    for case in all_400_cases_after_cleanup

    if focused_cleanup_case_family(
        case
    ) == "wrong_partner"
]


cleaned_lag_cases = [
    case

    for case in all_400_cases_after_cleanup

    if focused_cleanup_case_family(
        case
    ) == "lag"
]


cleaned_silent_cases = [
    case

    for case in all_400_cases_after_cleanup

    if focused_cleanup_case_family(
        case
    ) == "silent_partner"
]


assert len(
    cleaned_normal_cases
) == 100


assert len(
    cleaned_wrong_cases
) == 100


assert len(
    cleaned_lag_cases
) == 100


assert len(
    cleaned_silent_cases
) == 100


cleaned_normal_wrong_cases = (
    cleaned_normal_cases
    + cleaned_wrong_cases
)


# ============================================================
# CREATE BACKUPS BEFORE OVERWRITING
# ============================================================

def create_focused_speaks_backup_once(
    path,
):
    path = Path(
        path
    )


    backup_path = path.with_name(
        path.stem
        + ".before_removing_focused_summary_speaks"
        + path.suffix
    )


    if (
        path.exists()
        and not backup_path.exists()
    ):

        shutil.copy2(
            path,
            backup_path,
        )


    return backup_path


all_400_backup_path = (
    create_focused_speaks_backup_once(
        FINAL_ALL_400_DATABASE_PATH
    )
)


normal_wrong_backup_path = (
    create_focused_speaks_backup_once(
        FINAL_NORMAL_WRONG_DATABASE_PATH
    )
)


lag_backup_path = (
    create_focused_speaks_backup_once(
        FINAL_LAG_DATABASE_PATH
    )
)


silent_backup_path = (
    create_focused_speaks_backup_once(
        FINAL_SILENT_DATABASE_PATH
    )
)


# ============================================================
# SAVE CLEANED DATABASES
# ============================================================

FINAL_ALL_400_DATABASE_PATH.write_text(
    json.dumps(
        all_400_cases_after_cleanup,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


FINAL_NORMAL_WRONG_DATABASE_PATH.write_text(
    json.dumps(
        cleaned_normal_wrong_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


FINAL_LAG_DATABASE_PATH.write_text(
    json.dumps(
        cleaned_lag_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


FINAL_SILENT_DATABASE_PATH.write_text(
    json.dumps(
        cleaned_silent_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ============================================================
# SAVE CLEANUP AUDIT
# ============================================================

focused_speaks_cleanup_df = pd.DataFrame(
    focused_speaks_removal_records
)


FOCUSED_SPEAKS_CLEANUP_AUDIT_PATH = (
    FINAL_ALL_400_DATABASE_PATH.parent
    / "focused_summary_speaks_cleanup_audit.csv"
)


focused_speaks_cleanup_df.to_csv(
    FOCUSED_SPEAKS_CLEANUP_AUDIT_PATH,
    index=False,
)


if focused_speaks_removal_records:

    focused_speaks_cleanup_summary_df = (
        focused_speaks_cleanup_df
        .groupby(
            [
                "case_family",
                "participant_role",
            ],
            as_index=False,
        )
        .agg(
            removed_legacy_speaks_fields=(
                "case_id",
                "size",
            ),

            affected_samples=(
                "case_id",
                "nunique",
            ),
        )
    )

else:

    focused_speaks_cleanup_summary_df = (
        pd.DataFrame(
            columns=[
                "case_family",
                "participant_role",
                "removed_legacy_speaks_fields",
                "affected_samples",
            ]
        )
    )


# ============================================================
# RELOAD SAVED DATABASE AND VERIFY AGAIN
# ============================================================

reloaded_cleaned_cases = json.loads(
    FINAL_ALL_400_DATABASE_PATH.read_text(
        encoding="utf-8"
    )
)


assert len(
    reloaded_cleaned_cases
) == 400


assert (
    count_focused_speaks_fields(
        reloaded_cleaned_cases
    )
    == 0
)


for case in reloaded_cleaned_cases:

    assert (
        case[
            "participant_A"
        ][
            "speaks"
        ]
        ==
        (
            len(
                case[
                    "participant_A_filtered_turns"
                ]
            ) > 0
        )
    )


    assert (
        case[
            "participant_B"
        ][
            "speaks"
        ]
        ==
        (
            len(
                case[
                    "participant_B_filtered_turns"
                ]
            ) > 0
        )
    )


# ============================================================
# UPDATE IN-MEMORY VARIABLES
# ============================================================

completed_semantic_normal_cases = (
    cleaned_normal_cases
)


completed_semantic_wrong_cases = (
    cleaned_wrong_cases
)


completed_semantic_lag_cases = (
    cleaned_lag_cases
)


completed_semantic_silent_cases = (
    cleaned_silent_cases
)


completed_all_400_cases = (
    all_400_cases_after_cleanup
)


# ============================================================
# FINAL REPORT
# ============================================================

affected_case_ids = {
    record[
        "case_id"
    ]

    for record in focused_speaks_removal_records
}


print("\n" + "=" * 88)
print("LEGACY FOCUSED-SUMMARY SPEAKS CLEANUP COMPLETE")
print("=" * 88)

print(
    "Total cases checked:",
    len(
        all_400_cases_after_cleanup
    ),
)

print(
    "Focused summaries checked:",
    400 * 2 * 2,
)

print(
    "Legacy focused-summary speaks fields before:",
    legacy_focused_speaks_before,
)

print(
    "Legacy focused-summary speaks fields removed:",
    len(
        focused_speaks_removal_records
    ),
)

print(
    "Affected samples:",
    len(
        affected_case_ids
    ),
)

print(
    "Legacy focused-summary speaks fields remaining:",
    legacy_focused_speaks_after,
)

print(
    "Participant-level VAD speaks fields preserved:",
    400 * 2,
)


print(
    "\nREMOVALS BY CASE FAMILY AND PARTICIPANT ROLE"
)

display(
    focused_speaks_cleanup_summary_df
)


print(
    "\nUpdated combined database:",
    FINAL_ALL_400_DATABASE_PATH,
)

print(
    "Updated NORMAL + WRONG database:",
    FINAL_NORMAL_WRONG_DATABASE_PATH,
)

print(
    "Updated LAG database:",
    FINAL_LAG_DATABASE_PATH,
)

print(
    "Updated SILENT database:",
    FINAL_SILENT_DATABASE_PATH,
)

print(
    "Cleanup audit CSV:",
    FOCUSED_SPEAKS_CLEANUP_AUDIT_PATH,
)


print(
    "\nCombined database backup:",
    all_400_backup_path,
)

print(
    "NORMAL + WRONG backup:",
    normal_wrong_backup_path,
)

print(
    "LAG backup:",
    lag_backup_path,
)

print(
    "SILENT backup:",
    silent_backup_path,
)


print("\n" + "=" * 88)
print("VERIFICATION PASSED")
print("=" * 88)

print(
    "No focused_summary contains a speaks field."
)

print(
    "Only participant_A['speaks'] and "
    "participant_B['speaks'] remain."
)

print(
    "Both are derived strictly from the final VAD turns."
)

FOCUSED SUMMARY SPEAKS AUDIT — BEFORE CLEANUP
Total cases: 400
Legacy focused-summary speaks fields found: 742

LEGACY FOCUSED-SUMMARY SPEAKS CLEANUP COMPLETE
Total cases checked: 400
Focused summaries checked: 1600
Legacy focused-summary speaks fields before: 742
Legacy focused-summary speaks fields removed: 742
Affected samples: 240
Legacy focused-summary speaks fields remaining: 0
Participant-level VAD speaks fields preserved: 800

REMOVALS BY CASE FAMILY AND PARTICIPANT ROLE


,case_family,participant_role,removed_legacy_speaks_fields,affected_samples
0,lag,participant_A,106,53
1,lag,participant_B,106,53
2,normal,participant_A,106,53
3,normal,participant_B,106,53
4,silent_partner,participant_A,106,53
5,wrong_partner,participant_A,106,53
6,wrong_partner,participant_B,106,53



Updated combined database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Updated NORMAL + WRONG database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_normal_wrong_cases_with_temporal_and_completed_semantic_summaries.json
Updated LAG database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_lag_cases_with_temporal_and_completed_semantic_summaries.json
Updated SILENT database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_silent_partner_cases_with_temporal_and_completed_semantic_summaries.json
Cleanup audit CSV: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/focused_summary_speaks_cleanup_audit.csv

Combined database backup: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.before_removing_focu